# A Greedy GPT-PINN Reduced Digital Twin for Rapid Bridge Modal Updating — V0.6 FINAL R3

## 0. Final research scope, protocol freeze, and stopping rule

**Primary benchmark: Z24 only.** The ambient-vibration tests (AVT) are the development domain. The forced-vibration tests (FVT) remain a locked, no-target-refit cross-excitation audit. Route345, OpenLab, KW51, Vänersborg, FNO, diffusion, transformers, and reinforcement learning are outside this notebook.

**Final article concept**

> **A Greedy GPT-PINN Reduced Digital Twin for Rapid Bridge Modal Updating: Identifiability and Cross-Excitation Assessment on the Z24 Benchmark**

The notebook implements one reproducible chain:

\[
\text{Z24 AVT}
\rightarrow
\text{FDD/modal observations}
\rightarrow
\text{three-span structural model}
\rightarrow
\text{Full-PINN states}
\rightarrow
\text{greedy GPT-PINN}
\rightarrow
\text{identifiability-aware inverse updating}
\rightarrow
\text{locked AVT}\rightarrow\text{FVT audit}.
\]

### Scientific boundary frozen before this final run

The main evidence is rapid modal Digital-Twin updating, not autonomous prognosis. Gate H is retained only as a supplementary **conditional response-propagation** experiment. Gate I is retained as an honest locked audit: its original threshold is not relaxed after seeing FVT results. A Gate-I NO-GO can therefore coexist with a manuscript-ready methodological core.

The following elements are frozen:

- Z24 scenarios \(01,03,04,05,06\), corresponding to \(0,20,40,80,95\) mm controlled pier settlement;
- retained experimental modes \(1,3,4\);
- three-span geometry \(14+30+14=58\) m;
- AVT↔FVT physical setup map \(1\leftrightarrow9,\ 2\leftrightarrow8,\ 3\leftrightarrow7,\ 4\leftrightarrow6,\ 5\leftrightarrow5\);
- FVT driving-point exclusion (`DP1V`, `DP2V`);
- joint paired-setup modal covariance;
- one-dimensional identifiable trajectory \(\xi\);
- Gate-I maximum modal-ratio RMSE threshold of \(2.5\%\);
- Gate-E model-discrepancy cap of \(1\%\);
- all negative identifiability and prognosis findings.

#

### V0.6 FINAL R3 — JSON-safe final evidence and pipeline completion

The executed R2 smoke workflow reached the final protocol-freeze cell after
Gates A–I, but stopped while serializing a NumPy `bool_` returned by
`np.allclose`/`np.isclose`. R3 is a maintenance-only correction:

- every JSON payload is recursively converted to standard Python/JSON types;
- NumPy scalars and arrays, PyTorch tensors, `Path`, timestamps, tuples, and
  sets are supported;
- non-finite floating-point placeholders are written as JSON `null`, not the
  non-standard tokens `NaN` or `Infinity`;
- protocol-freeze checks are explicitly converted to Python `bool`;
- the serializer is self-tested at kernel startup and used by all JSON writes,
  including timing, ablation, sensitivity, certificates, evidence summaries,
  and code-freeze records.

R3 changes no governing equation, optimizer budget, physical parameter domain,
scientific threshold, AVT/FVT protocol, Gate-F proxy restriction, or article
claim. A smoke run using `FE_SMOKE_PROXY` remains workflow evidence only and
cannot authorize manuscript readiness or code freeze.

### V0.6 FINAL R2 — smoke-safe Gate-F execution

R2 corrects the case in which a Web-Jupyter run on `login02` is
automatically downgraded to `smoke`, no retained quality-validated Full-PINN
checkpoint is available, and the intentionally small smoke training budget
cannot satisfy publication thresholds. R2:

- scans every historical checkpoint layout recursively instead of relying on
  immediate version directories;
- independently validates any discovered checkpoint before reuse;
- avoids repeatedly training a known-underconverged smoke Full-PINN;
- uses a deterministic 116-element FE-snapshot reduced-basis proxy **only for
  smoke workflow testing** when no validated Full-PINN checkpoint exists;
- marks that proxy in every Gate-F diagnostic, timing table, final certificate,
  and readiness decision;
- prohibits the proxy in `full` mode and prevents any manuscript/code-freeze
  status while it is present.

The FE proxy is not a substitute for GPT-PINN evidence. The publication run
must still execute the actual Full-PINN/GPT-PINN chain in `full` mode inside a
Slurm compute-node allocation.

## V0.6 FINAL additions

1. Publication mode defaults to `auto`: a Slurm compute node runs `full`; a login-node Web session runs `smoke`. Explicit `full` execution on `login02` is blocked by default.
2. Compatible V0.6 FINAL/V0.4 checkpoints are imported only after each gate's existing metadata and convergence validator accepts them.
3. Timing uses warm-up, repeated per-call measurements, median and 5–95% quantiles.
4. Offline basis-construction cost and online query cost are separated, and the FE break-even query count is computed.
5. Final ablations evaluate basis size/order, modal covariance, model discrepancy, and mode subsets without changing the frozen model.
6. Final sensitivity checks evaluate FE mesh convergence and reduced-basis quadrature resolution.
7. All final tables, figures, claims, hashes, and the code-freeze certificate are generated from the current run.
8. After a successful Slurm full run, **algorithmic development stops** and manuscript preparation begins.

This notebook does **not** claim actual material stiffness loss, load-carrying capacity, autonomous deterioration, calendar-time remaining useful life, structural failure probability, maintenance savings, or an optimal maintenance policy.


### V0.6 FINAL R2 — Gate-F smoke/checkpoint correction

This maintenance revision changes no governing equation, physical domain,
acceptance threshold, AVT/FVT protocol, or article claim. It corrects a workflow
mismatch exposed by the executed Web-Jupyter run:

- `auto` correctly selected `smoke` on `login02`;
- the original Gate-F smoke budget (`200 + 450` Adam epochs and `80` L-BFGS
  iterations) was insufficient for the same strict Full-PINN quality gates used
  by the publication run;
- an unverified smoke checkpoint could be reloaded merely because its metadata
  matched, while a stronger compatible V0.5/V0.4 checkpoint was rejected because
  its training budget was larger than the smoke request;
- an invalid current checkpoint prevented the generic loader from continuing to
  the legacy fallback;
- the smoke greedy cap of two states was inconsistent with the previously
  executed residual history, which required three states to fall below 3%.

R1 therefore validates checkpoint **quality as well as metadata**, accepts a
physically identical overqualified full checkpoint for a smoke workflow, rejects
and records undertrained/unverified smoke checkpoints, continues to legacy
fallback after a current-checkpoint validation rejection, and uses at least three
states for the Gate-F smoke workflow. Full-run scientific thresholds remain
unchanged. A smoke run remains non-publication evidence.


In [1]:
from __future__ import annotations

import hashlib
import gc
import importlib.metadata as ilm
import json
import os
import platform
import shutil
import random
import socket
import re
import subprocess
import tempfile
import time
import uuid
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import scipy
import scipy.io as sio
import torch
from IPython.display import display
from scipy import signal, stats
from scipy.optimize import linear_sum_assignment


# Preserve the standard encoder before routing notebook JSON writes through
# the recursive sanitizer below.
_JSON_DUMPS_ORIGINAL = json.dumps

def json_safe_tree(value):
    """Recursively convert scientific Python objects into strict JSON data."""
    if value is None or isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, np.generic):
        return json_safe_tree(value.item())

    if isinstance(value, float):
        return value if np.isfinite(value) else None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if torch.is_tensor(value):
        detached = value.detach().cpu()
        if detached.ndim == 0:
            return json_safe_tree(detached.item())
        return json_safe_tree(detached.tolist())

    if isinstance(value, np.ndarray):
        return json_safe_tree(value.tolist())

    if isinstance(value, pd.DataFrame):
        return json_safe_tree(value.to_dict(orient="records"))

    if isinstance(value, pd.Series):
        return json_safe_tree(value.to_dict())

    if isinstance(value, dict):
        return {
            str(json_safe_tree(key)): json_safe_tree(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [json_safe_tree(item) for item in value]

    if isinstance(value, set):
        converted = [json_safe_tree(item) for item in value]
        return sorted(converted, key=lambda item: repr(item))

    raise TypeError(
        "Object of type "
        f"{type(value).__name__} is not supported by json_safe_tree"
    )

def json_dumps_safe(value, *args, **kwargs) -> str:
    """Serialize a recursively sanitized payload using the standard encoder."""
    # All generated evidence files must be valid RFC-compatible JSON.
    kwargs.setdefault("allow_nan", False)
    return _JSON_DUMPS_ORIGINAL(json_safe_tree(value), *args, **kwargs)

# Startup regression test for the exact scientific types that previously
# interrupted the final protocol/evidence stage.
_json_serializer_probe = {
    "numpy_bool": np.bool_(True),
    "numpy_integer": np.int64(7),
    "numpy_float": np.float64(0.25),
    "numpy_array": np.array([1.0, np.nan, np.inf]),
    "torch_scalar": torch.tensor(2.5),
    "torch_vector": torch.tensor([1, 2, 3]),
    "path": Path("/tmp/json-safe-probe"),
    "timestamp": pd.Timestamp("2026-09-05T00:00:00Z"),
    "tuple": (1, np.bool_(False)),
    "set": {"b", "a"},
}
_json_serializer_roundtrip = json.loads(
    json_dumps_safe(_json_serializer_probe, sort_keys=True)
)
if (
    _json_serializer_roundtrip["numpy_bool"] is not True
    or _json_serializer_roundtrip["numpy_array"][1] is not None
    or _json_serializer_roundtrip["numpy_array"][2] is not None
    or _json_serializer_roundtrip["torch_vector"] != [1, 2, 3]
):
    raise RuntimeError("R3 JSON serializer startup regression test failed")
del _json_serializer_probe, _json_serializer_roundtrip

VERSION = "V0_6_FINAL"
PATCH_LEVEL = "V0_6_FINAL_R3_JSON_SAFE_FINAL_EVIDENCE_PIPELINE"

PROJECT_ROOT = Path("/home/a.riyahi/GPT_PINN").resolve()
DATASET_ROOT = Path("/home/a.riyahi/EvoBridge_SHM/data/raw/z24").resolve()
NOTEBOOK_PATH = PROJECT_ROOT / "GPT_PINN_Z24_Physics_Digital_Twin_V0_6_FINAL_R3.ipynb"

HOSTNAME = socket.gethostname()
IS_LOGIN_NODE = HOSTNAME.lower().startswith("login")
SLURM_JOB_ID = os.environ.get("SLURM_JOB_ID")
IN_SLURM_ALLOCATION = bool(SLURM_JOB_ID)

REQUESTED_RUN_PROFILE = os.environ.get("GPT_PINN_RUN_PROFILE", "auto").strip().lower()
if REQUESTED_RUN_PROFILE not in {"auto", "smoke", "full"}:
    raise ValueError("GPT_PINN_RUN_PROFILE must be 'auto', 'smoke' or 'full'")

COMPUTE_GUARD_POLICY = os.environ.get("GPT_PINN_COMPUTE_GUARD", "strict").strip().lower()
if COMPUTE_GUARD_POLICY not in {"warn", "strict", "auto_smoke"}:
    raise ValueError("GPT_PINN_COMPUTE_GUARD must be 'warn', 'strict' or 'auto_smoke'")

ALLOW_LOGIN_NODE_HEAVY = os.environ.get("GPT_PINN_ALLOW_LOGIN_HEAVY", "0") == "1"
AUTO_DOWNGRADED_TO_SMOKE = False
if REQUESTED_RUN_PROFILE == "auto":
    RUN_PROFILE = "full" if (IN_SLURM_ALLOCATION and not IS_LOGIN_NODE) else "smoke"
else:
    RUN_PROFILE = REQUESTED_RUN_PROFILE
if (RUN_PROFILE == "full" and IS_LOGIN_NODE and not ALLOW_LOGIN_NODE_HEAVY
        and COMPUTE_GUARD_POLICY == "auto_smoke"):
    RUN_PROFILE = "smoke"
    AUTO_DOWNGRADED_TO_SMOKE = True

RUN_ID = os.environ.get(
    "GPT_PINN_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
).strip()
if not re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_ID):
    raise ValueError("GPT_PINN_RUN_ID may contain only letters, digits, dot, underscore and hyphen")

VERSION_ROOT = PROJECT_ROOT / "outputs" / "v0_6_final"
OUTPUT_ROOT = VERSION_ROOT / "runs" / RUN_ID
CACHE_ROOT = PROJECT_ROOT / "cache" / "v0_6_final"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
TABLE_ROOT = OUTPUT_ROOT / "tables"
CHECKPOINT_BASE_ROOT = VERSION_ROOT / "checkpoints"
CHECKPOINT_ROOT = CHECKPOINT_BASE_ROOT  # replaced by a config-hashed directory below
LOG_ROOT = PROJECT_ROOT / "logs" / "v0_6_final" / RUN_ID
for directory in (
    VERSION_ROOT, OUTPUT_ROOT, CACHE_ROOT, FIGURE_ROOT,
    TABLE_ROOT, CHECKPOINT_BASE_ROOT, LOG_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

RESOURCE_POLICY_EVENTS: list[dict] = []


CHECKPOINT_IO_EVENTS: list[dict] = []
CHECKPOINT_IO_EVENT_LOG = LOG_ROOT / "checkpoint_io_events.jsonl"

def _record_checkpoint_event(event: dict) -> None:
    CHECKPOINT_IO_EVENTS.append(event)
    try:
        with CHECKPOINT_IO_EVENT_LOG.open("a", encoding="utf-8") as handle:
            handle.write(json_dumps_safe(event, sort_keys=True) + "\n")
            handle.flush()
            os.fsync(handle.fileno())
    except Exception as exc:
        warnings.warn(
            f"Could not persist checkpoint I/O event log: {type(exc).__name__}: {exc}",
            RuntimeWarning,
            stacklevel=2,
        )

def _jsonable_checkpoint_value(value):
    """Convert diagnostic values to JSON-safe scalars without touching model tensors."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

def _checkpoint_sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def _cpu_checkpoint_tree(value):
    """Detach checkpoint tensors and move them to CPU before filesystem serialization."""
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: _cpu_checkpoint_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_cpu_checkpoint_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_cpu_checkpoint_tree(item) for item in value)
    return value

def _checkpoint_parent_diagnostics(path: Path) -> dict:
    parent = path.parent
    try:
        disk = shutil.disk_usage(parent)
        free_bytes = int(disk.free)
        total_bytes = int(disk.total)
    except Exception:
        free_bytes = -1
        total_bytes = -1
    return {
        "target": str(path),
        "parent": str(parent),
        "parent_exists": parent.exists(),
        "parent_is_directory": parent.is_dir(),
        "parent_writable": os.access(parent, os.W_OK) if parent.exists() else False,
        "free_bytes": free_bytes,
        "total_bytes": total_bytes,
    }

def _quarantine_path(path: Path, reason: str) -> Path | None:
    """Preserve an unreadable/conflicting artifact instead of silently deleting it."""
    if not path.exists():
        return None
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    quarantine = path.with_name(f"{path.name}.{reason}.{timestamp}.{uuid.uuid4().hex[:8]}")
    try:
        os.replace(path, quarantine)
    except Exception as exc:
        warnings.warn(
            f"Could not quarantine {path}: {type(exc).__name__}: {exc}",
            RuntimeWarning,
            stacklevel=2,
        )
        return None
    return quarantine

def safe_torch_load(
    path: str | Path,
    map_location="cpu",
    *,
    weights_only: bool = False,
    stage: str = "checkpoint_load",
    quarantine_on_error: bool = True,
):
    """
    Read a checkpoint defensively.

    A corrupt/truncated checkpoint is quarantined and treated as absent, allowing
    the corresponding training stage to restart rather than failing later.
    """
    path = Path(path)
    if not path.exists():
        return None
    if not path.is_file():
        quarantined = _quarantine_path(path, "not_a_file") if quarantine_on_error else None
        _record_checkpoint_event({
            "operation": "load",
            "stage": stage,
            "status": "INVALID_TARGET_TYPE",
            "path": str(path),
            "quarantined_to": str(quarantined) if quarantined else None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })
        return None
    try:
        loaded = torch.load(path, map_location=map_location, weights_only=weights_only)
        _record_checkpoint_event({
            "operation": "load",
            "stage": stage,
            "status": "PASS",
            "path": str(path),
            "bytes": int(path.stat().st_size),
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })
        return loaded
    except Exception as exc:
        quarantined = _quarantine_path(path, "corrupt") if quarantine_on_error else None
        _record_checkpoint_event({
            "operation": "load",
            "stage": stage,
            "status": "CORRUPT_OR_UNREADABLE",
            "path": str(path),
            "error_type": type(exc).__name__,
            "error": str(exc),
            "quarantined_to": str(quarantined) if quarantined else None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })
        warnings.warn(
            f"Checkpoint {path} could not be loaded and will be regenerated. "
            f"{type(exc).__name__}: {exc}",
            RuntimeWarning,
            stacklevel=2,
        )
        return None

def safe_torch_save(
    payload,
    target: str | Path,
    *,
    stage: str = "checkpoint_save",
    retries: int = 4,
    minimum_free_bytes: int = 128 * 1024 * 1024,
    verify: bool = True,
) -> Path:
    """
    Atomically save a PyTorch checkpoint on NFS/shared HPC storage.

    The write is made to a same-directory temporary file through a Python file
    handle using legacy stream serialization, then flushed, fsync'ed, reloaded
    for verification, and atomically replaced. This avoids the C++ zip-writer
    path-opening failure observed at Gate D and prevents half-written targets.
    """
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and not target.is_file():
        quarantined = _quarantine_path(target, "path_conflict")
        if target.exists():
            raise RuntimeError(
                f"Checkpoint target is not a regular file and could not be quarantined: {target}"
            )
        _record_checkpoint_event({
            "operation": "quarantine",
            "stage": stage,
            "status": "PATH_CONFLICT_MOVED",
            "path": str(target),
            "quarantined_to": str(quarantined) if quarantined else None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })

    diagnostics = _checkpoint_parent_diagnostics(target)
    if not diagnostics["parent_is_directory"] or not diagnostics["parent_writable"]:
        raise RuntimeError(
            "Checkpoint directory is unavailable or not writable: "
            + json_dumps_safe(diagnostics, sort_keys=True)
        )
    if 0 <= diagnostics["free_bytes"] < minimum_free_bytes:
        raise RuntimeError(
            "Insufficient free space for a verified checkpoint write: "
            + json_dumps_safe(diagnostics, sort_keys=True)
        )

    cpu_payload = _cpu_checkpoint_tree(payload)
    errors: list[str] = []
    for attempt in range(1, retries + 1):
        temp_path = target.parent / (
            f".{target.name}.partial-{os.getpid()}-{uuid.uuid4().hex}.tmp"
        )
        try:
            with temp_path.open("wb") as handle:
                torch.save(
                    cpu_payload,
                    handle,
                    pickle_protocol=4,
                    _use_new_zipfile_serialization=False,
                )
                handle.flush()
                os.fsync(handle.fileno())

            if not temp_path.is_file() or temp_path.stat().st_size <= 0:
                raise IOError(f"Temporary checkpoint is empty: {temp_path}")

            if verify:
                verified = torch.load(
                    temp_path,
                    map_location="cpu",
                    weights_only=False,
                )
                if isinstance(cpu_payload, dict) and isinstance(verified, dict):
                    expected_status = cpu_payload.get("status")
                    if expected_status is not None and verified.get("status") != expected_status:
                        raise IOError(
                            f"Checkpoint verification status mismatch: "
                            f"{verified.get('status')!r} != {expected_status!r}"
                        )

            os.replace(temp_path, target)
            try:
                directory_fd = os.open(target.parent, os.O_DIRECTORY)
                try:
                    os.fsync(directory_fd)
                finally:
                    os.close(directory_fd)
            except Exception:
                # Some NFS implementations do not support directory fsync.
                pass

            event = {
                "operation": "save",
                "stage": stage,
                "status": "PASS",
                "path": str(target),
                "bytes": int(target.stat().st_size),
                "sha256": _checkpoint_sha256(target),
                "attempt": attempt,
                "serialization": "python_file_handle_legacy_stream_atomic_replace",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            }
            _record_checkpoint_event(event)
            return target
        except Exception as exc:
            errors.append(f"attempt {attempt}: {type(exc).__name__}: {exc}")
            try:
                temp_path.unlink(missing_ok=True)
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if attempt < retries:
                time.sleep(min(8.0, 0.75 * (2 ** (attempt - 1))))

    diagnostics = _checkpoint_parent_diagnostics(target)
    failure = {
        "operation": "save",
        "stage": stage,
        "status": "FAIL",
        "path": str(target),
        "errors": errors,
        "diagnostics": diagnostics,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    _record_checkpoint_event(failure)
    raise RuntimeError(
        "Verified atomic checkpoint save failed after retries. "
        + json_dumps_safe(failure, sort_keys=True)
    )

def cleanup_stale_checkpoint_partials(root: str | Path, older_than_seconds: float = 3600.0) -> list[str]:
    """Remove only stale temporary checkpoint files, never completed checkpoints."""
    root = Path(root)
    removed: list[str] = []
    now = time.time()
    if not root.exists():
        return removed
    for candidate in root.rglob(".*.partial-*.tmp"):
        try:
            if candidate.is_file() and now - candidate.stat().st_mtime >= older_than_seconds:
                candidate.unlink()
                removed.append(str(candidate))
        except OSError:
            continue
    return removed

def checkpoint_io_probe(root: str | Path) -> dict:
    """Fail early if the selected MARWAN checkpoint directory cannot round-trip a tensor."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    probe = root / f".checkpoint_io_probe_{os.getpid()}_{uuid.uuid4().hex[:8]}.pt"
    payload = {
        "status": "probe",
        "tensor": torch.arange(8, dtype=torch.float64),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    safe_torch_save(payload, probe, stage="startup_checkpoint_io_probe", retries=2)
    loaded = safe_torch_load(
        probe,
        map_location="cpu",
        stage="startup_checkpoint_io_probe",
        quarantine_on_error=False,
    )
    passed = bool(
        isinstance(loaded, dict)
        and loaded.get("status") == "probe"
        and torch.equal(loaded["tensor"], payload["tensor"])
    )
    probe.unlink(missing_ok=True)
    result = {
        "passed": passed,
        "root": str(root),
        "diagnostics": _checkpoint_parent_diagnostics(root / "placeholder.pt"),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    if not passed:
        raise RuntimeError("Checkpoint I/O probe failed: " + json_dumps_safe(result, sort_keys=True))
    return result


def require_compute_node(stage: str) -> None:
    """Apply the configured resource policy without breaking normal Web execution."""
    if RUN_PROFILE != "full" or not IS_LOGIN_NODE or ALLOW_LOGIN_NODE_HEAVY:
        return
    event = {
        "stage": stage,
        "hostname": HOSTNAME,
        "policy": COMPUTE_GUARD_POLICY,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    RESOURCE_POLICY_EVENTS.append(event)
    message = (
        f"{stage} is a heavy full-profile stage and the current host is {HOSTNAME!r}. "
        "V0.6 FINAL protects publication runs; this execution will not "
        "be certified as a Slurm compute-node publication run. Set "
        "GPT_PINN_COMPUTE_GUARD=strict before starting the kernel to forbid this, or "
        "use GPT_PINN_COMPUTE_GUARD=auto_smoke for an automatic lightweight run."
    )
    if COMPUTE_GUARD_POLICY == "strict":
        raise RuntimeError(message)
    warnings.warn(message, RuntimeWarning, stacklevel=2)

# Use all CPUs actually allocated by Slurm. In a direct login-node Web session,
# cap library threads by default to avoid accidental oversubscription.
if IN_SLURM_ALLOCATION and os.environ.get("SLURM_CPUS_PER_TASK"):
    EFFECTIVE_CPU_THREADS = max(1, int(os.environ["SLURM_CPUS_PER_TASK"]))
else:
    login_default = min(8, os.cpu_count() or 1)
    EFFECTIVE_CPU_THREADS = max(1, int(os.environ.get(
        "GPT_PINN_CPU_THREADS", str(login_default if IS_LOGIN_NODE else (os.cpu_count() or 1))
    )))
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = str(EFFECTIVE_CPU_THREADS)
torch.set_num_threads(EFFECTIVE_CPU_THREADS)
try:
    torch.set_num_interop_threads(max(1, min(4, EFFECTIVE_CPU_THREADS)))
except RuntimeError:
    # Safe when the initialization cell is re-run in an already active kernel.
    pass

numpy_trapezoid = getattr(np, "trapezoid", np.trapz)
plt.close("all")

CONFIG = {
    "version": VERSION,
    "patch_level": PATCH_LEVEL,
    "run_id": RUN_ID,
    "requested_run_profile": REQUESTED_RUN_PROFILE,
    "run_profile": RUN_PROFILE,
    "compute_guard_policy": COMPUTE_GUARD_POLICY,
    "auto_downgraded_to_smoke": AUTO_DOWNGRADED_TO_SMOKE,
    "in_slurm_allocation": IN_SLURM_ALLOCATION,
    "slurm_job_id": SLURM_JOB_ID,
    "effective_cpu_threads": EFFECTIVE_CPU_THREADS,
    "project_root": str(PROJECT_ROOT),
    "notebook_path": str(NOTEBOOK_PATH),
    "hostname_at_start": HOSTNAME,
    "is_login_node": IS_LOGIN_NODE,
    "allow_login_node_heavy": ALLOW_LOGIN_NODE_HEAVY,
    "dataset_root": str(DATASET_ROOT),
    "seed": 20260828,
    "sampling_rate_hz": 100.0,
    "frequency_band_hz": [0.2, 25.0],
    "candidate_mode_count": 5,
    "candidate_mode_bands_hz": [[3.5, 4.3], [4.6, 5.6], [9.4, 10.15], [10.15, 10.95], [12.0, 13.6]],
    "published_sanity_modes_hz": [3.92, 5.12, 9.93, 10.52, 12.69],
    "primary_scenarios": ["01", "03", "04", "05", "06"],
    "pier_settlement_mm": [0.0, 20.0, 40.0, 80.0, 95.0],
    "fdd_nperseg": 4096,
    "fdd_noverlap": 2048,
    "fdd_smoothing_width_hz": 0.5,
    "mode_tracking_max_relative_shift": 0.22,
    "mode_tracking_frequency_weight": 3.0,
    "mode_tracking_mac_weight": 0.8,
    "minimum_setup_support": 7,
    "minimum_abs_spearman_for_primary_mode": 0.8,
    "minimum_monotonic_nonincrease_accuracy": 1.0,
    "healthy_frequency_sanity_tolerance_fraction": 0.02,
    "bootstrap_replicates": 2000,
    "reference_channels": ["R1V", "R2L", "R2T", "R2V", "R3V"],
    "fvt_shaker_channels": ["DP1V", "DP2V"],
    "expected_avt_to_fvt_setup_map": {"1": 9, "2": 8, "3": 7, "4": 6, "5": 5, "6": 4, "7": 3, "8": 2, "9": 1},
    "fdd_segment_detrend": "constant",
    "modal_ratio_covariance_shrinkage": 0.20,
    "checkpoint_save_retries": 4,
    "checkpoint_minimum_free_mib": 128,
    "import_compatible_legacy_checkpoints": os.environ.get(
        "GPT_PINN_IMPORT_LEGACY_CHECKPOINTS",
        os.environ.get("GPT_PINN_IMPORT_V04_CHECKPOINTS", "1"),
    ) == "1",
    "gate_b_mode": os.environ.get("GPT_PINN_GATE_B_MODE", RUN_PROFILE).strip().lower(),
    "gate_b_resume": os.environ.get("GPT_PINN_GATE_B_RESUME", "1") == "1",
    "beam_fe_elements": 80,
    "collocation_points": 192,
    "validation_points": 501,
    "network_architecture": [1, 32, 32, 32, 1],
    "learning_rate": 2e-3,
    "gate_b_full_adam_epochs": 1800,
    "gate_b_full_lbfgs_iterations": 220,
    "gate_b_smoke_adam_epochs": 250,
    "gate_b_smoke_lbfgs_iterations": 40,
    "gate_b_alpha_values": [0.6, 0.8, 1.0, 1.2, 1.4],
    "gate_b_seeds": [101, 202, 303, 404, 505],
    "gate_c_mode": os.environ.get("GPT_PINN_GATE_C_MODE", RUN_PROFILE).strip().lower(),
    "gate_c_resume": os.environ.get("GPT_PINN_GATE_C_RESUME", "1") == "1",
    "gate_c_mu_domain": [0.0, 0.7],
    "gate_c_stiffness_center": 44.0/58.0,
    "gate_c_stiffness_width": 0.14,
    "gate_c_full_candidate_count": 15,
    "gate_c_smoke_candidate_count": 5,
    "gate_c_full_max_basis": 5,
    "gate_c_smoke_max_basis": 5,
    "gate_c_min_basis": 2,
    "gate_c_greedy_loss_tolerance": 1e-5,
    "gate_c_full_adam_epochs": 1800,
    "gate_c_full_lbfgs_iterations": 220,
    "gate_c_smoke_adam_epochs": 800,
    "gate_c_smoke_lbfgs_iterations": 140,
    "gate_c_reduced_adam_steps": 180,
    "gate_c_reduced_lbfgs_iterations": 50,
    "gate_c_timing_repeats": 3,
    "gate_d_mode": os.environ.get("GPT_PINN_GATE_D_MODE", RUN_PROFILE).strip().lower(),
    "gate_d_resume": os.environ.get("GPT_PINN_GATE_D_RESUME", "1") == "1",
    "gate_d_equivalent_mode_orders": [1, 2, 3, 4],
    "gate_d_retained_equivalent_modes": [1, 3, 4],
    "gate_d_full_adam_epochs": 2200,
    "gate_d_full_lbfgs_iterations": 300,
    "gate_d_smoke_adam_epochs": 1000,
    "gate_d_smoke_lbfgs_iterations": 160,
    "gate_d_reduced_adam_steps": 300,
    "gate_d_reduced_lbfgs_iterations": 100,
    "gate_d_full_response_points": 141,
    "gate_d_smoke_response_points": 31,
    "gate_e_mode": os.environ.get("GPT_PINN_GATE_E_MODE", RUN_PROFILE).strip().lower(),
    "gate_e_fe_elements": 58,
    "gate_e_local_zone_width_m": 2.5,
    "gate_e_dimensionless_support_kappa": 3.0e6,
    "gate_e_fixed_local_exponent": 1.0,
    "gate_e_fixed_support_exponent": 0.5,
    "gate_e_calibration_max_state_rmse": 0.01,
    "gate_e_max_standardized_residual": 3.0,
    "gate_e_max_reduced_chi_square": 4.0,
    "gate_e_loso_interpolation_rmse": 0.01,
    "gate_e_loso_extrapolation_rmse": 0.015,
    "gate_e_max_jacobian_condition": 100.0,
    "gate_e_max_mesh_ratio_change": 5.0e-4,
    # Gate-E covariance-aware model-adequacy correction. These settings do not
    # alter the PINN/FE checkpointed solvers and are excluded from their hash.
    "gate_e_use_joint_modal_covariance": True,
    "gate_e_model_discrepancy_policy": "profiled_gaussian_plus_minimum_3sigma_adequacy",
    "gate_e_max_model_discrepancy_sd": 0.010,
    "gate_e_model_discrepancy_search_tolerance": 1.0e-6,
    "gate_e_feasibility_tolerance": 2.0e-4,
    "gate_e_profile_grid_points": 17,
    "gate_e_profile_max_iterations": 28,
    "gate_e_optimizer_max_nfev": 1200,
    "gate_f_mode": os.environ.get("GPT_PINN_GATE_F_MODE", RUN_PROFILE).strip().lower(),
    "gate_f_resume": os.environ.get("GPT_PINN_GATE_F_RESUME", "1") == "1",
    "gate_f_alpha_domain": [0.0, 0.65],
    "gate_f_log10_support_domain": [-2.0, 0.0],
    "gate_f_full_candidate_axis_count": 5,
    "gate_f_smoke_candidate_axis_count": 3,
    "gate_f_full_max_basis_states": 5,
    "gate_f_smoke_max_basis_states": 3,
    "gate_f_greedy_weak_residual_tolerance": 0.03,
    "gate_f_collocation_points": 321,
    "gate_f_basis_points": 501,
    "gate_f_full_warm_adam_epochs": 1800,
    "gate_f_full_projected_adam_epochs": 4000,
    "gate_f_full_lbfgs_iterations": 800,
    "gate_f_smoke_warm_adam_epochs": 200,
    "gate_f_smoke_projected_adam_epochs": 450,
    "gate_f_smoke_lbfgs_iterations": 80,
    # Smoke is a workflow test, but it still requires a numerically usable
    # reduced basis. V0.5.2 needed three validated states to cross the 3% weak
    # residual tolerance; two states were demonstrably insufficient.
    # A physically identical, quality-validated checkpoint trained with a
    # larger budget is safe to reuse for smoke. The converse is forbidden.
    "gate_f_smoke_prefer_overqualified_checkpoint": True,
    "gate_f_reject_unverified_exact_smoke_checkpoint": True,
    # R2 workflow controls. A smoke run may use an FE-snapshot proxy only
    # when no independently quality-validated Full-PINN checkpoint exists.
    # The proxy is never publication evidence and never participates in a
    # full-profile run.
    "gate_f_recursive_checkpoint_scan": True,
    "gate_f_smoke_fe_proxy_when_no_validated_checkpoint": (
        os.environ.get("GPT_PINN_GATE_F_SMOKE_FE_PROXY", "1") == "1"
    ),
    "gate_f_network_architecture": [1, 64, 64, 64, 4],
    "gate_f_max_full_pinn_frequency_error": 0.01,
    "gate_f_min_full_pinn_mac": 0.99,
    "gate_f_max_full_pinn_weak_residual": 0.03,
    "gate_f_max_gpt_frequency_error": 0.01,
    "gate_f_min_gpt_mac": 0.99,
    "gate_f_max_gpt_weak_residual": 0.03,
    "gate_f_max_mass_orthogonality_error": 1.0e-5,
    "gate_f_min_speedup_vs_full_pinn": 100.0,
    "gate_f_min_speedup_vs_fe": 2.0,
    "gate_g_mode": os.environ.get("GPT_PINN_GATE_G_MODE", RUN_PROFILE).strip().lower(),
    "gate_g_process_domain_fraction": 0.5,
    "gate_g_full_bootstrap_replicates": 1000,
    "gate_g_smoke_bootstrap_replicates": 100,
    "gate_g_max_state_modal_ratio_rmse": 0.01,
    "gate_g_max_standardized_residual": 3.0,
    "gate_g_nis_probability": 0.99,
    "gate_g_max_weighted_jacobian_condition": 300.0,
    "gate_g_max_quantile_split_scaled_difference": 0.15,
    "gate_g_max_performance_deviation_from_batch": 0.05,
    "gate_g_max_final_performance_index": 0.25,
    "gate_g_max_online_update_seconds": 2.0,
    "gate_g_min_bootstrap_valid_fraction": 0.99,
    "gate_g1_mode": os.environ.get("GPT_PINN_GATE_G1_MODE", RUN_PROFILE).strip().lower(),
    "gate_g1_process_sd": 0.5,
    "gate_g1_full_bootstrap_replicates": 1000,
    "gate_g1_smoke_bootstrap_replicates": 100,
    "gate_g1_full_loso_bootstrap_replicates": 400,
    "gate_g1_smoke_loso_bootstrap_replicates": 50,
    "gate_g1_max_calibration_standardized_residual": 3.0,
    "gate_g1_max_online_modal_ratio_rmse": 0.01,
    "gate_g1_max_online_standardized_residual": 3.0,
    "gate_g1_max_online_absolute_xi_error": 0.15,
    "gate_g1_max_loso_interpolation_absolute_xi_error": 0.15,
    "gate_g1_max_loso_extrapolation_absolute_xi_error": 0.15,
    "gate_g1_min_loso_ci_coverage": 1.0,
    "gate_g1_min_loso_spearman": 0.9,
    "gate_g1_min_physics_baseline_win_fraction": 0.75,
    "gate_g1_max_bootstrap_split_difference": 0.05,
    "gate_g1_max_loso_bootstrap_split_difference": 0.10,
    "gate_g1_max_kde_mode_count": 1,
    "gate_g1_max_ci_width": 0.5,
    "gate_g1_min_weighted_sensitivity": 2.0,
    "gate_g1_max_online_update_seconds": 1.0,
    "gate_h_mode": os.environ.get("GPT_PINN_GATE_H_MODE", RUN_PROFILE).strip().lower(),
    "gate_h_full_bootstrap_replicates": 1000,
    "gate_h_smoke_bootstrap_replicates": 100,
    "gate_h_beta_prior_mean": 1.0,
    "gate_h_beta_prior_sd": 0.5,
    "gate_h_beta_bounds": [0.0, 2.0],
    "gate_h_process_sd": 0.12,
    "gate_h_model_discrepancy_floor": 0.01,
    "gate_h_warning_health": 0.20,
    "gate_h_max_absolute_xi_error": 0.15,
    "gate_h_max_mean_modal_rmse": 0.015,
    "gate_h_max_single_modal_rmse": 0.020,
    "gate_h_min_xi_ci_coverage": 2.0/3.0,
    "gate_h_min_modal_ci_coverage": 0.80,
    "gate_h_max_mean_xi_crps": 0.15,
    "gate_h_max_split_quantile_difference": 0.08,
    "gate_h_max_xi_ci_width": 0.65,
    # Point forecast and uncertainty audit are distinct computational products.
    "gate_h_max_point_forecast_seconds": 0.10,
    "gate_h_max_probabilistic_forecast_seconds": 5.0,
    "gate_h_max_excess_mae_over_schedule_baseline": 0.05,
    "gate_h_max_excess_modal_rmse_over_schedule_baseline": 0.01,
    "gate_h_min_schedule_baseline_win_fraction_for_autonomy": 2.0/3.0,
    "basis_size": 8,
    "optimizer": "Adam_then_L-BFGS",
    "loss_weights": {"pde": 10.0, "bc": 10.0, "norm": 50.0, "rayleigh": 2.0},
    "gate_i_mode": os.environ.get("GPT_PINN_GATE_I_MODE", RUN_PROFILE).strip().lower(),
    "gate_i_excitation": "fvt",
    "gate_i_minimum_setup_support": 6,
    "gate_i_healthy_anchor_radius_hz": 0.45,
    "gate_i_tracking_max_relative_shift": 0.25,
    "gate_i_full_bootstrap_replicates": 1000,
    "gate_i_smoke_bootstrap_replicates": 100,
    "gate_i_min_spearman": 0.90,
    "gate_i_max_mean_absolute_xi_error": 0.15,
    "gate_i_max_absolute_xi_error": 0.25,
    "gate_i_max_modal_ratio_rmse": 0.025,
    "gate_i_min_ci_coverage": 0.80,
    "gate_i_max_bootstrap_split_difference": 0.10,
    "gate_i_max_physics_excess_mae_over_best_baseline": 0.05,
    "allow_network_clone": os.environ.get("GPT_PINN_ALLOW_NETWORK_CLONE", "0") == "1",
    "primary_article_scope": "IDENTIFIABILITY_AWARE_GPT_PINN_Z24_MODAL_DT_WITH_LOCKED_FVT_AUDIT",
    "current_execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
    "final_article_title": "A Greedy GPT-PINN Reduced Digital Twin for Rapid Bridge Modal Updating: Identifiability and Cross-Excitation Assessment on the Z24 Benchmark",
    "final_timing_warmup_calls": 30,
    "final_timing_gpt_repeats_full": 2000,
    "final_timing_gpt_repeats_smoke": 200,
    "final_timing_fe_repeats_full": 200,
    "final_timing_fe_repeats_smoke": 20,
    "final_timing_min_speedup_vs_fe": 10.0,
    "final_ablation_max_frequency_error": 0.002,
    "final_sensitivity_fe_meshes_full": [29, 58, 116, 232],
    "final_sensitivity_fe_meshes_smoke": [29, 58, 116],
    "final_sensitivity_basis_points_full": [251, 501, 1001],
    "final_sensitivity_basis_points_smoke": [251, 501],
    "final_sensitivity_max_mesh_frequency_change": 0.002,
    "final_sensitivity_max_basis_frequency_change": 0.002,
    "final_checkpoint_compatibility_generation": "V0_5_PHYSICS_CORE",
    "checkpoint_io_policy": "atomic same-directory temporary file; verified reload; NFS-safe stream serialization",
}

# A deterministic checkpoint namespace prevents silent reuse after any
# training/physics configuration change. Runtime-only fields are excluded.
_CHECKPOINT_RUNTIME_KEYS = {
    "version", "run_id", "requested_run_profile", "run_profile", "project_root",
    "notebook_path", "hostname_at_start", "is_login_node",
    "allow_login_node_heavy", "compute_guard_policy", "auto_downgraded_to_smoke",
    "in_slurm_allocation", "slurm_job_id", "effective_cpu_threads",
    "allow_network_clone", "current_execution_boundary", "primary_article_scope",
    "import_compatible_legacy_checkpoints", "checkpoint_save_retries",
    "checkpoint_minimum_free_mib", "checkpoint_io_policy",
    # Gate-F smoke/recovery controls do not alter an individual Full-PINN's
    # governing equations, architecture, collocation grid, or learned weights.
    "gate_f_smoke_max_basis_states",
    "gate_f_smoke_prefer_overqualified_checkpoint",
    "gate_f_reject_unverified_exact_smoke_checkpoint",
    "gate_f_recursive_checkpoint_scan",
    "gate_f_smoke_fe_proxy_when_no_validated_checkpoint",
    # Patch-only Gate-E likelihood/adequacy controls do not change trained
    # Full-PINN or GPT-PINN basis checkpoints.
    "patch_level",
    "gate_e_use_joint_modal_covariance",
    "gate_e_model_discrepancy_policy",
    "gate_e_max_model_discrepancy_sd",
    "gate_e_model_discrepancy_search_tolerance",
    "gate_e_feasibility_tolerance",
    "gate_e_profile_grid_points",
    "gate_e_profile_max_iterations",
    "gate_e_optimizer_max_nfev",
}
_checkpoint_payload = {
    key: value for key, value in CONFIG.items()
    if key not in _CHECKPOINT_RUNTIME_KEYS and not key.endswith("_resume") and not key.startswith("final_")
}
CHECKPOINT_SIGNATURE = hashlib.sha256(
    json_dumps_safe(_checkpoint_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]
CHECKPOINT_ROOT = CHECKPOINT_BASE_ROOT / CHECKPOINT_SIGNATURE
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
def discover_checkpoint_search_roots() -> list[Path]:
    """Return every plausible historical checkpoint container.

    R1 only kept immediate hash directories below v0_6_final/v0_5/v0_4.
    That produced an empty search list when a version directory existed but
    its files were stored directly, nested differently, or under the legacy
    outputs/checkpoints tree. R2 indexes the containers themselves and scans
    them recursively while excluding the active target file later.
    """
    candidates = [
        PROJECT_ROOT / "outputs" / "v0_6_final" / "checkpoints",
        PROJECT_ROOT / "outputs" / "v0_5" / "checkpoints",
        PROJECT_ROOT / "outputs" / "v0_4" / "checkpoints",
        PROJECT_ROOT / "outputs" / "checkpoints",
    ]
    roots: set[Path] = set()
    for candidate in candidates:
        if candidate.is_dir():
            roots.add(candidate.resolve())
    return sorted(roots)

LEGACY_CHECKPOINT_ROOTS = discover_checkpoint_search_roots()

def iter_legacy_checkpoint_candidates(filename: str, *, exclude: Path | None = None):
    """Yield matching checkpoint files from all prior output layouts."""
    seen: set[Path] = set()
    excluded = exclude.resolve() if exclude is not None else None
    for root in LEGACY_CHECKPOINT_ROOTS:
        for candidate in root.rglob(filename):
            try:
                resolved = candidate.resolve()
            except OSError:
                continue
            if excluded is not None and resolved == excluded:
                continue
            if resolved in seen or not candidate.is_file():
                continue
            seen.add(resolved)
            yield candidate

LEGACY_GATE_F_CHECKPOINTS_AT_START = sorted(
    str(path)
    for root in LEGACY_CHECKPOINT_ROOTS
    for path in root.rglob("multispan_full_pinn_a*_r*.pt")
    if path.is_file() and CHECKPOINT_ROOT.resolve() not in path.resolve().parents
)
STALE_CHECKPOINT_PARTIALS_REMOVED = cleanup_stale_checkpoint_partials(
    CHECKPOINT_BASE_ROOT, older_than_seconds=3600.0
)
CHECKPOINT_IO_PROBE = checkpoint_io_probe(CHECKPOINT_ROOT)
CONFIG["checkpoint_io_probe"] = CHECKPOINT_IO_PROBE
CONFIG["stale_checkpoint_partials_removed"] = STALE_CHECKPOINT_PARTIALS_REMOVED
CONFIG["legacy_checkpoint_roots"] = [str(path) for path in LEGACY_CHECKPOINT_ROOTS]
CONFIG["legacy_gate_f_checkpoints_at_start"] = LEGACY_GATE_F_CHECKPOINTS_AT_START

def load_checkpoint_with_legacy_fallback(
    target: str | Path,
    *,
    map_location,
    stage: str,
    validator,
):
    """
    Load the V0.6 FINAL checkpoint or import an explicitly validated compatible prior V0.6/V0.5/V0.4 file.

    Compatibility is decided by the caller from physical parameters, architecture,
    epoch counts and completion diagnostics. A corrupt legacy file is quarantined.
    """
    target = Path(target)
    loaded = safe_torch_load(target, map_location=map_location, stage=stage)
    if loaded is not None:
        if validator(loaded):
            return loaded
        # V0.6 FINAL originally returned here, so one metadata-valid but
        # scientifically unusable smoke checkpoint prevented inspection of a
        # stronger compatible legacy checkpoint. Record the rejection and
        # continue to the fallback search without deleting the artifact.
        _record_checkpoint_event({
            "operation": "load",
            "stage": stage,
            "status": "CURRENT_CHECKPOINT_VALIDATION_REJECTED_CONTINUE_TO_LEGACY",
            "path": str(target),
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        })
    if not CONFIG.get("import_compatible_legacy_checkpoints", False):
        return None
    for candidate in iter_legacy_checkpoint_candidates(
        target.name,
        exclude=target,
    ):
        legacy = safe_torch_load(
            candidate,
            map_location=map_location,
            stage=f"{stage}:legacy_checkpoint_probe",
            quarantine_on_error=False,
        )
        if legacy is None or not validator(legacy):
            continue
        imported = dict(legacy)
        imported["imported_from_legacy"] = str(candidate)
        imported["imported_into_version"] = VERSION
        safe_torch_save(
            imported,
            target,
            stage=f"{stage}:legacy_checkpoint_import",
            retries=CONFIG["checkpoint_save_retries"],
            minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2,
        )
        return imported
    return None

CONFIG["checkpoint_signature"] = CHECKPOINT_SIGNATURE
CONFIG["checkpoint_root"] = str(CHECKPOINT_ROOT)
CONFIG["support_stiffness_convention"] = (
    "kappa_K=k_K*L^3/EI0; physical FE spring k_K=kappa_K*EI0/L^3"
)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

CONFIG_PATH = OUTPUT_ROOT / "config.json"
CONFIG_PATH.write_text(json_dumps_safe(CONFIG, indent=2, sort_keys=True) + "\n", encoding="utf-8")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 320,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

REFERENCE_URLS = {
    "z24_official": "https://bwk.kuleuven.be/bwm/z24",
    "z24_fe_updating": "https://doi.org/10.1016/S0022-460X(03)01319-1",
    "z24_environmental_variability": "https://doi.org/10.1002/1096-9845(200102)30:2%3C149::AID-EQE1%3E3.0.CO;2-Z",
    "gpt_pinn_doi": "https://doi.org/10.1016/j.finel.2023.104047",
    "gpt_pinn_repo": "https://github.com/skoohy/GPT-PINN",
    "s2gpt_pinn_repo": "https://github.com/DuktigYajie/S2GPT-PINN",
    "z24_reference_repo": "https://github.com/Elios-Lab/WaveNet_MiniRocket_Z24_Bridge_Structural_Health_Monitoring",
    "explainable_z24_repo": "https://github.com/you-soof/explainable-shm-z24",
    "rom_gpt_pinn_repo": "https://github.com/ljq2025/ROM-GPT-PINN",
    "z24_anomaly_repo": "https://github.com/riyaarah/z24-shm-anomaly-detection",
    "z24_online_occ_repo": "https://github.com/AmroAbdrabo/application-of-online-anomaly-detection-using-one-class-classification-to-z24",
    "z24_online_occ_doi": "https://doi.org/10.3390/s24237866",
    "z24_hybrid_avt_fvt_doi": "https://doi.org/10.3390/jsan15020031",
    "z24_autoencoder_doi": "https://doi.org/10.1111/mice.12943",
}

print("VERSION:", VERSION)
print("PATCH_LEVEL:", PATCH_LEVEL)
print("RUN_ID:", RUN_ID)
print("REQUESTED_RUN_PROFILE:", REQUESTED_RUN_PROFILE)
print("EFFECTIVE RUN_PROFILE:", RUN_PROFILE)
print("COMPUTE_GUARD_POLICY:", COMPUTE_GUARD_POLICY)
print("IN_SLURM_ALLOCATION:", IN_SLURM_ALLOCATION, "SLURM_JOB_ID:", SLURM_JOB_ID)
print("EFFECTIVE_CPU_THREADS:", EFFECTIVE_CPU_THREADS)
print("HOSTNAME:", HOSTNAME)
print("IS_LOGIN_NODE:", IS_LOGIN_NODE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Z24 DATASET_ROOT:", DATASET_ROOT)
print("Primary article scope:", CONFIG["primary_article_scope"])
print("Run outputs:", OUTPUT_ROOT)
print("V0.6 FINAL checkpoint signature/root:", CHECKPOINT_SIGNATURE, CHECKPOINT_ROOT)
print("Checkpoint I/O probe:", CHECKPOINT_IO_PROBE)
print("Historical checkpoint search roots discovered:", len(LEGACY_CHECKPOINT_ROOTS))
print("Historical Gate-F checkpoint files indexed:", len(LEGACY_GATE_F_CHECKPOINTS_AT_START))
print("CONFIG saved:", CONFIG_PATH)

VERSION: V0_6_FINAL
PATCH_LEVEL: V0_6_FINAL_R3_JSON_SAFE_FINAL_EVIDENCE_PIPELINE
RUN_ID: V0_6_FINAL_R3_FULL_508150_20260906T011342Z
REQUESTED_RUN_PROFILE: full
EFFECTIVE RUN_PROFILE: full
COMPUTE_GUARD_POLICY: strict
IN_SLURM_ALLOCATION: True SLURM_JOB_ID: 508150
EFFECTIVE_CPU_THREADS: 8
HOSTNAME: data31
IS_LOGIN_NODE: False
PROJECT_ROOT: /home/a.riyahi/GPT_PINN
Z24 DATASET_ROOT: /home/a.riyahi/EvoBridge_SHM/data/raw/z24
Primary article scope: IDENTIFIABILITY_AWARE_GPT_PINN_Z24_MODAL_DT_WITH_LOCKED_FVT_AUDIT
Run outputs: /home/a.riyahi/GPT_PINN/outputs/v0_6_final/runs/V0_6_FINAL_R3_FULL_508150_20260906T011342Z
V0.6 FINAL checkpoint signature/root: 970ffcaa511ebcb7 /home/a.riyahi/GPT_PINN/outputs/v0_6_final/checkpoints/970ffcaa511ebcb7
Checkpoint I/O probe: {'passed': True, 'root': '/home/a.riyahi/GPT_PINN/outputs/v0_6_final/checkpoints/970ffcaa511ebcb7', 'diagnostics': {'target': '/home/a.riyahi/GPT_PINN/outputs/v0_6_final/checkpoints/970ffcaa511ebcb7/placeholder.pt', 'parent': '/home/

In [2]:
# Figure 1 — research workflow (conceptual; no inference claim is made at Gate A).
fig, ax = plt.subplots(figsize=(13.0, 2.8))
ax.axis("off")
labels = [
    "Z24 vibration\nmeasurements", "Modal / spectral\nobservations",
    "Parametric structural\nphysics", "Full PINN\nbasis states",
    "GPT-PINN reduced\nphysics surrogate", "Inverse Digital Twin\nstate update",
    "Conditional response\npropagation",
]
x = np.linspace(0.06, 0.94, len(labels))
colors = ["#264653", "#2a9d8f", "#457b9d", "#6d597a", "#e76f51", "#f4a261", "#8d6e63"]
for i, (xi, label, color) in enumerate(zip(x, labels, colors)):
    ax.text(xi, 0.5, label, ha="center", va="center", color="white", weight="bold",
            transform=ax.transAxes, bbox=dict(boxstyle="round,pad=0.55", fc=color, ec="none"))
    if i < len(labels) - 1:
        ax.annotate("", xy=(x[i+1] - 0.065, 0.5), xytext=(xi + 0.065, 0.5),
                    xycoords=ax.transAxes, arrowprops=dict(arrowstyle="->", lw=1.7, color="#333333"))
ax.text(0.5, 0.10, "V0.6 FINAL boundary: modal updating, identifiability, and locked FVT audit", ha="center",
        transform=ax.transAxes, color="#9d0208", weight="bold")
fig.savefig(FIGURE_ROOT / "figure_01_research_workflow.png", bbox_inches="tight", dpi=320)
fig.savefig(FIGURE_ROOT / "figure_01_research_workflow.pdf", bbox_inches="tight")
plt.show()
plt.close("all")

## 1. MARWAN environment audit

This audit runs before any installation or heavy computation. The current host is expected to be a login node; absence of a visible GPU here does not imply the cluster has no GPU nodes.

In [3]:
def run_shell(command: str) -> tuple[int, str]:
    completed = subprocess.run(command, shell=True, executable="/bin/bash", text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return completed.returncode, completed.stdout.rstrip()

audit_commands = [
    "pwd", "hostname", "date", "whoami",
    "python --version", "which python", "which pip", "which conda || true",
    "git --version", "nvidia-smi || true", "sinfo || true", 'squeue -u "$USER" || true',
    "df -h /home/a.riyahi || true", "df -i /home/a.riyahi || true",
    "mount | grep -E ' /home |nfsshare' || true", "ulimit -n || true",
    f"ls -ld {CHECKPOINT_BASE_ROOT} {CHECKPOINT_ROOT} || true",
    f"namei -om {CHECKPOINT_ROOT} || true",
    "du -sh /home/a.riyahi/EvoBridge_SHM/data/raw/z24/ || true",
]
environment_audit = []
for command in audit_commands:
    returncode, output = run_shell(command)
    environment_audit.append({"command": command, "returncode": returncode, "output": output})
    print(f"\n$ {command}\n{output}")


$ pwd
/home/a.riyahi/GPT_PINN

$ hostname
data31

$ date
Sun Sep  6 02:14:00 +01 2026

$ whoami
a.riyahi

$ python --version
Python 3.11.7

$ which python
/cm/shared/apps/eb/software/Anaconda3/2024.02-1/bin/python

$ which pip
/cm/shared/apps/eb/software/Anaconda3/2024.02-1/bin/pip

$ which conda || true
/cm/shared/apps/eb/software/Anaconda3/2024.02-1/bin/conda

$ git --version
git version 2.39.3

$ nvidia-smi || true
/bin/bash: nvidia-smi: command not found

$ sinfo || true
PARTITION AVAIL  TIMELIMIT  NODES  STATE NODELIST 
defq         up    1:00:00      1 drain* data32 
defq         up    1:00:00      3    mix bigmem41,node[03,08] 
defq         up    1:00:00     31  alloc bigmem[42-44],data[31,33-36],node[01-02,04-07,09-16,21-29] 
demoq        up   infinite      1   idle node30 
gpu-prodq    up 7-00:00:00      2    mix gpu[51-52] 
gpu-testq    up    1:00:00      2    mix gpu[51-52] 
long30dq     up 30-00:00:0      1 drain* data32 
long30dq     up 30-00:00:0      3    mix bigmem41,n

In [4]:
gpu_model = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
runtime_summary = {
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "python_version": sys.version.split()[0],
    "python_executable": sys.executable,
    "pytorch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_model": gpu_model,
    "cpu_logical_count": os.cpu_count(),
    "cpu_physical_count": psutil.cpu_count(logical=False),
    "ram_total_gib": psutil.virtual_memory().total / 2**30,
    "platform": platform.platform(),
    "hostname": HOSTNAME,
    "is_login_node": IS_LOGIN_NODE,
    "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    "slurm_job_name": os.environ.get("SLURM_JOB_NAME"),
    "slurm_node_list": os.environ.get("SLURM_NODELIST"),
    "allow_login_node_heavy": ALLOW_LOGIN_NODE_HEAVY,
    "checkpoint_io_probe": CHECKPOINT_IO_PROBE,
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "package_versions": {name: ilm.version(name) for name in
                         ["numpy", "scipy", "pandas", "matplotlib", "torch", "scikit-learn", "nbformat", "nbclient"]},
}
print(json_dumps_safe(runtime_summary, indent=2))
(OUTPUT_ROOT / "environment_audit.json").write_text(
    json_dumps_safe({"summary": runtime_summary, "commands": environment_audit}, indent=2) + "\n",
    encoding="utf-8",
)
print("Full-run compute-node requirement:", {
    "required": RUN_PROFILE == "full",
    "satisfied_now": (not IS_LOGIN_NODE) or ALLOW_LOGIN_NODE_HEAVY,
})

{
  "project_root": "/home/a.riyahi/GPT_PINN",
  "dataset_root": "/home/a.riyahi/EvoBridge_SHM/data/raw/z24",
  "python_version": "3.11.7",
  "python_executable": "/cm/shared/apps/eb/software/Anaconda3/2024.02-1/bin/python",
  "pytorch_version": "2.7.1+cu118",
  "cuda_available": false,
  "gpu_model": null,
  "cpu_logical_count": 48,
  "cpu_physical_count": 48,
  "ram_total_gib": 187.03497314453125,
  "platform": "Linux-4.18.0-477.27.1.el8_8.x86_64-x86_64-with-glibc2.28",
  "hostname": "data31",
  "is_login_node": false,
  "slurm_job_id": "508150",
  "slurm_job_name": "gptpinn_r3_full",
  "slurm_node_list": "data31",
  "allow_login_node_heavy": false,
  "checkpoint_io_probe": {
    "passed": true,
    "root": "/home/a.riyahi/GPT_PINN/outputs/v0_6_final/checkpoints/970ffcaa511ebcb7",
    "diagnostics": {
      "target": "/home/a.riyahi/GPT_PINN/outputs/v0_6_final/checkpoints/970ffcaa511ebcb7/placeholder.pt",
      "parent": "/home/a.riyahi/GPT_PINN/outputs/v0_6_final/checkpoints/970ffca

## 2. External repository inspection

Seven public repositories are treated as **audited references**, not dependencies of
the scientific execution. The four user-requested Z24 repositories are reviewed
together with GPT-PINN, S²GPT-PINN and ROM-GPT-PINN. Runtime cloning is optional
and non-blocking; when a local clone exists, its exact commit is recorded.

The notebook adopts only practices that survive scientific review: actual trace
lengths, channel/axis auditing, fixed-reference diagnostics, explicit FVT
driving-point removal, physical AVT–FVT setup mapping, leakage-safe calibration,
automatic-differentiation residuals, Full-PINN basis precomputation and greedy
maximum-residual enrichment. Padding/mirroring, global pre-split scaling, unrelated
PDEs and unvalidated classification assumptions are explicitly rejected.


In [5]:
repo_specs = {
    "GPT-PINN": {
        "url": REFERENCE_URLS["gpt_pinn_repo"] + ".git",
        "branch": None,
    },
    "S2GPT-PINN": {
        "url": REFERENCE_URLS["s2gpt_pinn_repo"] + ".git",
        "branch": None,
    },
    "ROM-GPT-PINN": {
        "url": REFERENCE_URLS["rom_gpt_pinn_repo"] + ".git",
        "branch": None,
    },
    "WaveNet_MiniRocket_Z24": {
        "url": REFERENCE_URLS["z24_reference_repo"] + ".git",
        "branch": "main",
    },
    "explainable-shm-z24": {
        "url": REFERENCE_URLS["explainable_z24_repo"] + ".git",
        "branch": "main",
    },
    "z24-shm-anomaly-detection": {
        "url": REFERENCE_URLS["z24_anomaly_repo"] + ".git",
        "branch": "main",
    },
    "online-one-class-z24": {
        "url": REFERENCE_URLS["z24_online_occ_repo"] + ".git",
        "branch": "jsv",
    },
}
external_root = PROJECT_ROOT / "external"
external_root.mkdir(parents=True, exist_ok=True)
repo_rows = []
for name, spec in repo_specs.items():
    url = spec["url"]
    branch = spec["branch"]
    path = external_root / name
    status = "AVAILABLE_LOCAL"
    error = ""
    if not (path / ".git").exists():
        if CONFIG["allow_network_clone"]:
            try:
                command = ["git", "clone", "--depth", "1"]
                if branch:
                    command += ["--branch", branch]
                command += [url, str(path)]
                subprocess.run(command, check=True, text=True, capture_output=True)
                status = "CLONED_NOW"
            except Exception as exc:
                status = "CLONE_FAILED_NONBLOCKING"
                error = repr(exc)
        else:
            status = "MISSING_CLONE_DISABLED_NONBLOCKING"
    commit = None
    if (path / ".git").exists():
        try:
            commit = subprocess.run(
                ["git", "-C", str(path), "rev-parse", "HEAD"],
                check=True, text=True, capture_output=True,
            ).stdout.strip()
        except Exception as exc:
            status = "LOCAL_GIT_AUDIT_FAILED_NONBLOCKING"
            error = repr(exc)
    repo_rows.append({
        "repository": name,
        "url": url.removesuffix(".git"),
        "requested_branch": branch,
        "path": str(path),
        "status": status,
        "commit": commit,
        "error": error,
    })

repo_audit = pd.DataFrame(repo_rows)
display(repo_audit)
repo_audit.to_csv(TABLE_ROOT / "reference_repository_commits.csv", index=False)

inspection_findings = pd.DataFrame([
    {
        "reference": "GPT-PINN",
        "decision": "ADOPT_METHOD",
        "adopted_idea": "Full-PINN basis, derivative precomputation, greedy maximum reduced-physics loss, low-dimensional online coefficients",
        "not_adopted": "Klein-Gordon/Burgers/Allen-Cahn equations and hard-coded CUDA",
    },
    {
        "reference": "ROM-GPT-PINN / S2GPT-PINN",
        "decision": "ADOPT_NUMERICAL_PRACTICES_ONLY",
        "adopted_idea": "float64 physics training, reduced-basis conditioning and residual-informed enrichment",
        "not_adopted": "problem-specific PDEs and unvalidated sparse extension",
    },
    {
        "reference": "Elios-Lab WaveNet/MiniRocket Z24",
        "decision": "REPRODUCTION_REFERENCE_ONLY",
        "adopted_idea": "MAT orientation, nominal 65,536 points, full-trace sanity and explicit scenario-03 QC",
        "not_adopted": "last-value padding, global scaling before split and random sensor-trace classification split",
    },
    {
        "reference": "you-soof/explainable-shm-z24",
        "decision": "ADOPT_DATA_QUIRKS_AND_DIAGNOSTICS",
        "adopted_idea": "actual trace lengths, varying channel counts, fixed references, FVT shaker separation, reversed AVT/FVT setup mapping, log-PSD/transmissibility audit",
        "not_adopted": "autoencoder/SHAP as the proposed Digital Twin or thesis-reported thresholds as ground truth",
    },
    {
        "reference": "riyaarah/z24-shm-anomaly-detection",
        "decision": "ADOPT_LEAKAGE_AND_BASELINE_PRINCIPLES",
        "adopted_idea": "healthy-only calibration discipline, explicit temporal gaps for windowed baselines, PCA/Mahalanobis diagnostic perspective and mechanism-stratified interpretation",
        "not_adopted": "setup-1-only restriction for the multimodal FDD core or window independence assumptions",
    },
    {
        "reference": "AmroAbdrabo online one-class Z24",
        "decision": "ADOPT_REFERENCE_FEATURE_AUDIT_ONLY",
        "adopted_idea": "fixed references R1V/R2L/R2T/R2V/R3V, DC/trend checks, spectral/transmissibility features and online-drift perspective",
        "not_adopted": "mirroring records to equal length, blanket per-trace standardization or fixed-support ANSYS assumptions as experimental truth",
    },
])
display(inspection_findings)
inspection_findings.to_csv(TABLE_ROOT / "reference_repository_decisions.csv", index=False)

if (repo_audit["status"] == "MISSING_CLONE_DISABLED_NONBLOCKING").any():
    print(
        "Reference repositories are optional at runtime. "
        "Set GPT_PINN_ALLOW_NETWORK_CLONE=1 before the kernel starts "
        "only when a clone is scientifically needed."
    )


,repository,url,requested_branch,path,status,commit,error
0,GPT-PINN,https://github.com/skoohy/GPT-PINN,None,/home/a.riyahi/GPT_PINN/external/GPT-PINN,MISSING_CLONE_DISABLED_NONBLOCKING,None,
1,S2GPT-PINN,https://github.com/DuktigYajie/S2GPT-PINN,None,/home/a.riyahi/GPT_PINN/external/S2GPT-PINN,MISSING_CLONE_DISABLED_NONBLOCKING,None,
2,ROM-GPT-PINN,https://github.com/ljq2025/ROM-GPT-PINN,None,/home/a.riyahi/GPT_PINN/external/ROM-GPT-PINN,MISSING_CLONE_DISABLED_NONBLOCKING,None,
3,WaveNet_MiniRocket_Z24,https://github.com/Elios-Lab/WaveNet_MiniRocke...,main,/home/a.riyahi/GPT_PINN/external/WaveNet_MiniR...,MISSING_CLONE_DISABLED_NONBLOCKING,None,
4,explainable-shm-z24,https://github.com/you-soof/explainable-shm-z24,main,/home/a.riyahi/GPT_PINN/external/explainable-s...,MISSING_CLONE_DISABLED_NONBLOCKING,None,
5,z24-shm-anomaly-detection,https://github.com/riyaarah/z24-shm-anomaly-de...,main,/home/a.riyahi/GPT_PINN/external/z24-shm-anoma...,MISSING_CLONE_DISABLED_NONBLOCKING,None,
6,online-one-class-z24,https://github.com/AmroAbdrabo/application-of-...,jsv,/home/a.riyahi/GPT_PINN/external/online-one-cl...,MISSING_CLONE_DISABLED_NONBLOCKING,None,


,reference,decision,adopted_idea,not_adopted
0,GPT-PINN,ADOPT_METHOD,"Full-PINN basis, derivative precomputation, gr...",Klein-Gordon/Burgers/Allen-Cahn equations and ...
1,ROM-GPT-PINN / S2GPT-PINN,ADOPT_NUMERICAL_PRACTICES_ONLY,"float64 physics training, reduced-basis condit...",problem-specific PDEs and unvalidated sparse e...
2,Elios-Lab WaveNet/MiniRocket Z24,REPRODUCTION_REFERENCE_ONLY,"MAT orientation, nominal 65,536 points, full-t...","last-value padding, global scaling before spli..."
3,you-soof/explainable-shm-z24,ADOPT_DATA_QUIRKS_AND_DIAGNOSTICS,"actual trace lengths, varying channel counts, ...",autoencoder/SHAP as the proposed Digital Twin ...
4,riyaarah/z24-shm-anomaly-detection,ADOPT_LEAKAGE_AND_BASELINE_PRINCIPLES,"healthy-only calibration discipline, explicit ...",setup-1-only restriction for the multimodal FD...
5,AmroAbdrabo online one-class Z24,ADOPT_REFERENCE_FEATURE_AUDIT_ONLY,"fixed references R1V/R2L/R2T/R2V/R3V, DC/trend...","mirroring records to equal length, blanket per..."


Reference repositories are optional at runtime. Set GPT_PINN_ALLOW_NETWORK_CLONE=1 before the kernel starts only when a clone is scientifically needed.


### 2.1 GPT-PINN fidelity contract and bridge-specific adaptation

V0.6 FINAL preserves the **algorithmic invariants** of the original GPT-PINN rather than copying its PDE-specific source line by line.

| Original GPT-PINN invariant | Bridge-specific implementation in V0.6 FINAL |
|---|---|
| Parametric PDE family \(u(\cdot;\mu)\) | Parametric bridge eigenproblem \((\phi_r,\omega_r)(\cdot;\theta)\) |
| Full PINN trained at selected parameters | Full modal PINN trained at selected rigidity/support states |
| Pretrained PINNs become reduced neurons/basis functions | Pretrained modal fields and derivatives become the reduced structural basis |
| Coefficients optimized for a new parameter | Reduced modal coefficients/eigenvalues solved for a new bridge state |
| Greedy next point maximizes reduced physics loss | Next bridge state maximizes the independent weak residual |
| Offline expensive training, online small query | Offline Full-PINN basis construction, online Digital Twin update |
| Original initial/boundary conditions | Bridge support constraints, modal normalization and orthogonality |
| Original test PDEs | Euler–Bernoulli validation, three-span weak formulation and Z24 synchronization |

The Klein–Gordon, Burgers and Allen–Cahn equations, hard-coded CUDA assumptions and problem-specific forcing terms are not reused. Every governing equation in this notebook is structural-dynamics code that is executed and independently checked.


### 2.2 Peer-reviewed Z24 literature and preprocessing audit

V0.6 FINAL separates **reproducible dataset facts**, **published methodological choices**, and
**choices adopted in this notebook**. The cited studies are used as methodological
comparators, not as sources of numerical targets to be hard-coded. Journal indexing
must be rechecked at submission, but the table focuses on peer-reviewed Z24 studies
published in established Scopus-covered engineering journals.

Key decisions for this notebook:

- preserve actual trace lengths rather than padding or mirroring records;
- treat one setup file as one simultaneous acquisition;
- use all physically consistent AVT setups for the primary FDD observation;
- keep fixed-reference spectral/transmissibility features as an auxiliary diagnostic;
- remove FVT driving-point channels from response-only FDD;
- map AVT and FVT setup numbers by physical location before external-excitation validation;
- estimate joint modal uncertainty by paired-setup bootstrap;
- never fit a scaler, threshold, or model using held-out damage observations.


In [6]:
literature_audit = pd.DataFrame([
    {
        "study": "Maeck & De Roeck (2003)",
        "venue": "Mechanical Systems and Signal Processing",
        "doi": "10.1006/mssp.2002.1550",
        "z24_scope": "progressive damage benchmark",
        "processing_or_method": "vibration/modal damage assessment",
        "v05_use": "physical/modal reference; no numerical value hard-coded as truth",
    },
    {
        "study": "Giglioni et al. (2023)",
        "venue": "Computer-Aided Civil and Infrastructure Engineering",
        "doi": "10.1111/mice.12943",
        "z24_scope": "healthy-trained unsupervised bridge assessment",
        "processing_or_method": "short raw acceleration sequences; single-sensor and ensemble autoencoders",
        "v05_use": "supports healthy-only and sensor-level baseline logic; not substituted for FDD physics",
    },
    {
        "study": "Dabbous et al. (2024)",
        "venue": "IEEE Open Journal of the Industrial Electronics Society",
        "doi": "10.1109/OJIES.2024.3434341",
        "z24_scope": "PDT AVT, 5-class and 15-class trace classification",
        "processing_or_method": "full nominal-length traces; WaveNet and MiniRocket; scenario-03 QC",
        "v05_use": "MAT/trace sanity reference; rejects global pre-split scaling and last-value padding",
    },
    {
        "study": "Abdrabo (2024)",
        "venue": "Sensors",
        "doi": "10.3390/s24237866",
        "z24_scope": "online one-class anomaly detection",
        "processing_or_method": "fixed reference sensors, engineered spectral/transmittance features, PCA/XGBoost feature selection",
        "v05_use": "auxiliary reference-channel spectral audit and online-drift perspective",
    },
    {
        "study": "Paoletti et al. (2026)",
        "venue": "Journal of Sensor and Actuator Networks",
        "doi": "10.3390/jsan15020031",
        "z24_scope": "hybrid AVT–FVT sensor optimization",
        "processing_or_method": "CPSD/SVD energy, AVT–FVT consistency, filtering and channel-quality control",
        "v05_use": "physical AVT–FVT setup mapping, driving-point exclusion and excitation-consistency audit",
    },
    {
        "study": "da Silva et al. (2022)",
        "venue": "Journal of Bridge Engineering",
        "doi": "10.1061/(ASCE)BE.1943-5592.0001949",
        "z24_scope": "damage detection under temperature effects",
        "processing_or_method": "Gaussian-process regression with hybrid experimental/numerical data",
        "v05_use": "environmental confounding retained as a stated limitation because EMS is absent locally",
    },
    {
        "study": "Sony et al. (2022)",
        "venue": "Journal of Computing in Civil Engineering",
        "doi": "10.1061/(ASCE)CP.1943-5487.0001003",
        "z24_scope": "multiclass full-scale bridge damage identification",
        "processing_or_method": "windowed vibration data and tuned one-dimensional CNN",
        "v05_use": "classification comparator only; windows are not treated as independent units here",
    },
])
literature_audit["indexing_note"] = (
    "peer-reviewed engineering-journal study; verify live Scopus coverage at submission"
)
display(literature_audit)
literature_audit.to_csv(TABLE_ROOT / "z24_peer_reviewed_literature_audit.csv", index=False)

data_practice_audit = pd.DataFrame([
    {
        "practice": "Use actual MAT trace lengths",
        "decision": "ADOPT",
        "reason": "local records vary from 65,026 to 65,536 samples",
    },
    {
        "practice": "Last-value padding to 65,536",
        "decision": "REJECT",
        "reason": "creates an artificial constant tail and can bias spectral estimates",
    },
    {
        "practice": "Mirror/append rows to equalize concatenated setup lengths",
        "decision": "REJECT",
        "reason": "duplicates observations and changes the spectral/statistical sample",
    },
    {
        "practice": "Global StandardScaler before train/test split",
        "decision": "REJECT",
        "reason": "information leakage; no such scaling is required for the FDD core",
    },
    {
        "practice": "Detrend and segment-wise mean removal",
        "decision": "ADOPT",
        "reason": "removes DC/linear drift while preserving modal frequencies",
    },
    {
        "practice": "All physically compatible AVT setups",
        "decision": "ADOPT",
        "reason": "setup is the acquisition unit; robust medians and support counts retain spatial coverage",
    },
    {
        "practice": "DP1V/DP2V in FVT response FDD",
        "decision": "REJECT",
        "reason": "driving-point shaker channels are inputs, not bridge-response accelerometers",
    },
    {
        "practice": "Same AVT and FVT file setup number",
        "decision": "REJECT",
        "reason": "physical setup numbering is reversed and must be mapped/audited",
    },
    {
        "practice": "Paired-setup bootstrap covariance",
        "decision": "ADOPT",
        "reason": "preserves cross-mode dependence and avoids fabricated independent uncertainties",
    },
])
display(data_practice_audit)
data_practice_audit.to_csv(TABLE_ROOT / "z24_data_practice_decisions.csv", index=False)


,study,venue,doi,z24_scope,processing_or_method,v05_use,indexing_note
0,Maeck & De Roeck (2003),Mechanical Systems and Signal Processing,10.1006/mssp.2002.1550,progressive damage benchmark,vibration/modal damage assessment,physical/modal reference; no numerical value h...,peer-reviewed engineering-journal study; verif...
1,Giglioni et al. (2023),Computer-Aided Civil and Infrastructure Engine...,10.1111/mice.12943,healthy-trained unsupervised bridge assessment,short raw acceleration sequences; single-senso...,supports healthy-only and sensor-level baselin...,peer-reviewed engineering-journal study; verif...
2,Dabbous et al. (2024),IEEE Open Journal of the Industrial Electronic...,10.1109/OJIES.2024.3434341,"PDT AVT, 5-class and 15-class trace classifica...",full nominal-length traces; WaveNet and MiniRo...,MAT/trace sanity reference; rejects global pre...,peer-reviewed engineering-journal study; verif...
3,Abdrabo (2024),Sensors,10.3390/s24237866,online one-class anomaly detection,"fixed reference sensors, engineered spectral/t...",auxiliary reference-channel spectral audit and...,peer-reviewed engineering-journal study; verif...
4,Paoletti et al. (2026),Journal of Sensor and Actuator Networks,10.3390/jsan15020031,hybrid AVT–FVT sensor optimization,"CPSD/SVD energy, AVT–FVT consistency, filterin...","physical AVT–FVT setup mapping, driving-point ...",peer-reviewed engineering-journal study; verif...
5,da Silva et al. (2022),Journal of Bridge Engineering,10.1061/(ASCE)BE.1943-5592.0001949,damage detection under temperature effects,Gaussian-process regression with hybrid experi...,environmental confounding retained as a stated...,peer-reviewed engineering-journal study; verif...
6,Sony et al. (2022),Journal of Computing in Civil Engineering,10.1061/(ASCE)CP.1943-5487.0001003,multiclass full-scale bridge damage identifica...,windowed vibration data and tuned one-dimensio...,classification comparator only; windows are no...,peer-reviewed engineering-journal study; verif...


,practice,decision,reason
0,Use actual MAT trace lengths,ADOPT,"local records vary from 65,026 to 65,536 samples"
1,"Last-value padding to 65,536",REJECT,creates an artificial constant tail and can bi...
2,Mirror/append rows to equalize concatenated se...,REJECT,duplicates observations and changes the spectr...
3,Global StandardScaler before train/test split,REJECT,information leakage; no such scaling is requir...
4,Detrend and segment-wise mean removal,ADOPT,removes DC/linear drift while preserving modal...
5,All physically compatible AVT setups,ADOPT,setup is the acquisition unit; robust medians ...
6,DP1V/DP2V in FVT response FDD,REJECT,"driving-point shaker channels are inputs, not ..."
7,Same AVT and FVT file setup number,REJECT,physical setup numbering is reversed and must ...
8,Paired-setup bootstrap covariance,ADOPT,preserves cross-mode dependence and avoids fab...


## 3. Z24 dataset audit

The authorized dataset root is inspected recursively without writing to it. A raw metadata signature is computed before and after the analysis. It covers relative paths, sizes, and nanosecond modification times; it is an immutability check for this run, not a content hash of 4.6 GB.

In [7]:
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Required Z24 dataset is absent: {DATASET_ROOT}")

def raw_metadata_signature(root: Path) -> tuple[str, int, int]:
    digest = hashlib.sha256()
    count = 0
    total_bytes = 0
    for path in sorted(p for p in root.rglob("*") if p.is_file()):
        stat = path.stat()
        row = f"{path.relative_to(root).as_posix()}|{stat.st_size}|{stat.st_mtime_ns}\n"
        digest.update(row.encode("utf-8"))
        count += 1
        total_bytes += stat.st_size
    return digest.hexdigest(), count, total_bytes

RAW_SIGNATURE_BEFORE = raw_metadata_signature(DATASET_ROOT)
directories = sorted(p.relative_to(DATASET_ROOT).as_posix() for p in DATASET_ROOT.rglob("*") if p.is_dir())
all_files = sorted(p for p in DATASET_ROOT.rglob("*") if p.is_file())
extension_counts = pd.Series([p.suffix.lower() or "[none]" for p in all_files]).value_counts().sort_index()
print("Folder hierarchy:")
for directory in directories:
    print("  ", directory)
print("\nFile-extension counts:")
display(extension_counts.rename("count").to_frame())
print("Raw signature before:", RAW_SIGNATURE_BEFORE)

mat_paths = sorted(DATASET_ROOT.rglob("*.mat"))
inventory_rows = []
for path in mat_paths:
    relative = path.relative_to(DATASET_ROOT)
    variables = {name: (tuple(shape), kind) for name, shape, kind in sio.whosmat(path)}
    data_shape = variables.get("data", (None, None))[0]
    labels_shape = variables.get("labelshulp", (None, None))[0]
    inventory_rows.append({
        "relative_path": relative.as_posix(),
        "scenario": relative.parts[0],
        "excitation": relative.parts[1].upper(),
        "setup": int(re.search(r"setup(\d{2})", path.stem, re.I).group(1)),
        "mat_keys": ",".join(sorted(variables)),
        "data_present": "data" in variables,
        "data_shape": str(data_shape),
        "n_samples": data_shape[0] if data_shape else np.nan,
        "n_channels": data_shape[1] if data_shape else np.nan,
        "labelshulp_shape": str(labels_shape),
        "bytes": path.stat().st_size,
    })
mat_inventory = pd.DataFrame(inventory_rows).sort_values(["scenario", "excitation", "setup"])
mat_inventory.to_csv(TABLE_ROOT / "z24_mat_inventory.csv", index=False)
print(f"\nComplete MAT inventory ({len(mat_inventory)} files):")
print("\n".join(mat_inventory["relative_path"]))
print("\nScenario/excitation acquisition counts:")
display(mat_inventory.groupby(["scenario", "excitation"]).size().unstack(fill_value=0))
print("\nObserved data-shape counts:")
display(mat_inventory.groupby(["n_samples", "n_channels"]).size().rename("file_count").reset_index())
expected_scenarios = {f"{i:02d}" for i in range(1, 18)}
layout_counts = mat_inventory.groupby(["scenario", "excitation"]).size()
complete_archive_layout = bool(
    len(mat_inventory) == 306
    and mat_inventory["data_present"].all()
    and set(mat_inventory["scenario"]) == expected_scenarios
    and (layout_counts == 9).all()
)
required_primary_pairs = {(scenario, "AVT") for scenario in CONFIG["primary_scenarios"]}
missing_primary_pairs = sorted(required_primary_pairs - set(layout_counts.index))
incorrect_primary_counts = {
    pair: int(layout_counts.loc[pair])
    for pair in required_primary_pairs & set(layout_counts.index)
    if int(layout_counts.loc[pair]) != 9
}
if missing_primary_pairs or incorrect_primary_counts:
    raise RuntimeError(
        "Primary Z24 AVT layout is incomplete: "
        f"missing={missing_primary_pairs}, incorrect_counts={incorrect_primary_counts}"
    )
if not complete_archive_layout:
    warnings.warn(
        "The full 17-scenario AVT/FVT archive differs from the previously audited 306-file layout. "
        "The primary five-scenario AVT analysis can continue only because its required files passed strict checks."
    )
print("Complete 306-file Z24 archive layout:", complete_archive_layout)

Folder hierarchy:
   01
   01/avt
   01/fvt
   02
   02/avt
   02/fvt
   03
   03/avt
   03/fvt
   04
   04/avt
   04/fvt
   05
   05/avt
   05/fvt
   06
   06/avt
   06/avt/01
   06/avt/02
   06/avt/03
   06/avt/04
   06/avt/05
   06/avt/06
   06/avt/07
   06/avt/08
   06/avt/09
   06/fvt
   07
   07/avt
   07/fvt
   08
   08/avt
   08/fvt
   09
   09/avt
   09/fvt
   10
   10/avt
   10/fvt
   11
   11/avt
   11/fvt
   12
   12/avt
   12/fvt
   13
   13/avt
   13/fvt
   14
   14/avt
   14/fvt
   15
   15/avt
   15/fvt
   16
   16/avt
   16/fvt
   17
   17/FVT
   17/FVT/01
   17/FVT/02
   17/FVT/03
   17/FVT/04
   17/FVT/05
   17/FVT/06
   17/FVT/07
   17/FVT/08
   17/FVT/09
   17/avt
   17/avt/01
   17/avt/02
   17/avt/03
   17/avt/04
   17/avt/05
   17/avt/06
   17/avt/07
   17/avt/08
   17/avt/09

File-extension counts:


,count
.aaa,891
.doc,30
.ini,66
.mat,306
.zip,2


Raw signature before: ('fa1e2cd04370ed3f9e2bd68d1220df56837aec89d0603f54cc66b017c67fad48', 1295, 4863554690)

Complete MAT inventory (306 files):
01/avt/01setup01.mat
01/avt/01setup02.mat
01/avt/01setup03.mat
01/avt/01setup04.mat
01/avt/01setup05.mat
01/avt/01setup06.mat
01/avt/01setup07.mat
01/avt/01setup08.mat
01/avt/01setup09.mat
01/fvt/01setup01.mat
01/fvt/01setup02.mat
01/fvt/01setup03.mat
01/fvt/01setup04.mat
01/fvt/01setup05.mat
01/fvt/01setup06.mat
01/fvt/01setup07.mat
01/fvt/01setup08.mat
01/fvt/01setup09.mat
02/avt/02setup01.mat
02/avt/02setup02.mat
02/avt/02setup03.mat
02/avt/02setup04.mat
02/avt/02setup05.mat
02/avt/02setup06.mat
02/avt/02setup07.mat
02/avt/02setup08.mat
02/avt/02setup09.mat
02/fvt/02setup01.mat
02/fvt/02setup02.mat
02/fvt/02setup03.mat
02/fvt/02setup04.mat
02/fvt/02setup05.mat
02/fvt/02setup06.mat
02/fvt/02setup07.mat
02/fvt/02setup08.mat
02/fvt/02setup09.mat
03/avt/03setup01.mat
03/avt/03setup02.mat
03/avt/03setup03.mat
03/avt/03setup04.mat
03/avt/03setup

excitation,AVT,FVT
scenario,,
01,9,9
02,9,9
03,9,9
04,9,9
05,9,9
06,9,9
07,9,9
08,9,9
09,9,9



Observed data-shape counts:


,n_samples,n_channels,file_count
0,65026,33,1
1,65410,33,1
2,65530,27,2
3,65530,28,2
4,65530,31,1
5,65530,32,1
6,65530,33,15
7,65530,34,2
8,65530,35,13
9,65536,27,15


Complete 306-file Z24 archive layout: True


In [8]:
representative_paths = [
    DATASET_ROOT / "01" / "avt" / "01setup01.mat",
    DATASET_ROOT / "03" / "avt" / "03setup01.mat",
    DATASET_ROOT / "06" / "fvt" / "06setup01.mat",
]
representative_rows = []
for path in representative_paths:
    mat = sio.loadmat(path)
    keys = sorted(k for k in mat if not k.startswith("__"))
    data = np.asarray(mat["data"])
    labels = [str(value).strip() for value in np.ravel(mat["labelshulp"])]
    orientation_verified = data.shape[1] == len(labels) and data.shape[0] > data.shape[1]
    representative_rows.append({
        "relative_path": path.relative_to(DATASET_ROOT).as_posix(),
        "keys": keys,
        "data_shape_on_disk": tuple(data.shape),
        "label_count": len(labels),
        "first_labels": labels[:8],
        "channel_major_shape_after_transpose": tuple(data.T.shape),
        "transpose_is_correct": orientation_verified,
    })
display(pd.DataFrame(representative_rows))
assert all(row["transpose_is_correct"] for row in representative_rows)
print("Verified local convention: data is (time, channel); data.T is (channel, time).")
print("Sampling rate is not encoded in the MAT variables. Local Z24 readme and EMPA report state 100 Hz.")
print("PDT layout is stored directly as scenario 01–17 folders. EMS is absent from the authorized raw root.")

,relative_path,keys,data_shape_on_disk,label_count,first_labels,channel_major_shape_after_transpose,transpose_is_correct
0,01/avt/01setup01.mat,"[data, labelshulp]","(65530, 33)",33,"[99V, 100V, 101V, 102V, 103V, 199L, 199T, 199V]","(33, 65530)",True
1,03/avt/03setup01.mat,"[data, labelshulp]","(65026, 33)",33,"[99V, 100V, 101V, 102V, 103V, 199L, 199T, 199V]","(33, 65026)",True
2,06/fvt/06setup01.mat,"[data, labelshulp]","(65536, 35)",35,"[139V, 140V, 141V, 142V, 143V, 239L, 239T, 239V]","(35, 65536)",True


Verified local convention: data is (time, channel); data.T is (channel, time).
Sampling rate is not encoded in the MAT variables. Local Z24 readme and EMPA report state 100 Hz.
PDT layout is stored directly as scenario 01–17 folders. EMS is absent from the authorized raw root.


## 4. Z24 canonical scenario catalog

The scenario chronology is corroborated by the official KU Leuven Z24 benchmark page. Local EMPA DOC files are checked separately because several are missing or stale; local contradictions are recorded rather than overwritten. The parameter $\delta_{pier}$ is the imposed benchmark settlement and is **not** assumed to equal stiffness loss.

In [9]:
conditions = {
    "01": "Initial healthy / first reference",
    "02": "Pier-settlement equipment installation",
    "03": "Koppigen pier lowering 20 mm",
    "04": "Koppigen pier lowering 40 mm",
    "05": "Koppigen pier lowering 80 mm",
    "06": "Koppigen pier lowering 95 mm",
    "07": "Pier lifting / foundation intervention",
    "08": "New reference condition",
    "09": "Concrete spalling at soffit, 12 m²",
    "10": "Concrete spalling at soffit, 24 m²",
    "11": "Landslide at abutment",
    "12": "Concrete-hinge failure",
    "13": "Failure of 2 anchor heads",
    "14": "Failure of 4 anchor heads",
    "15": "Rupture of 2 of 16 tendons",
    "16": "Rupture of 4 of 16 tendons",
    "17": "Rupture of 6 of 16 tendons",
}
settlement_map = {"01": 0.0, "03": 20.0, "04": 40.0, "05": 80.0, "06": 95.0}

def doc_evidence(scenario: str) -> tuple[str, str]:
    docs = sorted(path for path in (DATASET_ROOT / scenario).rglob("*") if path.is_file() and path.suffix.lower() == ".doc")
    if not docs:
        return "ABSENT", "No local EMPA DOC file"
    snippets = []
    for path in docs:
        text = subprocess.run(["strings", "-a", str(path)], text=True, capture_output=True, check=True).stdout
        found = re.findall(r"(?:Scenario:|Measurement:\s*|Type of Measurement:\s*)([^\r\n]+)", text)
        snippets.extend(value.strip() for value in found[:3])
    joined = " | ".join(dict.fromkeys(snippets))
    expected = settlement_map.get(scenario)
    if scenario == "01" and "Reference" in joined:
        return "LOCAL_DOC_CONFIRMED", joined
    if expected is not None and expected > 0 and re.search(rf"\b{int(expected)}\s*mm\b", joined, re.I):
        return "LOCAL_DOC_CONFIRMED", joined
    return "LOCAL_DOC_STALE_OR_CONTRADICTORY", joined

catalog_rows = []
for scenario, condition in conditions.items():
    local_status, local_note = doc_evidence(scenario)
    subset = mat_inventory[mat_inventory["scenario"] == scenario]
    catalog_rows.append({
        "scenario": scenario,
        "condition": condition,
        "delta_pier_mm": settlement_map.get(scenario, np.nan),
        "avt_files": int((subset["excitation"] == "AVT").sum()),
        "fvt_files": int((subset["excitation"] == "FVT").sum()),
        "local_doc_status": local_status,
        "local_doc_note": local_note,
        "chronology_source": REFERENCE_URLS["z24_official"],
    })
scenario_catalog = pd.DataFrame(catalog_rows)
scenario_catalog.to_csv(TABLE_ROOT / "table_01_z24_scenarios.csv", index=False)
display(scenario_catalog)

primary_catalog = scenario_catalog[scenario_catalog["scenario"].isin(CONFIG["primary_scenarios"])].copy()
assert primary_catalog["avt_files"].eq(9).all()

fig, ax = plt.subplots(figsize=(10.5, 3.5))
delta = np.asarray(CONFIG["pier_settlement_mm"])
ax.plot(delta, np.zeros_like(delta), color="#457b9d", lw=2.5, zorder=1)
ax.scatter(delta, np.zeros_like(delta), s=170, c=np.linspace(0.1, 0.9, len(delta)), cmap="YlOrRd", edgecolor="black", zorder=2)
for scenario, value in zip(CONFIG["primary_scenarios"], delta):
    ax.annotate(f"Scenario {scenario}\n{value:g} mm", (value, 0), xytext=(0, 18), textcoords="offset points", ha="center")
ax.set_xlim(-6, 102); ax.set_ylim(-0.45, 0.55); ax.set_yticks([])
ax.set_xlabel(r"Controlled Koppigen-pier settlement, $\delta_{pier}$ [mm]")
ax.set_title("Z24 primary controlled progression sequence")
fig.savefig(FIGURE_ROOT / "figure_02_controlled_settlement_scenarios.png", bbox_inches="tight", dpi=320)
fig.savefig(FIGURE_ROOT / "figure_02_controlled_settlement_scenarios.pdf", bbox_inches="tight")
plt.show()
plt.close("all")

,scenario,condition,delta_pier_mm,avt_files,fvt_files,local_doc_status,local_doc_note,chronology_source
0,01,Initial healthy / first reference,0.0,9,9,LOCAL_DOC_CONFIRMED,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
1,02,Pier-settlement equipment installation,NaN,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
2,03,Koppigen pier lowering 20 mm,20.0,9,9,ABSENT,No local EMPA DOC file,https://bwk.kuleuven.be/bwm/z24
3,04,Koppigen pier lowering 40 mm,40.0,9,9,LOCAL_DOC_CONFIRMED,"3.2 Pier Settlement | 04 | Ambient Test, 40 mm...",https://bwk.kuleuven.be/bwm/z24
4,05,Koppigen pier lowering 80 mm,80.0,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
5,06,Koppigen pier lowering 95 mm,95.0,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
6,07,Pier lifting / foundation intervention,NaN,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
7,08,New reference condition,NaN,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
8,09,"Concrete spalling at soffit, 12 m²",NaN,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24
9,10,"Concrete spalling at soffit, 24 m²",NaN,9,9,LOCAL_DOC_STALE_OR_CONTRADICTORY,First Reference Measurement | 01 | Forced Test...,https://bwk.kuleuven.be/bwm/z24


## 5. Z24 signal loading and quality control

One MAT setup is one simultaneous acquisition. Signals are mean-removed and linearly detrended for spectral analysis. No per-trace amplitude normalization is used in the primary FDD path. Location-aware vertical channels are girder `V`, pier `L` (4xx/5xx locations), and the common vertical references `R1V/R2V/R3V`. Physical units are not encoded in the MAT files and remain **TO-VERIFY**.

In [10]:
REFERENCE_CHANNELS = tuple(CONFIG["reference_channels"])
FVT_SHAKER_CHANNELS = frozenset(CONFIG["fvt_shaker_channels"])
EXPECTED_AVT_TO_FVT_SETUP = {
    int(key): int(value)
    for key, value in CONFIG["expected_avt_to_fvt_setup_map"].items()
}

def clean_matlab_label(value) -> str:
    """Normalize MATLAB char/cell labels without retaining brackets or whitespace."""
    while isinstance(value, np.ndarray) and value.size == 1:
        value = value.item()
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    label = str(value)
    label = re.sub(r"[\[\]'\"\s]+", "", label).upper()
    return label

def parse_labels(array: np.ndarray) -> list[str]:
    flat = np.asarray(array).ravel()
    labels = [clean_matlab_label(value) for value in flat]
    if any(not label for label in labels):
        raise ValueError("At least one empty Z24 channel label was found")
    return labels

def load_z24_mat(path: Path, dtype=np.float64) -> tuple[np.ndarray, list[str]]:
    mat = sio.loadmat(path, variable_names=["data", "labelshulp"])
    if "data" not in mat or "labelshulp" not in mat:
        raise KeyError(f"Required variables absent: {path}")
    data = np.asarray(mat["data"], dtype=dtype)
    labels = parse_labels(mat["labelshulp"])
    if data.ndim != 2 or data.shape[1] != len(labels):
        raise ValueError(f"Data/label mismatch at {path}: {data.shape} versus {len(labels)}")
    if len(set(labels)) != len(labels):
        duplicates = sorted({label for label in labels if labels.count(label) > 1})
        raise ValueError(f"Duplicate channel labels at {path}: {duplicates}")
    return data, labels

def load_z24_labels(path: Path) -> list[str]:
    mat = sio.loadmat(path, variable_names=["labelshulp"])
    if "labelshulp" not in mat:
        raise KeyError(f"labelshulp absent: {path}")
    return parse_labels(mat["labelshulp"])

def resolve_excitation_dir(scenario: str, excitation: str) -> Path:
    """Resolve AVT/FVT directories case-insensitively on Linux."""
    scenario_dir = DATASET_ROOT / str(scenario).zfill(2)
    if not scenario_dir.is_dir():
        raise FileNotFoundError(f"Missing Z24 scenario directory: {scenario_dir}")
    matches = [
        path for path in scenario_dir.iterdir()
        if path.is_dir() and path.name.casefold() == excitation.casefold()
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {excitation!r} directory below {scenario_dir}; found {matches}"
        )
    return matches[0]

def z24_setup_path(scenario: str, excitation: str, file_setup: int) -> Path:
    """Resolve a stored setup file number; this is not necessarily a physical AVT setup number."""
    directory = resolve_excitation_dir(scenario, excitation)
    candidate = directory / f"{str(scenario).zfill(2)}setup{int(file_setup):02d}.mat"
    if not candidate.is_file():
        matches = list(directory.rglob(candidate.name))
        if len(matches) != 1:
            raise FileNotFoundError(f"Could not uniquely resolve {candidate.name} below {directory}")
        candidate = matches[0]
    return candidate

def channel_metadata(label: str) -> dict:
    label = clean_matlab_label(label)
    shaker = label in FVT_SHAKER_CHANNELS
    reference = label in REFERENCE_CHANNELS
    match = re.fullmatch(r"(\d+)([LTV])", label)
    location = int(match.group(1)) if match else None
    axis = match.group(2) if match else (label[-1] if label and label[-1] in "LTV" else None)
    if shaker:
        family = "FVT_DRIVING_POINT"
    elif reference:
        family = "FIXED_REFERENCE"
    elif location is None:
        family = "UNKNOWN"
    elif location < 400:
        family = "GIRDER"
    else:
        family = "PIER"
    return {
        "label": label,
        "location_number": location,
        "axis": axis,
        "channel_family": family,
        "is_reference": reference,
        "is_fvt_shaker": shaker,
    }

def response_vertical_labels(labels: list[str]) -> list[str]:
    """
    Select bridge-response channels used by the FDD path.

    DP1V/DP2V are always excluded. The local-axis convention for pier channels
    is retained from the audited V0.2/V0.4 workflow: girder vertical channels
    end in V, while pier response channels numbered >=400 use local L.
    """
    selected: list[str] = []
    for raw_label in labels:
        meta = channel_metadata(raw_label)
        label = meta["label"]
        if meta["is_fvt_shaker"]:
            continue
        if label in {"R1V", "R2V", "R3V"}:
            selected.append(label)
        elif (meta["location_number"] is not None
              and meta["location_number"] < 400
              and meta["axis"] == "V"):
            selected.append(label)
        elif (meta["location_number"] is not None
              and meta["location_number"] >= 400
              and meta["axis"] == "L"):
            selected.append(label)
    return selected

# Backward-compatible name used by later cells.
global_vertical_labels = response_vertical_labels

# Build a metadata-only sensor/channel registry without loading 4.6 GB of signals.
sensor_registry_rows = []
for path in mat_paths:
    relative = path.relative_to(DATASET_ROOT)
    scenario = relative.parts[0]
    excitation = relative.parts[1].upper()
    file_setup = int(re.search(r"setup(\d{2})", path.stem, re.I).group(1))
    labels = load_z24_labels(path)
    for channel_index, label in enumerate(labels):
        sensor_registry_rows.append({
            "scenario": scenario,
            "excitation": excitation,
            "file_setup": file_setup,
            "channel_index": channel_index,
            **channel_metadata(label),
            "used_by_primary_fdd": label in response_vertical_labels(labels),
        })
sensor_registry = pd.DataFrame(sensor_registry_rows)
sensor_registry.to_csv(TABLE_ROOT / "z24_channel_registry.csv", index=False)

# Infer the physical AVT↔FVT setup correspondence from channel-label overlap,
# then compare it with the independently documented reversed setup mapping.
mapping_score = np.zeros((9, 9), dtype=float)
mapping_common_counts = np.zeros((9, 9), dtype=int)
for avt_setup in range(1, 10):
    avt_labels = set(response_vertical_labels(load_z24_labels(
        z24_setup_path("01", "avt", avt_setup)
    )))
    for fvt_setup in range(1, 10):
        fvt_labels = set(response_vertical_labels(load_z24_labels(
            z24_setup_path("01", "fvt", fvt_setup)
        )))
        union = avt_labels | fvt_labels
        common = avt_labels & fvt_labels
        mapping_common_counts[avt_setup-1, fvt_setup-1] = len(common)
        mapping_score[avt_setup-1, fvt_setup-1] = len(common) / max(1, len(union))

mapping_rows, mapping_columns = linear_sum_assignment(-mapping_score)
INFERRED_AVT_TO_FVT_SETUP = {
    int(row + 1): int(column + 1)
    for row, column in zip(mapping_rows, mapping_columns)
}
setup_mapping_rows = []
for avt_setup in range(1, 10):
    inferred_fvt = INFERRED_AVT_TO_FVT_SETUP[avt_setup]
    setup_mapping_rows.append({
        "physical_avt_setup": avt_setup,
        "expected_fvt_file_setup": EXPECTED_AVT_TO_FVT_SETUP[avt_setup],
        "inferred_fvt_file_setup": inferred_fvt,
        "jaccard_label_overlap": mapping_score[avt_setup-1, inferred_fvt-1],
        "common_response_channels": mapping_common_counts[avt_setup-1, inferred_fvt-1],
        "mapping_matches_documented_reversal":
            inferred_fvt == EXPECTED_AVT_TO_FVT_SETUP[avt_setup],
    })
avt_fvt_setup_mapping_audit = pd.DataFrame(setup_mapping_rows)
avt_fvt_setup_mapping_audit.to_csv(
    TABLE_ROOT / "z24_avt_fvt_physical_setup_mapping.csv", index=False
)
display(avt_fvt_setup_mapping_audit)

if avt_fvt_setup_mapping_audit["mapping_matches_documented_reversal"].all():
    ACTIVE_AVT_TO_FVT_SETUP = EXPECTED_AVT_TO_FVT_SETUP.copy()
    SETUP_MAPPING_STATUS = "DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL_OVERLAP"
else:
    minimum_overlap = float(avt_fvt_setup_mapping_audit["jaccard_label_overlap"].min())
    if minimum_overlap < 0.50:
        raise RuntimeError(
            "AVT/FVT physical setup mapping could not be established reliably; "
            f"minimum selected Jaccard overlap={minimum_overlap:.3f}"
        )
    ACTIVE_AVT_TO_FVT_SETUP = INFERRED_AVT_TO_FVT_SETUP.copy()
    SETUP_MAPPING_STATUS = "DATA_INFERRED_MAPPING_USED_DOCUMENTED_MAPPING_MISMATCH"

def z24_physical_setup_path(scenario: str, excitation: str, physical_avt_setup: int) -> Path:
    """Resolve the same physical bridge zone under AVT or FVT acquisition numbering."""
    excitation = excitation.lower()
    physical_avt_setup = int(physical_avt_setup)
    if excitation == "avt":
        file_setup = physical_avt_setup
    elif excitation == "fvt":
        file_setup = ACTIVE_AVT_TO_FVT_SETUP[physical_avt_setup]
    else:
        raise ValueError(f"Unsupported excitation: {excitation}")
    return z24_setup_path(scenario, excitation, file_setup)

shaker_audit = sensor_registry.groupby(
    ["excitation", "label", "is_fvt_shaker"], dropna=False
).size().reset_index(name="file_occurrences")
shaker_audit.to_csv(TABLE_ROOT / "z24_fvt_shaker_channel_audit.csv", index=False)
print("AVT/FVT setup mapping status:", SETUP_MAPPING_STATUS)
print("Active AVT→FVT map:", ACTIVE_AVT_TO_FVT_SETUP)
print("FVT shaker labels explicitly excluded:", sorted(FVT_SHAKER_CHANNELS))
print("Registered channel rows:", len(sensor_registry))


primary_qc_rows = []
representative_signals = {}
for scenario in CONFIG["primary_scenarios"]:
    paths = [z24_setup_path(scenario, "avt", setup) for setup in range(1, 10)]
    if len(paths) != 9:
        raise RuntimeError(f"Scenario {scenario} has {len(paths)} AVT MAT files, expected 9")
    for path in paths:
        data, labels = load_z24_mat(path)
        setup = int(re.search(r"setup(\d{2})", path.stem, re.I).group(1))
        channel_std = np.std(data, axis=0)
        channel_mean = np.mean(data, axis=0)
        channel_median = np.median(data, axis=0)
        channel_mad = np.median(np.abs(data-channel_median[None, :]), axis=0)
        robust_scale = np.maximum(1.4826*channel_mad, np.finfo(float).eps)
        robust_outlier_fraction = np.mean(
            np.abs(data-channel_median[None, :]) > 12.0*robust_scale[None, :],
            axis=0,
        )
        block = max(1, data.shape[0]//10)
        drift_fraction_of_std = np.abs(
            np.mean(data[-block:], axis=0)-np.mean(data[:block], axis=0)
        )/np.maximum(channel_std, np.finfo(float).eps)
        dc_fraction_of_std = np.abs(channel_mean)/np.maximum(
            channel_std, np.finfo(float).eps
        )
        flags = []
        if data.shape[0] != 65536:
            flags.append("NON_NOMINAL_SAMPLE_COUNT")
        if scenario == "03" and setup == 1:
            flags.append("LITERATURE_QC_EXCLUSION_CANDIDATE")
        if not np.isfinite(data).all():
            flags.append("NONFINITE")
        if np.any(channel_std <= np.finfo(float).eps):
            flags.append("CONSTANT_CHANNEL")
        if np.nanmax(robust_outlier_fraction) > 0.01:
            flags.append("ROBUST_OUTLIER_FRACTION_GT_1PCT")
        if np.nanmax(drift_fraction_of_std) > 2.0:
            flags.append("FIRST_LAST_DECILE_DRIFT_GT_2SD")
        primary_qc_rows.append({
            "scenario": scenario,
            "setup": setup,
            "relative_path": path.relative_to(DATASET_ROOT).as_posix(),
            "n_samples": data.shape[0],
            "n_channels": data.shape[1],
            "vertical_channel_count": len(global_vertical_labels(labels)),
            "all_finite": bool(np.isfinite(data).all()),
            "constant_channel_count": int(np.count_nonzero(
                channel_std <= np.finfo(float).eps
            )),
            "maximum_abs_mean_over_std": float(np.nanmax(dc_fraction_of_std)),
            "maximum_first_last_decile_drift_over_std": float(
                np.nanmax(drift_fraction_of_std)
            ),
            "maximum_robust_outlier_fraction": float(
                np.nanmax(robust_outlier_fraction)
            ),
            "qc_flags": ";".join(flags) if flags else "PASS",
        })
        if setup == 2:
            representative_label = "R1V" if "R1V" in labels else global_vertical_labels(labels)[0]
            index = labels.index(representative_label)
            representative_signals[scenario] = signal.detrend(data[:, index], type="linear")
        del data

primary_qc = pd.DataFrame(primary_qc_rows).sort_values(["scenario", "setup"])
primary_qc.to_csv(TABLE_ROOT / "z24_primary_avt_quality_control.csv", index=False)
display(primary_qc)
assert primary_qc["all_finite"].all()
assert primary_qc["constant_channel_count"].eq(0).all()
print("QC flags:", primary_qc["qc_flags"].value_counts().to_dict())

fig, axes = plt.subplots(len(CONFIG["primary_scenarios"]), 2, figsize=(12.5, 11.0), constrained_layout=True)
for row, scenario in enumerate(CONFIG["primary_scenarios"]):
    values = representative_signals[scenario]
    time_s = np.arange(len(values)) / CONFIG["sampling_rate_hz"]
    axes[row, 0].plot(time_s[:6000], values[:6000], lw=0.6, color="#264653")
    f, pxx = signal.welch(values, fs=CONFIG["sampling_rate_hz"], window="hann",
                          nperseg=CONFIG["fdd_nperseg"], noverlap=CONFIG["fdd_noverlap"], detrend=False)
    band = (f >= CONFIG["frequency_band_hz"][0]) & (f <= CONFIG["frequency_band_hz"][1])
    axes[row, 1].semilogy(f[band], pxx[band], lw=0.85, color="#e76f51")
    axes[row, 0].set_ylabel(f"S{scenario}\ndataset units")
    axes[row, 1].set_ylabel("PSD")
axes[0, 0].set_title("R1V, setup 02 — first 60 s (detrended, not normalized)")
axes[0, 1].set_title("Welch PSD — complete trace")
axes[-1, 0].set_xlabel("Time [s]")
axes[-1, 1].set_xlabel("Frequency [Hz]")
fig.savefig(FIGURE_ROOT / "figure_03_representative_signals_psd.png", bbox_inches="tight", dpi=320)
fig.savefig(FIGURE_ROOT / "figure_03_representative_signals_psd.pdf", bbox_inches="tight")
plt.show()
plt.close("all")

,physical_avt_setup,expected_fvt_file_setup,inferred_fvt_file_setup,jaccard_label_overlap,common_response_channels,mapping_matches_documented_reversal
0,1,9,9,0.95,19,True
1,2,8,8,1.00,20,True
2,3,7,7,1.00,20,True
3,4,6,6,1.00,20,True
4,5,5,5,1.00,18,True
5,6,4,4,1.00,20,True
6,7,3,3,0.95,19,True
7,8,2,2,1.00,20,True
8,9,1,1,1.00,20,True


AVT/FVT setup mapping status: DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL_OVERLAP
Active AVT→FVT map: {1: 9, 2: 8, 3: 7, 4: 6, 5: 5, 6: 4, 7: 3, 8: 2, 9: 1}
FVT shaker labels explicitly excluded: ['DP1V', 'DP2V']
Registered channel rows: 10189


,scenario,setup,relative_path,n_samples,n_channels,vertical_channel_count,all_finite,constant_channel_count,maximum_abs_mean_over_std,maximum_first_last_decile_drift_over_std,maximum_robust_outlier_fraction,qc_flags
0,01,1,01/avt/01setup01.mat,65530,33,20,True,0,262.759599,2.912832,0.005890,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
1,01,2,01/avt/01setup02.mat,65530,33,20,True,0,217.980468,2.957409,0.005265,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
2,01,3,01/avt/01setup03.mat,65530,33,20,True,0,1072.166060,3.028239,0.014802,NON_NOMINAL_SAMPLE_COUNT;ROBUST_OUTLIER_FRACTI...
3,01,4,01/avt/01setup04.mat,65530,33,20,True,0,83.526204,2.782455,0.002304,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
4,01,5,01/avt/01setup05.mat,65530,27,18,True,0,4.079434,2.430742,0.000000,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
5,01,6,01/avt/01setup06.mat,65530,33,20,True,0,86.554388,3.062285,0.062796,NON_NOMINAL_SAMPLE_COUNT;ROBUST_OUTLIER_FRACTI...
6,01,7,01/avt/01setup07.mat,65530,31,19,True,0,739.705107,3.226435,0.005097,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
7,01,8,01/avt/01setup08.mat,65530,33,20,True,0,205.305166,3.350245,0.000382,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
8,01,9,01/avt/01setup09.mat,65530,33,20,True,0,156.104793,3.439394,0.001419,NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRI...
9,03,1,03/avt/03setup01.mat,65026,33,20,True,0,104.986872,3.451813,0.000554,NON_NOMINAL_SAMPLE_COUNT;LITERATURE_QC_EXCLUSI...


QC flags: {'FIRST_LAST_DECILE_DRIFT_GT_2SD': 29, 'NON_NOMINAL_SAMPLE_COUNT;FIRST_LAST_DECILE_DRIFT_GT_2SD': 7, 'ROBUST_OUTLIER_FRACTION_GT_1PCT;FIRST_LAST_DECILE_DRIFT_GT_2SD': 5, 'NON_NOMINAL_SAMPLE_COUNT;ROBUST_OUTLIER_FRACTION_GT_1PCT;FIRST_LAST_DECILE_DRIFT_GT_2SD': 2, 'NON_NOMINAL_SAMPLE_COUNT;LITERATURE_QC_EXCLUSION_CANDIDATE;FIRST_LAST_DECILE_DRIFT_GT_2SD': 1, 'PASS': 1}


## 6. Published-data sanity check

The official Z24 benchmark reports 65,536 samples at 100 Hz with a 30 Hz antialiasing cutoff. The local snapshot contains legitimate length variation, so 65,536 is treated as nominal rather than universal. Healthy vertical-mode reference frequencies `[3.92, 5.12, 9.93, 10.52, 12.69] Hz` are used only as broad sanity anchors and initial mode bands; the extracted frequencies come from the local arrays.

FDD is computed independently inside every setup. For each setup, only vertical labels common to all five settlement scenarios are used, ensuring identical coordinates during state tracking. Target modes are matched sequentially using frequency proximity plus the Modal Assurance Criterion (MAC). Global mode shapes across the nine roving setups are not constructed.

In [11]:
def compute_fdd(path: Path, ordered_labels: list[str]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Estimate the leading FDD singular-value spectrum for one simultaneous setup.

    The actual trace length is preserved. Signals are linearly detrended once,
    then each Welch segment is mean-removed. CPSD scaling follows the standard
    one-sided density convention; this does not alter peak locations but makes
    cross-excitation energy audits interpretable.
    """
    data, labels = load_z24_mat(path)
    if any(label in FVT_SHAKER_CHANNELS for label in ordered_labels):
        raise ValueError(f"FVT driving-point channels cannot enter response FDD: {ordered_labels}")
    indices = [labels.index(label) for label in ordered_labels]
    values = signal.detrend(data[:, indices], axis=0, type="linear")
    nperseg = int(CONFIG["fdd_nperseg"])
    noverlap = int(CONFIG["fdd_noverlap"])
    step = nperseg - noverlap
    starts = np.arange(0, len(values) - nperseg + 1, step)
    if len(starts) < 4:
        raise ValueError(f"Insufficient Welch segments for {path}")

    window = signal.windows.hann(nperseg, sym=False)
    window_power = float(np.sum(window**2))
    segments = np.stack(
        [values[start:start+nperseg] for start in starts],
        axis=0,
    )
    segments = signal.detrend(
        segments,
        axis=1,
        type=CONFIG["fdd_segment_detrend"],
    )
    fft = np.fft.rfft(segments * window[None, :, None], axis=1)
    frequency = np.fft.rfftfreq(nperseg, d=1.0 / CONFIG["sampling_rate_hz"])

    scale = 1.0 / (
        CONFIG["sampling_rate_hz"] * window_power * len(starts)
    )
    cpsd_full = np.einsum(
        "sfc,sfd->fcd",
        fft.conj(),
        fft,
        optimize=True,
    ) * scale
    # One-sided density: double strictly positive, non-Nyquist bins.
    if nperseg % 2 == 0:
        cpsd_full[1:-1] *= 2.0
    else:
        cpsd_full[1:] *= 2.0

    band = (
        (frequency >= CONFIG["frequency_band_hz"][0])
        & (frequency <= CONFIG["frequency_band_hz"][1])
    )
    frequency = frequency[band]
    cpsd = cpsd_full[band]
    eigenvalues, eigenvectors = np.linalg.eigh(cpsd)
    leading = np.maximum(eigenvalues[:, -1].real, np.finfo(float).tiny)
    leading_vectors = eigenvectors[:, :, -1]
    return frequency, leading, leading_vectors

def smooth_log_spectrum(frequency: np.ndarray, power: np.ndarray) -> np.ndarray:
    spacing = float(np.median(np.diff(frequency)))
    width = max(5, int(round(CONFIG["fdd_smoothing_width_hz"] / spacing)))
    if width % 2 == 0:
        width += 1
    return signal.savgol_filter(np.log(np.maximum(power, np.finfo(float).tiny)), width, 2)

def modal_assurance_criterion(left: np.ndarray, right: np.ndarray) -> float:
    numerator = abs(np.vdot(left, right)) ** 2
    denominator = np.vdot(left, left).real * np.vdot(right, right).real
    return float(numerator / denominator)

scenarios = CONFIG["primary_scenarios"]
settlements = np.asarray(CONFIG["pier_settlement_mm"], dtype=float)
mode_count = CONFIG["candidate_mode_count"]
setup_count = 9
frequency_cube = np.full((len(scenarios), setup_count, mode_count), np.nan)
sequential_mac_cube = np.full_like(frequency_cube, np.nan)
selected_label_rows = []
fdd_spectra = {}
fdd_tracking_rows = []

for setup_index in range(setup_count):
    setup = setup_index + 1
    paths = {scenario: z24_setup_path(scenario, "avt", setup)
             for scenario in scenarios}
    vertical_by_scenario = {}
    for scenario, path in paths.items():
        _, labels = load_z24_mat(path)
        vertical_by_scenario[scenario] = global_vertical_labels(labels)
    common_labels = sorted(set.intersection(*(set(values) for values in vertical_by_scenario.values())))
    if len(common_labels) < 3:
        raise RuntimeError(f"Too few common vertical channels in setup {setup:02d}")
    selected_label_rows.append({"setup": setup, "common_vertical_count": len(common_labels),
                                "labels": ";".join(common_labels)})
    setup_data = {scenario: compute_fdd(path, common_labels) for scenario, path in paths.items()}
    for scenario, (frequency, leading, _) in setup_data.items():
        fdd_spectra[(scenario, setup)] = (frequency, leading)

    frequency, leading, vectors = setup_data["01"]
    smoothed = smooth_log_spectrum(frequency, leading)
    reference_modes = []
    for low, high in CONFIG["candidate_mode_bands_hz"]:
        indices = np.flatnonzero((frequency >= low) & (frequency <= high))
        local_peaks, properties = signal.find_peaks(smoothed[indices], prominence=0.0)
        if len(local_peaks):
            chosen = indices[local_peaks[int(np.argmax(properties["prominences"]))]]
        else:
            chosen = indices[int(np.argmax(smoothed[indices]))]
        reference_modes.append((float(frequency[chosen]), vectors[chosen]))
    frequency_cube[0, setup_index] = [item[0] for item in reference_modes]
    sequential_mac_cube[0, setup_index] = 1.0

    previous_modes = reference_modes
    for scenario_index, scenario in enumerate(scenarios[1:], start=1):
        frequency, leading, vectors = setup_data[scenario]
        smoothed = smooth_log_spectrum(frequency, leading)
        analysis_indices = np.flatnonzero((frequency >= 3.0) & (frequency <= 14.5))
        peaks, properties = signal.find_peaks(smoothed[analysis_indices], prominence=0.03, distance=4)
        candidates = analysis_indices[peaks]
        # Match only actual detected FDD peaks. If fewer peaks are available,
        # abstain for unsupported modes rather than forcing a spectral shoulder.
        if not len(candidates):
            fdd_tracking_rows.append({
                "excitation": "AVT", "setup": setup, "scenario": scenario,
                "detected_peak_count": 0, "matched_mode_count": 0,
                "status": "NO_PEAKS_EXPLICIT_ABSTENTION",
            })
            continue
        cost = np.full((mode_count, len(candidates)), 99.0)
        mac_values = np.full_like(cost, np.nan)
        for mode_index, (reference_frequency, reference_vector) in enumerate(previous_modes):
            for candidate_index, spectral_index in enumerate(candidates):
                relative_shift = abs(frequency[spectral_index] - reference_frequency) / reference_frequency
                mac = modal_assurance_criterion(reference_vector, vectors[spectral_index])
                mac_values[mode_index, candidate_index] = mac
                if relative_shift <= CONFIG["mode_tracking_max_relative_shift"]:
                    cost[mode_index, candidate_index] = (
                        CONFIG["mode_tracking_frequency_weight"] * relative_shift
                        + CONFIG["mode_tracking_mac_weight"] * (1.0 - mac)
                    )
        rows, columns = linear_sum_assignment(cost)
        matches = sorted(zip(rows, columns))
        # A mode may be unobservable in an individual output-only acquisition.
        # Record an explicit abstention and retain its last valid reference for
        # the next state; do not force an inadmissible spectral peak.
        next_modes = list(previous_modes)
        matched_frequency = np.full(mode_count, np.nan)
        matched_mac = np.full(mode_count, np.nan)
        for row, column in matches:
            if cost[row, column] < 99.0:
                spectral_index = candidates[column]
                matched_frequency[row] = float(frequency[spectral_index])
                matched_mac[row] = mac_values[row, column]
                next_modes[row] = (float(frequency[spectral_index]), vectors[spectral_index])
        previous_modes = next_modes
        frequency_cube[scenario_index, setup_index] = matched_frequency
        sequential_mac_cube[scenario_index, setup_index] = matched_mac
        matched_count = int(np.isfinite(matched_frequency).sum())
        fdd_tracking_rows.append({
            "excitation": "AVT", "setup": setup, "scenario": scenario,
            "detected_peak_count": int(len(candidates)),
            "matched_mode_count": matched_count,
            "status": "TRACKED" if matched_count == mode_count else "PARTIAL_EXPLICIT_ABSTENTION",
        })

selected_labels = pd.DataFrame(selected_label_rows)
fdd_tracking_log = pd.DataFrame(fdd_tracking_rows)
display(selected_labels)
selected_labels.to_csv(TABLE_ROOT / "z24_common_vertical_channels_by_setup.csv", index=False)
fdd_tracking_log.to_csv(TABLE_ROOT / "z24_avt_fdd_tracking_log.csv", index=False)
display(fdd_tracking_log)

published = np.asarray(CONFIG["published_sanity_modes_hz"], dtype=float)
healthy_median = np.nanmedian(frequency_cube[0], axis=0)
healthy_mad = np.nanmedian(np.abs(frequency_cube[0] - healthy_median), axis=0)
sanity_table = pd.DataFrame({
    "candidate_mode": np.arange(1, mode_count + 1),
    "published_reference_hz": published,
    "local_healthy_median_hz": healthy_median,
    "setup_mad_hz": healthy_mad,
    "relative_error_fraction": np.abs(healthy_median - published) / published,
})
sanity_table["passes_2pct_sanity"] = sanity_table["relative_error_fraction"] <= CONFIG["healthy_frequency_sanity_tolerance_fraction"]
display(sanity_table)
sanity_table.to_csv(TABLE_ROOT / "z24_healthy_frequency_sanity.csv", index=False)

,setup,common_vertical_count,labels
0,1,20,100V;101V;102V;103V;199V;200V;201V;202V;203V;2...
1,2,20,104V;105V;106V;107V;108V;204V;205V;206V;207V;2...
2,3,20,109V;110V;111V;112V;113V;209V;210V;211V;212V;2...
3,4,20,114V;115V;116V;117V;118V;214V;215V;216V;217V;2...
4,5,18,119V;120V;121V;122V;123V;219V;220V;221V;222V;2...
5,6,20,124V;125V;126V;127V;128V;224V;225V;226V;227V;2...
6,7,19,129V;130V;131V;132V;133V;229V;231V;232V;233V;3...
7,8,20,134V;135V;136V;137V;138V;234V;235V;236V;237V;2...
8,9,20,139V;140V;141V;142V;143V;239V;240V;241V;242V;2...


,excitation,setup,scenario,detected_peak_count,matched_mode_count,status
0,AVT,1,03,14,5,TRACKED
1,AVT,1,04,16,5,TRACKED
2,AVT,1,05,12,5,TRACKED
3,AVT,1,06,13,5,TRACKED
4,AVT,2,03,13,5,TRACKED
5,AVT,2,04,17,5,TRACKED
6,AVT,2,05,17,5,TRACKED
7,AVT,2,06,16,5,TRACKED
8,AVT,3,03,17,5,TRACKED
9,AVT,3,04,16,5,TRACKED


,candidate_mode,published_reference_hz,local_healthy_median_hz,setup_mad_hz,relative_error_fraction,passes_2pct_sanity
0,1,3.92,3.906250,0.000000,0.003508,True
1,2,5.12,5.078125,0.024414,0.008179,True
2,3,9.93,9.960938,0.000000,0.003116,True
3,4,10.52,10.522461,0.048828,0.000234,True
4,5,12.69,12.695312,0.122070,0.000419,True


## 7. Structural observations from Z24

Candidate modes are summarized robustly across the nine setup acquisitions. A setup estimate is supported when it lies within `max(0.15 Hz, 3 × scaled MAD)` of the scenario median. Primary modes must pass the healthy sanity check, have at least 7/9 setup support in every state, show an absolute Spearman association of at least 0.8 with imposed settlement, and remain non-increasing at every controlled step. This association is a progression check, not proof that settlement equals stiffness loss.

In [12]:
summary_rows = []
candidate_rows = []
for mode_index in range(mode_count):
    medians = np.nanmedian(frequency_cube[:, :, mode_index], axis=1)
    mads = np.nanmedian(np.abs(frequency_cube[:, :, mode_index] - medians[:, None]), axis=1)
    scaled_mads = 1.4826 * mads
    tolerances = np.maximum(0.15, 3.0 * scaled_mads)
    support = np.sum(np.abs(frequency_cube[:, :, mode_index] - medians[:, None]) <= tolerances[:, None], axis=1)
    rho, rho_p = stats.spearmanr(settlements, medians)
    tau, tau_p = stats.kendalltau(settlements, medians)
    progression_differences = np.diff(medians)
    monotonic_nonincrease_accuracy = float(np.mean(progression_differences <= 1e-12))
    all_support = bool(np.all(support >= CONFIG["minimum_setup_support"]))
    sanity_pass = bool(sanity_table.loc[mode_index, "passes_2pct_sanity"])
    retained = bool(sanity_pass and all_support and abs(rho) >= CONFIG["minimum_abs_spearman_for_primary_mode"]
                    and rho < 0
                    and monotonic_nonincrease_accuracy >= CONFIG["minimum_monotonic_nonincrease_accuracy"])
    candidate_rows.append({
        "candidate_mode": mode_index + 1,
        "healthy_sanity_pass": sanity_pass,
        "minimum_setup_support": int(np.min(support)),
        "median_sequential_mac": float(np.nanmedian(sequential_mac_cube[1:, :, mode_index])),
        "spearman_rho_vs_delta": float(rho), "spearman_p": float(rho_p),
        "kendall_tau_vs_delta": float(tau), "kendall_p": float(tau_p),
        "monotonic_nonincrease_accuracy": monotonic_nonincrease_accuracy,
        "retained_primary_observation": retained,
    })
    for scenario_index, scenario in enumerate(scenarios):
        summary_rows.append({
            "scenario": scenario, "delta_pier_mm": settlements[scenario_index],
            "candidate_mode": mode_index + 1, "median_frequency_hz": medians[scenario_index],
            "mad_frequency_hz": mads[scenario_index], "supported_setups": int(support[scenario_index]),
            "median_sequential_mac": float(np.nanmedian(sequential_mac_cube[scenario_index, :, mode_index])),
        })

candidate_assessment = pd.DataFrame(candidate_rows)
mode_summary = pd.DataFrame(summary_rows)
retained_modes = candidate_assessment.loc[candidate_assessment["retained_primary_observation"], "candidate_mode"].astype(int).tolist()
display(candidate_assessment)
display(mode_summary.pivot(index=["scenario", "delta_pier_mm"], columns="candidate_mode", values="median_frequency_hz"))
candidate_assessment.to_csv(TABLE_ROOT / "gate_a_candidate_mode_assessment.csv", index=False)
mode_summary.to_csv(TABLE_ROOT / "gate_a_mode_summary.csv", index=False)

# Sensitivity of scenario 03 medians to the literature-flagged short setup 01.
scenario03_index = scenarios.index("03")
with_flagged = np.median(frequency_cube[scenario03_index], axis=0)
without_flagged = np.median(frequency_cube[scenario03_index, 1:], axis=0)
qc_sensitivity = pd.DataFrame({
    "candidate_mode": np.arange(1, mode_count + 1),
    "scenario03_with_setup01_hz": with_flagged,
    "scenario03_without_setup01_hz": without_flagged,
    "absolute_difference_hz": np.abs(with_flagged - without_flagged),
})
display(qc_sensitivity)
qc_sensitivity.to_csv(TABLE_ROOT / "scenario03_setup01_sensitivity.csv", index=False)

scenario03_retained_sensitivity_hz = float(
    qc_sensitivity[
        qc_sensitivity["candidate_mode"].isin(retained_modes)
    ]["absolute_difference_hz"].max()
)
scenario03_setup01_influence_pass = bool(
    scenario03_retained_sensitivity_hz
    <= CONFIG["sampling_rate_hz"] / CONFIG["fdd_nperseg"] + 1e-12
)
print(
    "Scenario 03/setup 01 maximum influence on retained medians [Hz]:",
    scenario03_retained_sensitivity_hz,
    "pass:", scenario03_setup01_influence_pass,
)

# Deterministic setup bootstrap intervals for the retained structural observation vector.
rng = np.random.default_rng(CONFIG["seed"])
bootstrap_rows = []
for scenario_index, scenario in enumerate(scenarios):
    for mode in retained_modes:
        values = frequency_cube[scenario_index, :, mode - 1]
        values = values[np.isfinite(values)]
        draws = np.median(rng.choice(values, size=(CONFIG["bootstrap_replicates"], len(values)), replace=True), axis=1)
        bootstrap_rows.append({
            "scenario": scenario, "delta_pier_mm": settlements[scenario_index], "mode": mode,
            "median_frequency_hz": float(np.median(values)),
            "ci95_low_hz": float(np.quantile(draws, 0.025)),
            "ci95_high_hz": float(np.quantile(draws, 0.975)),
        })
bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table.to_csv(TABLE_ROOT / "z24_retained_modal_observations.csv", index=False)
display(bootstrap_table)

fig, axes = plt.subplots(1, len(retained_modes), figsize=(4.2 * len(retained_modes), 3.8), sharex=True)
if len(retained_modes) == 1:
    axes = [axes]
for ax, mode in zip(axes, retained_modes):
    rows = bootstrap_table[bootstrap_table["mode"] == mode]
    center = rows["median_frequency_hz"].to_numpy()
    lower = center - rows["ci95_low_hz"].to_numpy()
    upper = rows["ci95_high_hz"].to_numpy() - center
    ax.errorbar(rows["delta_pier_mm"], center, yerr=np.vstack([lower, upper]), fmt="o-",
                capsize=4, lw=1.8, color="#1d3557")
    rho = candidate_assessment.loc[candidate_assessment["candidate_mode"] == mode, "spearman_rho_vs_delta"].iloc[0]
    ax.set_title(f"Retained mode {mode} (Spearman ρ={rho:.2f})")
    ax.set_xlabel(r"$\delta_{pier}$ [mm]")
    ax.set_ylabel("Frequency [Hz]")
fig.suptitle("Z24 FDD observations retained for Digital Twin calibration", y=1.03, weight="bold")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "figure_04_extracted_structural_frequencies.png", bbox_inches="tight", dpi=320)
fig.savefig(FIGURE_ROOT / "figure_04_extracted_structural_frequencies.pdf", bbox_inches="tight")
plt.show()
plt.close("all")


# Joint paired-setup bootstrap: estimate cross-mode covariance without
# treating modal coordinates as independent pseudo-replicates.
ratio_covariance_rows = []
ratio_draw_rows = []
gate_a_ratio_covariance_by_scenario: dict[str, np.ndarray] = {}
ratio_bootstrap_rng = np.random.default_rng(CONFIG["seed"] + 1701)
retained_zero_based = np.asarray(retained_modes, dtype=int) - 1
frequency_resolution_hz = CONFIG["sampling_rate_hz"] / CONFIG["fdd_nperseg"]
healthy_retained = np.take(
    frequency_cube[0],
    retained_zero_based,
    axis=1,
)
expected_healthy_retained_shape = (setup_count, len(retained_zero_based))
if healthy_retained.shape != expected_healthy_retained_shape:
    raise RuntimeError(
        "Gate-A retained-mode orientation regression: "
        f"expected {expected_healthy_retained_shape}, got {healthy_retained.shape}"
    )

for state_index, scenario in enumerate(scenarios):
    draws = []
    attempts = 0
    while len(draws) < CONFIG["bootstrap_replicates"]:
        attempts += 1
        if attempts > CONFIG["bootstrap_replicates"] * 20:
            raise RuntimeError(
                f"Unable to form enough joint modal-ratio bootstrap draws for scenario {scenario}"
            )
        sampled_setups = ratio_bootstrap_rng.integers(0, setup_count, size=setup_count)
        ratios = []
        valid_draw = True
        for mode_position, mode_index in enumerate(retained_zero_based):
            current = frequency_cube[state_index, sampled_setups, mode_index]
            baseline = frequency_cube[0, sampled_setups, mode_index]
            valid = np.isfinite(current) & np.isfinite(baseline)
            if np.count_nonzero(valid) < max(3, CONFIG["minimum_setup_support"] // 2):
                valid_draw = False
                break
            ratios.append(
                float(np.median(current[valid]) / np.median(baseline[valid]))
            )
        if valid_draw:
            draws.append(ratios)

    draws = np.asarray(draws, dtype=float)
    raw_covariance = np.atleast_2d(np.cov(draws, rowvar=False, ddof=1))
    healthy_medians = np.nanmedian(healthy_retained, axis=0)
    expected_covariance_shape = (len(retained_zero_based), len(retained_zero_based))
    if raw_covariance.shape != expected_covariance_shape:
        raise RuntimeError(
            f"Gate-A covariance regression for scenario {scenario}: "
            f"expected {expected_covariance_shape}, got {raw_covariance.shape}"
        )
    if healthy_medians.shape != (len(retained_zero_based),):
        raise RuntimeError(
            f"Gate-A healthy median regression: expected {(len(retained_zero_based),)}, "
            f"got {healthy_medians.shape}"
        )
    resolution_variance = (
        0.5 * frequency_resolution_hz / np.maximum(healthy_medians, 1e-12)
    ) ** 2
    diagonal = np.maximum(np.diag(raw_covariance), resolution_variance)
    shrinkage = float(CONFIG["modal_ratio_covariance_shrinkage"])
    covariance = (
        (1.0 - shrinkage) * raw_covariance
        + shrinkage * np.diag(diagonal)
    )
    covariance[np.diag_indices_from(covariance)] = np.maximum(
        np.diag(covariance), resolution_variance
    )
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    covariance = (
        eigenvectors
        @ np.diag(np.maximum(eigenvalues, max(1e-12, 1e-6 * eigenvalues.max())))
        @ eigenvectors.T
    )
    gate_a_ratio_covariance_by_scenario[scenario] = covariance

    for row_index, mode_i in enumerate(retained_modes):
        for column_index, mode_j in enumerate(retained_modes):
            ratio_covariance_rows.append({
                "scenario": scenario,
                "mode_i": int(mode_i),
                "mode_j": int(mode_j),
                "covariance": float(covariance[row_index, column_index]),
                "correlation": float(
                    covariance[row_index, column_index]
                    / np.sqrt(
                        covariance[row_index, row_index]
                        * covariance[column_index, column_index]
                    )
                ),
                "bootstrap_replicates": int(len(draws)),
                "shrinkage_to_diagonal": shrinkage,
            })
    # Preserve compact draws for reproducibility and later covariance sensitivity checks.
    for replicate, row in enumerate(draws):
        ratio_draw_rows.append({
            "scenario": scenario,
            "replicate": replicate,
            **{f"ratio_f{mode}": float(value)
               for mode, value in zip(retained_modes, row)},
        })

gate_a_ratio_covariance = pd.DataFrame(ratio_covariance_rows)
gate_a_ratio_bootstrap_draws = pd.DataFrame(ratio_draw_rows)
gate_a_ratio_covariance.to_csv(
    TABLE_ROOT / "gate_a_joint_modal_ratio_covariance.csv", index=False
)
gate_a_ratio_bootstrap_draws.to_csv(
    TABLE_ROOT / "gate_a_joint_modal_ratio_bootstrap_draws.csv", index=False
)
print("Joint modal-ratio covariance estimated for scenarios:", sorted(
    gate_a_ratio_covariance_by_scenario
))

dataset_pass = bool(
    len(mat_inventory) == 306
    and mat_inventory["data_present"].all()
    and primary_qc["all_finite"].all()
    and primary_qc["constant_channel_count"].eq(0).all()
    and set(scenarios) == set(settlement_map)
)
orientation_pass = bool(all(row["transpose_is_correct"] for row in representative_rows))
modal_pass = len(retained_modes) >= 3
gate_a_pass = bool(
    dataset_pass and orientation_pass and modal_pass
    and scenario03_setup01_influence_pass
)
RAW_SIGNATURE_AFTER = raw_metadata_signature(DATASET_ROOT)
raw_unchanged = RAW_SIGNATURE_AFTER == RAW_SIGNATURE_BEFORE
gate_a_pass = bool(gate_a_pass and raw_unchanged)

gate_a_diagnostic = {
    "gate": "A_Z24_DATA_AND_STRUCTURAL_OBSERVATIONS",
    "status": "GO_TO_SYNTHETIC_PHYSICS_PILOT" if gate_a_pass else "NO_GO_STOP_AND_REVISE_DATA_STAGE",
    "dataset_pass": dataset_pass,
    "orientation_pass": orientation_pass,
    "raw_metadata_unchanged": raw_unchanged,
    "mat_file_count": int(len(mat_inventory)),
    "primary_avt_acquisition_count": int(len(primary_qc)),
    "sampling_frequency_hz": CONFIG["sampling_rate_hz"],
    "sampling_frequency_provenance": "local Z24 readme + local EMPA report; not encoded in MAT variables",
    "scenario_mapping_provenance": REFERENCE_URLS["z24_official"],
    "local_doc_discrepancy": "03 report absent; 05/06 reports contain stale scenario-01 text; 04 directly confirms 40 mm",
    "retained_primary_modes": retained_modes,
    "observation_vector": [f"f_{mode}" for mode in retained_modes],
    "mode_shape_scope": "within-setup MAC for tracking only; no fabricated global roving-setup mode shapes",
    "physical_units_status": "TO_VERIFY",
    "scenario03_setup01_policy": "included only after retained-mode median influence is no greater than one FDD frequency bin; raw file untouched",
    "scenario03_setup01_max_retained_median_influence_hz": scenario03_retained_sensitivity_hz,
    "scenario03_setup01_influence_pass": scenario03_setup01_influence_pass,
    "avt_fvt_setup_mapping_status": SETUP_MAPPING_STATUS,
    "avt_to_fvt_setup_map": ACTIVE_AVT_TO_FVT_SETUP,
    "fvt_shaker_channels_excluded": sorted(FVT_SHAKER_CHANNELS),
    "joint_modal_ratio_covariance": "paired-setup bootstrap with 20% shrinkage and resolution floor",
    "important_interpretation": "delta_pier is imposed settlement, not stiffness loss",
    "next_authorized_stage": "Section 8 synthetic SDOF and Euler-Bernoulli validation; still no Z24 inverse calibration until Full PINN and GPT-PINN gates pass",
}
(OUTPUT_ROOT / "gate_a_diagnostic.json").write_text(json_dumps_safe(gate_a_diagnostic, indent=2) + "\n", encoding="utf-8")
CONFIG["retained_mode_indices"] = retained_modes
CONFIG["retained_observation_vector"] = gate_a_diagnostic["observation_vector"]
CONFIG_PATH.write_text(json_dumps_safe(CONFIG, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("\n" + "=" * 78)
print("GATE A DECISION:", gate_a_diagnostic["status"])
print("Retained observation vector:", gate_a_diagnostic["observation_vector"])
print("Raw metadata unchanged:", raw_unchanged)
print("No Full PINN or GPT-PINN training was executed.")
print("=" * 78)

,candidate_mode,healthy_sanity_pass,minimum_setup_support,median_sequential_mac,spearman_rho_vs_delta,spearman_p,kendall_tau_vs_delta,kendall_p,monotonic_nonincrease_accuracy,retained_primary_observation
0,1,True,9,0.999402,-0.974679,4.818230e-03,-0.948683,0.022977,1.00,True
1,2,True,7,0.957694,-0.820783,8.858701e-02,-0.737865,0.076974,0.75,False
2,3,True,8,0.968530,-1.000000,1.404265e-24,-1.000000,0.016667,1.00,True
3,4,True,8,0.914001,-1.000000,1.404265e-24,-1.000000,0.016667,1.00,True
4,5,True,9,0.951394,0.300000,6.238377e-01,0.200000,0.816667,0.50,False


,candidate_mode,1,2,3,4,5
scenario,delta_pier_mm,,,,,
01,0.0,3.906250,5.078125,9.960938,10.522461,12.695312
03,20.0,3.857422,5.029297,9.790039,10.327148,12.890625
04,40.0,3.857422,4.907227,9.741211,10.253906,12.670898
05,80.0,3.759766,5.004883,9.350586,9.838867,12.963867
06,95.0,3.662109,4.907227,9.228516,9.692383,12.817383


,candidate_mode,scenario03_with_setup01_hz,scenario03_without_setup01_hz,absolute_difference_hz
0,1,3.857422,3.857422,0.000000
1,2,5.029297,5.041504,0.012207
2,3,9.790039,9.790039,0.000000
3,4,10.327148,10.327148,0.000000
4,5,12.890625,12.841797,0.048828


Scenario 03/setup 01 maximum influence on retained medians [Hz]: 0.0 pass: True


,scenario,delta_pier_mm,mode,median_frequency_hz,ci95_low_hz,ci95_high_hz
0,01,0.0,1,3.906250,3.906250,3.906250
1,01,0.0,3,9.960938,9.936523,9.960938
2,01,0.0,4,10.522461,10.473633,10.595703
3,03,20.0,1,3.857422,3.857422,3.881836
4,03,20.0,3,9.790039,9.790039,9.838867
5,03,20.0,4,10.327148,10.302734,10.375977
6,04,40.0,1,3.857422,3.833008,3.857422
7,04,40.0,3,9.741211,9.716797,9.765625
8,04,40.0,4,10.253906,10.205078,10.278320
9,05,80.0,1,3.759766,3.735352,3.759766


Joint modal-ratio covariance estimated for scenarios: ['01', '03', '04', '05', '06']

GATE A DECISION: GO_TO_SYNTHETIC_PHYSICS_PILOT
Retained observation vector: ['f_1', 'f_3', 'f_4']
Raw metadata unchanged: True
No Full PINN or GPT-PINN training was executed.


### 7.1 Auxiliary fixed-reference spectral and transmissibility audit

Several Z24 studies obtain useful damage-sensitive information from the fixed
reference channels. V0.6 FINAL therefore records an **auxiliary**, non-tuning audit using
`R1V`, `R2L`, `R2T`, `R2V`, and `R3V`. It does not replace the multichannel FDD
observation used by the Digital Twin.

For each setup and state, the notebook computes detrended Welch spectra, band
energy around each retained mode, peak frequency, and selected log-transmissibility
ratios. No padding, mirroring, per-trace standardization, threshold fitting, or
held-out-label optimization is performed.


In [13]:
reference_feature_rows = []
healthy_mode_centers = {
    int(mode): float(
        bootstrap_table[
            (bootstrap_table["scenario"] == "01")
            & (bootstrap_table["mode"] == mode)
        ]["median_frequency_hz"].iloc[0]
    )
    for mode in retained_modes
}

for scenario in scenarios:
    for setup in range(1, 10):
        path = z24_setup_path(scenario, "avt", setup)
        data, labels = load_z24_mat(path)
        available_references = [
            label for label in REFERENCE_CHANNELS if label in labels
        ]
        if len(available_references) < 3:
            continue
        reference_psd = {}
        reference_signals = {}
        reference_frequency = None
        for label in available_references:
            values = signal.detrend(data[:, labels.index(label)], type="linear")
            frequency, psd = signal.welch(
                values,
                fs=CONFIG["sampling_rate_hz"],
                window="hann",
                nperseg=CONFIG["fdd_nperseg"],
                noverlap=CONFIG["fdd_noverlap"],
                detrend="constant",
                scaling="density",
            )
            reference_frequency = frequency
            reference_psd[label] = psd
            reference_signals[label] = values
        for mode, center in healthy_mode_centers.items():
            band = np.abs(reference_frequency-center) <= 0.30
            if np.count_nonzero(band) < 2:
                continue
            row = {
                "scenario": scenario,
                "delta_pier_mm": settlement_map[scenario],
                "setup": setup,
                "mode": mode,
                "band_center_hz_from_healthy_fdd": center,
                "available_reference_channels": ";".join(available_references),
            }
            for label in available_references:
                psd = reference_psd[label]
                peak_local = np.flatnonzero(band)[int(np.argmax(psd[band]))]
                row[f"{label}_peak_hz"] = float(reference_frequency[peak_local])
                row[f"{label}_band_energy"] = float(
                    numpy_trapezoid(psd[band], reference_frequency[band])
                )
            for numerator, denominator in (("R3V", "R2V"), ("R2V", "R1V")):
                if numerator in reference_signals and denominator in reference_signals:
                    _, pxx = signal.welch(
                        reference_signals[denominator],
                        fs=CONFIG["sampling_rate_hz"],
                        window="hann",
                        nperseg=CONFIG["fdd_nperseg"],
                        noverlap=CONFIG["fdd_noverlap"],
                        detrend="constant",
                        scaling="density",
                    )
                    _, pxy = signal.csd(
                        reference_signals[denominator],
                        reference_signals[numerator],
                        fs=CONFIG["sampling_rate_hz"],
                        window="hann",
                        nperseg=CONFIG["fdd_nperseg"],
                        noverlap=CONFIG["fdd_noverlap"],
                        detrend="constant",
                        scaling="density",
                    )
                    h1 = pxy / np.maximum(pxx, np.finfo(float).tiny)
                    row[f"log_h1_transmissibility_{numerator}_over_{denominator}"] = float(
                        np.mean(np.log(np.maximum(np.abs(h1[band]), np.finfo(float).tiny)))
                    )
            reference_feature_rows.append(row)
        del data

reference_feature_audit = pd.DataFrame(reference_feature_rows)
reference_feature_audit.to_csv(
    TABLE_ROOT / "z24_fixed_reference_spectral_transmissibility_audit.csv",
    index=False,
)
display(reference_feature_audit.head(20))
print(
    "Auxiliary fixed-reference feature rows:",
    len(reference_feature_audit),
    "— not used to select retained modes or tune the Digital Twin.",
)


,scenario,delta_pier_mm,setup,mode,band_center_hz_from_healthy_fdd,available_reference_channels,R1V_peak_hz,R1V_band_energy,R2L_peak_hz,R2L_band_energy,R2T_peak_hz,R2T_band_energy,R2V_peak_hz,R2V_band_energy,R3V_peak_hz,R3V_band_energy,log_h1_transmissibility_R3V_over_R2V,log_h1_transmissibility_R2V_over_R1V
0,01,0.0,1,1,3.906250,R1V;R2L;R2T;R2V;R3V,3.906250,1.857894e-10,4.150391,4.031639e-14,3.906250,6.913181e-12,3.906250,9.584419e-09,3.906250,4.645915e-09,-0.363987,1.970394
1,01,0.0,1,3,9.960938,R1V;R2L;R2T;R2V;R3V,9.985352,1.769442e-11,9.985352,7.074758e-13,9.765625,5.406744e-13,10.229492,1.601595e-11,9.985352,8.973150e-11,0.481049,-0.651592
2,01,0.0,1,4,10.522461,R1V;R2L;R2T;R2V;R3V,10.278320,1.266041e-11,10.253906,4.667586e-13,10.278320,4.102383e-13,10.473633,2.866264e-11,10.253906,1.419481e-11,-1.123723,-0.415392
3,01,0.0,2,1,3.906250,R1V;R2L;R2T;R2V;R3V,3.930664,4.999201e-11,4.150391,3.358186e-14,3.930664,2.841657e-12,3.930664,2.620307e-09,3.930664,1.277476e-09,-0.361865,1.954464
4,01,0.0,2,3,9.960938,R1V;R2L;R2T;R2V;R3V,9.838867,5.338371e-12,9.960938,3.025479e-13,9.692383,2.211810e-13,10.180664,6.896890e-12,9.863281,2.561164e-11,0.335447,-0.860951
5,01,0.0,2,4,10.522461,R1V;R2L;R2T;R2V;R3V,10.400391,5.940722e-12,10.400391,2.362859e-13,10.498047,2.377910e-13,10.595703,1.608499e-11,10.229492,4.984369e-12,-1.432922,-0.154238
6,01,0.0,3,1,3.906250,R1V;R2L;R2T;R2V;R3V,3.906250,1.084691e-10,4.150391,2.742383e-14,3.906250,5.440637e-12,3.906250,5.791287e-09,3.906250,2.826164e-09,-0.359402,1.984377
7,01,0.0,3,3,9.960938,R1V;R2L;R2T;R2V;R3V,10.009766,1.097205e-12,9.814453,4.841079e-14,10.229492,3.866778e-14,10.205078,1.138602e-12,10.009766,5.796462e-12,0.503834,-0.685239
8,01,0.0,3,4,10.522461,R1V;R2L;R2T;R2V;R3V,10.644531,1.032255e-12,10.278320,4.427542e-14,10.351562,5.942350e-14,10.473633,2.766633e-12,10.229492,1.190640e-12,-1.554738,-0.222457
9,01,0.0,4,1,3.906250,R1V;R2L;R2T;R2V;R3V,3.906250,7.546573e-10,4.150391,2.479350e-13,3.906250,1.980408e-11,3.906250,3.957173e-08,3.906250,1.912654e-08,-0.366084,1.976366


Auxiliary fixed-reference feature rows: 135 — not used to select retained modes or tune the Digital Twin.


## 8. Synthetic physics validation

Gate B begins with two independent numerical checks before any neural solver is trusted. The SDOF frequency is compared with the generalized eigenvalue result. A conventional two-node Euler–Bernoulli finite-element model with consistent mass is then compared with the analytical simply supported spectrum

\[
\omega_n=\frac{n^2\pi^2}{L^2}\sqrt{\frac{EI}{\rho A}}.
\]

The finite-element implementation is intentionally independent of the PINN and also verifies the expected log–log slope (f\propto EI^{1/2}).

In [14]:
from scipy import linalg
import time

def sdof_frequency_hz(mass: float, stiffness: float) -> float:
    if mass <= 0.0 or stiffness <= 0.0:
        raise ValueError("SDOF mass and stiffness must be positive")
    return np.sqrt(stiffness / mass) / (2.0 * np.pi)

def euler_bernoulli_fe_frequencies(
    n_elements: int, length: float = 1.0, flexural_rigidity: float = 1.0,
    mass_per_length: float = 1.0, n_modes: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """Consistent-mass two-node beam FE frequencies; end vertical DOFs are fixed."""
    if n_elements < n_modes or min(length, flexural_rigidity, mass_per_length) <= 0.0:
        raise ValueError("Invalid Euler-Bernoulli FE parameters")
    element_length = length / n_elements
    k_e = (flexural_rigidity / element_length**3) * np.array([
        [12, 6*element_length, -12, 6*element_length],
        [6*element_length, 4*element_length**2, -6*element_length, 2*element_length**2],
        [-12, -6*element_length, 12, -6*element_length],
        [6*element_length, 2*element_length**2, -6*element_length, 4*element_length**2],
    ], dtype=np.float64)
    m_e = (mass_per_length * element_length / 420.0) * np.array([
        [156, 22*element_length, 54, -13*element_length],
        [22*element_length, 4*element_length**2, 13*element_length, -3*element_length**2],
        [54, 13*element_length, 156, -22*element_length],
        [-13*element_length, -3*element_length**2, -22*element_length, 4*element_length**2],
    ], dtype=np.float64)
    n_dof = 2 * (n_elements + 1)
    stiffness = np.zeros((n_dof, n_dof), dtype=np.float64)
    mass = np.zeros_like(stiffness)
    for element in range(n_elements):
        dof = np.array([2*element, 2*element+1, 2*element+2, 2*element+3])
        stiffness[np.ix_(dof, dof)] += k_e
        mass[np.ix_(dof, dof)] += m_e
    constrained = np.array([0, 2*n_elements])
    free = np.setdiff1d(np.arange(n_dof), constrained)
    eigenvalues, eigenvectors_free = linalg.eigh(
        stiffness[np.ix_(free, free)], mass[np.ix_(free, free)],
        subset_by_index=[0, n_modes-1], check_finite=True,
    )
    if np.any(eigenvalues <= 0.0):
        raise RuntimeError("Non-positive FE eigenvalue")
    eigenvectors = np.zeros((n_dof, n_modes), dtype=np.float64)
    eigenvectors[free, :] = eigenvectors_free
    return np.sqrt(eigenvalues), eigenvectors

sdof_mass, sdof_stiffness = 2.0, 800.0
sdof_analytic_hz = sdof_frequency_hz(sdof_mass, sdof_stiffness)
sdof_eigen_hz = np.sqrt(linalg.eigh(np.array([[sdof_stiffness]]),
                                    np.array([[sdof_mass]]), eigvals_only=True)[0]) / (2*np.pi)
sdof_relative_error = abs(sdof_eigen_hz - sdof_analytic_hz) / sdof_analytic_hz

beam_length = beam_ei = beam_rho_a = 1.0
beam_modes = np.arange(1, 5)
beam_analytic_omega = (beam_modes*np.pi/beam_length)**2 * np.sqrt(beam_ei/beam_rho_a)
beam_fe_omega, _ = euler_bernoulli_fe_frequencies(
    CONFIG["beam_fe_elements"], beam_length, beam_ei, beam_rho_a, len(beam_modes))
beam_fe_relative_errors = np.abs(beam_fe_omega-beam_analytic_omega)/beam_analytic_omega

element_counts = np.array([8, 16, 32, 64, CONFIG["beam_fe_elements"]])
fe_convergence_error = []
for n_elements in element_counts:
    omega_check, _ = euler_bernoulli_fe_frequencies(int(n_elements), n_modes=4)
    fe_convergence_error.append(np.max(np.abs(omega_check-beam_analytic_omega)/beam_analytic_omega))
fe_convergence_error = np.asarray(fe_convergence_error)

ei_sweep = np.array(CONFIG["gate_b_alpha_values"], dtype=np.float64)
fe_sweep_omega = np.array([
    euler_bernoulli_fe_frequencies(CONFIG["beam_fe_elements"], flexural_rigidity=alpha,
                                  n_modes=1)[0][0] for alpha in ei_sweep
])
ei_log_slope, ei_log_intercept = np.polyfit(np.log(ei_sweep), np.log(fe_sweep_omega), 1)

synthetic_validation = pd.DataFrame([
    {"check": "SDOF generalized eigenvalue", "value": sdof_eigen_hz,
     "reference": sdof_analytic_hz, "metric": sdof_relative_error, "tolerance": 1e-12,
     "pass": sdof_relative_error <= 1e-12},
    {"check": "Beam FE first-four maximum relative error", "value": beam_fe_relative_errors.max(),
     "reference": 0.0, "metric": beam_fe_relative_errors.max(), "tolerance": 1e-3,
     "pass": beam_fe_relative_errors.max() <= 1e-3},
    {"check": "FE log(f)-log(EI) slope", "value": ei_log_slope,
     "reference": 0.5, "metric": abs(ei_log_slope-0.5), "tolerance": 5e-3,
     "pass": abs(ei_log_slope-0.5) <= 5e-3},
])
display(synthetic_validation)
synthetic_validation.to_csv(TABLE_ROOT / "gate_b_synthetic_physics_checks.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2))
axes[0].loglog(element_counts, fe_convergence_error, "o-", color="#264653", label="maximum, modes 1–4")
axes[0].axhline(1e-3, color="#e76f51", linestyle="--", label="Gate B tolerance")
axes[0].set_xlabel("Euler–Bernoulli finite elements")
axes[0].set_ylabel("Maximum relative frequency error")
axes[0].set_title("Independent FE convergence")
axes[0].legend()
axes[1].loglog(ei_sweep, fe_sweep_omega/(2*np.pi), "o", color="#2a9d8f", label="FE mode 1")
ei_dense = np.linspace(ei_sweep.min(), ei_sweep.max(), 200)
axes[1].plot(ei_dense, np.exp(ei_log_intercept)*ei_dense**ei_log_slope/(2*np.pi),
             color="#264653", label=f"fit slope = {ei_log_slope:.6f}")
axes[1].set_xlabel(r"Stiffness retention $\alpha_{EI}$")
axes[1].set_ylabel("Fundamental frequency [dimensionless Hz]")
axes[1].set_title(r"Required $f\propto\sqrt{EI}$ scaling")
axes[1].legend()
fig.suptitle("Figure 5 — Synthetic structural-physics verification", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_05_synthetic_physics.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

if not bool(synthetic_validation["pass"].all()):
    raise RuntimeError("Gate B stopped: analytical/FE synthetic physics validation failed")

,check,value,reference,metric,tolerance,pass
0,SDOF generalized eigenvalue,3.183099e+00,3.183099,2.790295e-16,1.000000e-12,True
1,Beam FE first-four maximum relative error,4.225955e-07,0.000000,4.225955e-07,1.000000e-03,True
2,FE log(f)-log(EI) slope,5.000000e-01,0.500000,3.530950e-08,5.000000e-03,True


## 9. Parametric bridge physics model

The Gate B pilot is intentionally **not yet a Z24 bridge model**. It is a dimensionless, uniform, simply supported Euler–Bernoulli equivalent beam used to validate the eigen-PINN implementation:

\[
\alpha_{EI}\,\phi''''(x)-\omega^2\phi(x)=0,\quad x\in[0,1],
\qquad \phi(0)=\phi(1)=\phi''(0)=\phi''(1)=0.
\]

Here $\rho A=L=EI_0=1$ and $\alpha_{EI}>0$ is synthetic effective stiffness retention. Simply supported conditions are an explicit pilot assumption, not an inferred Z24 boundary condition. No mapping from pier settlement $\delta_{pier}$ to $\alpha_{EI}$ is fitted at this gate. Only the nodeless fundamental mode is solved; therefore the multi-mode orthogonality loss is not applicable and is recorded as zero.

In [15]:
beam_model_parameters = pd.DataFrame([
    {"symbol": "L", "value": 1.0, "role": "dimensionless span length", "status": "synthetic pilot"},
    {"symbol": "rhoA", "value": 1.0, "role": "dimensionless mass per length", "status": "synthetic pilot"},
    {"symbol": "EI0", "value": 1.0, "role": "reference flexural rigidity", "status": "synthetic pilot"},
    {"symbol": "alpha_EI", "value": CONFIG["gate_b_alpha_values"],
     "role": "effective stiffness retention", "status": "independent of delta_pier at Gate B"},
])
display(beam_model_parameters)
beam_model_parameters.to_csv(TABLE_ROOT / "gate_b_beam_model_parameters.csv", index=False)

gate_b_mode = CONFIG["gate_b_mode"]
if gate_b_mode not in {"smoke", "full"}:
    raise ValueError("GPT_PINN_GATE_B_MODE must be 'smoke' or 'full'")
if gate_b_mode == "smoke":
    gate_b_run_specs = [(1.0, CONFIG["gate_b_seeds"][0])]
    gate_b_adam_epochs = CONFIG["gate_b_smoke_adam_epochs"]
    gate_b_lbfgs_iterations = CONFIG["gate_b_smoke_lbfgs_iterations"]
else:
    gate_b_run_specs = [(1.0, seed) for seed in CONFIG["gate_b_seeds"]]
    gate_b_run_specs += [(float(alpha), CONFIG["gate_b_seeds"][0])
                         for alpha in CONFIG["gate_b_alpha_values"] if not np.isclose(alpha, 1.0)]
    gate_b_adam_epochs = CONFIG["gate_b_full_adam_epochs"]
    gate_b_lbfgs_iterations = CONFIG["gate_b_full_lbfgs_iterations"]
print("Gate B execution mode:", gate_b_mode)
print("Independent Full-PINN runs:", gate_b_run_specs)

,symbol,value,role,status
0,L,1.0,dimensionless span length,synthetic pilot
1,rhoA,1.0,dimensionless mass per length,synthetic pilot
2,EI0,1.0,reference flexural rigidity,synthetic pilot
3,alpha_EI,"[0.6, 0.8, 1.0, 1.2, 1.4]",effective stiffness retention,independent of delta_pier at Gate B


Gate B execution mode: full
Independent Full-PINN runs: [(1.0, 101), (1.0, 202), (1.0, 303), (1.0, 404), (1.0, 505), (0.6, 101), (0.8, 101), (1.2, 101), (1.4, 101)]


## 10. Full PINN solver

`FullPINN` uses PyTorch automatic differentiation in `float64`, because a fourth spatial derivative is sensitive to roundoff. The hard factor (x(1-x)) enforces displacement boundary conditions, while a positive `softplus` amplitude restricts this pilot to the nodeless fundamental mode. Mass normalization prevents the zero solution. The remaining moment boundary conditions, strong-form PDE, and Rayleigh-quotient consistency are optimized with Adam followed by L-BFGS. Each run starts independently with a seed-dependent frequency and saves the model and both optimizer states.

In [16]:
require_compute_node("Gate B Full-PINN training")
torch.set_default_dtype(torch.float64)
gate_b_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def inverse_softplus(value: float) -> float:
    value = float(value)
    if value <= 0.0 or not np.isfinite(value):
        raise ValueError("inverse_softplus requires a finite positive value")
    return value if value > 20.0 else float(np.log(np.expm1(value)))

class FullPINN(torch.nn.Module):
    def __init__(self, seed: int, initial_omega: float):
        super().__init__()
        torch.manual_seed(seed)
        widths = CONFIG["network_architecture"]
        layers = []
        for input_width, output_width in zip(widths[:-2], widths[1:-1]):
            linear = torch.nn.Linear(input_width, output_width)
            torch.nn.init.xavier_normal_(linear.weight)
            torch.nn.init.zeros_(linear.bias)
            layers.extend([linear, torch.nn.Tanh()])
        final = torch.nn.Linear(widths[-2], widths[-1])
        torch.nn.init.xavier_normal_(final.weight, gain=0.25)
        torch.nn.init.constant_(final.bias, 1.0)
        layers.append(final)
        self.network = torch.nn.Sequential(*layers)
        self.raw_omega = torch.nn.Parameter(torch.tensor(inverse_softplus(initial_omega)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positive_amplitude = torch.nn.functional.softplus(self.network(x)) + 1e-8
        return x * (1.0-x) * positive_amplitude

    @property
    def omega(self) -> torch.Tensor:
        return torch.nn.functional.softplus(self.raw_omega) + 1e-8

def differentiate(values: torch.Tensor, coordinates: torch.Tensor, order: int) -> torch.Tensor:
    derivative = values
    for _ in range(order):
        derivative = torch.autograd.grad(
            derivative, coordinates, grad_outputs=torch.ones_like(derivative),
            create_graph=True, retain_graph=True,
        )[0]
    return derivative

def full_pinn_loss(model: FullPINN, alpha_ei: float, x_col: torch.Tensor) -> tuple[torch.Tensor, dict]:
    phi = model(x_col)
    phi_2 = differentiate(phi, x_col, 2)
    phi_4 = differentiate(phi_2, x_col, 2)
    residual = alpha_ei*phi_4-model.omega.square()*phi
    x_boundary = torch.tensor([[0.0], [1.0]], device=x_col.device, requires_grad=True)
    phi_boundary = model(x_boundary)
    phi_2_boundary = differentiate(phi_boundary, x_boundary, 2)
    modal_mass = torch.trapezoid(phi.square().squeeze(-1), x_col.squeeze(-1))
    bending_energy = torch.trapezoid(phi_2.square().squeeze(-1), x_col.squeeze(-1))
    rayleigh_omega_sq = alpha_ei*bending_energy/modal_mass.clamp_min(1e-12)
    components = {
        "pde": torch.mean((residual/(np.pi**4*alpha_ei))**2),
        "bc": torch.mean((phi_2_boundary/np.pi**2)**2),
        "norm": (modal_mass-1.0).square(),
        "rayleigh": ((model.omega.square()-rayleigh_omega_sq)/(np.pi**4*alpha_ei)).square(),
        "orth": torch.zeros((), dtype=x_col.dtype, device=x_col.device),
    }
    total = sum(CONFIG["loss_weights"][name]*components[name]
                for name in ("pde", "bc", "norm", "rayleigh"))
    return total, components

def train_full_pinn(alpha_ei: float, seed: int, adam_epochs: int, lbfgs_iterations: int) -> dict:
    run_seed = int(CONFIG["seed"] + seed + round(1000*alpha_ei))
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    initial_omega = float(np.random.uniform(0.65, 1.35) * np.pi**2*np.sqrt(alpha_ei))
    model = FullPINN(run_seed, initial_omega).to(gate_b_device)
    checkpoint_path = CHECKPOINT_ROOT / f"full_pinn_alpha_{alpha_ei:.3f}_seed_{seed}.pt"
    x_col = torch.linspace(0.0, 1.0, CONFIG["collocation_points"], device=gate_b_device).reshape(-1, 1)
    x_col.requires_grad_(True)
    adam = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(adam, T_max=max(1, adam_epochs), eta_min=2e-5)
    start_epoch = 0
    history = []
    if CONFIG["gate_b_resume"]:
        def gate_b_checkpoint_compatible(saved):
            return bool(
                isinstance(saved, dict)
                and saved.get("status") in {"complete", "adam_checkpoint"}
                and saved.get("seed") == seed
                and np.isclose(saved.get("alpha_ei", np.nan), alpha_ei)
                and (
                    saved.get("status") != "complete"
                    or saved.get("adam_epochs") == adam_epochs
                )
            )
        saved = load_checkpoint_with_legacy_fallback(
            checkpoint_path,
            map_location=gate_b_device,
            stage=f"Gate B alpha={alpha_ei:.3f} seed={seed}",
            validator=gate_b_checkpoint_compatible,
        )
        if saved is not None and saved.get("status") == "complete":
            model.load_state_dict(saved["model_state_dict"])
            return {"model": model, "alpha_ei": alpha_ei, "seed": seed,
                    "initial_omega": saved["initial_omega"],
                    "runtime_seconds": saved["runtime_seconds"], "history": saved["history"],
                    "closure_calls": saved["lbfgs_closure_calls"],
                    "checkpoint_path": checkpoint_path, "loaded_checkpoint": True}
        if (saved is not None and saved.get("status") == "adam_checkpoint"
                and saved.get("adam_epoch", 0) < adam_epochs):
            model.load_state_dict(saved["model_state_dict"])
            adam.load_state_dict(saved["adam_optimizer_state_dict"])
            scheduler.load_state_dict(saved["scheduler_state_dict"])
            start_epoch = int(saved["adam_epoch"])
            history = list(saved.get("history", []))
    start = time.perf_counter()
    for epoch in range(start_epoch, adam_epochs):
        adam.zero_grad(set_to_none=True)
        total, components = full_pinn_loss(model, alpha_ei, x_col)
        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite Adam loss for alpha={alpha_ei}, seed={seed}")
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        adam.step()
        scheduler.step()
        if epoch == start_epoch or (epoch+1) % 25 == 0 or epoch+1 == adam_epochs:
            history.append({"stage": "Adam", "iteration": epoch+1, "total": float(total.detach().cpu()),
                            "omega": float(model.omega.detach().cpu()),
                            **{name: float(value.detach().cpu()) for name, value in components.items()}})
        if (epoch+1) % 500 == 0 and epoch+1 < adam_epochs:
            safe_torch_save({
                "status": "adam_checkpoint", "alpha_ei": alpha_ei, "seed": seed,
                "adam_epoch": epoch+1, "model_state_dict": model.state_dict(),
                "adam_optimizer_state_dict": adam.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(), "history": history,
            }, checkpoint_path,
                stage=f"Gate B Adam checkpoint alpha={alpha_ei:.3f} seed={seed}",
                retries=CONFIG["checkpoint_save_retries"],
                minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2)

    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=0.8, max_iter=lbfgs_iterations, max_eval=int(1.25*lbfgs_iterations),
        tolerance_grad=1e-11, tolerance_change=1e-13, history_size=80,
        line_search_fn="strong_wolfe",
    )
    closure_calls = 0
    def closure():
        nonlocal closure_calls
        lbfgs.zero_grad(set_to_none=True)
        total, components = full_pinn_loss(model, alpha_ei, x_col)
        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite L-BFGS loss for alpha={alpha_ei}, seed={seed}")
        total.backward()
        closure_calls += 1
        if closure_calls == 1 or closure_calls % 10 == 0:
            history.append({"stage": "L-BFGS", "iteration": closure_calls, "total": float(total.detach().cpu()),
                            "omega": float(model.omega.detach().cpu()),
                            **{name: float(value.detach().cpu()) for name, value in components.items()}})
        return total
    lbfgs.step(closure)
    runtime_seconds = time.perf_counter()-start
    final_total, final_components = full_pinn_loss(model, alpha_ei, x_col)
    safe_torch_save({
        "status": "complete", "alpha_ei": alpha_ei, "seed": seed,
        "initial_omega": initial_omega, "omega": float(model.omega.detach().cpu()),
        "model_state_dict": model.state_dict(),
        # Complete checkpoints omit optimizer histories; only interrupted Adam
        # checkpoints retain optimizer state for exact continuation.
        "adam_epochs": adam_epochs, "lbfgs_closure_calls": closure_calls,
        "runtime_seconds": runtime_seconds, "history": history,
        "final_loss": float(final_total.detach().cpu()),
        "final_components": {name: float(value.detach().cpu()) for name, value in final_components.items()},
        "torch_version": torch.__version__, "device": str(gate_b_device),
        "checkpoint_format": "V0.6_FINAL_atomic_cpu_stream",
    }, checkpoint_path,
        stage=f"Gate B complete alpha={alpha_ei:.3f} seed={seed}",
        retries=CONFIG["checkpoint_save_retries"],
        minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2)
    return {"model": model, "alpha_ei": alpha_ei, "seed": seed, "initial_omega": initial_omega,
            "runtime_seconds": runtime_seconds, "history": history,
            "closure_calls": closure_calls, "checkpoint_path": checkpoint_path}

full_pinn_runs = []
for run_number, (alpha_ei, seed) in enumerate(gate_b_run_specs, start=1):
    print(f"Full PINN {run_number}/{len(gate_b_run_specs)}: alpha_EI={alpha_ei:.3f}, seed={seed}")
    full_pinn_runs.append(train_full_pinn(alpha_ei, seed, gate_b_adam_epochs, gate_b_lbfgs_iterations))
    print(f"  omega={full_pinn_runs[-1]['model'].omega.item():.8f}, runtime={full_pinn_runs[-1]['runtime_seconds']:.2f} s")

Full PINN 1/9: alpha_EI=1.000, seed=101
  omega=9.86960402, runtime=480.40 s
Full PINN 2/9: alpha_EI=1.000, seed=202
  omega=9.86960102, runtime=182.43 s
Full PINN 3/9: alpha_EI=1.000, seed=303
  omega=9.86960275, runtime=198.07 s
Full PINN 4/9: alpha_EI=1.000, seed=404
  omega=9.86958712, runtime=260.19 s
Full PINN 5/9: alpha_EI=1.000, seed=505
  omega=9.86961404, runtime=162.78 s
Full PINN 6/9: alpha_EI=0.600, seed=101
  omega=7.64492926, runtime=128.60 s
Full PINN 7/9: alpha_EI=0.800, seed=101
  omega=8.82765420, runtime=199.27 s
Full PINN 8/9: alpha_EI=1.200, seed=101
  omega=10.81168632, runtime=102.91 s
Full PINN 9/9: alpha_EI=1.400, seed=101
  omega=11.67790800, runtime=100.85 s


## 11. Full PINN verification

Verification uses a denser grid that was not used as the training collocation set. Reported controls are frequency error, modal assurance criterion (MAC), mass normalization, both boundary-condition families, relative strong-form residual, nontrivial amplitude, finiteness, and seed stability. The analytical mode is used only for post-training verification. A smoke run can detect implementation failures but cannot issue `GO`; the full run requires five independent seeds at (alpha_{EI}=1).

In [17]:
def verify_full_pinn(run: dict) -> tuple[dict, np.ndarray, np.ndarray, np.ndarray]:
    model = run["model"]
    alpha_ei = run["alpha_ei"]
    x = torch.linspace(0.0, 1.0, CONFIG["validation_points"], device=gate_b_device).reshape(-1, 1)
    x.requires_grad_(True)
    phi = model(x)
    phi_2 = differentiate(phi, x, 2)
    phi_4 = differentiate(phi_2, x, 2)
    residual = alpha_ei*phi_4-model.omega.square()*phi
    x_np = x.detach().cpu().numpy().ravel()
    phi_np = phi.detach().cpu().numpy().ravel()
    phi_2_np = phi_2.detach().cpu().numpy().ravel()
    residual_np = residual.detach().cpu().numpy().ravel()
    analytic_phi = np.sqrt(2.0)*np.sin(np.pi*x_np)
    analytic_omega = np.pi**2*np.sqrt(alpha_ei)
    omega = float(model.omega.detach().cpu())
    modal_mass = float(numpy_trapezoid(phi_np**2, x_np))
    mac = float((numpy_trapezoid(phi_np*analytic_phi, x_np)**2) /
                (numpy_trapezoid(phi_np**2, x_np)*numpy_trapezoid(analytic_phi**2, x_np)))
    residual_scale = np.sqrt(np.mean((alpha_ei*phi_4.detach().cpu().numpy().ravel())**2))
    residual_scale += np.sqrt(np.mean((omega**2*phi_np)**2)) + 1e-15
    metrics = {
        "alpha_ei": alpha_ei,
        "seed": run["seed"],
        "initial_omega": run["initial_omega"],
        "omega": omega,
        "analytic_omega": analytic_omega,
        "frequency_relative_error": abs(omega-analytic_omega)/analytic_omega,
        "mac": mac,
        "modal_mass": modal_mass,
        "normalization_absolute_error": abs(modal_mass-1.0),
        "displacement_bc_max": float(max(abs(phi_np[0]), abs(phi_np[-1]))),
        "moment_bc_scaled_max": float(max(abs(phi_2_np[0]), abs(phi_2_np[-1]))/np.pi**2),
        "relative_residual_rms": float(np.sqrt(np.mean(residual_np**2))/residual_scale),
        "maximum_amplitude": float(np.max(np.abs(phi_np))),
        "nontrivial": bool(np.max(np.abs(phi_np)) >= 0.5),
        "finite": bool(np.isfinite(np.r_[phi_np, phi_2_np, residual_np, omega]).all()),
        "runtime_seconds": run["runtime_seconds"],
        "lbfgs_closure_calls": run["closure_calls"],
        "device": str(gate_b_device),
        "gpu_peak_memory_mib": (torch.cuda.max_memory_allocated()/2**20 if torch.cuda.is_available() else 0.0),
        "checkpoint": str(run["checkpoint_path"]),
    }
    return metrics, x_np, phi_np, residual_np

verification_rows = []
verification_curves = {}
for run in full_pinn_runs:
    metrics, x_values, phi_values, residual_values = verify_full_pinn(run)
    verification_rows.append(metrics)
    verification_curves[(run["alpha_ei"], run["seed"])] = (x_values, phi_values, residual_values)
full_pinn_verification = pd.DataFrame(verification_rows).sort_values(["alpha_ei", "seed"]).reset_index(drop=True)
display(full_pinn_verification)
full_pinn_verification.to_csv(TABLE_ROOT / f"gate_b_full_pinn_verification_{gate_b_mode}.csv", index=False)

alpha_one = full_pinn_verification[np.isclose(full_pinn_verification["alpha_ei"], 1.0)]
seed_omega_cv = float(alpha_one["omega"].std(ddof=1)/alpha_one["omega"].mean()) if len(alpha_one) >= 2 else np.nan
all_run_checks = {
    "finite": bool(full_pinn_verification["finite"].all()),
    "nontrivial": bool(full_pinn_verification["nontrivial"].all()),
    "frequency_relative_error_le_1pct": bool((full_pinn_verification["frequency_relative_error"] <= 0.01).all()),
    "mac_ge_0p995": bool((full_pinn_verification["mac"] >= 0.995).all()),
    "normalization_error_le_0p02": bool((full_pinn_verification["normalization_absolute_error"] <= 0.02).all()),
    "displacement_bc_le_1e-12": bool((full_pinn_verification["displacement_bc_max"] <= 1e-12).all()),
    "scaled_moment_bc_le_0p02": bool((full_pinn_verification["moment_bc_scaled_max"] <= 0.02).all()),
    "relative_residual_rms_le_0p05": bool((full_pinn_verification["relative_residual_rms"] <= 0.05).all()),
}
full_multi_seed = gate_b_mode == "full" and len(alpha_one) == len(CONFIG["gate_b_seeds"])
seed_stability_pass = bool(full_multi_seed and np.isfinite(seed_omega_cv) and seed_omega_cv <= 0.005)
synthetic_checks_pass = bool(synthetic_validation["pass"].all())
if gate_b_mode == "smoke":
    gate_b_decision = "SMOKE_PASS_REQUIRES_FULL_RUN" if synthetic_checks_pass and all(all_run_checks.values()) else "NO_GO_FIX_FULL_PINN"
else:
    gate_b_decision = "GO_TO_GPT_PINN_PILOT" if (synthetic_checks_pass and all(all_run_checks.values()) and seed_stability_pass) else "NO_GO_FIX_FULL_PINN"

gate_b_diagnostic = {
    "decision": gate_b_decision,
    "execution_mode": gate_b_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "model_scope": "dimensionless uniform simply supported Euler-Bernoulli equivalent beam; fundamental mode only",
    "z24_calibration_performed": False,
    "delta_pier_equated_to_alpha_ei": False,
    "synthetic_checks_pass": synthetic_checks_pass,
    "full_pinn_checks": all_run_checks,
    "full_multi_seed_requirement_met": full_multi_seed,
    "seed_omega_cv": seed_omega_cv if np.isfinite(seed_omega_cv) else None,
    "seed_stability_threshold": 0.005,
    "run_count": len(full_pinn_verification),
    "maximum_frequency_relative_error": float(full_pinn_verification["frequency_relative_error"].max()),
    "minimum_mac": float(full_pinn_verification["mac"].min()),
    "maximum_relative_residual_rms": float(full_pinn_verification["relative_residual_rms"].max()),
    "maximum_normalization_absolute_error": float(full_pinn_verification["normalization_absolute_error"].max()),
    "total_training_runtime_seconds": float(full_pinn_verification["runtime_seconds"].sum()),
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "gate_b_diagnostic.json").write_text(
    json_dumps_safe(gate_b_diagnostic, indent=2, sort_keys=True) + "\n", encoding="utf-8")
display(pd.DataFrame([gate_b_diagnostic]))

representative = full_pinn_runs[0]
history_frame = pd.DataFrame(representative["history"])
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))
axes[0].semilogy(np.arange(len(history_frame)), history_frame["total"], color="#264653")
axes[0].set_xlabel("Recorded optimizer state")
axes[0].set_ylabel("Weighted loss")
axes[0].set_title("Representative convergence")
for row in full_pinn_verification.itertuples():
    x_curve, phi_curve, _ = verification_curves[(row.alpha_ei, row.seed)]
    if gate_b_mode == "smoke" or row.seed == CONFIG["gate_b_seeds"][0]:
        axes[1].plot(x_curve, phi_curve, label=rf"$\alpha={row.alpha_ei:.1f}$")
axes[1].plot(x_curve, np.sqrt(2)*np.sin(np.pi*x_curve), "k--", linewidth=1.2, label="analytical")
axes[1].set_xlabel("Dimensionless coordinate x")
axes[1].set_ylabel(r"Mass-normalized $\phi_1(x)$")
axes[1].set_title("Independent-grid mode shapes")
axes[1].legend(ncol=2, fontsize=8)
axes[2].plot(full_pinn_verification["analytic_omega"], full_pinn_verification["omega"], "o", color="#2a9d8f")
limits = [full_pinn_verification[["analytic_omega", "omega"]].min().min()*0.98,
          full_pinn_verification[["analytic_omega", "omega"]].max().max()*1.02]
axes[2].plot(limits, limits, "k--", label="identity")
axes[2].set_xlim(limits); axes[2].set_ylim(limits)
axes[2].set_xlabel("Analytical angular frequency")
axes[2].set_ylabel("Full-PINN angular frequency")
axes[2].set_title("Frequency verification")
axes[2].legend()
fig.suptitle(f"Figure 6 — Full-PINN Gate B verification: {gate_b_decision}", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_06_full_pinn_gate_b_{gate_b_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_b_diagnostic, indent=2))
if gate_b_decision == "NO_GO_FIX_FULL_PINN":
    raise RuntimeError("Gate B NO-GO: the Full-PINN must be corrected before GPT-PINN construction")

,alpha_ei,seed,initial_omega,omega,analytic_omega,frequency_relative_error,mac,modal_mass,normalization_absolute_error,displacement_bc_max,moment_bc_scaled_max,relative_residual_rms,maximum_amplitude,nontrivial,finite,runtime_seconds,lbfgs_closure_calls,device,gpu_peak_memory_mib,checkpoint
0,0.6,101,6.471951,7.644929,7.644963,4.374163e-06,1.0,0.999999,9.699823e-07,0.0,4.924361e-06,0.000094,1.414213,True,True,128.602275,231,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
1,0.8,101,8.548468,8.827654,8.827643,1.320745e-06,1.0,1.000001,1.299513e-06,0.0,3.389012e-06,0.000057,1.414215,True,True,199.268211,237,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
2,1.0,101,8.841979,9.869604,9.869604,3.812103e-08,1.0,1.000000,3.394026e-08,0.0,4.012314e-07,0.000132,1.414214,True,True,480.403608,249,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
3,1.0,202,11.302245,9.869601,9.869604,3.423587e-07,1.0,0.999999,5.011664e-07,0.0,6.277314e-07,0.000111,1.414213,True,True,182.427883,232,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
4,1.0,303,12.731824,9.869603,9.869604,1.669143e-07,1.0,1.000000,4.259199e-08,0.0,1.047271e-06,0.000128,1.414214,True,True,198.066666,236,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
5,1.0,404,12.069209,9.869587,9.869604,1.751292e-06,1.0,1.000000,3.644218e-07,0.0,6.232129e-06,0.000122,1.414213,True,True,260.190990,229,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
6,1.0,505,6.452432,9.869614,9.869604,9.768590e-07,1.0,0.999996,3.983099e-06,0.0,5.202658e-06,0.000114,1.414211,True,True,162.779913,240,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
7,1.2,101,9.561971,10.811686,10.811610,7.065444e-06,1.0,1.000004,3.818048e-06,0.0,2.361182e-06,0.000225,1.414216,True,True,102.911378,231,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...
8,1.4,101,9.432466,11.677908,11.677873,2.961611e-06,1.0,0.999998,2.031524e-06,0.0,2.171933e-05,0.000130,1.414212,True,True,100.852723,235,cpu,0.0,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...


,decision,execution_mode,execution_boundary,model_scope,z24_calibration_performed,delta_pier_equated_to_alpha_ei,synthetic_checks_pass,full_pinn_checks,full_multi_seed_requirement_met,seed_omega_cv,seed_stability_threshold,run_count,maximum_frequency_relative_error,minimum_mac,maximum_relative_residual_rms,maximum_normalization_absolute_error,total_training_runtime_seconds,raw_z24_signature_unchanged,timestamp_utc
0,GO_TO_GPT_PINN_PILOT,full,STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_F...,dimensionless uniform simply supported Euler-B...,False,False,True,"{'finite': True, 'nontrivial': True, 'frequenc...",True,9.770310e-07,0.005,9,0.000007,1.0,0.000225,0.000004,1815.503646,True,2026-09-06T01:46:34.730445+00:00


{
  "decision": "GO_TO_GPT_PINN_PILOT",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "model_scope": "dimensionless uniform simply supported Euler-Bernoulli equivalent beam; fundamental mode only",
  "z24_calibration_performed": false,
  "delta_pier_equated_to_alpha_ei": false,
  "synthetic_checks_pass": true,
  "full_pinn_checks": {
    "finite": true,
    "nontrivial": true,
    "frequency_relative_error_le_1pct": true,
    "mac_ge_0p995": true,
    "normalization_error_le_0p02": true,
    "displacement_bc_le_1e-12": true,
    "scaled_moment_bc_le_0p02": true,
    "relative_residual_rms_le_0p05": true
  },
  "full_multi_seed_requirement_met": true,
  "seed_omega_cv": 9.770310188413953e-07,
  "seed_stability_threshold": 0.005,
  "run_count": 9,
  "maximum_frequency_relative_error": 7.06544418795534e-06,
  "minimum_mac": 0.9999999999994734,
  "maximum_relative_residual_rms": 0.00022532619360171387,
  "maximum_normalization_

## 12. GPT-PINN greedy basis construction

Uniform scaling (EI(x)=\alpha EI_0) was the correct Gate B verification problem, but it leaves the normalized mode shape invariant and would make basis enrichment artificially trivial. Gate C therefore introduces one still-synthetic but nonuniform parameter:

\[
EI(x;\mu)=1-\mu\exp\!\left[-\frac{1}{2}\left(\frac{x-44/58}{0.14}\right)^2\right],
\qquad 0\leq\mu\leq0.7.
\]

The zone is centered at the geometry-derived Koppigen-pier coordinate $44/58$. The parameter $\mu$ is a local effective stiffness-reduction coefficient, not measured Z24 stiffness loss and not pier settlement. The minimum rigidity is 0.3, so the eigenproblem remains positive. A conventional variable-rigidity FE model supplies independent truth only for verification. Starting at the median parameter, each reduced GPT query optimizes coefficients and frequency from precomputed Full-PINN fields and derivatives; the next Full PINN is trained where the reduced physics loss is largest. No FE error is used for greedy selection.

In [18]:
require_compute_node("Gate C greedy Full-PINN basis training")
from scipy.interpolate import CubicHermiteSpline

gate_c_mode = CONFIG["gate_c_mode"]
if gate_c_mode not in {"smoke", "full"}:
    raise ValueError("GPT_PINN_GATE_C_MODE must be 'smoke' or 'full'")
if gate_c_mode == "full" and gate_b_decision != "GO_TO_GPT_PINN_PILOT":
    raise RuntimeError("Full Gate C requires a full multi-seed Gate B pass")

mu_min, mu_max = map(float, CONFIG["gate_c_mu_domain"])
candidate_count = (CONFIG["gate_c_full_candidate_count"] if gate_c_mode == "full"
                   else CONFIG["gate_c_smoke_candidate_count"])
gate_c_max_basis = (CONFIG["gate_c_full_max_basis"] if gate_c_mode == "full"
                    else CONFIG["gate_c_smoke_max_basis"])
gate_c_adam_epochs = (CONFIG["gate_c_full_adam_epochs"] if gate_c_mode == "full"
                      else CONFIG["gate_c_smoke_adam_epochs"])
gate_c_lbfgs_iterations = (CONFIG["gate_c_full_lbfgs_iterations"] if gate_c_mode == "full"
                           else CONFIG["gate_c_smoke_lbfgs_iterations"])
candidate_mus = np.linspace(mu_min, mu_max, candidate_count)
collocation_x_np = np.linspace(0.0, 1.0, CONFIG["collocation_points"])

def local_stiffness_numpy(x: np.ndarray, mu: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    center = CONFIG["gate_c_stiffness_center"]
    width = CONFIG["gate_c_stiffness_width"]
    offset = np.asarray(x)-center
    gaussian = np.exp(-0.5*(offset/width)**2)
    gaussian_1 = -(offset/width**2)*gaussian
    gaussian_2 = ((offset**2/width**4)-(1.0/width**2))*gaussian
    return 1.0-mu*gaussian, -mu*gaussian_1, -mu*gaussian_2

def local_stiffness_torch(x: torch.Tensor, mu: float) -> torch.Tensor:
    center = CONFIG["gate_c_stiffness_center"]
    width = CONFIG["gate_c_stiffness_width"]
    return 1.0-mu*torch.exp(-0.5*((x-center)/width).square())

def variable_beam_fe_solution(
    mu: float, x_evaluation: np.ndarray, n_elements: int = 120,
) -> tuple[float, np.ndarray]:
    """Independent Hermite FE solution for the first variable-EI eigenmode."""
    element_length = 1.0/n_elements
    n_dof = 2*(n_elements+1)
    stiffness = np.zeros((n_dof, n_dof), dtype=np.float64)
    mass = np.zeros_like(stiffness)
    mass_element = (element_length/420.0)*np.array([
        [156, 22*element_length, 54, -13*element_length],
        [22*element_length, 4*element_length**2, 13*element_length, -3*element_length**2],
        [54, 13*element_length, 156, -22*element_length],
        [-13*element_length, -3*element_length**2, -22*element_length, 4*element_length**2],
    ], dtype=np.float64)
    for element in range(n_elements):
        midpoint = (element+0.5)*element_length
        element_ei = float(local_stiffness_numpy(np.array([midpoint]), mu)[0][0])
        stiffness_element = (element_ei/element_length**3)*np.array([
            [12, 6*element_length, -12, 6*element_length],
            [6*element_length, 4*element_length**2, -6*element_length, 2*element_length**2],
            [-12, -6*element_length, 12, -6*element_length],
            [6*element_length, 2*element_length**2, -6*element_length, 4*element_length**2],
        ], dtype=np.float64)
        dof = np.array([2*element, 2*element+1, 2*element+2, 2*element+3])
        stiffness[np.ix_(dof, dof)] += stiffness_element
        mass[np.ix_(dof, dof)] += mass_element
    constrained = np.array([0, 2*n_elements])
    free = np.setdiff1d(np.arange(n_dof), constrained)
    eigenvalue, eigenvector_free = linalg.eigh(
        stiffness[np.ix_(free, free)], mass[np.ix_(free, free)],
        subset_by_index=[0, 0], check_finite=True,
    )
    eigenvector = np.zeros(n_dof, dtype=np.float64)
    eigenvector[free] = eigenvector_free[:, 0]
    nodes = np.linspace(0.0, 1.0, n_elements+1)
    spline = CubicHermiteSpline(nodes, eigenvector[0::2], eigenvector[1::2])
    mode = np.asarray(spline(x_evaluation), dtype=np.float64)
    if mode[len(mode)//2] < 0.0:
        mode = -mode
    mode /= np.sqrt(numpy_trapezoid(mode**2, x_evaluation))
    return float(np.sqrt(eigenvalue[0])), mode

# Independent FE mesh convergence over the full parameter interval.
fe_mesh_rows = []
for mu_check in (mu_min, 0.5*(mu_min+mu_max), mu_max):
    omega_80, _ = variable_beam_fe_solution(mu_check, collocation_x_np, n_elements=80)
    omega_160, _ = variable_beam_fe_solution(mu_check, collocation_x_np, n_elements=160)
    fe_mesh_rows.append({"mu": mu_check, "omega_80": omega_80, "omega_160": omega_160,
                         "relative_difference": abs(omega_80-omega_160)/omega_160})
gate_c_fe_mesh_check = pd.DataFrame(fe_mesh_rows)
display(gate_c_fe_mesh_check)
gate_c_fe_mesh_check.to_csv(TABLE_ROOT / "gate_c_fe_mesh_convergence.csv", index=False)
if gate_c_fe_mesh_check["relative_difference"].max() > 2e-4:
    raise RuntimeError("Gate C stopped: variable-EI FE reference is not mesh-converged")

def parametric_full_pinn_loss(model: FullPINN, mu: float, x_col: torch.Tensor) -> tuple[torch.Tensor, dict]:
    phi = model(x_col)
    phi_2 = differentiate(phi, x_col, 2)
    ei = local_stiffness_torch(x_col, mu)
    bending_moment = ei*phi_2
    bending_moment_2 = differentiate(bending_moment, x_col, 2)
    residual = bending_moment_2-model.omega.square()*phi
    x_boundary = torch.tensor([[0.0], [1.0]], device=x_col.device, requires_grad=True)
    phi_boundary = model(x_boundary)
    phi_2_boundary = differentiate(phi_boundary, x_boundary, 2)
    moment_boundary = local_stiffness_torch(x_boundary, mu)*phi_2_boundary
    modal_mass = torch.trapezoid(phi.square().squeeze(-1), x_col.squeeze(-1))
    bending_energy = torch.trapezoid((ei*phi_2.square()).squeeze(-1), x_col.squeeze(-1))
    rayleigh_omega_sq = bending_energy/modal_mass.clamp_min(1e-12)
    components = {
        "pde": torch.mean((residual/np.pi**4)**2),
        "bc": torch.mean((moment_boundary/np.pi**2)**2),
        "norm": (modal_mass-1.0).square(),
        "rayleigh": ((model.omega.square()-rayleigh_omega_sq)/np.pi**4).square(),
        "orth": torch.zeros((), dtype=x_col.dtype, device=x_col.device),
    }
    total = sum(CONFIG["loss_weights"][name]*components[name]
                for name in ("pde", "bc", "norm", "rayleigh"))
    return total, components

def train_parametric_full_pinn(mu: float, basis_number: int) -> dict:
    seed = 700+basis_number
    run_seed = int(CONFIG["seed"]+seed+round(1000*mu))
    random.seed(run_seed); np.random.seed(run_seed); torch.manual_seed(run_seed)
    initial_center = np.pi**2*np.sqrt(max(0.2, 1.0-0.55*mu))
    initial_omega = float(np.random.uniform(0.75, 1.25)*initial_center)
    model = FullPINN(run_seed, initial_omega).to(gate_b_device)
    checkpoint_path = CHECKPOINT_ROOT / f"gate_c_{gate_c_mode}_basis_{basis_number:02d}_mu_{mu:.5f}.pt"
    if CONFIG["gate_c_resume"]:
        saved = load_checkpoint_with_legacy_fallback(
            checkpoint_path,
            map_location=gate_b_device,
            stage=f"Gate C basis={basis_number} mu={mu:.5f}",
            validator=lambda value: bool(
                isinstance(value, dict)
                and value.get("status") == "complete"
                and np.isclose(value.get("mu", np.nan), mu)
                and value.get("adam_epochs") == gate_c_adam_epochs
            ),
        )
        if saved is not None:
            model.load_state_dict(saved["model_state_dict"])
            return {"model": model, "mu": mu, "seed": seed, "initial_omega": saved["initial_omega"],
                    "runtime_seconds": saved["runtime_seconds"], "history": saved["history"],
                    "closure_calls": saved["lbfgs_closure_calls"], "checkpoint_path": checkpoint_path,
                    "loaded_checkpoint": True}
    x_col = torch.linspace(0.0, 1.0, CONFIG["collocation_points"], device=gate_b_device).reshape(-1, 1)
    x_col.requires_grad_(True)
    adam = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        adam, T_max=max(1, gate_c_adam_epochs), eta_min=2e-5)
    history = []
    start = time.perf_counter()
    for epoch in range(gate_c_adam_epochs):
        adam.zero_grad(set_to_none=True)
        total, components = parametric_full_pinn_loss(model, mu, x_col)
        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite parametric Full-PINN loss at mu={mu}")
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        adam.step(); scheduler.step()
        if epoch == 0 or (epoch+1) % 25 == 0 or epoch+1 == gate_c_adam_epochs:
            history.append({"stage": "Adam", "iteration": epoch+1, "total": float(total.detach().cpu()),
                            "omega": float(model.omega.detach().cpu()),
                            **{name: float(value.detach().cpu()) for name, value in components.items()}})
    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=0.8, max_iter=gate_c_lbfgs_iterations,
        max_eval=int(1.25*gate_c_lbfgs_iterations), tolerance_grad=1e-11,
        tolerance_change=1e-13, history_size=80, line_search_fn="strong_wolfe")
    closure_calls = 0
    def closure():
        nonlocal closure_calls
        lbfgs.zero_grad(set_to_none=True)
        total, components = parametric_full_pinn_loss(model, mu, x_col)
        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite L-BFGS loss at mu={mu}")
        total.backward(); closure_calls += 1
        if closure_calls == 1 or closure_calls % 10 == 0:
            history.append({"stage": "L-BFGS", "iteration": closure_calls,
                            "total": float(total.detach().cpu()),
                            "omega": float(model.omega.detach().cpu()),
                            **{name: float(value.detach().cpu()) for name, value in components.items()}})
        return total
    lbfgs.step(closure)
    runtime_seconds = time.perf_counter()-start
    final_total, final_components = parametric_full_pinn_loss(model, mu, x_col)
    safe_torch_save({
        "status": "complete", "mu": mu, "seed": seed, "initial_omega": initial_omega,
        "omega": float(model.omega.detach().cpu()), "model_state_dict": model.state_dict(),
        "adam_epochs": gate_c_adam_epochs,
        "lbfgs_closure_calls": closure_calls, "runtime_seconds": runtime_seconds, "history": history,
        "final_loss": float(final_total.detach().cpu()),
        "final_components": {name: float(value.detach().cpu()) for name, value in final_components.items()},
        "torch_version": torch.__version__, "device": str(gate_b_device),
        "checkpoint_format": "V0.6_FINAL_atomic_cpu_stream",
    }, checkpoint_path,
        stage=f"Gate C complete basis={basis_number} mu={mu:.5f}",
        retries=CONFIG["checkpoint_save_retries"],
        minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2)
    return {"model": model, "mu": mu, "seed": seed, "initial_omega": initial_omega,
            "runtime_seconds": runtime_seconds, "history": history, "closure_calls": closure_calls,
            "checkpoint_path": checkpoint_path, "loaded_checkpoint": False}

def precompute_basis_state(models: list[torch.nn.Module], x_values: np.ndarray) -> dict:
    fields = {name: [] for name in ("phi", "phi_2", "phi_3", "phi_4")}
    for model in models:
        x = torch.as_tensor(x_values, device=gate_b_device).reshape(-1, 1).clone().requires_grad_(True)
        phi = model(x)
        phi_2 = differentiate(phi, x, 2)
        phi_3 = differentiate(phi_2, x, 1)
        phi_4 = differentiate(phi_3, x, 1)
        for name, value in (("phi", phi), ("phi_2", phi_2), ("phi_3", phi_3), ("phi_4", phi_4)):
            fields[name].append(value.detach().cpu().numpy().ravel())
    return {"x": np.asarray(x_values), **{name: np.column_stack(values) for name, values in fields.items()}}

def reduced_components(
    coefficients: torch.Tensor, raw_omega: torch.Tensor, mu: float, basis: dict,
) -> tuple[torch.Tensor, dict]:
    x = basis["x"]
    phi = basis["phi"]@coefficients
    phi_2 = basis["phi_2"]@coefficients
    phi_3 = basis["phi_3"]@coefficients
    phi_4 = basis["phi_4"]@coefficients
    ei, ei_1, ei_2 = local_stiffness_numpy(x.detach().cpu().numpy(), mu)
    ei = torch.as_tensor(ei, device=x.device); ei_1 = torch.as_tensor(ei_1, device=x.device)
    ei_2 = torch.as_tensor(ei_2, device=x.device)
    omega = torch.nn.functional.softplus(raw_omega)+1e-8
    residual = ei*phi_4+2.0*ei_1*phi_3+ei_2*phi_2-omega.square()*phi
    modal_mass = torch.trapezoid(phi.square(), x)
    bending_energy = torch.trapezoid(ei*phi_2.square(), x)
    rayleigh_omega_sq = bending_energy/modal_mass.clamp_min(1e-12)
    moment_boundary = torch.stack((ei[0]*phi_2[0], ei[-1]*phi_2[-1]))
    components = {
        "pde": torch.mean((residual/np.pi**4)**2),
        "bc": torch.mean((moment_boundary/np.pi**2)**2),
        "norm": (modal_mass-1.0).square(),
        "rayleigh": ((omega.square()-rayleigh_omega_sq)/np.pi**4).square(),
    }
    total = (10.0*components["pde"]+10.0*components["bc"]+
             50.0*components["norm"]+2.0*components["rayleigh"]+
             1e-10*torch.mean(coefficients.square()))
    return total, {**components, "omega": omega, "mass": modal_mass, "residual": residual}

def solve_reduced_gpt(
    mu: float, basis_numpy: dict, selected_mus: list[float], selected_omegas: list[float],
    adam_steps: int | None = None, lbfgs_iterations: int | None = None,
) -> dict:
    adam_steps = CONFIG["gate_c_reduced_adam_steps"] if adam_steps is None else adam_steps
    lbfgs_iterations = CONFIG["gate_c_reduced_lbfgs_iterations"] if lbfgs_iterations is None else lbfgs_iterations
    basis = {"x": torch.as_tensor(basis_numpy["x"], device=gate_b_device),
             **{name: torch.as_tensor(basis_numpy[name], device=gate_b_device)
                for name in ("phi", "phi_2", "phi_3", "phi_4")}}
    nearest = int(np.argmin(np.abs(np.asarray(selected_mus)-mu)))
    initial_coefficients = np.zeros(len(selected_mus)); initial_coefficients[nearest] = 1.0
    coefficients = torch.nn.Parameter(torch.as_tensor(initial_coefficients, device=gate_b_device))
    raw_omega = torch.nn.Parameter(torch.tensor(inverse_softplus(selected_omegas[nearest]), device=gate_b_device))
    adam = torch.optim.Adam([coefficients, raw_omega], lr=3e-2)
    start = time.perf_counter()
    for _ in range(adam_steps):
        adam.zero_grad(set_to_none=True)
        total, _ = reduced_components(coefficients, raw_omega, mu, basis)
        total.backward(); adam.step()
    lbfgs = torch.optim.LBFGS([coefficients, raw_omega], lr=0.8, max_iter=lbfgs_iterations,
                              max_eval=int(1.4*lbfgs_iterations), tolerance_grad=1e-12,
                              tolerance_change=1e-14, history_size=40, line_search_fn="strong_wolfe")
    def closure():
        lbfgs.zero_grad(set_to_none=True)
        total, _ = reduced_components(coefficients, raw_omega, mu, basis)
        total.backward()
        return total
    lbfgs.step(closure)
    with torch.no_grad():
        _, before = reduced_components(coefficients, raw_omega, mu, basis)
        coefficients.div_(torch.sqrt(before["mass"].clamp_min(1e-12)))
        if (basis["phi"]@coefficients)[len(basis["x"])//2] < 0.0:
            coefficients.mul_(-1.0)
        total, final = reduced_components(coefficients, raw_omega, mu, basis)
    runtime_seconds = time.perf_counter()-start
    residual = final["residual"].detach().cpu().numpy()
    phi = (basis["phi"]@coefficients).detach().cpu().numpy()
    scale = (np.sqrt(np.mean((residual+float(final["omega"].cpu())**2*phi)**2))+
             np.sqrt(np.mean((float(final["omega"].cpu())**2*phi)**2))+1e-15)
    return {
        "mu": mu, "coefficients": coefficients.detach().cpu().numpy(),
        "omega": float(final["omega"].detach().cpu()), "physics_loss": float(total.detach().cpu()),
        "relative_residual_rms": float(np.sqrt(np.mean(residual**2))/scale),
        "normalization_error": float(abs(final["mass"].detach().cpu()-1.0)),
        "runtime_seconds": runtime_seconds,
    }

def verify_parametric_full_pinn(run: dict) -> dict:
    model, mu = run["model"], run["mu"]
    x_np = np.linspace(0.0, 1.0, CONFIG["validation_points"])
    state = precompute_basis_state([model], x_np)
    phi = state["phi"][:, 0]
    phi /= np.sqrt(numpy_trapezoid(phi**2, x_np))
    fe_omega, fe_phi = variable_beam_fe_solution(mu, x_np, n_elements=160)
    omega = float(model.omega.detach().cpu())
    mac = float(numpy_trapezoid(phi*fe_phi, x_np)**2/(numpy_trapezoid(phi**2, x_np)*numpy_trapezoid(fe_phi**2, x_np)))
    ei, ei_1, ei_2 = local_stiffness_numpy(x_np, mu)
    residual = (ei*state["phi_4"][:, 0]+2*ei_1*state["phi_3"][:, 0]+
                ei_2*state["phi_2"][:, 0]-omega**2*state["phi"][:, 0])
    scale = (np.sqrt(np.mean((residual+omega**2*state["phi"][:, 0])**2))+
             np.sqrt(np.mean((omega**2*state["phi"][:, 0])**2))+1e-15)
    return {"basis_number": len(gate_c_basis_runs)+1, "mu": mu, "seed": run["seed"],
            "omega": omega, "fe_omega": fe_omega,
            "frequency_relative_error": abs(omega-fe_omega)/fe_omega, "mac": mac,
            "shape_relative_l2": float(np.sqrt(numpy_trapezoid((phi-fe_phi)**2, x_np))),
            "relative_residual_rms": float(np.sqrt(np.mean(residual**2))/scale),
            "runtime_seconds": run["runtime_seconds"], "checkpoint": str(run["checkpoint_path"]),
            "loaded_checkpoint": run["loaded_checkpoint"]}

gate_c_basis_runs = []
gate_c_basis_verification_rows = []
gate_c_greedy_rows = []
gate_c_candidate_rows = []
gate_c_snapshots = []
selected_mus = []
next_mu = float(candidate_mus[len(candidate_mus)//2])

for basis_number in range(1, gate_c_max_basis+1):
    print(f"Gate C basis {basis_number}/{gate_c_max_basis}: train Full PINN at mu={next_mu:.5f}")
    basis_run = train_parametric_full_pinn(next_mu, basis_number)
    verification = verify_parametric_full_pinn(basis_run)
    gate_c_basis_runs.append(basis_run)
    gate_c_basis_verification_rows.append(verification)
    print("  Full-PINN verification:", {key: verification[key] for key in ["frequency_relative_error", "mac", "shape_relative_l2", "relative_residual_rms"]})
    if (verification["frequency_relative_error"] > 5e-3 or verification["mac"] < 0.995 or
            verification["relative_residual_rms"] > 0.02):
        raise RuntimeError(f"Gate C stopped: selected Full PINN failed FE verification at mu={next_mu}")
    selected_mus.append(next_mu)
    basis_models = [run["model"] for run in gate_c_basis_runs]
    selected_omegas = [float(run["model"].omega.detach().cpu()) for run in gate_c_basis_runs]
    basis_collocation = precompute_basis_state(basis_models, collocation_x_np)
    candidate_solutions = []
    for candidate_mu in candidate_mus:
        solution = solve_reduced_gpt(float(candidate_mu), basis_collocation, selected_mus, selected_omegas)
        candidate_solutions.append(solution)
        gate_c_candidate_rows.append({"basis_size": basis_number, **{k: v for k, v in solution.items()
                                      if k not in {"coefficients"}}})
    available = [index for index, mu in enumerate(candidate_mus)
                 if not np.any(np.isclose(mu, np.asarray(selected_mus), atol=1e-12))]
    if available:
        worst_index = max(available, key=lambda index: candidate_solutions[index]["physics_loss"])
        maximum_loss = float(candidate_solutions[worst_index]["physics_loss"])
        proposed_next_mu = float(candidate_mus[worst_index])
    else:
        maximum_loss, proposed_next_mu = 0.0, np.nan
    gate_c_greedy_rows.append({"basis_size": basis_number, "selected_mu": next_mu,
                               "maximum_unselected_physics_loss": maximum_loss,
                               "proposed_next_mu": proposed_next_mu,
                               "cumulative_full_pinn_seconds": sum(run["runtime_seconds"] for run in gate_c_basis_runs)})
    gate_c_snapshots.append({"basis_size": basis_number, "models": list(basis_models),
                             "selected_mus": list(selected_mus), "selected_omegas": list(selected_omegas),
                             "basis_collocation": basis_collocation})
    print(f"  verified FE error={verification['frequency_relative_error']:.3e}, MAC={verification['mac']:.8f}; "
          f"max reduced loss={maximum_loss:.3e} at mu={proposed_next_mu}")
    if basis_number >= CONFIG["gate_c_min_basis"] and maximum_loss <= CONFIG["gate_c_greedy_loss_tolerance"]:
        print("  greedy physics-loss tolerance reached")
        break
    if not available:
        break
    next_mu = proposed_next_mu

gate_c_basis_verification = pd.DataFrame(gate_c_basis_verification_rows)
gate_c_greedy_history = pd.DataFrame(gate_c_greedy_rows)
gate_c_candidate_scan = pd.DataFrame(gate_c_candidate_rows)
display(gate_c_basis_verification)
display(gate_c_greedy_history)
gate_c_basis_verification.to_csv(TABLE_ROOT / f"gate_c_full_pinn_basis_verification_{gate_c_mode}.csv", index=False)
gate_c_greedy_history.to_csv(TABLE_ROOT / f"gate_c_greedy_history_{gate_c_mode}.csv", index=False)
gate_c_candidate_scan.to_csv(TABLE_ROOT / f"gate_c_candidate_physics_scan_{gate_c_mode}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
x_profile = np.linspace(0.0, 1.0, 500)
for mu in selected_mus:
    axes[0].plot(x_profile, local_stiffness_numpy(x_profile, mu)[0], label=rf"$\mu={mu:.3f}$")
axes[0].set_xlabel("Dimensionless coordinate x")
axes[0].set_ylabel(r"$EI(x;\mu)/EI_0$")
axes[0].set_title("Greedily selected stiffness profiles")
axes[0].legend(fontsize=8)
axes[1].semilogy(gate_c_greedy_history["basis_size"],
                 np.maximum(gate_c_greedy_history["maximum_unselected_physics_loss"], 1e-16),
                 "o-", color="#264653")
axes[1].axhline(CONFIG["gate_c_greedy_loss_tolerance"], color="#e76f51", linestyle="--", label="stopping tolerance")
for row in gate_c_greedy_history.itertuples():
    axes[1].annotate(rf"$\mu={row.selected_mu:.3f}$", (row.basis_size, max(row.maximum_unselected_physics_loss, 1e-16)),
                     xytext=(5, 6), textcoords="offset points", fontsize=8)
axes[1].set_xlabel("Full-PINN basis size")
axes[1].set_ylabel("Maximum unselected reduced physics loss")
axes[1].set_title("Residual-driven greedy enrichment")
axes[1].legend()
fig.suptitle("Figure 7 — Gate C greedy GPT-PINN basis construction", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_07_gate_c_greedy_basis_{gate_c_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

,mu,omega_80,omega_160,relative_difference
0,0.00,9.869604,9.869608,3.807930e-07
1,0.35,9.162629,9.162634,5.122964e-07
2,0.70,7.938740,7.938728,1.497026e-06


Gate C basis 1/5: train Full PINN at mu=0.35000
  Full-PINN verification: {'frequency_relative_error': 2.355638423574574e-05, 'mac': 0.9999999998290168, 'shape_relative_l2': 1.307606689913067e-05, 'relative_residual_rms': 0.002046777386596256}
  verified FE error=2.356e-05, MAC=1.00000000; max reduced loss=2.555e+01 at mu=0.7
Gate C basis 2/5: train Full PINN at mu=0.70000
  Full-PINN verification: {'frequency_relative_error': 5.598512776768119e-05, 'mac': 0.9999999999186922, 'shape_relative_l2': 9.017090621119184e-06, 'relative_residual_rms': 0.0007346476149995182}
  verified FE error=5.599e-05, MAC=1.00000000; max reduced loss=9.185e+00 at mu=0.0
Gate C basis 3/5: train Full PINN at mu=0.00000
  Full-PINN verification: {'frequency_relative_error': 3.9071687829823363e-07, 'mac': 0.9999999999999987, 'shape_relative_l2': 3.6597840272770447e-08, 'relative_residual_rms': 6.820114363222945e-05}
  verified FE error=3.907e-07, MAC=1.00000000; max reduced loss=1.778e-01 at mu=0.6
Gate C basis

,basis_number,mu,seed,omega,fe_omega,frequency_relative_error,mac,shape_relative_l2,relative_residual_rms,runtime_seconds,checkpoint,loaded_checkpoint
0,1,0.35,701,9.162418,9.162634,2.355638e-05,1.0,1.307607e-05,0.002047,117.515257,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,False
1,2,0.70,702,7.938284,7.938728,5.598513e-05,1.0,9.017091e-06,0.000735,178.535261,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,False
2,3,0.00,703,9.869604,9.869608,3.907169e-07,1.0,3.659784e-08,0.000068,110.887635,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,False
3,4,0.60,704,8.386781,8.386637,1.718447e-05,1.0,6.091102e-06,0.000982,130.714687,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,False
4,5,0.15,705,9.600946,9.600948,2.162486e-07,1.0,1.045866e-06,0.000272,158.882220,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,False


,basis_size,selected_mu,maximum_unselected_physics_loss,proposed_next_mu,cumulative_full_pinn_seconds
0,1,0.35,25.547159,0.70,117.515257
1,2,0.70,9.185204,0.00,296.050519
2,3,0.00,0.177805,0.60,406.938153
3,4,0.60,0.002174,0.15,537.652840
4,5,0.15,0.000138,0.45,696.535060


## 13. GPT-PINN accuracy evaluation

Accuracy is evaluated at the midpoints between greedy-candidate parameters. These values are never eligible for Full-PINN basis training. For every retained basis size, coefficient/frequency optimization uses only the strong-form physics loss on the reduced collocation grid. Frequency and mode-shape errors are then computed against the independent 160-element FE solution on a denser grid. This separates greedy selection from truth-based validation.

In [19]:
validation_mus = 0.5*(candidate_mus[:-1]+candidate_mus[1:])
validation_x_np = np.linspace(0.0, 1.0, CONFIG["validation_points"])
gate_c_accuracy_rows = []
gate_c_final_curves = {}

for snapshot in gate_c_snapshots:
    basis_validation = precompute_basis_state(snapshot["models"], validation_x_np)
    for mu in validation_mus:
        reduced = solve_reduced_gpt(float(mu), snapshot["basis_collocation"],
                                    snapshot["selected_mus"], snapshot["selected_omegas"])
        coefficients = reduced["coefficients"]
        phi = basis_validation["phi"]@coefficients
        phi /= np.sqrt(numpy_trapezoid(phi**2, validation_x_np))
        omega = reduced["omega"]
        fe_start = time.perf_counter()
        fe_omega, fe_phi = variable_beam_fe_solution(float(mu), validation_x_np, n_elements=160)
        fe_runtime = time.perf_counter()-fe_start
        if numpy_trapezoid(phi*fe_phi, validation_x_np) < 0.0:
            phi = -phi
        mac = float(numpy_trapezoid(phi*fe_phi, validation_x_np)**2 /
                    (numpy_trapezoid(phi**2, validation_x_np)*numpy_trapezoid(fe_phi**2, validation_x_np)))
        phi_2 = basis_validation["phi_2"]@coefficients
        phi_3 = basis_validation["phi_3"]@coefficients
        phi_4 = basis_validation["phi_4"]@coefficients
        ei, ei_1, ei_2 = local_stiffness_numpy(validation_x_np, float(mu))
        residual = ei*phi_4+2*ei_1*phi_3+ei_2*phi_2-omega**2*(basis_validation["phi"]@coefficients)
        residual_scale = (np.sqrt(np.mean((residual+omega**2*(basis_validation["phi"]@coefficients))**2))+
                          np.sqrt(np.mean((omega**2*(basis_validation["phi"]@coefficients))**2))+1e-15)
        row = {
            "basis_size": snapshot["basis_size"], "mu": float(mu), "omega": omega, "fe_omega": fe_omega,
            "frequency_relative_error": abs(omega-fe_omega)/fe_omega, "mac": mac,
            "shape_relative_l2": float(np.sqrt(numpy_trapezoid((phi-fe_phi)**2, validation_x_np))),
            "relative_residual_rms": float(np.sqrt(np.mean(residual**2))/residual_scale),
            "normalization_error": abs(numpy_trapezoid(phi**2, validation_x_np)-1.0),
            "gpt_query_seconds": reduced["runtime_seconds"], "fe_solve_seconds": fe_runtime,
            "coefficients": json_dumps_safe(coefficients.tolist()),
        }
        gate_c_accuracy_rows.append(row)
        if snapshot is gate_c_snapshots[-1]:
            gate_c_final_curves[float(mu)] = {"phi": phi, "fe_phi": fe_phi, "residual": residual}

gate_c_accuracy = pd.DataFrame(gate_c_accuracy_rows)
gate_c_accuracy.to_csv(TABLE_ROOT / f"gate_c_heldout_accuracy_{gate_c_mode}.csv", index=False)
accuracy_by_basis = gate_c_accuracy.groupby("basis_size").agg(
    maximum_frequency_relative_error=("frequency_relative_error", "max"),
    minimum_mac=("mac", "min"),
    maximum_shape_relative_l2=("shape_relative_l2", "max"),
    maximum_relative_residual_rms=("relative_residual_rms", "max"),
    median_query_seconds=("gpt_query_seconds", "median"),
).reset_index()
display(accuracy_by_basis)
accuracy_by_basis.to_csv(TABLE_ROOT / f"gate_c_accuracy_by_basis_size_{gate_c_mode}.csv", index=False)

final_basis_size = gate_c_snapshots[-1]["basis_size"]
gate_c_final_accuracy = gate_c_accuracy[gate_c_accuracy["basis_size"] == final_basis_size].copy()

fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.2))
axes[0].semilogy(accuracy_by_basis["basis_size"], accuracy_by_basis["maximum_frequency_relative_error"],
                 "o-", label="max frequency error")
axes[0].semilogy(accuracy_by_basis["basis_size"], accuracy_by_basis["maximum_shape_relative_l2"],
                 "s-", label="max shape L2 error")
axes[0].set_xlabel("Basis size"); axes[0].set_ylabel("Held-out error")
axes[0].set_title("Error versus basis size"); axes[0].legend(fontsize=8)
plot_mus = [float(validation_mus[0]), float(validation_mus[len(validation_mus)//2]), float(validation_mus[-1])]
for mu in plot_mus:
    curve = gate_c_final_curves[mu]
    axes[1].plot(validation_x_np, curve["phi"], label=rf"GPT $\mu={mu:.3f}$")
    axes[1].plot(validation_x_np, curve["fe_phi"], "--", color=axes[1].lines[-1].get_color(), linewidth=1.0)
axes[1].set_xlabel("Dimensionless coordinate x"); axes[1].set_ylabel(r"Normalized $\phi_1$")
axes[1].set_title("Held-out modes: GPT (solid), FE (dashed)"); axes[1].legend(fontsize=7)
axes[2].semilogy(gate_c_final_accuracy["mu"], gate_c_final_accuracy["frequency_relative_error"],
                 "o-", label="frequency error")
axes[2].semilogy(gate_c_final_accuracy["mu"], gate_c_final_accuracy["relative_residual_rms"],
                 "s-", label="physics residual")
axes[2].set_xlabel(r"Held-out $\mu$"); axes[2].set_ylabel("Relative metric")
axes[2].set_title("Final-basis validation across domain"); axes[2].legend(fontsize=8)
fig.suptitle("Figure 8 — Held-out GPT-PINN accuracy", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_08_gate_c_heldout_accuracy_{gate_c_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

,basis_size,maximum_frequency_relative_error,minimum_mac,maximum_shape_relative_l2,maximum_relative_residual_rms,median_query_seconds
0,1,0.030967,0.996323,0.060666,0.631076,0.406290
1,2,0.004473,0.999474,0.022929,0.378964,0.472329
2,3,0.000029,0.999997,0.001811,0.089379,0.380412
3,4,0.000032,1.000000,0.000246,0.007877,0.396141
4,5,0.000039,1.000000,0.000029,0.002213,0.366177


## 14. Computational speed-up evaluation

Costs are separated explicitly. *Offline* cost is the sum of Full-PINN basis training and greedy reduced scans. *Online solve* time includes coefficient/frequency optimization for a new (mu); *online assembly* is only the final basis combination after coefficients are known. Speed-up is reported against measured Full-PINN retraining time. The FE timing is also reported honestly: this small 1-D FE model can be faster than a physics-optimization query and remains a required classical baseline for later gates.

In [20]:
final_snapshot = gate_c_snapshots[-1]
timing_rows = []
timing_repeats = CONFIG["gate_c_timing_repeats"] if gate_c_mode == "full" else 1
for repeat in range(timing_repeats):
    for mu in validation_mus:
        reduced = solve_reduced_gpt(float(mu), final_snapshot["basis_collocation"],
                                    final_snapshot["selected_mus"], final_snapshot["selected_omegas"])
        timing_rows.append({"repeat": repeat, "mu": float(mu),
                            "gpt_online_solve_seconds": reduced["runtime_seconds"]})
gate_c_timing = pd.DataFrame(timing_rows)

# Measure pure reduced field assembly after coefficients have been identified.
representative_mu = float(validation_mus[len(validation_mus)//2])
representative_solution = solve_reduced_gpt(representative_mu, final_snapshot["basis_collocation"],
                                            final_snapshot["selected_mus"], final_snapshot["selected_omegas"])
final_basis_validation = precompute_basis_state(final_snapshot["models"], validation_x_np)
assembly_repeats = 5000
assembly_start = time.perf_counter()
for _ in range(assembly_repeats):
    _assembled_mode = final_basis_validation["phi"]@representative_solution["coefficients"]
assembly_seconds = (time.perf_counter()-assembly_start)/assembly_repeats

median_full_pinn_seconds = float(gate_c_basis_verification["runtime_seconds"].median())
offline_full_pinn_seconds = float(gate_c_basis_verification["runtime_seconds"].sum())
median_gpt_online_seconds = float(gate_c_timing["gpt_online_solve_seconds"].median())
median_fe_seconds = float(gate_c_final_accuracy["fe_solve_seconds"].median())
speedup_vs_full_pinn = median_full_pinn_seconds/median_gpt_online_seconds
speedup_vs_fe = median_fe_seconds/median_gpt_online_seconds

gate_c_speed = pd.DataFrame([
    {"quantity": "offline Full-PINN basis training", "seconds": offline_full_pinn_seconds,
     "interpretation": f"{final_basis_size} greedily selected Full PINNs"},
    {"quantity": "median new-parameter Full-PINN retraining", "seconds": median_full_pinn_seconds,
     "interpretation": "measured selected-basis training runtime"},
    {"quantity": "median GPT online coefficient/frequency solve", "seconds": median_gpt_online_seconds,
     "interpretation": f"median of {len(gate_c_timing)} timed held-out queries"},
    {"quantity": "median independent FE solve", "seconds": median_fe_seconds,
     "interpretation": "160-element 1-D reference; accuracy validation baseline"},
    {"quantity": "GPT online field assembly after solve", "seconds": assembly_seconds,
     "interpretation": f"mean of {assembly_repeats} matrix-vector products"},
    {"quantity": "speed-up GPT solve versus Full-PINN retraining", "seconds": speedup_vs_full_pinn,
     "interpretation": "dimensionless ratio"},
    {"quantity": "speed-up GPT solve versus FE", "seconds": speedup_vs_fe,
     "interpretation": "dimensionless ratio; below one means FE is faster"},
])
display(gate_c_speed)
gate_c_speed.to_csv(TABLE_ROOT / f"gate_c_speed_{gate_c_mode}.csv", index=False)
gate_c_timing.to_csv(TABLE_ROOT / f"gate_c_query_timings_{gate_c_mode}.csv", index=False)

final_checks = {
    "basis_full_pinn_frequency_error_le_0p5pct": bool((gate_c_basis_verification["frequency_relative_error"] <= 5e-3).all()),
    "basis_full_pinn_mac_ge_0p995": bool((gate_c_basis_verification["mac"] >= 0.995).all()),
    "heldout_frequency_error_le_0p5pct": bool((gate_c_final_accuracy["frequency_relative_error"] <= 5e-3).all()),
    "heldout_mac_ge_0p995": bool((gate_c_final_accuracy["mac"] >= 0.995).all()),
    "heldout_shape_l2_le_0p02": bool((gate_c_final_accuracy["shape_relative_l2"] <= 0.02).all()),
    "heldout_relative_residual_le_0p02": bool((gate_c_final_accuracy["relative_residual_rms"] <= 0.02).all()),
    "full_pinn_speedup_ge_10": bool(speedup_vs_full_pinn >= 10.0),
    "gpt_online_median_le_5s": bool(median_gpt_online_seconds <= 5.0),
    "finite": bool(np.isfinite(gate_c_final_accuracy.select_dtypes(include=[np.number]).to_numpy()).all()),
}
full_gate_c_requirement = gate_c_mode == "full" and gate_b_decision == "GO_TO_GPT_PINN_PILOT"
if gate_c_mode == "smoke":
    gate_c_decision = "SMOKE_PASS_REQUIRES_FULL_RUN" if all(final_checks.values()) else "NO_GO_FIX_GPT_PINN"
else:
    gate_c_decision = ("GO_TO_Z24_CALIBRATION_PILOT_WITH_FE_BASELINE_REQUIRED"
                       if full_gate_c_requirement and all(final_checks.values()) else "NO_GO_FIX_GPT_PINN")

gate_c_diagnostic = {
    "decision": gate_c_decision, "execution_mode": gate_c_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "model_scope": "dimensionless simply supported beam with one smooth local stiffness-reduction parameter",
    "z24_calibration_performed": False, "mu_equated_to_delta_pier": False,
    "selected_mus": selected_mus, "final_basis_size": final_basis_size,
    "greedy_stopping_tolerance": CONFIG["gate_c_greedy_loss_tolerance"],
    "final_maximum_unselected_physics_loss": float(gate_c_greedy_history.iloc[-1]["maximum_unselected_physics_loss"]),
    "maximum_heldout_frequency_relative_error": float(gate_c_final_accuracy["frequency_relative_error"].max()),
    "minimum_heldout_mac": float(gate_c_final_accuracy["mac"].min()),
    "maximum_heldout_shape_relative_l2": float(gate_c_final_accuracy["shape_relative_l2"].max()),
    "maximum_heldout_relative_residual_rms": float(gate_c_final_accuracy["relative_residual_rms"].max()),
    "median_full_pinn_retraining_seconds": median_full_pinn_seconds,
    "median_gpt_online_solve_seconds": median_gpt_online_seconds,
    "median_fe_solve_seconds": median_fe_seconds,
    "gpt_speedup_vs_full_pinn": speedup_vs_full_pinn,
    "gpt_speedup_vs_fe": speedup_vs_fe,
    "online_assembly_seconds": assembly_seconds,
    "checks": final_checks, "full_gate_c_requirement_met": full_gate_c_requirement,
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "gate_c_diagnostic.json").write_text(
    json_dumps_safe(gate_c_diagnostic, indent=2, sort_keys=True)+"\n", encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2))
timing_plot = pd.DataFrame({
    "method": ["Full-PINN\nretraining", "GPT-PINN\nonline solve", "FE\nreference"],
    "seconds": [median_full_pinn_seconds, median_gpt_online_seconds, median_fe_seconds],
})
axes[0].bar(timing_plot["method"], timing_plot["seconds"], color=["#6d597a", "#2a9d8f", "#457b9d"])
axes[0].set_yscale("log"); axes[0].set_ylabel("Measured wall time [s]")
axes[0].set_title("Per-parameter computational cost")
axes[1].boxplot([gate_c_timing["gpt_online_solve_seconds"], gate_c_final_accuracy["fe_solve_seconds"]],
                labels=["GPT online", "FE reference"], showmeans=True)
axes[1].set_yscale("log"); axes[1].set_ylabel("Wall time [s]")
axes[1].set_title(rf"GPT versus Full-PINN speed-up = {speedup_vs_full_pinn:.1f}$\times$")
fig.suptitle(f"Figure 9 — Gate C timing: {gate_c_decision.replace('_', ' ')}", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_09_gate_c_speed_{gate_c_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_c_diagnostic, indent=2))
if gate_c_decision == "NO_GO_FIX_GPT_PINN":
    raise RuntimeError("Gate C NO-GO: correct the reduced GPT-PINN before Z24 calibration")

,quantity,seconds,interpretation
0,offline Full-PINN basis training,696.535060,5 greedily selected Full PINNs
1,median new-parameter Full-PINN retraining,130.714687,measured selected-basis training runtime
2,median GPT online coefficient/frequency solve,0.367280,median of 42 timed held-out queries
3,median independent FE solve,0.014986,160-element 1-D reference; accuracy validation...
4,GPT online field assembly after solve,0.000003,mean of 5000 matrix-vector products
5,speed-up GPT solve versus Full-PINN retraining,355.899356,dimensionless ratio
6,speed-up GPT solve versus FE,0.040802,dimensionless ratio; below one means FE is faster


{
  "decision": "GO_TO_Z24_CALIBRATION_PILOT_WITH_FE_BASELINE_REQUIRED",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "model_scope": "dimensionless simply supported beam with one smooth local stiffness-reduction parameter",
  "z24_calibration_performed": false,
  "mu_equated_to_delta_pier": false,
  "selected_mus": [
    0.35,
    0.7,
    0.0,
    0.6,
    0.15
  ],
  "final_basis_size": 5,
  "greedy_stopping_tolerance": 1e-05,
  "final_maximum_unselected_physics_loss": 0.00013758262062880982,
  "maximum_heldout_frequency_relative_error": 3.912583568880979e-05,
  "minimum_heldout_mac": 0.9999999991357165,
  "maximum_heldout_shape_relative_l2": 2.9398701593012042e-05,
  "maximum_heldout_relative_residual_rms": 0.0022125700503381044,
  "median_full_pinn_retraining_seconds": 130.71468697302043,
  "median_gpt_online_solve_seconds": 0.3672799200285226,
  "median_fe_solve_seconds": 0.014985758578404784,
  "gpt_speedup_vs_full_p

## 15. Z24-to-Digital-Twin calibration

### 15.1 Multimodal extension and independent verification

The official benchmark describes a 58 m continuous bridge with 14–30–14 m spans; the Koppigen pier is therefore represented at the normalized coordinate $44/58\approx0.759$. This geometry is used only to locate the smooth effective-rigidity zone inherited from Gate C. The pilot is still a one-dimensional simply supported equivalent beam and does not reproduce the true pier restraint, skew, box-girder section, cracking, or static settlement mechanics.

Gate D resolves equivalent modes 1–4. Mode 1 reuses the verified positive Full-PINN from Gate C. Modes 2–4 are trained sequentially with independent signed networks, mass normalization, boundary moments, Rayleigh consistency, and mass-orthogonality against all lower modes. Uniform-beam sine modes are initialization functions only. A 160-element variable-rigidity FE solution remains the independent verification reference. Experimental tracks `[f_1,f_3,f_4]` are associated by rank with equivalent modes `[1,3,4]`; equality of physical mode shapes is not claimed.

**V0.6 FINAL checkpoint protocol.** Every completed mode is saved immediately through
a verified atomic NFS-safe stream write. A progress JSON is updated after each
successful mode. Compatible V0.4 completed checkpoints may be imported after
parameter/epoch/convergence checks; the unreadable
`gate_d_full_basis_03_mu_0.00000_mode_3.pt` file is treated as absent and the
corresponding mode is retrained rather than terminating the notebook.


In [21]:
require_compute_node("Gate D multimodal Full-PINN training")
from scipy.optimize import least_squares, minimize_scalar
from scipy.interpolate import PchipInterpolator

gate_d_mode = CONFIG["gate_d_mode"]
if gate_d_mode not in {"smoke", "full"}:
    raise ValueError("GPT_PINN_GATE_D_MODE must be 'smoke' or 'full'")
if gate_d_mode == "full" and gate_c_decision != "GO_TO_Z24_CALIBRATION_PILOT_WITH_FE_BASELINE_REQUIRED":
    raise RuntimeError("Full Gate D requires a full Gate C pass")
gate_d_adam_epochs = (CONFIG["gate_d_full_adam_epochs"] if gate_d_mode == "full"
                      else CONFIG["gate_d_smoke_adam_epochs"])
gate_d_lbfgs_iterations = (CONFIG["gate_d_full_lbfgs_iterations"] if gate_d_mode == "full"
                           else CONFIG["gate_d_smoke_lbfgs_iterations"])
equivalent_mode_orders = list(CONFIG["gate_d_equivalent_mode_orders"])
retained_equivalent_modes = list(CONFIG["gate_d_retained_equivalent_modes"])
retained_experimental_modes = list(retained_modes)
if retained_experimental_modes != [1, 3, 4] or retained_equivalent_modes != [1, 3, 4]:
    raise RuntimeError("Gate D mapping audit expected experimental/equivalent ranks [1, 3, 4]")

geometry_audit = pd.DataFrame([
    {"quantity": "total girder length", "value": 58.0, "unit": "m", "source": REFERENCE_URLS["z24_official"]},
    {"quantity": "span lengths", "value": "14 / 30 / 14", "unit": "m", "source": REFERENCE_URLS["z24_official"]},
    {"quantity": "normalized Koppigen-pier coordinate", "value": CONFIG["gate_c_stiffness_center"],
     "unit": "x/L", "source": "derived as (14+30)/58"},
    {"quantity": "effective-zone width", "value": CONFIG["gate_c_stiffness_width"],
     "unit": "x/L", "source": "pilot assumption; not a measured crack length"},
])
display(geometry_audit)
geometry_audit.to_csv(TABLE_ROOT / "gate_d_geometry_and_assumptions.csv", index=False)

def variable_beam_fe_modes(
    mu: float, x_evaluation: np.ndarray, n_elements: int = 160, n_modes: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """Independent consistent-mass Hermite FE modes for the Gate D equivalent beam."""
    element_length = 1.0/n_elements
    n_dof = 2*(n_elements+1)
    stiffness = np.zeros((n_dof, n_dof), dtype=np.float64)
    mass = np.zeros_like(stiffness)
    mass_element = (element_length/420.0)*np.array([
        [156, 22*element_length, 54, -13*element_length],
        [22*element_length, 4*element_length**2, 13*element_length, -3*element_length**2],
        [54, 13*element_length, 156, -22*element_length],
        [-13*element_length, -3*element_length**2, -22*element_length, 4*element_length**2],
    ], dtype=np.float64)
    for element in range(n_elements):
        midpoint = (element+0.5)*element_length
        element_ei = float(local_stiffness_numpy(np.array([midpoint]), mu)[0][0])
        stiffness_element = (element_ei/element_length**3)*np.array([
            [12, 6*element_length, -12, 6*element_length],
            [6*element_length, 4*element_length**2, -6*element_length, 2*element_length**2],
            [-12, -6*element_length, 12, -6*element_length],
            [6*element_length, 2*element_length**2, -6*element_length, 4*element_length**2],
        ], dtype=np.float64)
        dof = np.array([2*element, 2*element+1, 2*element+2, 2*element+3])
        stiffness[np.ix_(dof, dof)] += stiffness_element
        mass[np.ix_(dof, dof)] += mass_element
    constrained = np.array([0, 2*n_elements])
    free = np.setdiff1d(np.arange(n_dof), constrained)
    eigenvalues, eigenvectors_free = linalg.eigh(
        stiffness[np.ix_(free, free)], mass[np.ix_(free, free)],
        subset_by_index=[0, n_modes-1], check_finite=True)
    nodes = np.linspace(0.0, 1.0, n_elements+1)
    modes = []
    for mode_index in range(n_modes):
        vector = np.zeros(n_dof, dtype=np.float64)
        vector[free] = eigenvectors_free[:, mode_index]
        spline = CubicHermiteSpline(nodes, vector[0::2], vector[1::2])
        shape = np.asarray(spline(x_evaluation), dtype=np.float64)
        reference = np.sin((mode_index+1)*np.pi*x_evaluation)
        if numpy_trapezoid(shape*reference, x_evaluation) < 0.0:
            shape = -shape
        shape /= np.sqrt(numpy_trapezoid(shape**2, x_evaluation))
        modes.append(shape)
    return np.sqrt(eigenvalues), np.column_stack(modes)

class DeflatedModePINN(torch.nn.Module):
    """One signed eigenmode, initialized by but not restricted to a sine reference."""
    def __init__(self, mode_order: int, seed: int, initial_omega: float):
        super().__init__()
        self.mode_order = int(mode_order)
        torch.manual_seed(seed)
        widths = [1, 32, 32, 32, 1]
        layers = []
        for input_width, output_width in zip(widths[:-2], widths[1:-1]):
            layer = torch.nn.Linear(input_width, output_width)
            torch.nn.init.xavier_normal_(layer.weight); torch.nn.init.zeros_(layer.bias)
            layers.extend([layer, torch.nn.Tanh()])
        final = torch.nn.Linear(widths[-2], widths[-1])
        torch.nn.init.zeros_(final.weight); torch.nn.init.zeros_(final.bias)
        layers.append(final)
        self.correction = torch.nn.Sequential(*layers)
        self.raw_omega = torch.nn.Parameter(torch.tensor(inverse_softplus(initial_omega)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        reference = np.sqrt(2.0)*torch.sin(self.mode_order*np.pi*x)
        return reference+x*(1.0-x)*self.correction(x)

    @property
    def omega(self) -> torch.Tensor:
        return torch.nn.functional.softplus(self.raw_omega)+1e-8

def deflated_mode_loss(
    model: DeflatedModePINN, mu: float, x_col: torch.Tensor,
    lower_models: list[torch.nn.Module],
) -> tuple[torch.Tensor, dict]:
    order = model.mode_order
    phi = model(x_col)
    phi_2 = differentiate(phi, x_col, 2)
    ei = local_stiffness_torch(x_col, mu)
    moment = ei*phi_2
    moment_2 = differentiate(moment, x_col, 2)
    residual = moment_2-model.omega.square()*phi
    x_boundary = torch.tensor([[0.0], [1.0]], device=x_col.device, requires_grad=True)
    phi_boundary = model(x_boundary)
    phi_2_boundary = differentiate(phi_boundary, x_boundary, 2)
    moment_boundary = local_stiffness_torch(x_boundary, mu)*phi_2_boundary
    modal_mass = torch.trapezoid(phi.square().squeeze(-1), x_col.squeeze(-1))
    bending_energy = torch.trapezoid((ei*phi_2.square()).squeeze(-1), x_col.squeeze(-1))
    rayleigh_omega_sq = bending_energy/modal_mass.clamp_min(1e-12)
    orth_terms = []
    for lower_model in lower_models:
        lower_phi = lower_model(x_col).detach().squeeze(-1)
        lower_phi = lower_phi/torch.sqrt(torch.trapezoid(lower_phi.square(), x_col.squeeze(-1)).clamp_min(1e-12))
        orth_terms.append(torch.trapezoid(phi.squeeze(-1)*lower_phi, x_col.squeeze(-1)).square())
    orthogonality = torch.stack(orth_terms).sum() if orth_terms else torch.zeros((), device=x_col.device)
    reference = np.sqrt(2.0)*torch.sin(order*np.pi*x_col.squeeze(-1))
    signed_correlation = torch.trapezoid(phi.squeeze(-1)*reference, x_col.squeeze(-1))
    orientation = torch.nn.functional.relu(0.25-signed_correlation).square()
    scale_4 = (order*np.pi)**4
    components = {
        "pde": torch.mean((residual/scale_4)**2),
        "bc": torch.mean((moment_boundary/(order*np.pi)**2)**2),
        "norm": (modal_mass-1.0).square(),
        "orth": orthogonality,
        "rayleigh": ((model.omega.square()-rayleigh_omega_sq)/scale_4).square(),
        "orientation": orientation,
        "spectral": rayleigh_omega_sq/scale_4,
    }
    total = (50.0*components["pde"]+20.0*components["bc"]+50.0*components["norm"]+
             100.0*components["orth"]+200.0*components["rayleigh"]+
             components["orientation"]+0.001*components["spectral"])
    return total, components

def gate_d_internal_converged(components: dict) -> bool:
    # Optimization diagnostic only; the independent FE checks below remain authoritative.
    return (float(components.get("pde", np.inf)) <= 2e-3 and
            float(components.get("bc", np.inf)) <= 5e-3 and
            float(components.get("rayleigh", np.inf)) <= 1e-4)

def train_deflated_mode(
    mu: float, mode_order: int, basis_number: int, lower_models: list[torch.nn.Module],
) -> dict:
    seed = 9000+100*basis_number+mode_order
    run_seed = int(CONFIG["seed"]+seed+round(1000*mu))
    random.seed(run_seed); np.random.seed(run_seed); torch.manual_seed(run_seed)
    initial_center = (mode_order*np.pi)**2*np.sqrt(max(0.2, 1.0-0.5*mu))
    initial_omega = float(np.random.uniform(0.9, 1.1)*initial_center)
    model = DeflatedModePINN(mode_order, run_seed, initial_omega).to(gate_b_device)
    checkpoint_path = CHECKPOINT_ROOT / (
        f"gate_d_{gate_d_mode}_basis_{basis_number:02d}_mu_{mu:.5f}_mode_{mode_order}.pt")
    gate_d_training_signature = {
        "mode_order": int(mode_order),
        "basis_number": int(basis_number),
        "mu": float(mu),
        "adam_epochs": int(gate_d_adam_epochs),
        "lbfgs_iterations": int(gate_d_lbfgs_iterations),
        "collocation_points": int(CONFIG["collocation_points"]),
        "architecture": [1, 32, 32, 32, 1],
        "physics": "variable_EI_Euler_Bernoulli_deflated_modes_v05",
    }
    if CONFIG["gate_d_resume"]:
        def gate_d_checkpoint_compatible(value):
            if not isinstance(value, dict):
                return False
            basic = bool(
                value.get("status") == "complete"
                and value.get("mode_order") == mode_order
                and np.isclose(value.get("mu", np.nan), mu)
                and value.get("adam_epochs") == gate_d_adam_epochs
                and gate_d_internal_converged(value.get("final_components", {}))
            )
            recorded_signature = value.get("training_signature")
            return basic and (
                recorded_signature is None
                or recorded_signature == gate_d_training_signature
            )
        saved = load_checkpoint_with_legacy_fallback(
            checkpoint_path,
            map_location=gate_b_device,
            stage=f"Gate D basis={basis_number} mu={mu:.5f} mode={mode_order}",
            validator=gate_d_checkpoint_compatible,
        )
        if saved is not None:
            model.load_state_dict(saved["model_state_dict"])
            return {"model": model, "mu": mu, "mode_order": mode_order,
                    "runtime_seconds": saved["runtime_seconds"], "history": saved["history"],
                    "checkpoint_path": checkpoint_path, "loaded_checkpoint": True}
    x_col = torch.linspace(0.0, 1.0, CONFIG["collocation_points"], device=gate_b_device).reshape(-1, 1)
    x_col.requires_grad_(True)
    adam = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        adam, T_max=max(1, gate_d_adam_epochs), eta_min=2e-5)
    history = []
    start = time.perf_counter()
    for epoch in range(gate_d_adam_epochs):
        adam.zero_grad(set_to_none=True)
        total, components = deflated_mode_loss(model, mu, x_col, lower_models)
        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite Gate D loss at mu={mu}, mode={mode_order}")
        total.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
        adam.step(); scheduler.step()
        if epoch == 0 or (epoch+1) % 25 == 0 or epoch+1 == gate_d_adam_epochs:
            history.append({"stage": "Adam", "iteration": epoch+1, "total": float(total.detach().cpu()),
                            "omega": float(model.omega.detach().cpu()),
                            **{name: float(value.detach().cpu()) for name, value in components.items()}})
    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=0.8, max_iter=gate_d_lbfgs_iterations,
        max_eval=int(1.3*gate_d_lbfgs_iterations), tolerance_grad=1e-11,
        tolerance_change=1e-13, history_size=80, line_search_fn="strong_wolfe")
    closure_calls = 0
    def closure():
        nonlocal closure_calls
        lbfgs.zero_grad(set_to_none=True)
        total, components = deflated_mode_loss(model, mu, x_col, lower_models)
        total.backward(); closure_calls += 1
        if closure_calls == 1 or closure_calls % 10 == 0:
            history.append({"stage": "L-BFGS", "iteration": closure_calls,
                            "total": float(total.detach().cpu()),
                            "omega": float(model.omega.detach().cpu()),
                            **{name: float(value.detach().cpu()) for name, value in components.items()}})
        return total
    lbfgs.step(closure)
    final_total, final_components = deflated_mode_loss(model, mu, x_col, lower_models)
    continuation_rounds = 0
    while not gate_d_internal_converged(final_components) and continuation_rounds < 3:
        continuation_rounds += 1
        print(f"  adaptive continuation {continuation_rounds}/3 for mu={mu:.5f}, mode={mode_order}")
        refinement_epochs = max(300, gate_d_adam_epochs//2)
        adam = torch.optim.Adam(model.parameters(), lr=5e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            adam, T_max=refinement_epochs, eta_min=1e-5)
        for epoch in range(refinement_epochs):
            adam.zero_grad(set_to_none=True)
            total, components = deflated_mode_loss(model, mu, x_col, lower_models)
            if not torch.isfinite(total):
                raise FloatingPointError(f"Non-finite Gate D continuation at mu={mu}, mode={mode_order}")
            total.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
            adam.step(); scheduler.step()
            if epoch == 0 or (epoch+1) % 25 == 0 or epoch+1 == refinement_epochs:
                history.append({"stage": f"Adam-cont-{continuation_rounds}",
                                "iteration": epoch+1, "total": float(total.detach().cpu()),
                                "omega": float(model.omega.detach().cpu()),
                                **{name: float(value.detach().cpu()) for name, value in components.items()}})
        lbfgs = torch.optim.LBFGS(
            model.parameters(), lr=0.8, max_iter=gate_d_lbfgs_iterations,
            max_eval=int(1.3*gate_d_lbfgs_iterations), tolerance_grad=1e-11,
            tolerance_change=1e-13, history_size=80, line_search_fn="strong_wolfe")
        closure_calls = 0
        def continuation_closure():
            nonlocal closure_calls
            lbfgs.zero_grad(set_to_none=True)
            total, components = deflated_mode_loss(model, mu, x_col, lower_models)
            total.backward(); closure_calls += 1
            if closure_calls == 1 or closure_calls % 10 == 0:
                history.append({"stage": f"L-BFGS-cont-{continuation_rounds}",
                                "iteration": closure_calls, "total": float(total.detach().cpu()),
                                "omega": float(model.omega.detach().cpu()),
                                **{name: float(value.detach().cpu()) for name, value in components.items()}})
            return total
        lbfgs.step(continuation_closure)
        final_total, final_components = deflated_mode_loss(model, mu, x_col, lower_models)
    runtime_seconds = time.perf_counter()-start
    safe_torch_save({
        "status": "complete", "mu": mu, "mode_order": mode_order,
        "initial_omega": initial_omega, "omega": float(model.omega.detach().cpu()),
        "model_state_dict": model.state_dict(),
        "adam_epochs": gate_d_adam_epochs, "lbfgs_closure_calls": closure_calls,
        "adaptive_continuation_rounds": continuation_rounds,
        "runtime_seconds": runtime_seconds, "history": history,
        "final_loss": float(final_total.detach().cpu()),
        "final_components": {name: float(value.detach().cpu()) for name, value in final_components.items()},
        "training_signature": gate_d_training_signature,
        "torch_version": torch.__version__,
        "device": str(gate_b_device),
        "checkpoint_format": "V0.6_FINAL_atomic_cpu_stream",
    }, checkpoint_path,
        stage=f"Gate D complete basis={basis_number} mu={mu:.5f} mode={mode_order}",
        retries=CONFIG["checkpoint_save_retries"],
        minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2)
    return {"model": model, "mu": mu, "mode_order": mode_order,
            "runtime_seconds": runtime_seconds, "history": history,
            "checkpoint_path": checkpoint_path, "loaded_checkpoint": False}

def model_state(model: torch.nn.Module, x_values: np.ndarray) -> dict:
    x = torch.as_tensor(x_values, device=gate_b_device).reshape(-1, 1).clone().requires_grad_(True)
    phi = model(x); phi_2 = differentiate(phi, x, 2)
    phi_3 = differentiate(phi_2, x, 1); phi_4 = differentiate(phi_3, x, 1)
    return {"phi": phi.detach().cpu().numpy().ravel(),
            "phi_2": phi_2.detach().cpu().numpy().ravel(),
            "phi_3": phi_3.detach().cpu().numpy().ravel(),
            "phi_4": phi_4.detach().cpu().numpy().ravel()}

gate_d_models_by_basis = []
gate_d_training_rows = []
gate_d_verification_rows = []
gate_d_validation_x = np.linspace(0.0, 1.0, CONFIG["validation_points"])

for basis_index, gate_c_run in enumerate(gate_c_basis_runs, start=1):
    mu = float(gate_c_run["mu"])
    models = [gate_c_run["model"]]
    for mode_order in equivalent_mode_orders[1:]:
        print(f"Gate D Full PINN: basis {basis_index}/{len(gate_c_basis_runs)}, mu={mu:.5f}, mode={mode_order}")
        trained = train_deflated_mode(mu, mode_order, basis_index, models)
        models.append(trained["model"])
        gate_d_training_rows.append({"basis_number": basis_index, "mu": mu, "mode_order": mode_order,
                                     "runtime_seconds": trained["runtime_seconds"],
                                     "checkpoint": str(trained["checkpoint_path"]),
                                     "checkpoint_bytes": int(trained["checkpoint_path"].stat().st_size),
                                     "loaded_checkpoint": trained["loaded_checkpoint"]})
        # Persist progress after every successfully verified checkpoint. A later
        # disconnect or filesystem retry therefore resumes at the next mode.
        gate_d_progress_path = OUTPUT_ROOT / "gate_d_progress.json"
        gate_d_progress_path.write_text(
            json_dumps_safe({
                "version": VERSION,
                "last_completed_basis": basis_index,
                "last_completed_mu": mu,
                "last_completed_mode": mode_order,
                "completed_model_count": len(gate_d_training_rows),
                "checkpoint": str(trained["checkpoint_path"]),
                "checkpoint_bytes": int(trained["checkpoint_path"].stat().st_size),
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            }, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    gate_d_models_by_basis.append(models)
    fe_omegas, fe_modes = variable_beam_fe_modes(mu, gate_d_validation_x, n_elements=160, n_modes=4)
    states = [model_state(model, gate_d_validation_x) for model in models]
    normalized_shapes = []
    for mode_index, (model, state) in enumerate(zip(models, states)):
        shape = state["phi"].copy()
        shape /= np.sqrt(numpy_trapezoid(shape**2, gate_d_validation_x))
        if numpy_trapezoid(shape*fe_modes[:, mode_index], gate_d_validation_x) < 0.0:
            shape = -shape
        normalized_shapes.append(shape)
        omega = float(model.omega.detach().cpu())
        ei, ei_1, ei_2 = local_stiffness_numpy(gate_d_validation_x, mu)
        residual = (ei*state["phi_4"]+2*ei_1*state["phi_3"]+ei_2*state["phi_2"]-
                    omega**2*state["phi"])
        scale = (np.sqrt(np.mean((residual+omega**2*state["phi"])**2))+
                 np.sqrt(np.mean((omega**2*state["phi"])**2))+1e-15)
        mac = float(numpy_trapezoid(shape*fe_modes[:, mode_index], gate_d_validation_x)**2)
        gate_d_verification_rows.append({
            "basis_number": basis_index, "mu": mu, "mode_order": mode_index+1,
            "omega": omega, "fe_omega": fe_omegas[mode_index],
            "frequency_relative_error": abs(omega-fe_omegas[mode_index])/fe_omegas[mode_index],
            "mac": mac, "shape_relative_l2": float(np.sqrt(numpy_trapezoid(
                (shape-fe_modes[:, mode_index])**2, gate_d_validation_x))),
            "relative_residual_rms": float(np.sqrt(np.mean(residual**2))/scale),
        })
    gram = np.array([[numpy_trapezoid(a*b, gate_d_validation_x) for b in normalized_shapes]
                     for a in normalized_shapes])
    off_diagonal = np.max(np.abs(gram-np.eye(4)))
    for row in gate_d_verification_rows[-4:]:
        row["maximum_mass_orthogonality_error"] = float(off_diagonal)

gate_d_training = pd.DataFrame(gate_d_training_rows)
gate_d_full_pinn_verification = pd.DataFrame(gate_d_verification_rows)
display(gate_d_full_pinn_verification)
gate_d_training.to_csv(TABLE_ROOT / f"gate_d_full_pinn_training_{gate_d_mode}.csv", index=False)
gate_d_full_pinn_verification.to_csv(
    TABLE_ROOT / f"gate_d_full_pinn_verification_{gate_d_mode}.csv", index=False)

gate_d_full_pinn_checks = {
    "frequency_error_le_1pct": bool((gate_d_full_pinn_verification["frequency_relative_error"] <= 0.01).all()),
    "mac_ge_0p99": bool((gate_d_full_pinn_verification["mac"] >= 0.99).all()),
    "relative_residual_le_0p03": bool((gate_d_full_pinn_verification["relative_residual_rms"] <= 0.03).all()),
    "orthogonality_error_le_0p01": bool((gate_d_full_pinn_verification["maximum_mass_orthogonality_error"] <= 0.01).all()),
}
if not all(gate_d_full_pinn_checks.values()):
    raise RuntimeError(f"Gate D stopped: multimodal Full-PINN verification failed: {gate_d_full_pinn_checks}")

# Figure 10: representative multimodal verification at the median selected parameter.
representative_index = int(np.argmin(np.abs(np.asarray(selected_mus)-np.median(candidate_mus))))
representative_mu = selected_mus[representative_index]
representative_models = gate_d_models_by_basis[representative_index]
fe_omegas_rep, fe_modes_rep = variable_beam_fe_modes(
    representative_mu, gate_d_validation_x, n_elements=160, n_modes=4)
fig, axes = plt.subplots(2, 2, figsize=(11.4, 7.2), sharex=True)
for mode_index, ax in enumerate(axes.ravel()):
    state = model_state(representative_models[mode_index], gate_d_validation_x)
    shape = state["phi"]/np.sqrt(numpy_trapezoid(state["phi"]**2, gate_d_validation_x))
    if numpy_trapezoid(shape*fe_modes_rep[:, mode_index], gate_d_validation_x) < 0.0:
        shape = -shape
    ax.plot(gate_d_validation_x, shape, label="Full PINN", color="#2a9d8f")
    ax.plot(gate_d_validation_x, fe_modes_rep[:, mode_index], "k--", label="160-element FE")
    ax.set_title(f"Equivalent mode {mode_index+1}")
    ax.set_ylabel("Normalized mode")
    ax.legend(fontsize=8)
for ax in axes[-1, :]: ax.set_xlabel("Dimensionless coordinate x")
fig.suptitle(f"Figure 10 — Multimodal Full-PINN verification at μ={representative_mu:.3f}", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_10_gate_d_multimodal_full_pinn_{gate_d_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

,quantity,value,unit,source
0,total girder length,58.0,m,https://bwk.kuleuven.be/bwm/z24
1,span lengths,14 / 30 / 14,m,https://bwk.kuleuven.be/bwm/z24
2,normalized Koppigen-pier coordinate,0.758621,x/L,derived as (14+30)/58
3,effective-zone width,0.14,x/L,pilot assumption; not a measured crack length


Gate D Full PINN: basis 1/5, mu=0.35000, mode=2
Gate D Full PINN: basis 1/5, mu=0.35000, mode=3
Gate D Full PINN: basis 1/5, mu=0.35000, mode=4
Gate D Full PINN: basis 2/5, mu=0.70000, mode=2
  adaptive continuation 1/3 for mu=0.70000, mode=2
Gate D Full PINN: basis 2/5, mu=0.70000, mode=3
  adaptive continuation 1/3 for mu=0.70000, mode=3
Gate D Full PINN: basis 2/5, mu=0.70000, mode=4
  adaptive continuation 1/3 for mu=0.70000, mode=4
Gate D Full PINN: basis 3/5, mu=0.00000, mode=2
Gate D Full PINN: basis 3/5, mu=0.00000, mode=3
Gate D Full PINN: basis 3/5, mu=0.00000, mode=4
Gate D Full PINN: basis 4/5, mu=0.60000, mode=2
  adaptive continuation 1/3 for mu=0.60000, mode=2
Gate D Full PINN: basis 4/5, mu=0.60000, mode=3
  adaptive continuation 1/3 for mu=0.60000, mode=3
Gate D Full PINN: basis 4/5, mu=0.60000, mode=4
Gate D Full PINN: basis 5/5, mu=0.15000, mode=2
Gate D Full PINN: basis 5/5, mu=0.15000, mode=3
Gate D Full PINN: basis 5/5, mu=0.15000, mode=4


,basis_number,mu,mode_order,omega,fe_omega,frequency_relative_error,mac,shape_relative_l2,relative_residual_rms,maximum_mass_orthogonality_error
0,1,0.35,1,9.162418,9.162625,2.258293e-05,1.000000,1.307607e-05,0.002047,3.808644e-06
1,1,0.35,2,36.180680,36.180842,4.473213e-06,1.000000,1.146928e-05,0.000037,3.808644e-06
2,1,0.35,3,82.772543,82.772435,1.306587e-06,1.000000,6.777482e-06,0.000090,3.808644e-06
3,1,0.35,4,147.224826,147.224715,7.545517e-07,1.000000,5.485912e-06,0.000086,3.808644e-06
4,2,0.70,1,7.938284,7.938724,5.536471e-05,1.000000,9.017086e-06,0.000735,2.591004e-03
5,2,0.70,2,31.524853,31.527189,7.410291e-05,1.000000,9.060478e-05,0.003839,2.591004e-03
6,2,0.70,3,74.453384,74.234219,2.952349e-03,0.999961,6.264921e-03,0.017938,2.591004e-03
7,2,0.70,4,131.883051,131.881403,1.249455e-05,0.999997,1.700127e-03,0.000704,2.591004e-03
8,3,0.00,1,9.869604,9.869601,3.383149e-07,1.000000,3.660020e-08,0.000068,1.016700e-05
9,3,0.00,2,39.478411,39.478417,1.663466e-07,1.000000,5.311232e-07,0.000003,1.016700e-05


### 15.2 Joint multimodal GPT-PINN and Z24 inverse calibration

For each equivalent mode, the five greedily selected Full-PINNs form a separate basis. A joint reduced solve optimizes all modal coefficients and eigenfrequencies while enforcing normalization, mutual orthogonality, boundary moments, frequency ordering, and the precomputed strong-form residual. The multimodal surrogate is checked at the Gate C held-out parameter midpoints before it is used for Z24.

Absolute equivalent-beam frequencies are not fitted to the Z24. Instead, both physics and measurements are divided by their healthy values. Calibration therefore uses the dimensionless shift vector

\[
q_s=[f_{1,s}/f_{1,0},\ f_{3,s}/f_{3,0},\ f_{4,s}/f_{4,0}],
\]

with uncertainty weights derived from the setup-bootstrap intervals plus a spectral-resolution floor. The inferred $\mu_s$ is an effective model-updating coordinate only.

In [22]:
def precompute_multimode_basis(models_by_basis: list[list[torch.nn.Module]], x_values: np.ndarray) -> list[dict]:
    result = []
    for mode_index in range(4):
        states = [model_state(models[mode_index], x_values) for models in models_by_basis]
        result.append({"x": np.asarray(x_values),
                       **{name: np.column_stack([state[name] for state in states])
                          for name in ("phi", "phi_2", "phi_3", "phi_4")}})
    return result

gate_d_basis_collocation = precompute_multimode_basis(gate_d_models_by_basis, collocation_x_np)
gate_d_selected_omegas = np.array([
    [float(model.omega.detach().cpu()) for model in models]
    for models in gate_d_models_by_basis
])

def joint_reduced_loss(
    coefficients: torch.Tensor, raw_omegas: torch.Tensor, mu: float, bases: list[dict],
) -> tuple[torch.Tensor, dict]:
    x = bases[0]["x"]
    ei_np, ei_1_np, ei_2_np = local_stiffness_numpy(x.detach().cpu().numpy(), mu)
    ei = torch.as_tensor(ei_np, device=x.device); ei_1 = torch.as_tensor(ei_1_np, device=x.device)
    ei_2 = torch.as_tensor(ei_2_np, device=x.device)
    omegas = torch.nn.functional.softplus(raw_omegas)+1e-8
    shapes, second_derivatives, residuals = [], [], []
    total = torch.zeros((), device=x.device)
    component_rows = []
    for mode_index, order in enumerate(equivalent_mode_orders):
        basis = bases[mode_index]
        coeff = coefficients[:, mode_index]
        phi = basis["phi"]@coeff; phi_2 = basis["phi_2"]@coeff
        phi_3 = basis["phi_3"]@coeff; phi_4 = basis["phi_4"]@coeff
        residual = ei*phi_4+2*ei_1*phi_3+ei_2*phi_2-omegas[mode_index].square()*phi
        mass = torch.trapezoid(phi.square(), x)
        energy = torch.trapezoid(ei*phi_2.square(), x)
        rayleigh = energy/mass.clamp_min(1e-12)
        moment_bc = torch.stack((ei[0]*phi_2[0], ei[-1]*phi_2[-1]))
        scale_4 = (order*np.pi)**4
        pde = torch.mean((residual/scale_4)**2)
        bc = torch.mean((moment_bc/(order*np.pi)**2)**2)
        norm = (mass-1.0).square()
        rayleigh_loss = ((omegas[mode_index].square()-rayleigh)/scale_4).square()
        total = total+50.0*pde+20.0*bc+50.0*norm+50.0*rayleigh_loss
        shapes.append(phi); second_derivatives.append(phi_2); residuals.append(residual)
        component_rows.append({"pde": pde, "bc": bc, "norm": norm,
                               "rayleigh": rayleigh_loss, "mass": mass})
    orth = torch.zeros((), device=x.device)
    for left in range(4):
        for right in range(left+1, 4):
            orth = orth+torch.trapezoid(shapes[left]*shapes[right], x).square()
    ordering = sum(torch.nn.functional.relu(
        omegas[index]-omegas[index+1]+1e-3).square() for index in range(3))
    total = total+100.0*orth+10.0*ordering+1e-10*torch.mean(coefficients.square())
    return total, {"omegas": omegas, "shapes": shapes, "phi_2": second_derivatives,
                   "residuals": residuals, "orth": orth, "ordering": ordering,
                   "components": component_rows}

def solve_joint_multimode_gpt(
    mu: float, bases_numpy: list[dict], adam_steps: int | None = None,
    lbfgs_iterations: int | None = None,
) -> dict:
    adam_steps = CONFIG["gate_d_reduced_adam_steps"] if adam_steps is None else adam_steps
    lbfgs_iterations = CONFIG["gate_d_reduced_lbfgs_iterations"] if lbfgs_iterations is None else lbfgs_iterations
    bases = []
    for basis_numpy in bases_numpy:
        bases.append({"x": torch.as_tensor(basis_numpy["x"], device=gate_b_device),
                      **{name: torch.as_tensor(basis_numpy[name], device=gate_b_device)
                         for name in ("phi", "phi_2", "phi_3", "phi_4")}})
    nearest = int(np.argmin(np.abs(np.asarray(selected_mus)-mu)))
    initial_coeff = np.zeros((len(selected_mus), 4)); initial_coeff[nearest, :] = 1.0
    coefficients = torch.nn.Parameter(torch.as_tensor(initial_coeff, device=gate_b_device))
    initial_omegas = gate_d_selected_omegas[nearest]
    raw_omegas = torch.nn.Parameter(torch.as_tensor([inverse_softplus(value) for value in initial_omegas],
                                                    device=gate_b_device))
    adam = torch.optim.Adam([coefficients, raw_omegas], lr=3e-2)
    start = time.perf_counter()
    for _ in range(adam_steps):
        adam.zero_grad(set_to_none=True)
        total, _ = joint_reduced_loss(coefficients, raw_omegas, mu, bases)
        total.backward(); adam.step()
    lbfgs = torch.optim.LBFGS([coefficients, raw_omegas], lr=0.8, max_iter=lbfgs_iterations,
                              max_eval=int(1.4*lbfgs_iterations), tolerance_grad=1e-12,
                              tolerance_change=1e-14, history_size=50, line_search_fn="strong_wolfe")
    def closure():
        lbfgs.zero_grad(set_to_none=True)
        total, _ = joint_reduced_loss(coefficients, raw_omegas, mu, bases)
        total.backward(); return total
    lbfgs.step(closure)
    with torch.no_grad():
        _, before = joint_reduced_loss(coefficients, raw_omegas, mu, bases)
        for mode_index in range(4):
            mass = before["components"][mode_index]["mass"]
            coefficients[:, mode_index].div_(torch.sqrt(mass.clamp_min(1e-12)))
            shape = bases[mode_index]["phi"]@coefficients[:, mode_index]
            reference = torch.sin((mode_index+1)*np.pi*bases[mode_index]["x"])
            if torch.trapezoid(shape*reference, bases[mode_index]["x"]) < 0.0:
                coefficients[:, mode_index].mul_(-1.0)
        total, final = joint_reduced_loss(coefficients, raw_omegas, mu, bases)
    runtime = time.perf_counter()-start
    relative_residuals = []
    for mode_index in range(4):
        residual = final["residuals"][mode_index].detach().cpu().numpy()
        phi = final["shapes"][mode_index].detach().cpu().numpy()
        omega = float(final["omegas"][mode_index].detach().cpu())
        stiff = residual+omega**2*phi
        scale = np.sqrt(np.mean(stiff**2))+np.sqrt(np.mean((omega**2*phi)**2))+1e-15
        relative_residuals.append(float(np.sqrt(np.mean(residual**2))/scale))
    return {"mu": mu, "coefficients": coefficients.detach().cpu().numpy(),
            "omegas": final["omegas"].detach().cpu().numpy(),
            "physics_loss": float(total.detach().cpu()),
            "maximum_relative_residual_rms": max(relative_residuals),
            "orthogonality_loss": float(final["orth"].detach().cpu()),
            "runtime_seconds": runtime}

# Independent midpoint validation of the joint reduced four-mode surrogate.
gate_d_basis_validation = precompute_multimode_basis(gate_d_models_by_basis, gate_d_validation_x)
gate_d_reduced_rows = []
for mu in validation_mus:
    solution = solve_joint_multimode_gpt(float(mu), gate_d_basis_collocation)
    fe_omegas, fe_modes = variable_beam_fe_modes(float(mu), gate_d_validation_x, n_elements=160, n_modes=4)
    predicted_shapes = []
    for mode_index in range(4):
        shape = gate_d_basis_validation[mode_index]["phi"]@solution["coefficients"][:, mode_index]
        shape /= np.sqrt(numpy_trapezoid(shape**2, gate_d_validation_x))
        if numpy_trapezoid(shape*fe_modes[:, mode_index], gate_d_validation_x) < 0.0: shape = -shape
        predicted_shapes.append(shape)
        gate_d_reduced_rows.append({
            "mu": float(mu), "mode_order": mode_index+1,
            "omega": solution["omegas"][mode_index], "fe_omega": fe_omegas[mode_index],
            "frequency_relative_error": abs(solution["omegas"][mode_index]-fe_omegas[mode_index])/fe_omegas[mode_index],
            "mac": float(numpy_trapezoid(shape*fe_modes[:, mode_index], gate_d_validation_x)**2),
            "shape_relative_l2": float(np.sqrt(numpy_trapezoid(
                (shape-fe_modes[:, mode_index])**2, gate_d_validation_x))),
            "maximum_query_relative_residual_rms": solution["maximum_relative_residual_rms"],
            "orthogonality_loss": solution["orthogonality_loss"],
            "query_seconds": solution["runtime_seconds"],
        })
gate_d_reduced_validation = pd.DataFrame(gate_d_reduced_rows)
display(gate_d_reduced_validation.groupby("mode_order").agg(
    maximum_frequency_error=("frequency_relative_error", "max"),
    minimum_mac=("mac", "min"), maximum_shape_l2=("shape_relative_l2", "max")))
gate_d_reduced_validation.to_csv(
    TABLE_ROOT / f"gate_d_multimodal_gpt_validation_{gate_d_mode}.csv", index=False)
gate_d_reduced_checks = {
    "frequency_error_le_1pct": bool((gate_d_reduced_validation["frequency_relative_error"] <= 0.01).all()),
    "mac_ge_0p99": bool((gate_d_reduced_validation["mac"] >= 0.99).all()),
    "shape_l2_le_0p03": bool((gate_d_reduced_validation["shape_relative_l2"] <= 0.03).all()),
    "relative_residual_le_0p03": bool((gate_d_reduced_validation["maximum_query_relative_residual_rms"] <= 0.03).all()),
    "orthogonality_loss_le_1e-3": bool((gate_d_reduced_validation["orthogonality_loss"] <= 1e-3).all()),
}
if not all(gate_d_reduced_checks.values()):
    raise RuntimeError(f"Gate D stopped: multimodal GPT-PINN verification failed: {gate_d_reduced_checks}")

# Dense physics response used for inverse calibration.
response_count = (CONFIG["gate_d_full_response_points"] if gate_d_mode == "full"
                  else CONFIG["gate_d_smoke_response_points"])
gate_d_response_mus = np.linspace(mu_min, mu_max, response_count)
response_rows = []
for index, mu in enumerate(gate_d_response_mus):
    if index % 20 == 0: print(f"Multimodal response {index+1}/{response_count}")
    solution = solve_joint_multimode_gpt(float(mu), gate_d_basis_collocation)
    response_rows.append({"mu": float(mu), "physics_loss": solution["physics_loss"],
                          "maximum_relative_residual_rms": solution["maximum_relative_residual_rms"],
                          "orthogonality_loss": solution["orthogonality_loss"],
                          "query_seconds": solution["runtime_seconds"],
                          **{f"omega_{order}": solution["omegas"][order-1]
                             for order in equivalent_mode_orders}})
gate_d_physics_response = pd.DataFrame(response_rows)
for order in equivalent_mode_orders:
    baseline = float(gate_d_physics_response.loc[0, f"omega_{order}"])
    gate_d_physics_response[f"ratio_{order}"] = gate_d_physics_response[f"omega_{order}"]/baseline
gate_d_physics_response.to_csv(
    TABLE_ROOT / f"gate_d_multimodal_physics_response_{gate_d_mode}.csv", index=False)

# Assemble observed ratios and resolution-aware uncertainty.
observation = bootstrap_table.copy()
observation["scenario"] = observation["scenario"].astype(str).str.zfill(2)
observation = observation[observation["mode"].isin(retained_experimental_modes)].copy()
healthy = observation[observation["scenario"] == "01"].set_index("mode")
frequency_resolution_hz = CONFIG["sampling_rate_hz"]/CONFIG["fdd_nperseg"]
observed_rows = []
for row in observation.itertuples():
    base = healthy.loc[row.mode]
    ratio = row.median_frequency_hz/base.median_frequency_hz
    half_width = max(0.5*(row.ci95_high_hz-row.ci95_low_hz), 0.5*frequency_resolution_hz)
    base_half_width = max(0.5*(base.ci95_high_hz-base.ci95_low_hz), 0.5*frequency_resolution_hz)
    legacy_marginal_sigma = np.sqrt(
        (half_width/base.median_frequency_hz)**2
        + (ratio*base_half_width/base.median_frequency_hz)**2
    )
    covariance = gate_a_ratio_covariance_by_scenario[str(row.scenario)]
    covariance_position = retained_experimental_modes.index(int(row.mode))
    joint_bootstrap_sigma = float(np.sqrt(covariance[
        covariance_position, covariance_position
    ]))
    legacy_one_sigma = legacy_marginal_sigma / float(stats.norm.ppf(0.975))
    sigma_ratio = max(joint_bootstrap_sigma, legacy_one_sigma)
    observed_rows.append({
        "scenario": row.scenario,
        "delta_pier_mm": row.delta_pier_mm,
        "experimental_mode": int(row.mode),
        "equivalent_mode": retained_equivalent_modes[
            retained_experimental_modes.index(int(row.mode))
        ],
        "frequency_hz": row.median_frequency_hz,
        "frequency_ratio": ratio,
        "sigma_frequency_ratio": sigma_ratio,
        "joint_bootstrap_sigma": joint_bootstrap_sigma,
        "legacy_ci_propagation_sigma": legacy_marginal_sigma,
    })
gate_d_observed_ratios = pd.DataFrame(observed_rows)
gate_d_observed_ratios.to_csv(TABLE_ROOT / "gate_d_z24_observed_frequency_ratios.csv", index=False)

ratio_interpolators = {
    order: PchipInterpolator(gate_d_physics_response["mu"], gate_d_physics_response[f"ratio_{order}"],
                             extrapolate=False)
    for order in retained_equivalent_modes
}

def model_retained_ratios(mu: float) -> np.ndarray:
    return np.array([float(ratio_interpolators[order](mu)) for order in retained_equivalent_modes])

calibration_rows = []
calibration_residual_rows = []
for scenario in scenarios:
    scenario_observed = gate_d_observed_ratios[gate_d_observed_ratios["scenario"] == scenario]
    q_observed = scenario_observed.set_index("experimental_mode").loc[retained_experimental_modes, "frequency_ratio"].to_numpy()
    sigma = scenario_observed.set_index("experimental_mode").loc[retained_experimental_modes, "sigma_frequency_ratio"].to_numpy()
    def objective(mu_value):
        return float(np.mean(((q_observed-model_retained_ratios(float(mu_value)))/sigma)**2))
    if scenario == "01":
        mu_hat, objective_value = 0.0, objective(0.0)
    else:
        optimized = minimize_scalar(objective, bounds=(mu_min, mu_max), method="bounded",
                                    options={"xatol": 1e-8})
        mu_hat, objective_value = float(optimized.x), float(optimized.fun)
    q_model = model_retained_ratios(mu_hat)
    derivative = np.array([float(ratio_interpolators[order].derivative()(mu_hat))
                           for order in retained_equivalent_modes])
    information = float(np.sum((derivative/sigma)**2))
    sigma_mu = 1.0/np.sqrt(information) if information > 0.0 else np.inf
    calibration_rows.append({"scenario": scenario,
                             "delta_pier_mm": float(scenario_observed["delta_pier_mm"].iloc[0]),
                             "mu_effective": mu_hat, "sigma_mu_linearized": sigma_mu,
                             "weighted_mean_square_residual": objective_value,
                             "modal_ratio_rmse": float(np.sqrt(np.mean((q_observed-q_model)**2))),
                             "at_lower_bound": bool(mu_hat <= mu_min+1e-5),
                             "at_upper_bound": bool(mu_hat >= mu_max-1e-5)})
    for experimental_mode, equivalent_order, observed_ratio, predicted_ratio, sigma_value in zip(
            retained_experimental_modes, retained_equivalent_modes, q_observed, q_model, sigma):
        calibration_residual_rows.append({"scenario": scenario,
                                          "delta_pier_mm": float(scenario_observed["delta_pier_mm"].iloc[0]),
                                          "experimental_mode": experimental_mode,
                                          "equivalent_mode": equivalent_order,
                                          "observed_ratio": observed_ratio, "predicted_ratio": predicted_ratio,
                                          "residual": observed_ratio-predicted_ratio,
                                          "standardized_residual": (observed_ratio-predicted_ratio)/sigma_value})

gate_d_calibration = pd.DataFrame(calibration_rows).sort_values("delta_pier_mm").reset_index(drop=True)
gate_d_calibration_residuals = pd.DataFrame(calibration_residual_rows)
display(gate_d_calibration)
gate_d_calibration.to_csv(TABLE_ROOT / f"gate_d_z24_calibration_{gate_d_mode}.csv", index=False)
gate_d_calibration_residuals.to_csv(
    TABLE_ROOT / f"gate_d_z24_calibration_residuals_{gate_d_mode}.csv", index=False)

inferred_mu_differences = np.diff(gate_d_calibration["mu_effective"].to_numpy())
gate_d_identifiability = {
    "maximum_sigma_mu": float(gate_d_calibration["sigma_mu_linearized"].replace([np.inf], np.nan).max()),
    "monotonic_non_decreasing_fraction": float(np.mean(inferred_mu_differences >= -1e-6)),
    "maximum_weighted_mean_square_residual": float(gate_d_calibration["weighted_mean_square_residual"].max()),
    "maximum_modal_ratio_rmse": float(gate_d_calibration["modal_ratio_rmse"].max()),
    "upper_bound_saturation_count": int(gate_d_calibration["at_upper_bound"].sum()),
}

fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.3))
colors = ["#264653", "#2a9d8f", "#e76f51"]
for color, experimental_mode, equivalent_order in zip(colors, retained_experimental_modes, retained_equivalent_modes):
    axes[0].plot(gate_d_physics_response["mu"], gate_d_physics_response[f"ratio_{equivalent_order}"],
                 color=color, label=f"equivalent mode {equivalent_order}")
    mode_obs = gate_d_observed_ratios[gate_d_observed_ratios["experimental_mode"] == experimental_mode]
    axes[0].scatter(gate_d_calibration["mu_effective"], mode_obs.sort_values("delta_pier_mm")["frequency_ratio"],
                    color=color, marker="o", edgecolor="white", zorder=3,
                    label=f"Z24 track f{experimental_mode}")
axes[0].set_xlabel(r"Effective parameter $\mu$"); axes[0].set_ylabel("Healthy-normalized frequency")
axes[0].set_title("Physics response and calibrated observations"); axes[0].legend(fontsize=7, ncol=2)
axes[1].errorbar(gate_d_calibration["delta_pier_mm"], gate_d_calibration["mu_effective"],
                 yerr=gate_d_calibration["sigma_mu_linearized"], fmt="o-", capsize=3, color="#6d597a")
axes[1].set_xlabel(r"Imposed settlement $\delta_{pier}$ [mm]")
axes[1].set_ylabel(r"Inferred effective $\mu$")
axes[1].set_title("Inverse model-updating coordinate")
for experimental_mode, group in gate_d_calibration_residuals.groupby("experimental_mode"):
    axes[2].plot(group["delta_pier_mm"], group["standardized_residual"], "o-", label=f"f{experimental_mode}")
axes[2].axhline(3.0, color="#e76f51", linestyle="--"); axes[2].axhline(-3.0, color="#e76f51", linestyle="--")
axes[2].set_xlabel(r"Imposed settlement $\delta_{pier}$ [mm]")
axes[2].set_ylabel("Standardized residual")
axes[2].set_title("Mode-wise calibration adequacy"); axes[2].legend()
fig.suptitle("Figure 11 — Z24 multimodal effective-parameter calibration", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_11_gate_d_z24_calibration_{gate_d_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

,maximum_frequency_error,minimum_mac,maximum_shape_l2
mode_order,,,
1,0.000322,1.000000,0.000105
2,0.004955,0.999991,0.003081
3,0.001125,0.999987,0.003657
4,0.003756,0.999991,0.002996


Multimodal response 1/141
Multimodal response 21/141
Multimodal response 41/141
Multimodal response 61/141
Multimodal response 81/141
Multimodal response 101/141
Multimodal response 121/141
Multimodal response 141/141


,scenario,delta_pier_mm,mu_effective,sigma_mu_linearized,weighted_mean_square_residual,modal_ratio_rmse,at_lower_bound,at_upper_bound
0,01,0.0,0.000000,0.006558,0.000000,0.000000,True,False
1,03,20.0,0.093933,0.006550,0.706327,0.002638,False,False
2,04,40.0,0.116487,0.006792,2.996557,0.005603,False,False
3,05,80.0,0.289511,0.007148,17.837735,0.013299,False,False
4,06,95.0,0.362872,0.005266,6.522567,0.008713,False,False


## 16. Leave-one-severity-out experiments

Each fold removes one nonzero settlement state from the monotonic settlement-to-effective-parameter relationship. A bounded power law $\mu(\delta)=a(\delta/95)^p$ is fitted only to the remaining modal-inverse estimates. The held-out settlement then predicts $\mu$ and the complete three-mode frequency-shift vector. The independently inferred held-out $\mu$, which uses that fold's modal observation, is reported only as a consistency reference and is never used in the fold fit. Folds at 20, 40, and 80 mm are interpolation; 95 mm is explicitly labeled extrapolation.

In [23]:
def monotonic_settlement_mapping(parameters: np.ndarray, delta_mm: np.ndarray) -> np.ndarray:
    amplitude, exponent = parameters
    return amplitude*np.power(np.asarray(delta_mm, dtype=float)/95.0, exponent)

loso_rows = []
nonzero_calibration = gate_d_calibration[gate_d_calibration["delta_pier_mm"] > 0.0].copy()
for heldout in nonzero_calibration.itertuples():
    training = gate_d_calibration[gate_d_calibration["scenario"] != heldout.scenario].copy()
    train_nonzero = training[training["delta_pier_mm"] > 0.0]
    weights = 1.0/np.maximum(train_nonzero["sigma_mu_linearized"].to_numpy(), 1e-3)
    fitted = least_squares(
        lambda parameters: weights*(monotonic_settlement_mapping(
            parameters, train_nonzero["delta_pier_mm"].to_numpy())-
            train_nonzero["mu_effective"].to_numpy()),
        x0=np.array([min(mu_max, max(0.05, train_nonzero["mu_effective"].max())), 1.0]),
        bounds=([0.0, 0.15], [mu_max, 4.0]), method="trf")
    mu_predicted = float(monotonic_settlement_mapping(fitted.x, np.array([heldout.delta_pier_mm]))[0])
    mu_predicted = float(np.clip(mu_predicted, mu_min, mu_max))
    predicted_ratios = model_retained_ratios(mu_predicted)
    heldout_observed_frame = gate_d_observed_ratios[
        gate_d_observed_ratios["scenario"] == heldout.scenario].set_index("experimental_mode")
    heldout_observed = heldout_observed_frame.loc[retained_experimental_modes, "frequency_ratio"].to_numpy()
    fold_type = "extrapolation" if heldout.delta_pier_mm == gate_d_calibration["delta_pier_mm"].max() else "interpolation"
    row = {"heldout_scenario": heldout.scenario, "heldout_delta_pier_mm": heldout.delta_pier_mm,
           "fold_type": fold_type, "training_scenarios": json_dumps_safe(training["scenario"].tolist()),
           "mapping_amplitude": fitted.x[0], "mapping_exponent": fitted.x[1],
           "predicted_mu_from_settlement": mu_predicted,
           "independent_mu_from_heldout_modes": heldout.mu_effective,
           "mu_consistency_absolute_error": abs(mu_predicted-heldout.mu_effective),
           "modal_ratio_rmse": float(np.sqrt(np.mean((predicted_ratios-heldout_observed)**2))),
           "modal_ratio_max_absolute_error": float(np.max(np.abs(predicted_ratios-heldout_observed))),
           "mapping_fit_cost": float(fitted.cost)}
    for experimental_mode, observed_ratio, predicted_ratio in zip(
            retained_experimental_modes, heldout_observed, predicted_ratios):
        row[f"observed_ratio_f{experimental_mode}"] = observed_ratio
        row[f"predicted_ratio_f{experimental_mode}"] = predicted_ratio
    loso_rows.append(row)

gate_d_loso = pd.DataFrame(loso_rows).sort_values("heldout_delta_pier_mm").reset_index(drop=True)
display(gate_d_loso)
gate_d_loso.to_csv(TABLE_ROOT / f"gate_d_leave_one_severity_out_{gate_d_mode}.csv", index=False)
gate_d_loso_interpolation = gate_d_loso[gate_d_loso["fold_type"] == "interpolation"]
gate_d_loso_extrapolation = gate_d_loso[gate_d_loso["fold_type"] == "extrapolation"]

gate_d_calibration_checks = {
    "effective_mu_monotonic": bool((np.diff(gate_d_calibration["mu_effective"]) >= -1e-6).all()),
    "maximum_sigma_mu_le_0p10": bool(gate_d_identifiability["maximum_sigma_mu"] <= 0.10),
    "maximum_weighted_mse_le_9": bool(gate_d_identifiability["maximum_weighted_mean_square_residual"] <= 9.0),
    "maximum_modal_ratio_rmse_le_1pct": bool(gate_d_identifiability["maximum_modal_ratio_rmse"] <= 0.01),
    "no_upper_bound_saturation": bool(gate_d_identifiability["upper_bound_saturation_count"] == 0),
    "interpolation_mu_consistency_le_0p12": bool(
        (gate_d_loso_interpolation["mu_consistency_absolute_error"] <= 0.12).all()),
    "interpolation_modal_ratio_rmse_le_1p5pct": bool(
        (gate_d_loso_interpolation["modal_ratio_rmse"] <= 0.015).all()),
}
full_gate_d_requirement = gate_d_mode == "full" and all(gate_d_full_pinn_checks.values()) and all(gate_d_reduced_checks.values())
if gate_d_mode == "smoke":
    gate_d_decision = ("SMOKE_COMPLETED_REQUIRES_FULL_RUN" if full_gate_d_requirement is False
                       else "SMOKE_PASS_REQUIRES_FULL_RUN")
else:
    gate_d_decision = ("GO_TO_STRUCTURAL_STATE_ESTIMATION_PILOT"
                       if full_gate_d_requirement and all(gate_d_calibration_checks.values())
                       else "NO_GO_REVISE_PHYSICAL_PARAMETERIZATION")

gate_d_diagnostic = {
    "decision": gate_d_decision, "execution_mode": gate_d_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "equivalent_model": "one-dimensional simply supported beam with Koppigen-centered smooth effective rigidity zone",
    "experimental_to_equivalent_mode_mapping": dict(zip(
        map(str, retained_experimental_modes), retained_equivalent_modes)),
    "mapping_interpretation": "rank association of healthy-normalized shifts; physical mode-shape identity is not claimed",
    "mu_interpretation": "effective model-updating coordinate; not measured stiffness loss and not pier settlement",
    "geometry_source": REFERENCE_URLS["z24_official"],
    "multimodal_full_pinn_checks": gate_d_full_pinn_checks,
    "multimodal_gpt_checks": gate_d_reduced_checks,
    "calibration_checks": gate_d_calibration_checks,
    "identifiability": gate_d_identifiability,
    "maximum_interpolation_mu_consistency_error": float(
        gate_d_loso_interpolation["mu_consistency_absolute_error"].max()),
    "maximum_interpolation_modal_ratio_rmse": float(gate_d_loso_interpolation["modal_ratio_rmse"].max()),
    "extrapolation_95mm_mu_consistency_error": float(
        gate_d_loso_extrapolation["mu_consistency_absolute_error"].iloc[0]),
    "extrapolation_95mm_modal_ratio_rmse": float(gate_d_loso_extrapolation["modal_ratio_rmse"].iloc[0]),
    "z24_calibration_performed": True, "capacity_or_rul_claim_performed": False,
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "gate_d_diagnostic.json").write_text(
    json_dumps_safe(gate_d_diagnostic, indent=2, sort_keys=True)+"\n", encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.3))
axes[0].plot(gate_d_calibration["delta_pier_mm"], gate_d_calibration["mu_effective"],
             "o-", color="#6d597a", label="modal inverse")
axes[0].scatter(gate_d_loso["heldout_delta_pier_mm"], gate_d_loso["predicted_mu_from_settlement"],
                marker="s", color="#e76f51", label="LOSO prediction")
for row in gate_d_loso.itertuples():
    axes[0].plot([row.heldout_delta_pier_mm, row.heldout_delta_pier_mm],
                 [row.independent_mu_from_heldout_modes, row.predicted_mu_from_settlement],
                 color="#777777", linewidth=1)
axes[0].set_xlabel(r"Imposed settlement $\delta_{pier}$ [mm]")
axes[0].set_ylabel(r"Effective $\mu$"); axes[0].set_title("Leave-one-severity-out parameter consistency")
axes[0].legend()
colors = {"interpolation": "#2a9d8f", "extrapolation": "#e76f51"}
for fold_type, group in gate_d_loso.groupby("fold_type"):
    axes[1].scatter(group["heldout_delta_pier_mm"], 100*group["modal_ratio_rmse"],
                    s=70, color=colors[fold_type], label=fold_type)
axes[1].axhline(1.5, color="#555555", linestyle="--", label="interpolation gate")
axes[1].set_xlabel("Held-out settlement [mm]"); axes[1].set_ylabel("Three-mode ratio RMSE [%]")
axes[1].set_title("Forward prediction of held-out modal shifts"); axes[1].legend()
fig.suptitle(f"Figure 12 — Z24 leave-one-severity-out: {gate_d_decision.replace('_', ' ')}", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT / f"figure_12_gate_d_loso_{gate_d_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_d_diagnostic, indent=2))

,heldout_scenario,heldout_delta_pier_mm,fold_type,training_scenarios,mapping_amplitude,mapping_exponent,predicted_mu_from_settlement,independent_mu_from_heldout_modes,mu_consistency_absolute_error,modal_ratio_rmse,modal_ratio_max_absolute_error,mapping_fit_cost,observed_ratio_f1,predicted_ratio_f1,observed_ratio_f3,predicted_ratio_f3,observed_ratio_f4,predicted_ratio_f4
0,03,20.0,interpolation,"[""01"", ""04"", ""05"", ""06""]",0.362865,1.313689,0.046858,0.093933,0.047076,0.008401,0.010466,0.000007,0.9875,0.991794,0.982843,0.991994,0.981439,0.991904
1,04,40.0,interpolation,"[""01"", ""03"", ""05"", ""06""]",0.356205,0.891499,0.164739,0.116487,0.048252,0.011390,0.017638,3.643698,0.9875,0.969862,0.977941,0.970086,0.974478,0.970426
2,05,80.0,interpolation,"[""01"", ""03"", ""04"", ""06""]",0.360266,1.081721,0.299151,0.289511,0.009640,0.013682,0.021708,15.409374,0.9625,0.940792,0.938725,0.943167,0.935035,0.943439
3,06,95.0,extrapolation,"[""01"", ""03"", ""04"", ""05""]",0.335522,0.998813,0.335522,0.362872,0.027350,0.010223,0.014407,13.457103,0.9375,0.931986,0.926471,0.935164,0.921114,0.935521


{
  "decision": "NO_GO_REVISE_PHYSICAL_PARAMETERIZATION",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "equivalent_model": "one-dimensional simply supported beam with Koppigen-centered smooth effective rigidity zone",
  "experimental_to_equivalent_mode_mapping": {
    "1": 1,
    "3": 3,
    "4": 4
  },
  "mapping_interpretation": "rank association of healthy-normalized shifts; physical mode-shape identity is not claimed",
  "mu_interpretation": "effective model-updating coordinate; not measured stiffness loss and not pier settlement",
  "geometry_source": "https://bwk.kuleuven.be/bwm/z24",
  "multimodal_full_pinn_checks": {
    "frequency_error_le_1pct": true,
    "mac_ge_0p99": true,
    "relative_residual_le_0p03": true,
    "orthogonality_error_le_0p01": true
  },
  "multimodal_gpt_checks": {
    "frequency_error_le_1pct": true,
    "mac_ge_0p99": true,
    "shape_l2_le_0p03": true,
    "relative_residual_le_0p03": tru

## 17. V0.6 FINAL covariance-aware physical parameterization — Gate E

Gate D is not overwritten: its one-coordinate simply supported equivalent beam remains a documented **NO-GO** when it fails the predeclared 1% multimodal criterion. Gate E uses the continuous three-span Euler–Bernoulli model with the official 14 m + 30 m + 14 m geometry, a local effective rigidity coordinate near the Koppigen pier, and a dimensionless vertical-restraint coordinate at 44 m.

The earlier V0.6 FINAL implementation treated the marginal ±3σ condition as a hard SLSQP feasibility box. After the corrected one-standard-deviation uncertainty and paired-setup covariance were propagated, no hard-box candidate was found and the notebook raised an exception. That exception mixed two different facts: numerical optimization and reduced-model adequacy.

V0.6 FINAL separates them:

1. The physical amplitudes are fitted by generalized least squares using the full paired-setup modal covariance.
2. A nonnegative isotropic model-form discrepancy standard deviation, \(\tau_E\), is profiled with a Gaussian likelihood:
   \[
   \Sigma_{s,\mathrm{eff}}=\Sigma_{s,\mathrm{obs}}+\tau_E^2 I.
   \]
3. The notebook records both the likelihood estimate and the **minimum** \(\tau_E\) required to satisfy the predeclared marginal criterion \(|r_{s,m}|/\sqrt{\Sigma_{s,\mathrm{eff},mm}}\le 3\).
4. The fit is eligible to proceed only when the selected discrepancy does not exceed the fixed cap of 1% in healthy-normalized modal-frequency ratio. If the cap is insufficient, Gate E is a scientific NO-GO rather than an optimizer crash.

The discrepancy is a reduced-model error term, not additional experimental noise and not permission to claim material stiffness, bearing stiffness, load capacity, failure probability, remaining life, or maintenance savings. It is estimated on AVT development data and must remain locked during the later FVT validation.


**V0.6 FINAL control-flow correction.** The positive-discrepancy feasible branch
now explicitly sets the feasibility flag, initializes all adequacy outputs before
branching, guards the grid size, handles the grid-origin case defensively, and
returns an auditable `adequacy_status`. This changes no physical equation,
training checkpoint, likelihood definition, discrepancy cap, or acceptance
threshold.


In [24]:
from functools import lru_cache
from scipy.linalg import eigh
from scipy.optimize import least_squares, minimize_scalar

gate_e_mode = CONFIG["gate_e_mode"]
if gate_e_mode not in {"full", "smoke"}:
    raise ValueError("GPT_PINN_GATE_E_MODE must be 'full' or 'smoke'")
if gate_e_mode == "full" and gate_d_mode != "full":
    raise RuntimeError("Full Gate E requires the audited full Gate D observations")

CONFIG_PATH.write_text(json_dumps_safe(CONFIG, indent=2, sort_keys=True) + "\n", encoding="utf-8")

GATE_E_LENGTH_M = 58.0
GATE_E_SUPPORTS_M = (0.0, 14.0, 44.0, 58.0)
GATE_E_RETAINED_RANKS_ZERO_BASED = np.array([0, 2, 3], dtype=int)
GATE_E_PARAMETER_BOUNDS = (
    np.array([0.0, 0.0]),
    np.array([0.9, 8.0]),
)

@lru_cache(maxsize=None)
def gate_e_mesh(n_elements: int) -> dict:
    if n_elements % 58 != 0:
        raise ValueError("Gate E mesh must place nodes exactly at 14 and 44 m")
    length = GATE_E_LENGTH_M
    element_length = length/n_elements
    nodes = np.linspace(0.0, length, n_elements + 1)
    support_nodes = [int(round(position/element_length)) for position in GATE_E_SUPPORTS_M]
    fixed_vertical_dofs = [2*support_nodes[index] for index in (0, 1, 3)]
    free_dofs = np.setdiff1d(np.arange(2*(n_elements + 1)), fixed_vertical_dofs)
    ke_unit = np.array([
        [12, 6*element_length, -12, 6*element_length],
        [6*element_length, 4*element_length**2, -6*element_length, 2*element_length**2],
        [-12, -6*element_length, 12, -6*element_length],
        [6*element_length, 2*element_length**2, -6*element_length, 4*element_length**2],
    ])/element_length**3
    me_unit = element_length*np.array([
        [156, 22*element_length, 54, -13*element_length],
        [22*element_length, 4*element_length**2, 13*element_length, -3*element_length**2],
        [54, 13*element_length, 156, -22*element_length],
        [-13*element_length, -3*element_length**2, -22*element_length, 4*element_length**2],
    ])/420.0
    size = 2*(n_elements + 1)
    mass = np.zeros((size, size))
    element_dofs = []
    element_centers = []
    for element in range(n_elements):
        dofs = np.arange(2*element, 2*element + 4)
        mass[np.ix_(dofs, dofs)] += me_unit
        element_dofs.append(dofs)
        element_centers.append(0.5*(nodes[element] + nodes[element + 1]))
    koppigen_vertical_dof = 2*support_nodes[2]
    koppigen_reduced_dof = int(np.flatnonzero(free_dofs == koppigen_vertical_dof)[0])
    return {
        "nodes": nodes, "free_dofs": free_dofs, "ke_unit": ke_unit,
        "mass_reduced": mass[np.ix_(free_dofs, free_dofs)],
        "element_dofs": element_dofs, "element_centers": np.asarray(element_centers),
        "koppigen_reduced_dof": koppigen_reduced_dof,
    }

def gate_e_modes(alpha: float, log10_support_ratio: float, n_elements: int = 58,
                 n_modes: int = 6) -> np.ndarray:
    mesh = gate_e_mesh(int(n_elements))
    size = 2*(int(n_elements) + 1)
    stiffness = np.zeros((size, size))
    width = CONFIG["gate_e_local_zone_width_m"]
    zone = np.exp(-0.5*((mesh["element_centers"] - 44.0)/width)**2)
    rigidity = np.maximum(0.08, 1.0 - float(alpha)*zone)
    for element, dofs in enumerate(mesh["element_dofs"]):
        stiffness[np.ix_(dofs, dofs)] += rigidity[element]*mesh["ke_unit"]
    free = mesh["free_dofs"]
    stiffness_reduced = stiffness[np.ix_(free, free)]
    spring_reference = CONFIG["gate_e_dimensionless_support_kappa"]/GATE_E_LENGTH_M**3
    spring = spring_reference*10.0**float(log10_support_ratio)
    k_index = mesh["koppigen_reduced_dof"]
    stiffness_reduced[k_index, k_index] += spring
    eigenvalues = eigh(stiffness_reduced, mesh["mass_reduced"],
                       subset_by_index=(0, n_modes - 1), check_finite=False,
                       eigvals_only=True)
    return np.sqrt(np.maximum(eigenvalues, 0.0))

gate_e_observations = gate_d_observed_ratios[
    (gate_d_observed_ratios["delta_pier_mm"] > 0.0) &
    (gate_d_observed_ratios["experimental_mode"].isin(retained_experimental_modes))
].sort_values(["delta_pier_mm", "experimental_mode"]).reset_index(drop=True)
gate_e_deltas = np.sort(gate_e_observations["delta_pier_mm"].unique())
gate_e_scenarios = [
    str(gate_e_observations[
        gate_e_observations["delta_pier_mm"] == delta
    ]["scenario"].iloc[0]).zfill(2)
    for delta in gate_e_deltas
]
gate_e_observed_matrix = np.vstack([
    gate_e_observations[gate_e_observations["delta_pier_mm"] == delta]
    .sort_values("experimental_mode")["frequency_ratio"].to_numpy(dtype=float)
    for delta in gate_e_deltas
])
gate_e_sigma_matrix = np.vstack([
    gate_e_observations[gate_e_observations["delta_pier_mm"] == delta]
    .sort_values("experimental_mode")["sigma_frequency_ratio"].to_numpy(dtype=float)
    for delta in gate_e_deltas
])

if gate_e_observed_matrix.shape != (len(gate_e_deltas), len(retained_experimental_modes)):
    raise RuntimeError(f"Unexpected Gate-E observation shape: {gate_e_observed_matrix.shape}")
if gate_e_sigma_matrix.shape != gate_e_observed_matrix.shape:
    raise RuntimeError("Gate-E marginal sigma shape does not match observations")
if not np.isfinite(gate_e_observed_matrix).all() or not np.isfinite(gate_e_sigma_matrix).all():
    raise RuntimeError("Gate-E observations or uncertainties contain non-finite values")
if np.any(gate_e_sigma_matrix <= 0.0):
    raise RuntimeError("Gate-E marginal standard deviations must be strictly positive")


def gate_e_stabilize_covariance(covariance: np.ndarray) -> np.ndarray:
    covariance = np.asarray(covariance, dtype=float)
    covariance = 0.5*(covariance + covariance.T)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    scale = max(1.0, float(np.max(np.abs(eigenvalues))))
    floor = max(1e-12, 1e-10*scale)
    return eigenvectors @ np.diag(np.maximum(eigenvalues, floor)) @ eigenvectors.T

# Full paired-setup covariance, with a conservative marginal floor inherited
# from Gate D. This keeps cross-mode dependence without understating any mode's
# audited one-standard-deviation uncertainty.
gate_e_observation_covariances = np.stack([
    gate_a_ratio_covariance_by_scenario[scenario].copy()
    for scenario in gate_e_scenarios
])
for state_index in range(len(gate_e_observation_covariances)):
    covariance = gate_e_stabilize_covariance(gate_e_observation_covariances[state_index])
    target_diagonal = np.maximum(
        np.diag(covariance),
        gate_e_sigma_matrix[state_index]**2,
    )
    covariance += np.diag(target_diagonal - np.diag(covariance))
    gate_e_observation_covariances[state_index] = gate_e_stabilize_covariance(covariance)
# The covariance diagonal is the authoritative marginal measurement variance.
gate_e_sigma_matrix = np.sqrt(np.diagonal(
    gate_e_observation_covariances, axis1=1, axis2=2
))


def gate_e_state(parameters: np.ndarray, delta_mm: float) -> tuple[float, float]:
    amplitude_ei, log_amplitude_support = np.asarray(parameters, dtype=float)
    normalized_delta = float(delta_mm)/95.0
    return (amplitude_ei*normalized_delta**CONFIG["gate_e_fixed_local_exponent"],
            -log_amplitude_support*normalized_delta**CONFIG["gate_e_fixed_support_exponent"])

@lru_cache(maxsize=None)
def gate_e_healthy_modes(n_elements: int) -> tuple:
    return tuple(gate_e_modes(0.0, 0.0, n_elements=n_elements))

def gate_e_ratio_vector(parameters: np.ndarray, delta_mm: float,
                        n_elements: int = 58) -> np.ndarray:
    alpha, log_ratio = gate_e_state(parameters, delta_mm)
    healthy = np.asarray(gate_e_healthy_modes(int(n_elements)))[GATE_E_RETAINED_RANKS_ZERO_BASED]
    current = gate_e_modes(alpha, log_ratio, n_elements=int(n_elements))[
        GATE_E_RETAINED_RANKS_ZERO_BASED]
    return current/healthy

def gate_e_indices(selected_indices=None) -> list[int]:
    return (list(range(len(gate_e_deltas))) if selected_indices is None
            else [int(index) for index in selected_indices])

def gate_e_raw_residual(parameters: np.ndarray, selected_indices=None,
                        n_elements: int = 58) -> np.ndarray:
    indices = gate_e_indices(selected_indices)
    return np.vstack([
        gate_e_ratio_vector(parameters, gate_e_deltas[index], n_elements=n_elements) -
        gate_e_observed_matrix[index]
        for index in indices
    ])

def gate_e_effective_covariances(model_discrepancy_sd: float,
                                 selected_indices=None) -> np.ndarray:
    indices = gate_e_indices(selected_indices)
    tau = float(model_discrepancy_sd)
    if not np.isfinite(tau) or tau < 0.0:
        raise ValueError("Gate-E model discrepancy must be finite and nonnegative")
    identity = np.eye(gate_e_observed_matrix.shape[1])
    return np.stack([
        gate_e_stabilize_covariance(
            gate_e_observation_covariances[index] + tau**2*identity
        )
        for index in indices
    ])

def gate_e_whitened_residual(parameters: np.ndarray, selected_indices=None,
                             model_discrepancy_sd: float = 0.0) -> np.ndarray:
    raw = gate_e_raw_residual(parameters, selected_indices)
    covariances = gate_e_effective_covariances(
        model_discrepancy_sd, selected_indices
    )
    whitened = [
        np.linalg.solve(np.linalg.cholesky(covariance), residual)
        for covariance, residual in zip(covariances, raw)
    ]
    return np.concatenate(whitened)

def gate_e_marginal_standardized_residual(
    parameters: np.ndarray,
    selected_indices=None,
    model_discrepancy_sd: float = 0.0,
) -> np.ndarray:
    raw = gate_e_raw_residual(parameters, selected_indices)
    covariance = gate_e_effective_covariances(
        model_discrepancy_sd, selected_indices
    )
    sigma = np.sqrt(np.diagonal(covariance, axis1=1, axis2=2))
    return (raw/sigma).ravel()

# Backward-compatible diagnostic function. After profiling, calls without an
# explicit discrepancy use the selected Gate-E value.
def gate_e_standardized_residual(
    parameters: np.ndarray,
    selected_indices=None,
    model_discrepancy_sd: float | None = None,
) -> np.ndarray:
    if model_discrepancy_sd is None:
        model_discrepancy_sd = float(globals().get("gate_e_model_discrepancy_sd", 0.0))
    return gate_e_marginal_standardized_residual(
        parameters, selected_indices, model_discrepancy_sd
    )

gate_e_starts = (
    np.array([0.20, 0.5]),
    np.array([0.55, 1.8]),
    np.array([0.80, 4.0]),
    np.array([0.62, 1.71]),
)

def gate_e_unique_starts(extra_starts=(), exhaustive: bool = False) -> list[np.ndarray]:
    starts = [np.asarray(start, dtype=float) for start in (*gate_e_starts, *tuple(extra_starts))]
    if exhaustive:
        a_grid = np.linspace(0.08, 0.84, 4)
        b_grid = np.array([0.15, 0.8, 2.4, 6.5])
        starts.extend(np.array([a, b]) for a in a_grid for b in b_grid)
    lower, upper = GATE_E_PARAMETER_BOUNDS
    unique = []
    seen = set()
    for start in starts:
        clipped = np.clip(start, lower + 1e-10, upper - 1e-10)
        key = tuple(np.round(clipped, 12))
        if key not in seen:
            seen.add(key)
            unique.append(clipped)
    return unique

def gate_e_fit_at_discrepancy(
    model_discrepancy_sd: float,
    selected_indices=None,
    starts=(),
    exhaustive: bool = False,
) -> dict:
    candidate_starts = gate_e_unique_starts(starts, exhaustive=exhaustive)
    candidates = []
    candidate_rows = []
    for start_index, start in enumerate(candidate_starts):
        try:
            fit = least_squares(
                lambda parameters: gate_e_whitened_residual(
                    parameters,
                    selected_indices=selected_indices,
                    model_discrepancy_sd=model_discrepancy_sd,
                ),
                start,
                bounds=GATE_E_PARAMETER_BOUNDS,
                method="trf",
                x_scale="jac",
                max_nfev=CONFIG["gate_e_optimizer_max_nfev"],
                xtol=1e-11,
                ftol=1e-11,
                gtol=1e-11,
            )
            chi_square = float(np.dot(fit.fun, fit.fun))
            marginal = gate_e_marginal_standardized_residual(
                fit.x, selected_indices, model_discrepancy_sd
            )
            maximum_marginal_z = float(np.max(np.abs(marginal)))
            finite = bool(
                np.isfinite(chi_square)
                and np.isfinite(maximum_marginal_z)
                and np.all(np.isfinite(fit.x))
            )
            candidate_rows.append({
                "model_discrepancy_sd": float(model_discrepancy_sd),
                "start_index": start_index,
                "start_a": float(start[0]),
                "start_b": float(start[1]),
                "a": float(fit.x[0]),
                "b": float(fit.x[1]),
                "chi_square": chi_square,
                "maximum_marginal_standardized_residual": maximum_marginal_z,
                "success": bool(fit.success),
                "finite": finite,
                "nfev": int(fit.nfev),
                "message": str(fit.message),
            })
            if finite:
                candidates.append(fit)
        except Exception as exc:
            candidate_rows.append({
                "model_discrepancy_sd": float(model_discrepancy_sd),
                "start_index": start_index,
                "start_a": float(start[0]),
                "start_b": float(start[1]),
                "a": np.nan,
                "b": np.nan,
                "chi_square": np.inf,
                "maximum_marginal_standardized_residual": np.inf,
                "success": False,
                "finite": False,
                "nfev": 0,
                "message": f"{type(exc).__name__}: {exc}",
            })
    if not candidates:
        raise RuntimeError(
            "Gate-E generalized least-squares optimization returned no finite candidate"
        )
    successful = [candidate for candidate in candidates if candidate.success]
    pool = successful if successful else candidates
    best = min(pool, key=lambda candidate: float(candidate.fun @ candidate.fun))
    covariance = gate_e_effective_covariances(
        model_discrepancy_sd, selected_indices
    )
    sign_and_logdet = [np.linalg.slogdet(value) for value in covariance]
    if any(sign <= 0 for sign, _ in sign_and_logdet):
        raise RuntimeError("Gate-E effective covariance is not positive definite")
    chi_square = float(best.fun @ best.fun)
    log_determinant = float(sum(logdet for _, logdet in sign_and_logdet))
    observation_count = int(best.fun.size)
    gaussian_nll = 0.5*(
        chi_square + log_determinant + observation_count*np.log(2.0*np.pi)
    )
    maximum_marginal_z = float(np.max(np.abs(
        gate_e_marginal_standardized_residual(
            best.x, selected_indices, model_discrepancy_sd
        )
    )))
    return {
        "fit": best,
        "candidates": candidates,
        "candidate_rows": candidate_rows,
        "model_discrepancy_sd": float(model_discrepancy_sd),
        "chi_square": chi_square,
        "gaussian_nll": float(gaussian_nll),
        "maximum_marginal_z": maximum_marginal_z,
    }

def gate_e_profiled_optimizer(selected_indices=None, starts=()) -> dict:
    target = float(CONFIG["gate_e_max_standardized_residual"])
    feasibility_tolerance = float(CONFIG["gate_e_feasibility_tolerance"])
    max_tau = float(CONFIG["gate_e_max_model_discrepancy_sd"])
    search_tolerance = float(CONFIG["gate_e_model_discrepancy_search_tolerance"])
    profile_cache: dict[float, dict] = {}

    def evaluate(tau: float, exhaustive: bool = False, warm_starts=()) -> dict:
        tau = float(np.clip(tau, 0.0, max_tau))
        key = float(np.round(tau, 12))
        if key not in profile_cache:
            profile_cache[key] = gate_e_fit_at_discrepancy(
                tau,
                selected_indices=selected_indices,
                starts=(*tuple(starts), *tuple(warm_starts)),
                exhaustive=exhaustive,
            )
        return profile_cache[key]

    measurement_only = evaluate(0.0, exhaustive=True)

    # Profiled Gaussian-likelihood discrepancy. Tau=0 and the cap are always
    # evaluated explicitly because bounded scalar optimizers do not include
    # endpoints by construction.
    def profiled_nll(tau: float) -> float:
        return evaluate(tau, warm_starts=(measurement_only["fit"].x,))["gaussian_nll"]

    scalar = minimize_scalar(
        profiled_nll,
        bounds=(0.0, max_tau),
        method="bounded",
        options={
            "xatol": search_tolerance,
            "maxiter": int(CONFIG["gate_e_profile_max_iterations"]),
        },
    )
    mle_candidates = [
        measurement_only,
        evaluate(float(scalar.x), warm_starts=(measurement_only["fit"].x,)),
        evaluate(max_tau, warm_starts=(measurement_only["fit"].x,)),
    ]
    mle = min(mle_candidates, key=lambda result: result["gaussian_nll"])

    # Independent adequacy audit: find the smallest tau for which the best GLS
    # fit lies inside the fixed marginal ±3σ envelope. A coarse grid provides a
    # robust bracket even if the refitted maximum residual is not perfectly
    # monotone in tau.
    adequacy_limit = target + feasibility_tolerance

    # Initialise every output of the adequacy search before branching.
    # V0.6 FINAL did not assign `adequacy_feasible` when a feasible positive-tau
    # grid point was found, causing an UnboundLocalError after a successful
    # bracket. These defaults also make future refactors fail safely.
    adequacy_feasible = False
    adequacy_status = "not_evaluated"
    minimum_adequacy_tau = float(max_tau)
    adequacy_fit = evaluate(
        max_tau,
        warm_starts=(measurement_only["fit"].x,),
    )

    if measurement_only["maximum_marginal_z"] <= adequacy_limit:
        minimum_adequacy_tau = 0.0
        adequacy_fit = measurement_only
        adequacy_feasible = True
        adequacy_status = "measurement_only_feasible"
    else:
        grid_point_count = max(
            2,
            int(CONFIG["gate_e_profile_grid_points"]),
        )
        tau_grid = np.linspace(0.0, max_tau, grid_point_count)
        grid_results = [measurement_only]
        previous_fit = measurement_only["fit"]

        for tau in tau_grid[1:]:
            result = evaluate(float(tau), warm_starts=(previous_fit.x,))
            grid_results.append(result)
            previous_fit = result["fit"]

        first_feasible = next(
            (
                index
                for index, result in enumerate(grid_results)
                if result["maximum_marginal_z"] <= adequacy_limit
            ),
            None,
        )

        if first_feasible is None:
            # The complete predeclared discrepancy interval is inadequate.
            # Return a scientific NO-GO at the cap; do not raise or leave
            # control-flow variables undefined.
            minimum_adequacy_tau = float(max_tau)
            adequacy_fit = grid_results[-1]
            adequacy_feasible = False
            adequacy_status = "infeasible_within_cap"
        elif first_feasible == 0:
            # Defensive case. It is normally handled by the measurement-only
            # branch above, but prevents invalid negative grid indexing after
            # future modifications.
            minimum_adequacy_tau = 0.0
            adequacy_fit = grid_results[0]
            adequacy_feasible = True
            adequacy_status = "grid_origin_feasible"
        else:
            adequacy_feasible = True
            adequacy_status = "positive_tau_bracket_found"
            low = float(tau_grid[first_feasible - 1])
            high = float(tau_grid[first_feasible])
            adequacy_fit = grid_results[first_feasible]

            for _ in range(int(CONFIG["gate_e_profile_max_iterations"])):
                if high - low <= search_tolerance:
                    break
                midpoint = 0.5 * (low + high)
                result = evaluate(
                    midpoint,
                    warm_starts=(adequacy_fit["fit"].x,),
                )
                if result["maximum_marginal_z"] <= adequacy_limit:
                    high = midpoint
                    adequacy_fit = result
                else:
                    low = midpoint

            minimum_adequacy_tau = float(high)

    if not isinstance(adequacy_feasible, (bool, np.bool_)):
        raise RuntimeError("Gate-E adequacy feasibility flag is not boolean")
    if not np.isfinite(minimum_adequacy_tau):
        raise RuntimeError("Gate-E minimum adequacy discrepancy is non-finite")

    if adequacy_feasible:
        selected_tau = max(
            float(mle["model_discrepancy_sd"]),
            float(minimum_adequacy_tau),
        )
        selected = evaluate(
            selected_tau,
            warm_starts=(mle["fit"].x, adequacy_fit["fit"].x),
        )
        profile_feasible = bool(
            selected["maximum_marginal_z"] <= adequacy_limit
            and selected_tau <= max_tau + search_tolerance
        )
    else:
        selected_tau = max_tau
        selected = adequacy_fit
        profile_feasible = False

    profile_rows = []
    for tau, result in sorted(profile_cache.items()):
        profile_rows.append({
            "model_discrepancy_sd": float(tau),
            "a": float(result["fit"].x[0]),
            "b": float(result["fit"].x[1]),
            "chi_square": float(result["chi_square"]),
            "gaussian_nll": float(result["gaussian_nll"]),
            "maximum_marginal_standardized_residual": float(
                result["maximum_marginal_z"]
            ),
            "optimizer_success": bool(result["fit"].success),
            "nfev": int(result["fit"].nfev),
        })
    candidate_rows = [
        row
        for result in profile_cache.values()
        for row in result["candidate_rows"]
    ]
    return {
        "fit": selected["fit"],
        "selected": selected,
        "measurement_only": measurement_only,
        "mle": mle,
        "minimum_adequacy_tau": float(minimum_adequacy_tau),
        "selected_model_discrepancy_sd": float(selected_tau),
        "profile_feasible": profile_feasible,
        "adequacy_feasible_within_cap": bool(adequacy_feasible),
        "adequacy_status": adequacy_status,
        "profile_rows": profile_rows,
        "candidate_rows": candidate_rows,
    }

gate_e_profile = gate_e_profiled_optimizer(starts=gate_e_starts)
gate_e_fit = gate_e_profile["fit"]
gate_e_parameters = gate_e_fit.x.copy()
gate_e_model_discrepancy_sd = float(
    gate_e_profile["selected_model_discrepancy_sd"]
)
gate_e_measurement_only_max_standardized_residual = float(
    gate_e_profile["measurement_only"]["maximum_marginal_z"]
)
gate_e_mle_model_discrepancy_sd = float(
    gate_e_profile["mle"]["model_discrepancy_sd"]
)
gate_e_minimum_3sigma_model_discrepancy_sd = float(
    gate_e_profile["minimum_adequacy_tau"]
)
gate_e_profile_feasible = bool(gate_e_profile["profile_feasible"])
gate_e_adequacy_status = str(gate_e_profile["adequacy_status"])

# Compatibility names used by later diagnostics.
gate_e_fits = gate_e_profile["measurement_only"]["candidates"]
gate_e_unconstrained_fit = gate_e_profile["measurement_only"]["fit"]
gate_e_constrained_candidates = gate_e_profile["selected"]["candidates"]

gate_e_profile_table = pd.DataFrame(gate_e_profile["profile_rows"]).sort_values(
    "model_discrepancy_sd"
)
gate_e_candidate_table = pd.DataFrame(gate_e_profile["candidate_rows"])
gate_e_profile_table.to_csv(
    TABLE_ROOT/f"gate_e_model_discrepancy_profile_{gate_e_mode}.csv", index=False
)
gate_e_candidate_table.to_csv(
    TABLE_ROOT/f"gate_e_optimizer_candidates_{gate_e_mode}.csv", index=False
)
display(gate_e_profile_table)

if not gate_e_profile_feasible:
    warnings.warn(
        "Gate E requires more than the predeclared 1% model-discrepancy cap "
        "to satisfy the marginal ±3σ adequacy criterion. The notebook records "
        "a scientific NO-GO instead of misreporting an optimizer failure.",
        RuntimeWarning,
        stacklevel=2,
    )

gate_e_predicted_matrix = np.vstack([
    gate_e_ratio_vector(gate_e_parameters, delta) for delta in gate_e_deltas
])
gate_e_raw_matrix = gate_e_predicted_matrix - gate_e_observed_matrix
gate_e_effective_covariance = gate_e_effective_covariances(
    gate_e_model_discrepancy_sd
)
gate_e_total_sigma_matrix = np.sqrt(np.diagonal(
    gate_e_effective_covariance, axis1=1, axis2=2
))
gate_e_standardized_matrix = gate_e_raw_matrix/gate_e_total_sigma_matrix
gate_e_measurement_only_standardized_matrix = gate_e_raw_matrix/gate_e_sigma_matrix
gate_e_whitened_matrix = np.vstack([
    np.linalg.solve(np.linalg.cholesky(covariance), residual)
    for covariance, residual in zip(gate_e_effective_covariance, gate_e_raw_matrix)
])
gate_e_state_rmse = np.sqrt(np.mean(gate_e_raw_matrix**2, axis=1))
gate_e_estimated_parameter_count = len(gate_e_parameters) + int(
    gate_e_model_discrepancy_sd > CONFIG["gate_e_model_discrepancy_search_tolerance"]
)
gate_e_degrees_of_freedom = max(
    1, gate_e_raw_matrix.size - gate_e_estimated_parameter_count
)
gate_e_reduced_chi_square = float(
    np.sum(gate_e_whitened_matrix**2)/gate_e_degrees_of_freedom
)

gate_e_calibration_rows = []
for state_index, delta in enumerate(gate_e_deltas):
    alpha, log_ratio = gate_e_state(gate_e_parameters, delta)
    scenario = gate_e_scenarios[state_index]
    for mode_index, experimental_mode in enumerate(retained_experimental_modes):
        gate_e_calibration_rows.append({
            "scenario": scenario,
            "delta_pier_mm": delta,
            "experimental_mode": experimental_mode,
            "equivalent_fe_mode": retained_equivalent_modes[mode_index],
            "alpha_local_ei": alpha,
            "koppigen_support_ratio": 10.0**log_ratio,
            "observed_frequency_ratio": gate_e_observed_matrix[state_index, mode_index],
            "predicted_frequency_ratio": gate_e_predicted_matrix[state_index, mode_index],
            "raw_residual": gate_e_raw_matrix[state_index, mode_index],
            "sigma_measurement_ratio": gate_e_sigma_matrix[state_index, mode_index],
            "sigma_model_discrepancy_ratio": gate_e_model_discrepancy_sd,
            "sigma_total_ratio": gate_e_total_sigma_matrix[state_index, mode_index],
            "measurement_only_standardized_residual":
                gate_e_measurement_only_standardized_matrix[state_index, mode_index],
            "standardized_residual": gate_e_standardized_matrix[state_index, mode_index],
            "whitened_residual_component": gate_e_whitened_matrix[state_index, mode_index],
            "state_modal_ratio_rmse": gate_e_state_rmse[state_index],
        })
gate_e_calibration = pd.DataFrame(gate_e_calibration_rows)
gate_e_calibration.to_csv(
    TABLE_ROOT/f"gate_e_z24_calibration_residuals_{gate_e_mode}.csv", index=False
)

# Conditional parameter covariance at the selected model-discrepancy value.
gate_e_jacobian_steps = np.maximum(1e-6, 1e-4*np.maximum(np.abs(gate_e_parameters), 1.0))
gate_e_weighted_jacobian = np.column_stack([
    (
        gate_e_whitened_residual(
            gate_e_parameters + gate_e_jacobian_steps*np.eye(len(gate_e_parameters))[index],
            model_discrepancy_sd=gate_e_model_discrepancy_sd,
        )
        - gate_e_whitened_residual(
            gate_e_parameters - gate_e_jacobian_steps*np.eye(len(gate_e_parameters))[index],
            model_discrepancy_sd=gate_e_model_discrepancy_sd,
        )
    )/(2.0*gate_e_jacobian_steps[index])
    for index in range(len(gate_e_parameters))
])
gate_e_singular_values = np.linalg.svd(gate_e_weighted_jacobian, compute_uv=False)
gate_e_jacobian_rank = int(np.linalg.matrix_rank(gate_e_weighted_jacobian))
gate_e_jacobian_condition = float(
    gate_e_singular_values[0]/max(gate_e_singular_values[-1], np.finfo(float).tiny)
)
gate_e_covariance = max(1.0, gate_e_reduced_chi_square)*np.linalg.pinv(
    gate_e_weighted_jacobian.T@gate_e_weighted_jacobian
)
gate_e_parameter_se = np.sqrt(np.maximum(np.diag(gate_e_covariance), 0.0))
gate_e_parameter_audit = pd.DataFrame({
    "parameter": ["a_local_ei_amplitude", "b_support_log_amplitude"],
    "estimate": gate_e_parameters,
    "standard_error_linearized_conditional_on_tau": gate_e_parameter_se,
    "lower_bound": GATE_E_PARAMETER_BOUNDS[0],
    "upper_bound": GATE_E_PARAMETER_BOUNDS[1],
})
gate_e_discrepancy_audit = pd.DataFrame([{
    "policy": CONFIG["gate_e_model_discrepancy_policy"],
    "measurement_only_max_standardized_residual":
        gate_e_measurement_only_max_standardized_residual,
    "profile_likelihood_mle_sd": gate_e_mle_model_discrepancy_sd,
    "minimum_sd_for_3sigma_adequacy": gate_e_minimum_3sigma_model_discrepancy_sd,
    "selected_model_discrepancy_sd": gate_e_model_discrepancy_sd,
    "predeclared_maximum_sd": CONFIG["gate_e_max_model_discrepancy_sd"],
    "profile_feasible_within_cap": gate_e_profile_feasible,
    "adequacy_search_status": gate_e_adequacy_status,
}])
gate_e_fixed_exponent_audit = pd.DataFrame([
    {"coordinate": "local_EI", "fixed_exponent": CONFIG["gate_e_fixed_local_exponent"],
     "status": "model-form assumption; not estimated"},
    {"coordinate": "Koppigen_support", "fixed_exponent": CONFIG["gate_e_fixed_support_exponent"],
     "status": "model-form assumption; not estimated"},
])
gate_e_parameter_audit.to_csv(
    TABLE_ROOT/f"gate_e_parameter_identifiability_{gate_e_mode}.csv", index=False
)
gate_e_discrepancy_audit.to_csv(
    TABLE_ROOT/f"gate_e_model_discrepancy_audit_{gate_e_mode}.csv", index=False
)
gate_e_fixed_exponent_audit.to_csv(
    TABLE_ROOT/f"gate_e_fixed_exponents_{gate_e_mode}.csv", index=False
)

gate_e_mesh_rows = []
gate_e_reference_ratios = None
for n_elements in (58, 116, 232):
    ratios = np.vstack([
        gate_e_ratio_vector(gate_e_parameters, delta, n_elements=n_elements)
        for delta in gate_e_deltas
    ])
    if gate_e_reference_ratios is None:
        gate_e_reference_ratios = ratios.copy()
    residuals = ratios - gate_e_observed_matrix
    gate_e_mesh_rows.append({
        "n_elements": n_elements,
        "maximum_state_rmse": float(np.max(np.sqrt(np.mean(residuals**2, axis=1)))),
        "maximum_absolute_residual": float(np.max(np.abs(residuals))),
        "maximum_ratio_change_from_58_elements": float(np.max(np.abs(ratios-gate_e_reference_ratios))),
    })
gate_e_mesh_audit = pd.DataFrame(gate_e_mesh_rows)
gate_e_mesh_audit.to_csv(
    TABLE_ROOT/f"gate_e_mesh_convergence_{gate_e_mode}.csv", index=False
)

gate_e_geometry = pd.DataFrame([
    {"item": "span_lengths_m", "value": "14, 30, 14", "status": "official Z24 geometry"},
    {"item": "support_positions_m", "value": "0, 14, 44, 58", "status": "derived from span lengths"},
    {"item": "Koppigen_support", "value": "vertical spring at x=44 m", "status": "V0.6 FINAL model assumption"},
    {"item": "local_EI_zone", "value": "Gaussian sigma=2.5 m centered x=44 m", "status": "V0.6 FINAL model assumption"},
    {"item": "absolute_EI_and_mass", "value": "normalized", "status": "not identifiable from ratios"},
    {"item": "model_discrepancy", "value": "isotropic ratio-space SD tau_E", "status": "profiled AVT development nuisance term; locked for FVT"},
])
gate_e_geometry.to_csv(TABLE_ROOT/"gate_e_geometry_and_assumptions.csv", index=False)

display(gate_e_geometry)
display(gate_e_parameter_audit)
display(gate_e_discrepancy_audit)
display(gate_e_fixed_exponent_audit)
display(gate_e_mesh_audit)
display(gate_e_calibration)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))
axes[0].plot([0, 58], [0, 0], color="#264653", linewidth=5)
for x_support in GATE_E_SUPPORTS_M:
    axes[0].scatter(x_support, 0, marker="v", s=100,
                    color="#e76f51" if x_support == 44 else "#2a9d8f", zorder=4)
axes[0].axvspan(41.5, 46.5, color="#e76f51", alpha=0.16, label="local EI zone")
axes[0].annotate("Koppigen spring", xy=(44, 0), xytext=(38, 0.35),
                 arrowprops=dict(arrowstyle="->", color="#555555"))
axes[0].set_xlim(-1, 59)
axes[0].set_ylim(-0.25, 0.65)
axes[0].set_yticks([])
axes[0].set_xlabel("Bridge coordinate x [m]")
axes[0].set_title("V0.6 FINAL three-span physical model")
axes[0].legend(loc="upper left")
mode_colors = ["#264653", "#2a9d8f", "#e76f51"]
for mode_index, (experimental_mode, color) in enumerate(zip(retained_experimental_modes, mode_colors)):
    axes[1].errorbar(
        gate_e_deltas,
        gate_e_observed_matrix[:, mode_index],
        yerr=1.96*gate_e_total_sigma_matrix[:, mode_index],
        fmt="o",
        capsize=3,
        color=color,
        label=f"observed f{experimental_mode}",
    )
    axes[1].plot(
        gate_e_deltas,
        gate_e_predicted_matrix[:, mode_index],
        "-",
        color=color,
        label=f"FE mode {retained_equivalent_modes[mode_index]}",
    )
axes[1].set_xlabel("Imposed settlement [mm]")
axes[1].set_ylabel("Healthy-normalized frequency")
axes[1].set_title("Covariance-aware multimodal Z24 calibration")
axes[1].legend(ncol=2, fontsize=8)
fig.suptitle("Figure 13 — Gate E multispan calibration with explicit model discrepancy", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT/f"figure_13_gate_e_multispan_calibration_{gate_e_mode}.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

print("Gate E fitted amplitudes [a,b]:", gate_e_parameters)
print("Gate E fixed exponents [p,q]:", [
    CONFIG["gate_e_fixed_local_exponent"],
    CONFIG["gate_e_fixed_support_exponent"],
])
print("Measurement-only maximum standardized residual:",
      gate_e_measurement_only_max_standardized_residual)
print("Profile-likelihood model-discrepancy SD:", gate_e_mle_model_discrepancy_sd)
print("Minimum model-discrepancy SD for 3-sigma adequacy:",
      gate_e_minimum_3sigma_model_discrepancy_sd)
print("Selected Gate-E model-discrepancy SD:", gate_e_model_discrepancy_sd)
print("Adequacy-search status:", gate_e_adequacy_status)
print("Profile feasible within 1% cap:", gate_e_profile_feasible)
print("Maximum calibration state RMSE:", float(gate_e_state_rmse.max()))
print("Maximum effective standardized residual:", float(np.abs(gate_e_standardized_matrix).max()))
print("Reduced chi-square:", gate_e_reduced_chi_square)
print("Jacobian rank / condition:", gate_e_jacobian_rank, gate_e_jacobian_condition)


,model_discrepancy_sd,a,b,chi_square,gaussian_nll,maximum_marginal_standardized_residual,optimizer_success,nfev
0,0.000000,0.587033,1.765401,73.507035,-22.098824,6.117264,True,25
1,0.000625,0.584451,1.775647,64.790592,-26.013586,5.591929,True,29
2,0.001250,0.577056,1.800816,49.094069,-32.802086,4.590024,True,22
3,0.001875,0.572883,1.817734,36.264880,-37.925544,3.688133,True,22
4,0.002500,0.567242,1.836349,27.293643,-41.082568,3.024052,True,29
5,0.002520,0.568875,1.832870,27.061673,-41.157419,3.000858,True,29
6,0.002521,0.568462,1.833691,27.047021,-41.162175,3.001120,True,18
7,0.002521,0.568462,1.833691,27.039874,-41.164464,3.000563,True,18
8,0.002522,0.568462,1.833691,27.032730,-41.166751,3.000007,True,17
9,0.002524,0.568493,1.833735,27.004190,-41.175883,2.997604,True,20


,item,value,status
0,span_lengths_m,"14, 30, 14",official Z24 geometry
1,support_positions_m,"0, 14, 44, 58",derived from span lengths
2,Koppigen_support,vertical spring at x=44 m,V0.6 FINAL model assumption
3,local_EI_zone,Gaussian sigma=2.5 m centered x=44 m,V0.6 FINAL model assumption
4,absolute_EI_and_mass,normalized,not identifiable from ratios
5,model_discrepancy,isotropic ratio-space SD tau_E,profiled AVT development nuisance term; locked...


,parameter,estimate,standard_error_linearized_conditional_on_tau,lower_bound,upper_bound
0,a_local_ei_amplitude,0.560345,0.046656,0.0,0.9
1,b_support_log_amplitude,1.866403,0.110133,0.0,8.0


,policy,measurement_only_max_standardized_residual,profile_likelihood_mle_sd,minimum_sd_for_3sigma_adequacy,selected_model_discrepancy_sd,predeclared_maximum_sd,profile_feasible_within_cap,adequacy_search_status
0,profiled_gaussian_plus_minimum_3sigma_adequacy,6.117264,0.004948,0.002522,0.004948,0.01,True,positive_tau_bracket_found


,coordinate,fixed_exponent,status
0,local_EI,1.0,model-form assumption; not estimated
1,Koppigen_support,0.5,model-form assumption; not estimated


,n_elements,maximum_state_rmse,maximum_absolute_residual,maximum_ratio_change_from_58_elements
0,58,0.006463,0.008687,0.000000
1,116,0.006425,0.008683,0.000117
2,232,0.006416,0.008683,0.000141


,scenario,delta_pier_mm,experimental_mode,equivalent_fe_mode,alpha_local_ei,koppigen_support_ratio,observed_frequency_ratio,predicted_frequency_ratio,raw_residual,sigma_measurement_ratio,sigma_model_discrepancy_ratio,sigma_total_ratio,measurement_only_standardized_residual,standardized_residual,whitened_residual_component,state_modal_ratio_rmse
0,03,20.0,1,1,0.117967,0.139199,0.987500,0.990756,0.003256,0.003125,0.004948,0.005852,1.041979,0.556414,0.556414,0.006252
1,03,20.0,3,3,0.117967,0.139199,0.982843,0.991530,0.008687,0.001393,0.004948,0.005140,6.234591,1.689987,1.679938,0.006252
2,03,20.0,4,4,0.117967,0.139199,0.981439,0.987023,0.005585,0.003839,0.004948,0.006263,1.454653,0.891749,0.826310,0.006252
3,04,40.0,1,1,0.235935,0.061506,0.987500,0.980453,-0.007047,0.003125,0.004948,0.005852,-2.254926,-1.204125,-1.204125,0.004413
4,04,40.0,3,3,0.235935,0.061506,0.977941,0.980437,0.002496,0.001392,0.004948,0.005140,1.793215,0.485644,0.465389,0.004413
5,04,40.0,4,4,0.235935,0.061506,0.974478,0.972888,-0.001590,0.004530,0.004948,0.006708,-0.350995,-0.237008,-0.261986,0.004413
6,05,80.0,1,1,0.471869,0.019376,0.962500,0.955218,-0.007282,0.003125,0.004948,0.005852,-2.330204,-1.244324,-1.244324,0.006015
7,05,80.0,3,3,0.471869,0.019376,0.938725,0.943232,0.004507,0.001965,0.004948,0.005324,2.292961,0.846506,0.849191,0.006015
8,05,80.0,4,4,0.471869,0.019376,0.935035,0.940968,0.005933,0.005380,0.004948,0.007310,1.102771,0.811724,0.838050,0.006015
9,06,95.0,1,1,0.560345,0.013602,0.937500,0.943543,0.006043,0.003125,0.004948,0.005852,1.933629,1.032553,1.032553,0.006463


Gate E fitted amplitudes [a,b]: [0.56034484 1.86640304]
Gate E fixed exponents [p,q]: [1.0, 0.5]
Measurement-only maximum standardized residual: 6.117263629158133
Profile-likelihood model-discrepancy SD: 0.0049478563946875114
Minimum model-discrepancy SD for 3-sigma adequacy: 0.00252197265625
Selected Gate-E model-discrepancy SD: 0.0049478563946875114
Adequacy-search status: positive_tau_bracket_found
Profile feasible within 1% cap: True
Maximum calibration state RMSE: 0.006462731080725595
Maximum effective standardized residual: 1.6899866561788668
Reduced chi-square: 1.2444717330344703
Jacobian rank / condition: 2 5.166404011469925


## 18. Gate E leave-one-severity-out validation with training-only discrepancy profiling

Each nonzero severity is held out in turn. The two physical amplitudes and the Gate-E model-discrepancy standard deviation are re-estimated **only from the remaining training severities**, using the same covariance-aware profile and the same fixed 1% discrepancy cap. No held-out frequency enters that fold's optimizer or discrepancy estimate.

The held-out residual is then scored with the training-estimated discrepancy and the held-out paired-setup observation covariance. Thus, the correction avoids both the earlier optimizer crash and the invalid alternative of inflating a held-out uncertainty after seeing its residual. The 20, 40 and 80 mm folds are interpolation tests; the 95 mm fold is explicitly extrapolation.


In [25]:
gate_e_loso_rows = []
gate_e_loso_parameter_rows = []
for heldout_index, heldout_delta in enumerate(gate_e_deltas):
    training_indices = [
        index for index in range(len(gate_e_deltas))
        if index != heldout_index
    ]
    fold_profile = gate_e_profiled_optimizer(
        selected_indices=training_indices,
        starts=(gate_e_parameters, *gate_e_starts),
    )
    fold_fit = fold_profile["fit"]
    fold_model_discrepancy_sd = float(
        fold_profile["selected_model_discrepancy_sd"]
    )
    heldout_prediction = gate_e_ratio_vector(fold_fit.x, heldout_delta)
    heldout_residual = heldout_prediction-gate_e_observed_matrix[heldout_index]
    heldout_covariance = gate_e_stabilize_covariance(
        gate_e_observation_covariances[heldout_index]
        + fold_model_discrepancy_sd**2*np.eye(gate_e_observed_matrix.shape[1])
    )
    heldout_sigma = np.sqrt(np.diag(heldout_covariance))
    heldout_standardized = heldout_residual/heldout_sigma
    heldout_whitened = np.linalg.solve(
        np.linalg.cholesky(heldout_covariance), heldout_residual
    )
    fold_type = (
        "extrapolation"
        if heldout_delta == gate_e_deltas.max()
        else "interpolation"
    )
    fold_training_whitened = gate_e_whitened_residual(
        fold_fit.x,
        selected_indices=training_indices,
        model_discrepancy_sd=fold_model_discrepancy_sd,
    )
    fold_training_marginal = gate_e_marginal_standardized_residual(
        fold_fit.x,
        selected_indices=training_indices,
        model_discrepancy_sd=fold_model_discrepancy_sd,
    )
    gate_e_loso_parameter_rows.append({
        "heldout_delta_pier_mm": heldout_delta,
        "fold_type": fold_type,
        "training_deltas_mm": json_dumps_safe(gate_e_deltas[training_indices].tolist()),
        "a": float(fold_fit.x[0]),
        "p_fixed": CONFIG["gate_e_fixed_local_exponent"],
        "b": float(fold_fit.x[1]),
        "q_fixed": CONFIG["gate_e_fixed_support_exponent"],
        "training_model_discrepancy_sd": fold_model_discrepancy_sd,
        "training_profile_feasible_within_cap": bool(fold_profile["profile_feasible"]),
        "training_measurement_only_max_standardized_residual": float(
            fold_profile["measurement_only"]["maximum_marginal_z"]
        ),
        "training_minimum_3sigma_discrepancy_sd": float(
            fold_profile["minimum_adequacy_tau"]
        ),
        "training_chi_square": float(fold_training_whitened @ fold_training_whitened),
        "training_nfev": int(fold_fit.nfev),
        "training_max_standardized_residual": float(
            np.max(np.abs(fold_training_marginal))
        ),
        "heldout_modal_ratio_rmse": float(np.sqrt(np.mean(heldout_residual**2))),
        "heldout_modal_ratio_max_absolute_error": float(np.max(np.abs(heldout_residual))),
        "heldout_max_standardized_residual": float(np.max(np.abs(heldout_standardized))),
        "heldout_normalized_innovation_squared": float(heldout_whitened@heldout_whitened),
    })
    for mode_index, experimental_mode in enumerate(retained_experimental_modes):
        gate_e_loso_rows.append({
            "heldout_delta_pier_mm": heldout_delta,
            "fold_type": fold_type,
            "experimental_mode": experimental_mode,
            "equivalent_fe_mode": retained_equivalent_modes[mode_index],
            "observed_frequency_ratio": gate_e_observed_matrix[heldout_index, mode_index],
            "predicted_frequency_ratio": heldout_prediction[mode_index],
            "raw_residual": heldout_residual[mode_index],
            "measurement_sigma": gate_e_sigma_matrix[heldout_index, mode_index],
            "training_model_discrepancy_sd": fold_model_discrepancy_sd,
            "heldout_total_sigma": heldout_sigma[mode_index],
            "standardized_residual": heldout_standardized[mode_index],
            "whitened_residual_component": heldout_whitened[mode_index],
        })

gate_e_loso = pd.DataFrame(gate_e_loso_rows)
gate_e_loso_parameters = pd.DataFrame(gate_e_loso_parameter_rows).sort_values(
    "heldout_delta_pier_mm"
).reset_index(drop=True)
gate_e_loso.to_csv(
    TABLE_ROOT/f"gate_e_leave_one_severity_out_residuals_{gate_e_mode}.csv",
    index=False,
)
gate_e_loso_parameters.to_csv(
    TABLE_ROOT/f"gate_e_leave_one_severity_out_{gate_e_mode}.csv",
    index=False,
)
display(gate_e_loso_parameters)
display(gate_e_loso)

gate_e_interpolation = gate_e_loso_parameters[
    gate_e_loso_parameters["fold_type"] == "interpolation"
]
gate_e_extrapolation = gate_e_loso_parameters[
    gate_e_loso_parameters["fold_type"] == "extrapolation"
]
gate_e_max_mesh_change = float(
    gate_e_mesh_audit["maximum_ratio_change_from_58_elements"].max()
)
gate_e_checks = {
    "full_execution": gate_e_mode == "full",
    "optimizer_converged": bool(gate_e_fit.success),
    "measurement_only_3sigma_feasible": bool(
        gate_e_measurement_only_max_standardized_residual
        <= CONFIG["gate_e_max_standardized_residual"]
        + CONFIG["gate_e_feasibility_tolerance"]
    ),
    # Measurement-only feasibility is diagnostic. Eligibility is based on the
    # bounded, explicit model-discrepancy policy below.
    "profiled_model_discrepancy_feasible_within_1pct_cap": bool(
        gate_e_profile_feasible
        and gate_e_model_discrepancy_sd
        <= CONFIG["gate_e_max_model_discrepancy_sd"]
        + CONFIG["gate_e_model_discrepancy_search_tolerance"]
    ),
    "all_loso_training_profiles_feasible_within_cap": bool(
        gate_e_loso_parameters["training_profile_feasible_within_cap"].all()
    ),
    "maximum_calibration_state_rmse_le_1pct": bool(
        gate_e_state_rmse.max() <= CONFIG["gate_e_calibration_max_state_rmse"]
    ),
    "maximum_effective_standardized_residual_le_3sigma": bool(
        np.abs(gate_e_standardized_matrix).max()
        <= CONFIG["gate_e_max_standardized_residual"]
        + CONFIG["gate_e_feasibility_tolerance"]
    ),
    "reduced_chi_square_le_4": bool(
        gate_e_reduced_chi_square <= CONFIG["gate_e_max_reduced_chi_square"]
    ),
    "jacobian_full_column_rank": gate_e_jacobian_rank == len(gate_e_parameters),
    "jacobian_condition_le_100": bool(
        gate_e_jacobian_condition <= CONFIG["gate_e_max_jacobian_condition"]
    ),
    "mesh_ratio_change_le_0p05pct": bool(
        gate_e_max_mesh_change <= CONFIG["gate_e_max_mesh_ratio_change"]
    ),
    "interpolation_loso_rmse_le_1pct": bool(
        (gate_e_interpolation["heldout_modal_ratio_rmse"]
         <= CONFIG["gate_e_loso_interpolation_rmse"]).all()
    ),
    "extrapolation_95mm_rmse_le_1p5pct": bool(
        (gate_e_extrapolation["heldout_modal_ratio_rmse"]
         <= CONFIG["gate_e_loso_extrapolation_rmse"]).all()
    ),
    "raw_z24_signature_unchanged": (
        raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE
    ),
}

# The measurement-only hard-box result is intentionally NOT a GO requirement;
# it is retained as evidence that reduced-model discrepancy was necessary.
gate_e_required_checks = {
    key: value for key, value in gate_e_checks.items()
    if key != "measurement_only_3sigma_feasible"
}
if gate_e_mode == "smoke":
    gate_e_decision = "SMOKE_COMPLETED_REQUIRES_FULL_RUN"
else:
    gate_e_decision = (
        "GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL"
        if all(gate_e_required_checks.values())
        else "NO_GO_REVISE_MULTISPAN_PARAMETERIZATION"
    )

gate_d_prior_decision = json.loads(
    (OUTPUT_ROOT/"gate_d_diagnostic.json").read_text(encoding="utf-8")
)["decision"]
gate_e_diagnostic = {
    "version": "V0.6 FINAL",
    "decision": gate_e_decision,
    "execution_mode": gate_e_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "prior_gate_d_decision_preserved": gate_d_prior_decision,
    "physical_model": (
        "continuous 14+30+14 m Euler-Bernoulli FE with local deck EI "
        "coordinate and Koppigen support spring"
    ),
    "uncertainty_model": {
        "measurement_covariance": (
            "paired physical-setup bootstrap covariance with marginal floors"
        ),
        "model_discrepancy_policy": CONFIG["gate_e_model_discrepancy_policy"],
        "measurement_only_max_standardized_residual":
            gate_e_measurement_only_max_standardized_residual,
        "profile_likelihood_mle_sd": gate_e_mle_model_discrepancy_sd,
        "minimum_sd_for_3sigma_adequacy":
            gate_e_minimum_3sigma_model_discrepancy_sd,
        "selected_model_discrepancy_sd": gate_e_model_discrepancy_sd,
        "predeclared_maximum_model_discrepancy_sd":
            CONFIG["gate_e_max_model_discrepancy_sd"],
        "selected_discrepancy_feasible_within_cap": gate_e_profile_feasible,
        "fvt_policy": "selected AVT discrepancy is locked and not refitted on FVT",
    },
    "parameter_interpretation": {
        "alpha": "effective local deck rigidity-reduction coordinate",
        "support_ratio": "effective Koppigen vertical-restraint ratio",
        "warning": "model-updating coordinates; not measured component properties",
        "fixed_exponents_status": "development-selected model assumptions; not inferred",
    },
    "estimated_amplitudes_a_b": gate_e_parameters.tolist(),
    "fixed_exponents_p_q": [
        CONFIG["gate_e_fixed_local_exponent"],
        CONFIG["gate_e_fixed_support_exponent"],
    ],
    "calibration": {
        "maximum_state_modal_ratio_rmse": float(gate_e_state_rmse.max()),
        "maximum_absolute_modal_ratio_residual": float(np.abs(gate_e_raw_matrix).max()),
        "maximum_effective_standardized_residual": float(
            np.abs(gate_e_standardized_matrix).max()
        ),
        "reduced_chi_square": gate_e_reduced_chi_square,
    },
    "identifiability": {
        "weighted_jacobian_singular_values": gate_e_singular_values.tolist(),
        "rank": gate_e_jacobian_rank,
        "condition_number": gate_e_jacobian_condition,
        "standard_errors_are_conditional_on_selected_discrepancy": True,
    },
    "maximum_mesh_ratio_change": gate_e_max_mesh_change,
    "maximum_interpolation_loso_rmse": float(
        gate_e_interpolation["heldout_modal_ratio_rmse"].max()
    ),
    "extrapolation_95mm_loso_rmse": float(
        gate_e_extrapolation["heldout_modal_ratio_rmse"].iloc[0]
    ),
    "maximum_loso_training_model_discrepancy_sd": float(
        gate_e_loso_parameters["training_model_discrepancy_sd"].max()
    ),
    "checks": gate_e_checks,
    "required_checks": gate_e_required_checks,
    "z24_calibration_performed": True,
    "loso_performed": True,
    "capacity_or_rul_claim_performed": False,
    "raw_z24_signature_unchanged": gate_e_checks["raw_z24_signature_unchanged"],
    "limitations": [
        "measurement-only three-sigma infeasibility is retained as a model-adequacy finding",
        "the isotropic discrepancy is a parsimonious development model and may hide mode-specific model error",
        "the discrepancy is estimated on AVT and must be locked for FVT validation",
        "rank-based experimental-to-FE mode association remains provisional",
        "full experimental mode-shape assembly and environmental correction remain future work",
        "LOSO validates observed severities only; it is not an independent bridge validation",
        "fixed model-form assumptions are not structural ground truth",
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT/"gate_e_diagnostic.json").write_text(
    json_dumps_safe(gate_e_diagnostic, indent=2, sort_keys=True)+"\n",
    encoding="utf-8",
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.5))
delta_grid = np.linspace(0.0, 95.0, 200)
alpha_grid = np.array([
    gate_e_state(gate_e_parameters, delta)[0]
    for delta in delta_grid
])
support_grid = np.array([
    10.0**gate_e_state(gate_e_parameters, delta)[1]
    for delta in delta_grid
])
axes[0].plot(delta_grid, alpha_grid, color="#e76f51", label=r"local $EI$ coordinate $\alpha$")
axes[0].plot(delta_grid, support_grid, color="#264653", label=r"support ratio $r_K$")
axes[0].set_xlabel("Imposed settlement [mm]")
axes[0].set_ylabel("Effective coordinate")
axes[0].set_title("Monotone V0.6 FINAL state map")
axes[0].legend()
fold_colors = {"interpolation": "#2a9d8f", "extrapolation": "#e76f51"}
for fold_type, group in gate_e_loso_parameters.groupby("fold_type"):
    axes[1].scatter(
        group["heldout_delta_pier_mm"],
        100*group["heldout_modal_ratio_rmse"],
        s=75,
        color=fold_colors[fold_type],
        label=fold_type,
    )
axes[1].axhline(
    100*CONFIG["gate_e_loso_interpolation_rmse"],
    color="#2a9d8f",
    linestyle="--",
    linewidth=1,
    label="1% interpolation gate",
)
axes[1].axhline(
    100*CONFIG["gate_e_loso_extrapolation_rmse"],
    color="#e76f51",
    linestyle=":",
    linewidth=1,
    label="1.5% extrapolation gate",
)
axes[1].set_xlabel("Held-out settlement [mm]")
axes[1].set_ylabel("Held-out 3-mode RMSE [%]")
axes[1].set_title("Training-only discrepancy LOSO")
axes[1].legend(fontsize=8)
fig.suptitle(
    f"Figure 14 — Gate E: {gate_e_decision.replace('_', ' ')}",
    weight="bold",
)
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT/f"figure_14_gate_e_loso_{gate_e_mode}.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

print(json_dumps_safe(gate_e_diagnostic, indent=2))


,heldout_delta_pier_mm,fold_type,training_deltas_mm,a,p_fixed,b,q_fixed,training_model_discrepancy_sd,training_profile_feasible_within_cap,training_measurement_only_max_standardized_residual,training_minimum_3sigma_discrepancy_sd,training_chi_square,training_nfev,training_max_standardized_residual,heldout_modal_ratio_rmse,heldout_modal_ratio_max_absolute_error,heldout_max_standardized_residual,heldout_normalized_innovation_squared
0,20.0,interpolation,"[40.0, 80.0, 95.0]",0.551957,1.0,1.877661,0.5,0.004527,True,3.125834,0.000498,8.311953,23,1.263229,0.006342,0.008777,1.853118,4.530204
1,40.0,interpolation,"[20.0, 80.0, 95.0]",0.568081,1.0,1.853547,0.5,0.005399,True,6.099824,0.002496,8.242654,26,1.543821,0.004531,0.007302,1.170496,1.598149
2,80.0,interpolation,"[20.0, 40.0, 95.0]",0.604403,1.0,1.701882,0.5,0.004258,True,5.962060,0.002423,8.706023,23,1.875285,0.008666,0.010468,1.982037,7.422650
3,95.0,extrapolation,"[20.0, 40.0, 80.0]",0.482246,1.0,2.066967,0.5,0.004124,True,6.282020,0.002735,7.709012,24,2.129112,0.012065,0.016905,3.887766,20.709221


,heldout_delta_pier_mm,fold_type,experimental_mode,equivalent_fe_mode,observed_frequency_ratio,predicted_frequency_ratio,raw_residual,measurement_sigma,training_model_discrepancy_sd,heldout_total_sigma,standardized_residual,whitened_residual_component
0,20.0,interpolation,1,1,0.987500,0.990886,0.003386,0.003125,0.004527,0.005501,0.615560,0.615560
1,20.0,interpolation,3,3,0.982843,0.991620,0.008777,0.001393,0.004527,0.004736,1.853118,1.840334
2,20.0,interpolation,4,4,0.981439,0.987109,0.005670,0.003839,0.004527,0.005935,0.955321,0.874333
3,40.0,interpolation,1,1,0.987500,0.980198,-0.007302,0.003125,0.005399,0.006239,-1.170496,-1.170496
4,40.0,interpolation,3,3,0.977941,0.980277,0.002336,0.001392,0.005399,0.005576,0.418933,0.401894
5,40.0,interpolation,4,4,0.974478,0.972805,-0.001673,0.004530,0.005399,0.007048,-0.237398,-0.258009
6,80.0,interpolation,1,1,0.962500,0.952032,-0.010468,0.003125,0.004258,0.005281,-1.982037,-1.982037
7,80.0,interpolation,3,3,0.938725,0.944794,0.006069,0.001965,0.004258,0.004689,1.294174,1.299555
8,80.0,interpolation,4,4,0.935035,0.943917,0.008882,0.005380,0.004258,0.006861,1.294554,1.343629
9,95.0,extrapolation,1,1,0.937500,0.949765,0.012265,0.003125,0.004124,0.005174,2.370450,2.370450


{
  "version": "V0.6 FINAL",
  "decision": "GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "prior_gate_d_decision_preserved": "NO_GO_REVISE_PHYSICAL_PARAMETERIZATION",
  "physical_model": "continuous 14+30+14 m Euler-Bernoulli FE with local deck EI coordinate and Koppigen support spring",
  "uncertainty_model": {
    "measurement_covariance": "paired physical-setup bootstrap covariance with marginal floors",
    "model_discrepancy_policy": "profiled_gaussian_plus_minimum_3sigma_adequacy",
    "measurement_only_max_standardized_residual": 6.117263629158133,
    "profile_likelihood_mle_sd": 0.0049478563946875114,
    "minimum_sd_for_3sigma_adequacy": 0.00252197265625,
    "selected_model_discrepancy_sd": 0.0049478563946875114,
    "predeclared_maximum_model_discrepancy_sd": 0.01,
    "selected_discrepancy_feasible_within_cap": true,
    "fvt_policy": "selected AVT discrepancy is lock

## 19. Multispan variational Full-PINN — Gate F

Gate E returned `GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL`. Gate F therefore replaces the strong-form simply-supported network used in the earlier equivalent-beam pilot with a weak/variational four-output eigenmode PINN for the validated three-span model.

For normalized coordinate `s=x/L`, the network trial function contains the hard factor

`s (s-14/58) (1-s)`,

so vertical displacement is exactly zero at `x=0,14,58 m`. The Koppigen restraint at `x=44 m` enters the potential energy as a point spring; the local Gaussian `EI` reduction uses the same Gate E definition. Four network outputs learn the first modal subspace simultaneously. Training first enforces mass normalization/orthogonality, then minimizes the invariant Ritz trace `tr(M^{-1}K)`. A `4×4` generalized eigensolve rotates and orders the learned subspace.

This weak formulation is intentional: point reactions at the internal supports create shear jumps, so a global strong/nodal residual is not an appropriate acceptance statistic. Physics adequacy is evaluated independently against 20 analytical weak test functions satisfying the essential boundary conditions. FE frequencies and displacement MAC are used only after training; they never enter the Full-PINN loss.

In [26]:
require_compute_node("Gate F multispan Full-PINN training")
from scipy.linalg import eigh
import time

gate_f_mode = CONFIG["gate_f_mode"]
if gate_f_mode not in {"full", "smoke"}:
    raise ValueError("GPT_PINN_GATE_F_MODE must be 'full' or 'smoke'")
if gate_f_mode == "full" and gate_e_decision != "GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL":
    raise RuntimeError("Full Gate F requires the executed Gate E GO decision")

GATE_F_CANDIDATE_AXIS_COUNT = (
    CONFIG["gate_f_full_candidate_axis_count"] if gate_f_mode == "full"
    else CONFIG["gate_f_smoke_candidate_axis_count"]
)
GATE_F_MAX_BASIS_STATES = (
    CONFIG["gate_f_full_max_basis_states"] if gate_f_mode == "full"
    else CONFIG["gate_f_smoke_max_basis_states"]
)
GATE_F_WARM_EPOCHS = (
    CONFIG["gate_f_full_warm_adam_epochs"] if gate_f_mode == "full"
    else CONFIG["gate_f_smoke_warm_adam_epochs"]
)
GATE_F_PROJECTED_EPOCHS = (
    CONFIG["gate_f_full_projected_adam_epochs"] if gate_f_mode == "full"
    else CONFIG["gate_f_smoke_projected_adam_epochs"]
)
GATE_F_LBFGS_ITERATIONS = (
    CONFIG["gate_f_full_lbfgs_iterations"] if gate_f_mode == "full"
    else CONFIG["gate_f_smoke_lbfgs_iterations"]
)

GATE_F_LENGTH = 58.0
GATE_F_FIXED_SUPPORT = 14.0/GATE_F_LENGTH
GATE_F_KOPPIGEN = 44.0/GATE_F_LENGTH
GATE_F_ZONE_WIDTH = CONFIG["gate_e_local_zone_width_m"]/GATE_F_LENGTH
GATE_F_SPRING_REFERENCE = CONFIG["gate_e_dimensionless_support_kappa"]
GATE_F_CHECKPOINT_ROOT = CHECKPOINT_ROOT/"gate_f_v03"
GATE_F_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
gate_f_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def gate_f_scalar_derivative(values: torch.Tensor, coordinates: torch.Tensor,
                             order: int) -> torch.Tensor:
    derivative_value = values
    for _ in range(order):
        derivative_value = torch.autograd.grad(
            derivative_value, coordinates, torch.ones_like(derivative_value),
            create_graph=True, retain_graph=True)[0]
    return derivative_value

def gate_f_multi_derivative(values: torch.Tensor, coordinates: torch.Tensor,
                            order: int) -> torch.Tensor:
    return torch.cat([
        gate_f_scalar_derivative(values[:, index:index+1], coordinates, order)
        for index in range(values.shape[1])
    ], dim=1)

class MultispanVariationalFullPINN(torch.nn.Module):
    def __init__(self, seed: int):
        super().__init__()
        torch.manual_seed(seed)
        widths = CONFIG["gate_f_network_architecture"]
        layers = []
        for input_width, output_width in zip(widths[:-2], widths[1:-1]):
            layer = torch.nn.Linear(input_width, output_width)
            torch.nn.init.xavier_normal_(layer.weight)
            torch.nn.init.zeros_(layer.bias)
            layers.extend([layer, torch.nn.Tanh()])
        final = torch.nn.Linear(widths[-2], widths[-1])
        torch.nn.init.xavier_normal_(final.weight)
        torch.nn.init.zeros_(final.bias)
        layers.append(final)
        self.network = torch.nn.Sequential(*layers)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        hard_boundary = s*(s-GATE_F_FIXED_SUPPORT)*(1.0-s)/0.055
        return hard_boundary*self.network(s)

def gate_f_trapezoid_weights(count: int, device: torch.device) -> torch.Tensor:
    weights = torch.ones(count, device=device)/(count-1)
    weights[0] *= 0.5; weights[-1] *= 0.5
    return weights

def gate_f_modal_matrices(model: MultispanVariationalFullPINN,
                          s: torch.Tensor, alpha: float,
                          log10_support_ratio: float):
    phi = model(s)
    phi_2 = gate_f_multi_derivative(phi, s, 2)
    weights = gate_f_trapezoid_weights(len(s), s.device)
    zone = torch.exp(-0.5*((s[:, 0]-GATE_F_KOPPIGEN)/GATE_F_ZONE_WIDTH).square())
    ei = torch.clamp(1.0-alpha*zone, min=0.08).detach()
    mass = phi.T@(weights[:, None]*phi)
    bending = phi_2.T@((weights*ei)[:, None]*phi_2)
    support_coordinate = torch.tensor([[GATE_F_KOPPIGEN]], dtype=s.dtype, device=s.device)
    support_values = model(support_coordinate)[0]
    stiffness = bending + GATE_F_SPRING_REFERENCE*10.0**log10_support_ratio*torch.outer(
        support_values, support_values)
    return mass, stiffness, phi, phi_2, support_values

def gate_f_checkpoint_path(alpha: float, log10_support_ratio: float) -> Path:
    return GATE_F_CHECKPOINT_ROOT/f"multispan_full_pinn_a{alpha:.4f}_r{log10_support_ratio:.4f}.pt"

def gate_f_checkpoint_payload_compatible(
    value: object,
    *,
    alpha: float,
    log10_support_ratio: float,
    requested_signature: dict,
    allow_unverified_exact_smoke: bool = False,
) -> bool:
    """Validate Gate-F physics metadata and one-way training-budget compatibility.

    In smoke mode a stronger checkpoint is admissible only when every training
    budget is at least the requested smoke budget. A weak checkpoint can never
    satisfy a full request. Exact-budget smoke checkpoints additionally require
    a post-training quality certificate, preventing an underconverged artifact
    from being reloaded forever.
    """
    if not isinstance(value, dict) or value.get("status") != "complete":
        return False
    if not np.isclose(value.get("alpha", np.nan), alpha):
        return False
    if not np.isclose(value.get("log10_support_ratio", np.nan), log10_support_ratio):
        return False

    saved_signature = value.get("training_signature", {})
    exact_keys = (
        "architecture",
        "collocation_points",
        "support_stiffness_convention",
        "physics",
    )
    if any(saved_signature.get(key) != requested_signature.get(key) for key in exact_keys):
        return False

    budget_keys = ("warm_epochs", "projected_epochs", "lbfgs_iterations")
    try:
        saved_budget = {key: int(saved_signature.get(key, -1)) for key in budget_keys}
        requested_budget = {key: int(requested_signature[key]) for key in budget_keys}
    except (TypeError, ValueError, KeyError):
        return False

    if gate_f_mode == "full":
        budget_compatible = all(
            saved_budget[key] == requested_budget[key] for key in budget_keys
        )
    elif CONFIG["gate_f_smoke_prefer_overqualified_checkpoint"]:
        budget_compatible = all(
            saved_budget[key] >= requested_budget[key] for key in budget_keys
        )
    else:
        budget_compatible = all(
            saved_budget[key] == requested_budget[key] for key in budget_keys
        )
    if not budget_compatible:
        return False

    exact_smoke_budget = (
        gate_f_mode == "smoke"
        and all(saved_budget[key] == requested_budget[key] for key in budget_keys)
    )
    if (
        exact_smoke_budget
        and CONFIG["gate_f_reject_unverified_exact_smoke_checkpoint"]
        and not allow_unverified_exact_smoke
        and not bool(value.get("quality_verified", False))
    ):
        return False
    return True


def gate_f_state_from_payload(payload: dict, checkpoint: Path) -> dict:
    model = MultispanVariationalFullPINN(int(payload["seed"])).to(gate_f_device)
    model.load_state_dict(payload["model_state_dict"])
    model.eval()
    return {
        "model": model,
        "coefficients": np.asarray(payload["coefficients"]),
        "eigenvalues_scaled": np.asarray(payload["eigenvalues_scaled"]),
        "runtime_seconds": float(payload["runtime_seconds"]),
        "loss": float(payload["loss"]),
        "normalization_loss": float(payload["normalization_loss"]),
        "orthogonality_loss": float(payload["orthogonality_loss"]),
        "alpha": float(payload["alpha"]),
        "log10_support_ratio": float(payload["log10_support_ratio"]),
        "seed": int(payload["seed"]),
        "checkpoint": Path(checkpoint),
        "resumed": True,
        "training_signature": dict(payload.get("training_signature", {})),
        "checkpoint_quality_verified": bool(payload.get("quality_verified", False)),
        "checkpoint_provenance": payload.get(
            "imported_from_legacy",
            payload.get("quality_source_checkpoint", str(checkpoint)),
        ),
    }


def gate_f_train_full_pinn(alpha: float, log10_support_ratio: float,
                           initial_state: dict | None = None) -> dict:
    seed = int(CONFIG["seed"] + round(1000*alpha) - round(100*log10_support_ratio))
    checkpoint = gate_f_checkpoint_path(alpha, log10_support_ratio)
    signature = {
        "warm_epochs": GATE_F_WARM_EPOCHS,
        "projected_epochs": GATE_F_PROJECTED_EPOCHS,
        "lbfgs_iterations": GATE_F_LBFGS_ITERATIONS,
        "architecture": CONFIG["gate_f_network_architecture"],
        "collocation_points": CONFIG["gate_f_collocation_points"],
        "support_stiffness_convention": CONFIG["support_stiffness_convention"],
        "physics": "three_span_variational_subspace_v05",
    }
    if CONFIG["gate_f_resume"]:
        saved = load_checkpoint_with_legacy_fallback(
            checkpoint,
            map_location=gate_f_device,
            stage=f"Gate F alpha={alpha:.4f} log10_rK={log10_support_ratio:.4f}",
            validator=lambda value: gate_f_checkpoint_payload_compatible(
                value,
                alpha=alpha,
                log10_support_ratio=log10_support_ratio,
                requested_signature=signature,
            ),
        )
        if saved is not None:
            state = gate_f_state_from_payload(saved, checkpoint)
            state["alpha"] = float(alpha)
            state["log10_support_ratio"] = float(log10_support_ratio)
            return state

    model = MultispanVariationalFullPINN(seed).to(gate_f_device)
    if initial_state is not None:
        model.load_state_dict(initial_state["model"].state_dict())
    s = torch.linspace(0.0, 1.0, CONFIG["gate_f_collocation_points"],
                       device=gate_f_device).reshape(-1, 1)
    s.requires_grad_(True)

    def loss_parts(projected: bool):
        mass, stiffness, *_ = gate_f_modal_matrices(
            model, s, alpha, log10_support_ratio)
        diagonal = torch.diag(mass)
        off_diagonal = mass-torch.diag(diagonal)
        normalization = torch.mean((diagonal-1.0).square())
        orthogonality = torch.mean(off_diagonal.square())
        if projected:
            invariant_energy = torch.trace(torch.linalg.solve(
                mass+1e-8*torch.eye(4, dtype=mass.dtype, device=mass.device), stiffness))
            total = invariant_energy/1.5e5+150.0*normalization+300.0*orthogonality
        else:
            total = torch.trace(stiffness)/1.5e5+250.0*normalization+500.0*orthogonality
        return total, normalization, orthogonality, mass, stiffness

    start_time = time.perf_counter()
    warm_optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    for _ in range(GATE_F_WARM_EPOCHS):
        warm_optimizer.zero_grad(set_to_none=True)
        loss, *_ = loss_parts(projected=False)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite Gate F warm Full-PINN loss")
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
        warm_optimizer.step()

    projected_optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    for _ in range(GATE_F_PROJECTED_EPOCHS):
        projected_optimizer.zero_grad(set_to_none=True)
        loss, *_ = loss_parts(projected=True)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite Gate F projected Full-PINN loss")
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
        projected_optimizer.step()

    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=0.8, max_iter=GATE_F_LBFGS_ITERATIONS,
        max_eval=int(1.4*GATE_F_LBFGS_ITERATIONS), tolerance_grad=1e-10,
        tolerance_change=1e-13, history_size=80, line_search_fn="strong_wolfe")
    def closure():
        lbfgs.zero_grad(set_to_none=True)
        loss, *_ = loss_parts(projected=True)
        loss.backward()
        return loss
    lbfgs.step(closure)
    loss, normalization, orthogonality, mass, stiffness = loss_parts(projected=True)
    eigenvalues, coefficients = eigh(
        stiffness.detach().cpu().numpy(), mass.detach().cpu().numpy(), check_finite=False)
    runtime_seconds = time.perf_counter()-start_time
    model.eval()
    saved = {
        "status": "complete", "alpha": alpha,
        "log10_support_ratio": log10_support_ratio, "seed": seed,
        "model_state_dict": model.state_dict(), "coefficients": coefficients,
        "eigenvalues_scaled": eigenvalues, "runtime_seconds": runtime_seconds,
        "loss": float(loss.detach().cpu()),
        "normalization_loss": float(normalization.detach().cpu()),
        "orthogonality_loss": float(orthogonality.detach().cpu()),
        "training_signature": signature,
        "initial_parameter": (None if initial_state is None else
                              [initial_state["alpha"], initial_state["log10_support_ratio"]]),
        "device": str(gate_f_device), "torch_version": torch.__version__,
        # This flag is set only after the independent FE/MAC/weak-residual
        # validation performed in the next cell.
        "quality_verified": False,
        "quality_verification": None,
    }
    safe_torch_save(
        saved,
        checkpoint,
        stage=f"Gate F complete alpha={alpha:.4f} log10_rK={log10_support_ratio:.4f}",
        retries=CONFIG["checkpoint_save_retries"],
        minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2,
    )
    return {"model": model, "coefficients": coefficients,
            "eigenvalues_scaled": eigenvalues, "runtime_seconds": runtime_seconds,
            "loss": saved["loss"], "normalization_loss": saved["normalization_loss"],
            "orthogonality_loss": saved["orthogonality_loss"],
            "alpha": alpha, "log10_support_ratio": log10_support_ratio,
            "seed": seed, "checkpoint": checkpoint, "resumed": False,
            "training_signature": signature,
            "checkpoint_quality_verified": False,
            "checkpoint_provenance": "trained_in_current_run"}

def gate_f_solution(state: dict, s: torch.Tensor, orders=(0, 1, 2)) -> dict:
    raw = state["model"](s)
    coefficients = torch.as_tensor(state["coefficients"], dtype=s.dtype, device=s.device)
    result = {0: raw@coefficients}
    if 1 in orders or 2 in orders:
        result[1] = gate_f_multi_derivative(raw, s, 1)@coefficients
    if 2 in orders:
        result[2] = gate_f_multi_derivative(raw, s, 2)@coefficients
    return result

def gate_f_fe_reference(alpha: float, log10_support_ratio: float,
                        n_elements: int = 116):
    length = GATE_F_LENGTH
    element_length = length/n_elements
    nodes = np.linspace(0.0, length, n_elements+1)
    support_nodes = [int(round(value/element_length)) for value in (0.0, 14.0, 44.0, 58.0)]
    fixed = [2*support_nodes[index] for index in (0, 1, 3)]
    free = np.setdiff1d(np.arange(2*(n_elements+1)), fixed)
    ke = np.array([
        [12, 6*element_length, -12, 6*element_length],
        [6*element_length, 4*element_length**2, -6*element_length, 2*element_length**2],
        [-12, -6*element_length, 12, -6*element_length],
        [6*element_length, 2*element_length**2, -6*element_length, 4*element_length**2],
    ])/element_length**3
    me = element_length*np.array([
        [156, 22*element_length, 54, -13*element_length],
        [22*element_length, 4*element_length**2, 13*element_length, -3*element_length**2],
        [54, 13*element_length, 156, -22*element_length],
        [-13*element_length, -3*element_length**2, -22*element_length, 4*element_length**2],
    ])/420.0
    size = 2*(n_elements+1)
    stiffness = np.zeros((size, size)); mass = np.zeros_like(stiffness)
    for element in range(n_elements):
        center = 0.5*(nodes[element]+nodes[element+1])
        zone = np.exp(-0.5*((center-44.0)/CONFIG["gate_e_local_zone_width_m"])**2)
        rigidity = max(0.08, 1.0-alpha*zone)
        dofs = np.arange(2*element, 2*element+4)
        stiffness[np.ix_(dofs, dofs)] += rigidity*ke
        mass[np.ix_(dofs, dofs)] += me
    stiffness_reduced = stiffness[np.ix_(free, free)]
    mass_reduced = mass[np.ix_(free, free)]
    spring_index = int(np.flatnonzero(free == 2*support_nodes[2])[0])
    stiffness_reduced[spring_index, spring_index] += (
        GATE_F_SPRING_REFERENCE/GATE_F_LENGTH**3*10.0**log10_support_ratio)
    values, vectors = eigh(stiffness_reduced, mass_reduced,
                           subset_by_index=(0, 5), check_finite=False)
    full_vectors = np.zeros((size, 6)); full_vectors[free] = vectors
    return nodes, np.sqrt(np.maximum(values, 0.0)), full_vectors

def gate_f_numpy_weights(count: int) -> np.ndarray:
    weights = np.ones(count)/(count-1)
    weights[[0, -1]] *= 0.5
    return weights

def gate_f_test_functions(s_values: np.ndarray, count: int = 20):
    s = torch.as_tensor(s_values, dtype=torch.get_default_dtype(),
                        device=gate_f_device).reshape(-1, 1)
    s.requires_grad_(True)
    tests, second_derivatives, support_values = [], [], []
    for order in range(1, count+1):
        hard = s*(s-GATE_F_FIXED_SUPPORT)*(1.0-s)/0.055
        test = hard*torch.sin(order*np.pi*s)
        test_2 = gate_f_scalar_derivative(test, s, 2)
        tests.append(test.detach().cpu().numpy()[:, 0])
        second_derivatives.append(test_2.detach().cpu().numpy()[:, 0])
        support = GATE_F_KOPPIGEN*(GATE_F_KOPPIGEN-GATE_F_FIXED_SUPPORT)*(
            1.0-GATE_F_KOPPIGEN)/0.055*np.sin(order*np.pi*GATE_F_KOPPIGEN)
        support_values.append(support)
    return np.column_stack(tests), np.column_stack(second_derivatives), np.asarray(support_values)

def gate_f_weak_residuals(result: dict, basis: dict,
                          alpha: float, log10_support_ratio: float) -> list[dict]:
    s = basis["s"]; weights = basis["weights"]
    zone = np.exp(-0.5*((s-GATE_F_KOPPIGEN)/GATE_F_ZONE_WIDTH)**2)
    ei = np.maximum(0.08, 1.0-alpha*zone)
    tests, tests_2, test_support = gate_f_test_functions(s)
    rows = []
    for mode_index in range(4):
        phi = result["shapes"][:, mode_index]
        phi_2 = result["curvatures"][:, mode_index]
        phi_support = result["support_values"][mode_index]
        eigenvalue = result["eigenvalues_scaled"][mode_index]
        energy_phi = float(np.sum(weights*ei*phi_2**2) +
                           GATE_F_SPRING_REFERENCE*10.0**log10_support_ratio*phi_support**2)
        mass_phi = float(np.sum(weights*phi**2))
        weak_stiffness = (tests_2.T@(weights*ei*phi_2) +
                          GATE_F_SPRING_REFERENCE*10.0**log10_support_ratio*test_support*phi_support)
        weak_mass = tests.T@(weights*phi)
        test_energy = (np.sum((weights*ei)[:, None]*tests_2**2, axis=0) +
                       GATE_F_SPRING_REFERENCE*10.0**log10_support_ratio*test_support**2)
        test_mass = np.sum(weights[:, None]*tests**2, axis=0)
        denominator = (np.sqrt(energy_phi*test_energy) +
                       eigenvalue*np.sqrt(mass_phi*test_mass) + 1e-15)
        relative = np.abs(weak_stiffness-eigenvalue*weak_mass)/denominator
        rows.append({"mode_order": mode_index+1,
                     "weak_residual_max": float(relative.max()),
                     "weak_residual_rms": float(np.sqrt(np.mean(relative**2)))})
    return rows

## 20. Residual-driven GPT-PINN basis in the two-dimensional parameter domain

The candidate rectangle is `alpha in [0,0.65]` and `log10(r_K) in [-2,0]`. It deliberately includes combinations away from the one-dimensional calibrated Z24 trajectory. The greedy algorithm starts from the healthy state only. At each iteration it:

1. samples the four Full-PINN modes from every selected state;
2. mass-orthonormalizes the union by an eigendecomposition of the snapshot Gram matrix;
3. solves the projected generalized eigenproblem at all unselected candidate states;
4. evaluates the maximum independent weak residual;
5. trains a new Full-PINN at the worst candidate, initialized by continuation from the nearest selected network.

The process stops only when the maximum candidate residual is at most 3% or five Full-PINN states have been used. The online coefficients are therefore obtained by a closed-form reduced Ritz solve, not by FE labels or interpolation.

In [27]:

def gate_f_hermite_fe_fields(
    nodes: np.ndarray,
    full_vectors: np.ndarray,
    s_values: np.ndarray,
    *,
    mode_count: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """Evaluate cubic-Hermite FE displacement and d²/ds² fields.

    The FE rotational degrees of freedom are physical derivatives dphi/dx.
    Curvatures are converted to the nondimensional Gate-F coordinate
    s=x/L through d²phi/ds²=L² d²phi/dx².
    """
    nodes = np.asarray(nodes, dtype=float)
    s_values = np.asarray(s_values, dtype=float)
    if nodes.ndim != 1 or len(nodes) < 2:
        raise ValueError("Gate-F FE proxy requires a one-dimensional node grid")
    if full_vectors.ndim != 2 or full_vectors.shape[0] != 2 * len(nodes):
        raise ValueError("Gate-F FE vector shape is incompatible with Hermite interpolation")
    if not np.all(np.diff(nodes) > 0.0):
        raise ValueError("Gate-F FE nodes must be strictly increasing")

    x_values = np.clip(s_values, 0.0, 1.0) * GATE_F_LENGTH
    element_indices = np.searchsorted(nodes, x_values, side="right") - 1
    element_indices = np.clip(element_indices, 0, len(nodes) - 2)
    left = nodes[element_indices]
    right = nodes[element_indices + 1]
    element_lengths = right - left
    local = (x_values - left) / element_lengths

    n1 = 1.0 - 3.0 * local**2 + 2.0 * local**3
    n2 = element_lengths * (local - 2.0 * local**2 + local**3)
    n3 = 3.0 * local**2 - 2.0 * local**3
    n4 = element_lengths * (-local**2 + local**3)

    d2n1_dx2 = (-6.0 + 12.0 * local) / element_lengths**2
    d2n2_dx2 = (-4.0 + 6.0 * local) / element_lengths
    d2n3_dx2 = (6.0 - 12.0 * local) / element_lengths**2
    d2n4_dx2 = (-2.0 + 6.0 * local) / element_lengths

    shapes = np.empty((len(s_values), mode_count), dtype=float)
    curvatures = np.empty_like(shapes)
    for mode_index in range(mode_count):
        w_left = full_vectors[2 * element_indices, mode_index]
        theta_left = full_vectors[2 * element_indices + 1, mode_index]
        w_right = full_vectors[2 * (element_indices + 1), mode_index]
        theta_right = full_vectors[2 * (element_indices + 1) + 1, mode_index]

        shape = n1 * w_left + n2 * theta_left + n3 * w_right + n4 * theta_right
        curvature_x = (
            d2n1_dx2 * w_left
            + d2n2_dx2 * theta_left
            + d2n3_dx2 * w_right
            + d2n4_dx2 * theta_right
        )
        curvature_s = GATE_F_LENGTH**2 * curvature_x

        norm = float(np.sqrt(
            max(numpy_trapezoid(shape**2, s_values), 1.0e-30)
        ))
        shapes[:, mode_index] = shape / norm
        curvatures[:, mode_index] = curvature_s / norm

    return shapes, curvatures


def gate_f_make_smoke_fe_proxy_state(
    alpha: float,
    log10_support_ratio: float,
) -> dict:
    """Create a deterministic FE-snapshot state for smoke workflow testing.

    This object is deliberately not called a Full-PINN checkpoint. It exists
    only so a login-node smoke run can exercise the reduced-basis, inverse,
    plotting, and artifact paths when no validated Full-PINN checkpoint is
    available. A full Slurm run never uses this branch.
    """
    if gate_f_mode != "smoke":
        raise RuntimeError("The Gate-F FE proxy is permitted only in smoke mode")

    start = time.perf_counter()
    proxy_point_count = max(1001, int(CONFIG["gate_f_basis_points"]))
    proxy_s = np.linspace(0.0, 1.0, proxy_point_count)
    nodes, fe_omegas, fe_vectors = gate_f_fe_reference(
        alpha,
        log10_support_ratio,
        n_elements=116,
    )
    proxy_shapes, proxy_curvatures = gate_f_hermite_fe_fields(
        nodes,
        fe_vectors,
        proxy_s,
        mode_count=4,
    )
    proxy_support = np.array([
        np.interp(GATE_F_KOPPIGEN, proxy_s, proxy_shapes[:, mode_index])
        for mode_index in range(4)
    ])
    proxy_weights = gate_f_numpy_weights(len(proxy_s))
    gram = proxy_shapes.T @ (proxy_weights[:, None] * proxy_shapes)
    diagonal = np.diag(gram)
    off_diagonal = gram - np.diag(diagonal)
    normalization_loss = float(np.mean((diagonal - 1.0) ** 2))
    orthogonality_loss = float(np.mean(off_diagonal**2))
    runtime_seconds = float(time.perf_counter() - start)

    marker_root = OUTPUT_ROOT / "gate_f_smoke_fe_proxy"
    marker_root.mkdir(parents=True, exist_ok=True)
    marker_path = marker_root / (
        f"fe_proxy_a{alpha:.4f}_r{log10_support_ratio:.4f}.json"
    )
    marker = {
        "status": "SMOKE_FE_PROXY_ONLY",
        "alpha": float(alpha),
        "log10_support_ratio": float(log10_support_ratio),
        "source": "gate_f_fe_reference_116_element_cubic_hermite_snapshot",
        "publication_evidence": False,
        "reason": (
            "No independently quality-validated Full-PINN checkpoint was "
            "available during a login-node smoke run."
        ),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(
        json_dumps_safe(marker, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    return {
        "model": None,
        "coefficients": np.eye(4),
        "eigenvalues_scaled": (np.asarray(fe_omegas[:4]) * GATE_F_LENGTH**2) ** 2,
        "runtime_seconds": runtime_seconds,
        "loss": 0.0,
        "normalization_loss": normalization_loss,
        "orthogonality_loss": orthogonality_loss,
        "alpha": float(alpha),
        "log10_support_ratio": float(log10_support_ratio),
        "seed": int(CONFIG["seed"]),
        "checkpoint": marker_path,
        "resumed": False,
        "training_signature": {
            "warm_epochs": 0,
            "projected_epochs": 0,
            "lbfgs_iterations": 0,
            "architecture": "FE_SMOKE_PROXY",
            "collocation_points": int(proxy_point_count),
            "support_stiffness_convention": CONFIG["support_stiffness_convention"],
            "physics": "three_span_FE_smoke_proxy_v1",
        },
        "checkpoint_quality_verified": False,
        "checkpoint_provenance": str(marker_path),
        "state_source": "FE_SMOKE_PROXY",
        "publication_evidence": False,
        "proxy_s": proxy_s,
        "proxy_shapes": proxy_shapes,
        "proxy_curvatures": proxy_curvatures,
        "proxy_support": proxy_support,
    }


def gate_f_snapshot(state: dict, s_values: np.ndarray):
    if state.get("state_source") == "FE_SMOKE_PROXY":
        source_s = np.asarray(state["proxy_s"], dtype=float)
        target_s = np.asarray(s_values, dtype=float)
        shapes = np.column_stack([
            np.interp(target_s, source_s, state["proxy_shapes"][:, mode_index])
            for mode_index in range(4)
        ])
        curvatures = np.column_stack([
            np.interp(target_s, source_s, state["proxy_curvatures"][:, mode_index])
            for mode_index in range(4)
        ])
        support = np.asarray(state["proxy_support"], dtype=float)
        return shapes, curvatures, support

    s = torch.as_tensor(
        s_values,
        dtype=torch.get_default_dtype(),
        device=gate_f_device,
    ).reshape(-1, 1)
    s.requires_grad_(True)
    solved = gate_f_solution(state, s, orders=(0, 1, 2))
    support_coordinate = torch.tensor(
        [[GATE_F_KOPPIGEN]],
        dtype=s.dtype,
        device=s.device,
    )
    support = gate_f_solution(state, support_coordinate, orders=(0,))[0]
    return (
        solved[0].detach().cpu().numpy(),
        solved[2].detach().cpu().numpy(),
        support.detach().cpu().numpy()[0],
    )


def gate_f_build_reduced_basis(states: list[dict], s_values: np.ndarray) -> dict:
    snapshots = [gate_f_snapshot(state, s_values) for state in states]
    shapes = np.column_stack([snapshot[0] for snapshot in snapshots])
    curvatures = np.column_stack([snapshot[1] for snapshot in snapshots])
    supports = np.concatenate([snapshot[2] for snapshot in snapshots])
    weights = gate_f_numpy_weights(len(s_values))
    gram = shapes.T@(weights[:, None]*shapes)
    values, vectors = eigh(gram, check_finite=False)
    retained = values > max(values)*1e-9
    transform = vectors[:, retained]/np.sqrt(values[retained])[None, :]
    basis_shapes = shapes@transform
    basis_curvatures = curvatures@transform
    basis_support = supports@transform
    mass_error = float(np.max(np.abs(
        basis_shapes.T@(weights[:, None]*basis_shapes)-np.eye(int(retained.sum())))))
    return {"s": np.asarray(s_values), "weights": weights,
            "shapes": basis_shapes, "curvatures": basis_curvatures,
            "support": basis_support, "rank": int(retained.sum()),
            "mass_orthogonality_error": mass_error,
            "snapshot_singular_values": np.sqrt(np.maximum(values[retained], 0.0))}

def gate_f_gpt_query(basis: dict, alpha: float,
                     log10_support_ratio: float) -> dict:
    s = basis["s"]; weights = basis["weights"]
    zone = np.exp(-0.5*((s-GATE_F_KOPPIGEN)/GATE_F_ZONE_WIDTH)**2)
    ei = np.maximum(0.08, 1.0-alpha*zone)
    stiffness = (basis["curvatures"].T@(
        (weights*ei)[:, None]*basis["curvatures"]) +
        GATE_F_SPRING_REFERENCE*10.0**log10_support_ratio*np.outer(
            basis["support"], basis["support"]))
    eigenvalues, coefficients = eigh(stiffness, subset_by_index=(0, 3), check_finite=False)
    return {"alpha": alpha, "log10_support_ratio": log10_support_ratio,
            "eigenvalues_scaled": eigenvalues,
            "omegas": np.sqrt(np.maximum(eigenvalues, 0.0))/GATE_F_LENGTH**2,
            "coefficients": coefficients,
            "shapes": basis["shapes"]@coefficients,
            "curvatures": basis["curvatures"]@coefficients,
            "support_values": basis["support"]@coefficients}

def gate_f_validate_full_state(state: dict) -> list[dict]:
    s_values = np.linspace(0.0, 1.0, CONFIG["gate_f_basis_points"])
    shapes, curvatures, support = gate_f_snapshot(state, s_values)
    result = {"eigenvalues_scaled": state["eigenvalues_scaled"],
              "omegas": np.sqrt(np.maximum(state["eigenvalues_scaled"], 0.0))/GATE_F_LENGTH**2,
              "shapes": shapes, "curvatures": curvatures,
              "support_values": support}
    temporary_basis = {"s": s_values, "weights": gate_f_numpy_weights(len(s_values))}
    weak_rows = gate_f_weak_residuals(
        result, temporary_basis, state["alpha"], state["log10_support_ratio"])
    nodes, fe_omegas, fe_modes = gate_f_fe_reference(
        state["alpha"], state["log10_support_ratio"], n_elements=116)
    x = nodes/GATE_F_LENGTH
    rows = []
    for mode_index in range(4):
        predicted = np.interp(x, s_values, shapes[:, mode_index])
        reference = fe_modes[0::2, mode_index]
        predicted /= np.sqrt(numpy_trapezoid(predicted**2, x))
        reference /= np.sqrt(numpy_trapezoid(reference**2, x))
        if numpy_trapezoid(predicted*reference, x) < 0.0:
            predicted = -predicted
        mac = float(numpy_trapezoid(predicted*reference, x)**2)
        rows.append({
            "alpha": state["alpha"],
            "log10_support_ratio": state["log10_support_ratio"],
            "mode_order": mode_index+1,
            "full_pinn_omega": result["omegas"][mode_index],
            "fe_omega": fe_omegas[mode_index],
            "frequency_relative_error": float(abs(
                result["omegas"][mode_index]-fe_omegas[mode_index])/fe_omegas[mode_index]),
            "displacement_mac": mac,
            **weak_rows[mode_index],
            "training_runtime_seconds": state["runtime_seconds"],
            "training_loss": state["loss"],
            "normalization_loss": state["normalization_loss"],
            "orthogonality_loss": state["orthogonality_loss"],
            "resumed": state["resumed"],
            "checkpoint": str(state["checkpoint"]),
        })
    return rows


def gate_f_full_state_quality_report(rows: list[dict]) -> dict:
    frame = pd.DataFrame(rows)
    required_columns = {
        "frequency_relative_error",
        "displacement_mac",
        "weak_residual_max",
        "normalization_loss",
        "orthogonality_loss",
    }
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise RuntimeError(f"Gate F quality report is missing columns: {missing}")
    finite = bool(np.isfinite(frame[list(required_columns)].to_numpy(dtype=float)).all())
    metrics = {
        "maximum_frequency_relative_error": float(frame["frequency_relative_error"].max()),
        "minimum_displacement_mac": float(frame["displacement_mac"].min()),
        "maximum_weak_residual": float(frame["weak_residual_max"].max()),
        "maximum_normalization_loss": float(frame["normalization_loss"].max()),
        "maximum_orthogonality_loss": float(frame["orthogonality_loss"].max()),
    }
    checks = {
        "finite": finite,
        "frequency_error_le_1pct": bool(
            metrics["maximum_frequency_relative_error"]
            <= CONFIG["gate_f_max_full_pinn_frequency_error"]
        ),
        "mac_ge_0p99": bool(
            metrics["minimum_displacement_mac"] >= CONFIG["gate_f_min_full_pinn_mac"]
        ),
        "weak_residual_le_3pct": bool(
            metrics["maximum_weak_residual"]
            <= CONFIG["gate_f_max_full_pinn_weak_residual"]
        ),
        "mass_normalization_loss_le_1e_5": bool(
            metrics["maximum_normalization_loss"]
            <= CONFIG["gate_f_max_mass_orthogonality_error"]
        ),
        "orthogonality_loss_le_1e_5": bool(
            metrics["maximum_orthogonality_loss"]
            <= CONFIG["gate_f_max_mass_orthogonality_error"]
        ),
    }
    return {
        "passed": bool(all(checks.values())),
        "checks": checks,
        "failed_checks": [name for name, passed in checks.items() if not passed],
        "metrics": metrics,
    }


def gate_f_full_state_passes(rows: list[dict]) -> bool:
    return bool(gate_f_full_state_quality_report(rows)["passed"])


gate_f_quality_audit_rows: list[dict] = []


def gate_f_record_quality_audit(
    *,
    state: dict,
    rows: list[dict],
    report: dict,
    role: str,
    candidate_source: str,
) -> None:
    signature = state.get("training_signature", {})
    for row in rows:
        gate_f_quality_audit_rows.append({
            "role": role,
            "candidate_source": candidate_source,
            "alpha": float(state["alpha"]),
            "log10_support_ratio": float(state["log10_support_ratio"]),
            "mode_order": int(row["mode_order"]),
            "passed_state_gate": bool(report["passed"]),
            "failed_checks": ",".join(report["failed_checks"]),
            "frequency_relative_error": float(row["frequency_relative_error"]),
            "displacement_mac": float(row["displacement_mac"]),
            "weak_residual_max": float(row["weak_residual_max"]),
            "weak_residual_rms": float(row["weak_residual_rms"]),
            "normalization_loss": float(row["normalization_loss"]),
            "orthogonality_loss": float(row["orthogonality_loss"]),
            "warm_epochs": signature.get("warm_epochs"),
            "projected_epochs": signature.get("projected_epochs"),
            "lbfgs_iterations": signature.get("lbfgs_iterations"),
            "resumed": bool(state.get("resumed", False)),
            "checkpoint": str(state["checkpoint"]),
            "checkpoint_provenance": str(state.get("checkpoint_provenance", "")),
        })



def gate_f_mark_quality_verified(
    state: dict,
    report: dict,
    *,
    quality_source_checkpoint: str,
) -> None:
    if state.get("state_source") == "FE_SMOKE_PROXY":
        # A deterministic FE proxy is useful only for workflow execution.
        # It must never be converted into, or certified as, a Full-PINN checkpoint.
        state["checkpoint_quality_verified"] = False
        state["publication_evidence"] = False
        return

    checkpoint = Path(state["checkpoint"])
    payload = safe_torch_load(
        checkpoint,
        map_location="cpu",
        stage=f"Gate F quality certificate alpha={state['alpha']:.4f} "
              f"log10_rK={state['log10_support_ratio']:.4f}",
        quarantine_on_error=False,
    )
    if not isinstance(payload, dict):
        warnings.warn(
            f"Gate F state passed quality validation but checkpoint {checkpoint} "
            "could not be reopened for certification.",
            RuntimeWarning,
            stacklevel=2,
        )
        return
    payload["quality_verified"] = True
    payload["quality_verification"] = report
    payload["quality_verified_at_utc"] = datetime.now(timezone.utc).isoformat()
    payload["quality_source_checkpoint"] = str(quality_source_checkpoint)
    safe_torch_save(
        payload,
        checkpoint,
        stage=f"Gate F quality-certified checkpoint alpha={state['alpha']:.4f} "
              f"log10_rK={state['log10_support_ratio']:.4f}",
        retries=CONFIG["checkpoint_save_retries"],
        minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2,
    )
    state["checkpoint_quality_verified"] = True
    state["checkpoint_provenance"] = str(quality_source_checkpoint)


def gate_f_checkpoint_candidate_files(
    alpha: float,
    log10_support_ratio: float,
) -> list[Path]:
    """Discover matching Gate-F checkpoints across every historical layout."""
    target = gate_f_checkpoint_path(alpha, log10_support_ratio)
    filename = target.name
    paths: list[Path] = []
    seen: set[Path] = set()

    if target.is_file():
        paths.append(target)
        seen.add(target.resolve())

    for candidate in iter_legacy_checkpoint_candidates(
        filename,
        exclude=target,
    ):
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        paths.append(candidate)

    # Defensive global scan: older development notebooks sometimes wrote
    # checkpoints below outputs/checkpoints or a run-specific subtree.
    if CONFIG.get("gate_f_recursive_checkpoint_scan", True):
        output_tree = PROJECT_ROOT / "outputs"
        if output_tree.is_dir():
            for candidate in output_tree.rglob(filename):
                try:
                    resolved = candidate.resolve()
                except OSError:
                    continue
                if resolved == target.resolve() or resolved in seen:
                    continue
                if not candidate.is_file():
                    continue
                seen.add(resolved)
                paths.append(candidate)

    return paths


def gate_f_payload_candidates(
    alpha: float,
    log10_support_ratio: float,
    requested_signature: dict,
):
    candidates = []
    for candidate in gate_f_checkpoint_candidate_files(
        alpha,
        log10_support_ratio,
    ):
        payload = safe_torch_load(
            candidate,
            map_location=gate_f_device,
            stage=f"Gate F checkpoint candidate probe {candidate.name}",
            quarantine_on_error=False,
        )
        if not gate_f_checkpoint_payload_compatible(
            payload,
            alpha=alpha,
            log10_support_ratio=log10_support_ratio,
            requested_signature=requested_signature,
            # Every candidate is independently checked below; allowing an
            # unverified exact-smoke payload here does not certify it.
            allow_unverified_exact_smoke=True,
        ):
            continue
        signature = payload.get("training_signature", {})
        budget_score = sum(
            int(signature.get(key, 0))
            for key in ("warm_epochs", "projected_epochs", "lbfgs_iterations")
        )
        quality_score = int(bool(payload.get("quality_verified", False)))
        candidates.append(
            (quality_score, budget_score, candidate, payload)
        )

    # Prefer an already certified checkpoint, then the largest training budget.
    return sorted(
        candidates,
        key=lambda item: (item[0], item[1], str(item[2])),
        reverse=True,
    )


def gate_f_try_checkpoint_candidates(
    alpha: float,
    log10_support_ratio: float,
    requested_signature: dict,
    *,
    role: str,
) -> tuple[dict, list[dict], dict] | None:
    target = gate_f_checkpoint_path(alpha, log10_support_ratio)

    for _, _, candidate, payload in gate_f_payload_candidates(
        alpha,
        log10_support_ratio,
        requested_signature,
    ):
        candidate_state = gate_f_state_from_payload(payload, candidate)
        candidate_rows = gate_f_validate_full_state(candidate_state)
        candidate_report = gate_f_full_state_quality_report(candidate_rows)
        gate_f_record_quality_audit(
            state=candidate_state,
            rows=candidate_rows,
            report=candidate_report,
            role=role,
            candidate_source=str(candidate),
        )
        if not candidate_report["passed"]:
            continue

        imported_payload = dict(payload)
        imported_payload["imported_from_legacy"] = str(candidate)
        imported_payload["imported_into_version"] = VERSION
        imported_payload["quality_verified"] = True
        imported_payload["quality_verification"] = candidate_report
        imported_payload["quality_verified_at_utc"] = datetime.now(timezone.utc).isoformat()
        imported_payload["quality_source_checkpoint"] = str(candidate)
        safe_torch_save(
            imported_payload,
            target,
            stage=f"Gate F quality-validated import alpha={alpha:.4f} "
                  f"log10_rK={log10_support_ratio:.4f}",
            retries=CONFIG["checkpoint_save_retries"],
            minimum_free_bytes=CONFIG["checkpoint_minimum_free_mib"] * 1024**2,
        )
        accepted_state = gate_f_state_from_payload(imported_payload, target)
        accepted_state["checkpoint_provenance"] = str(candidate)
        accepted_state["checkpoint_quality_verified"] = True
        accepted_state["state_source"] = "FULL_PINN_CHECKPOINT"
        accepted_state["publication_evidence"] = gate_f_mode == "full"
        for row in candidate_rows:
            row["checkpoint"] = str(target)
            row["state_source"] = "FULL_PINN_CHECKPOINT"
        print(
            f"Gate F {role}: accepted independently validated checkpoint "
            f"{candidate}"
        )
        return accepted_state, candidate_rows, candidate_report

    return None


def gate_f_get_quality_validated_state(
    alpha: float,
    log10_support_ratio: float,
    *,
    initial_state: dict | None,
    role: str,
) -> tuple[dict, list[dict], dict]:
    requested_signature = {
        "warm_epochs": GATE_F_WARM_EPOCHS,
        "projected_epochs": GATE_F_PROJECTED_EPOCHS,
        "lbfgs_iterations": GATE_F_LBFGS_ITERATIONS,
        "architecture": CONFIG["gate_f_network_architecture"],
        "collocation_points": CONFIG["gate_f_collocation_points"],
        "support_stiffness_convention": CONFIG["support_stiffness_convention"],
        "physics": "three_span_variational_subspace_v05",
    }

    # Search and independently validate every historical layout before
    # spending time on a new optimization.
    recovered = gate_f_try_checkpoint_candidates(
        alpha,
        log10_support_ratio,
        requested_signature,
        role=role,
    )
    if recovered is not None:
        return recovered

    # A smoke run is a workflow test, not a publication-quality Full-PINN
    # training run. When no valid checkpoint exists, use a deterministic
    # FE-snapshot proxy rather than repeatedly training a known-underconverged
    # neural state. This branch is prohibited in full mode and is surfaced in
    # every final certificate.
    if (
        gate_f_mode == "smoke"
        and CONFIG.get(
            "gate_f_smoke_fe_proxy_when_no_validated_checkpoint",
            True,
        )
    ):
        proxy_state = gate_f_make_smoke_fe_proxy_state(
            alpha,
            log10_support_ratio,
        )
        proxy_rows = gate_f_validate_full_state(proxy_state)
        for row in proxy_rows:
            row["state_source"] = "FE_SMOKE_PROXY"
        proxy_report = gate_f_full_state_quality_report(proxy_rows)
        gate_f_record_quality_audit(
            state=proxy_state,
            rows=proxy_rows,
            report=proxy_report,
            role=role,
            candidate_source="FE_SMOKE_PROXY",
        )
        if not proxy_report["passed"]:
            raise RuntimeError(
                "Internal Gate-F FE smoke proxy failed its deterministic "
                "regression checks: "
                + json_dumps_safe(proxy_report, sort_keys=True)
            )
        warnings.warn(
            "Gate F is using an FE-snapshot smoke proxy because no "
            "quality-validated Full-PINN checkpoint was found. This permits "
            "workflow execution only; it is not GPT-PINN publication evidence.",
            RuntimeWarning,
            stacklevel=2,
        )
        return proxy_state, proxy_rows, proxy_report

    # Full mode trains the actual bridge Full-PINN when no compatible,
    # independently validated checkpoint is available.
    state = gate_f_train_full_pinn(
        alpha,
        log10_support_ratio,
        initial_state=initial_state,
    )
    state.setdefault("state_source", "FULL_PINN_TRAINED")
    state.setdefault("publication_evidence", gate_f_mode == "full")
    rows = gate_f_validate_full_state(state)
    for row in rows:
        row["state_source"] = state["state_source"]
    report = gate_f_full_state_quality_report(rows)
    gate_f_record_quality_audit(
        state=state,
        rows=rows,
        report=report,
        role=role,
        candidate_source=str(state.get("checkpoint_provenance", "current")),
    )
    if report["passed"]:
        gate_f_mark_quality_verified(
            state,
            report,
            quality_source_checkpoint=str(
                state.get("checkpoint_provenance", state["checkpoint"])
            ),
        )
        return state, rows, report

    quality_audit = pd.DataFrame(gate_f_quality_audit_rows)
    quality_audit.to_csv(
        TABLE_ROOT / f"gate_f_checkpoint_quality_audit_{gate_f_mode}.csv",
        index=False,
    )
    failure = {
        "role": role,
        "execution_mode": gate_f_mode,
        "alpha": float(alpha),
        "log10_support_ratio": float(log10_support_ratio),
        "strict_thresholds": {
            "frequency_relative_error": CONFIG["gate_f_max_full_pinn_frequency_error"],
            "minimum_mac": CONFIG["gate_f_min_full_pinn_mac"],
            "maximum_weak_residual": CONFIG["gate_f_max_full_pinn_weak_residual"],
            "maximum_mass_error": CONFIG["gate_f_max_mass_orthogonality_error"],
        },
        "initial_report": report,
        "checkpoint_search_roots": [str(path) for path in LEGACY_CHECKPOINT_ROOTS],
        "checkpoint_files_discovered": [
            str(path)
            for path in gate_f_checkpoint_candidate_files(
                alpha,
                log10_support_ratio,
            )
        ],
        "diagnosis": (
            "A full-profile bridge Full-PINN failed its independent "
            "frequency/MAC/weak-form/mass audit."
        ),
        "required_action": (
            "Inspect the quality table and rerun in full mode on an allocated "
            "Slurm compute node. Scientific thresholds are intentionally unchanged."
        ),
    }
    (OUTPUT_ROOT / "gate_f_checkpoint_quality_failure.json").write_text(
        json_dumps_safe(failure, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    raise RuntimeError(
        "Gate F Full-PINN quality validation failed in full mode. "
        + json_dumps_safe(failure, sort_keys=True)
    )


candidate_alpha = np.linspace(
    *CONFIG["gate_f_alpha_domain"],
    GATE_F_CANDIDATE_AXIS_COUNT,
)
candidate_log_support = np.linspace(
    *CONFIG["gate_f_log10_support_domain"],
    GATE_F_CANDIDATE_AXIS_COUNT,
)
gate_f_candidates = [
    (float(alpha), float(log_ratio))
    for alpha in candidate_alpha
    for log_ratio in candidate_log_support
]
gate_f_selected_parameters = [(0.0, 0.0)]
healthy_state, healthy_rows, healthy_report = gate_f_get_quality_validated_state(
    0.0,
    0.0,
    initial_state=None,
    role="healthy_initial_state",
)
gate_f_states = [healthy_state]
gate_f_full_rows = list(healthy_rows)
gate_f_greedy_rows = []
gate_f_candidate_score_rows = []
gate_f_s_values = np.linspace(0.0, 1.0, CONFIG["gate_f_basis_points"])

while True:
    gate_f_basis = gate_f_build_reduced_basis(gate_f_states, gate_f_s_values)
    unselected = [
        candidate
        for candidate in gate_f_candidates
        if candidate not in gate_f_selected_parameters
    ]
    scored = []
    for alpha, log_ratio in unselected:
        query = gate_f_gpt_query(gate_f_basis, alpha, log_ratio)
        weak = gate_f_weak_residuals(query, gate_f_basis, alpha, log_ratio)
        maximum = max(row["weak_residual_max"] for row in weak)
        rms_maximum = max(row["weak_residual_rms"] for row in weak)
        scored.append((maximum, alpha, log_ratio, rms_maximum))
        gate_f_candidate_score_rows.append({
            "basis_size": len(gate_f_states),
            "reduced_rank": gate_f_basis["rank"],
            "alpha": alpha,
            "log10_support_ratio": log_ratio,
            "maximum_weak_residual": maximum,
            "maximum_weak_residual_rms": rms_maximum,
        })
    if not scored:
        break
    worst = max(scored)
    gate_f_greedy_rows.append({
        "basis_size": len(gate_f_states),
        "reduced_rank": gate_f_basis["rank"],
        "selected_alpha": gate_f_selected_parameters[-1][0],
        "selected_log10_support_ratio": gate_f_selected_parameters[-1][1],
        "maximum_unselected_weak_residual": worst[0],
        "worst_candidate_alpha": worst[1],
        "worst_candidate_log10_support_ratio": worst[2],
        "basis_mass_orthogonality_error": gate_f_basis["mass_orthogonality_error"],
    })
    print(
        "Gate F basis",
        len(gate_f_states),
        "rank",
        gate_f_basis["rank"],
        "max weak residual",
        worst[0],
        "at",
        (worst[1], worst[2]),
    )
    if (
        worst[0] <= CONFIG["gate_f_greedy_weak_residual_tolerance"]
        or len(gate_f_states) >= GATE_F_MAX_BASIS_STATES
    ):
        break

    next_parameter = (worst[1], worst[2])
    alpha_scale = (
        CONFIG["gate_f_alpha_domain"][1] - CONFIG["gate_f_alpha_domain"][0]
    )
    support_scale = (
        CONFIG["gate_f_log10_support_domain"][1]
        - CONFIG["gate_f_log10_support_domain"][0]
    )
    nearest_index = int(np.argmin([
        ((state["alpha"] - next_parameter[0]) / alpha_scale) ** 2
        + (
            (state["log10_support_ratio"] - next_parameter[1])
            / support_scale
        ) ** 2
        for state in gate_f_states
    ]))
    new_state, new_rows, new_report = gate_f_get_quality_validated_state(
        next_parameter[0],
        next_parameter[1],
        initial_state=gate_f_states[nearest_index],
        role=f"greedy_state_{len(gate_f_states) + 1}",
    )
    gate_f_states.append(new_state)
    gate_f_selected_parameters.append(next_parameter)
    gate_f_full_rows.extend(new_rows)

GATE_F_SMOKE_PROXY_USED = any(
    state.get("state_source") == "FE_SMOKE_PROXY"
    for state in gate_f_states
)
GATE_F_STATE_SOURCE_CLASSES = sorted({
    str(state.get("state_source", "FULL_PINN"))
    for state in gate_f_states
})
print("Gate F state sources:", GATE_F_STATE_SOURCE_CLASSES)
print("Gate F FE smoke proxy used:", GATE_F_SMOKE_PROXY_USED)

gate_f_full_verification = pd.DataFrame(gate_f_full_rows)
gate_f_greedy_history = pd.DataFrame(gate_f_greedy_rows)
gate_f_candidate_scores = pd.DataFrame(gate_f_candidate_score_rows)
gate_f_checkpoint_quality_audit = pd.DataFrame(gate_f_quality_audit_rows)

gate_f_full_verification.to_csv(
    TABLE_ROOT / f"gate_f_full_pinn_verification_{gate_f_mode}.csv",
    index=False,
)
gate_f_greedy_history.to_csv(
    TABLE_ROOT / f"gate_f_greedy_history_{gate_f_mode}.csv",
    index=False,
)
gate_f_candidate_scores.to_csv(
    TABLE_ROOT / f"gate_f_candidate_physics_scores_{gate_f_mode}.csv",
    index=False,
)
gate_f_checkpoint_quality_audit.to_csv(
    TABLE_ROOT / f"gate_f_checkpoint_quality_audit_{gate_f_mode}.csv",
    index=False,
)
display(gate_f_full_verification)
display(gate_f_greedy_history)
display(gate_f_checkpoint_quality_audit)

gate_f_full_checks = {
    "frequency_error_le_1pct": bool(
        (
            gate_f_full_verification["frequency_relative_error"]
            <= CONFIG["gate_f_max_full_pinn_frequency_error"]
        ).all()
    ),
    "mac_ge_0p99": bool(
        (
            gate_f_full_verification["displacement_mac"]
            >= CONFIG["gate_f_min_full_pinn_mac"]
        ).all()
    ),
    "weak_residual_le_3pct": bool(
        (
            gate_f_full_verification["weak_residual_max"]
            <= CONFIG["gate_f_max_full_pinn_weak_residual"]
        ).all()
    ),
    "mass_normalization_loss_le_1e_5": bool(
        (
            gate_f_full_verification["normalization_loss"]
            <= CONFIG["gate_f_max_mass_orthogonality_error"]
        ).all()
    ),
    "orthogonality_loss_le_1e_5": bool(
        (
            gate_f_full_verification["orthogonality_loss"]
            <= CONFIG["gate_f_max_mass_orthogonality_error"]
        ).all()
    ),
}
if not all(gate_f_full_checks.values()):
    raise RuntimeError(
        f"Gate F Full-PINN checks failed after validated recovery: "
        f"{gate_f_full_checks}"
    )

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
scores_last = gate_f_candidate_scores[
    gate_f_candidate_scores["basis_size"]
    == gate_f_candidate_scores["basis_size"].max()
]
scatter = axes[0].scatter(
    scores_last["alpha"],
    scores_last["log10_support_ratio"],
    c=100 * scores_last["maximum_weak_residual"],
    cmap="viridis",
    s=85,
)
for index, (alpha, log_ratio) in enumerate(
    gate_f_selected_parameters,
    start=1,
):
    axes[0].scatter(
        alpha,
        log_ratio,
        marker="*",
        s=210,
        color="#e76f51",
        edgecolor="white",
    )
    axes[0].annotate(
        str(index),
        (alpha, log_ratio),
        xytext=(5, 5),
        textcoords="offset points",
    )
fig.colorbar(scatter, ax=axes[0], label="final weak residual [%]")
axes[0].set_xlabel(r"Local $EI$ coordinate $\alpha$")
axes[0].set_ylabel(r"$\log_{10}(r_K)$")
axes[0].set_title("Residual-driven states in the 2-D domain")
axes[1].semilogy(
    gate_f_greedy_history["basis_size"],
    gate_f_greedy_history["maximum_unselected_weak_residual"],
    "o-",
    color="#264653",
)
axes[1].axhline(
    CONFIG["gate_f_greedy_weak_residual_tolerance"],
    linestyle="--",
    color="#e76f51",
    label="3% stopping tolerance",
)
axes[1].set_xlabel("Full-PINN state count")
axes[1].set_ylabel("Maximum candidate weak residual")
axes[1].set_title("Greedy convergence")
axes[1].legend()
fig.suptitle(
    "Figure 15 — Gate F two-parameter greedy GPT-PINN construction",
    weight="bold",
)
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT / f"figure_15_gate_f_greedy_{gate_f_mode}.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


Gate F basis 1 rank 4 max weak residual 0.2087431201592475 at (0.0, -2.0)
Gate F basis 2 rank 8 max weak residual 0.07498395996478886 at (0.65, -2.0)
Gate F basis 3 rank 12 max weak residual 0.015831125984835777 at (0.65, -1.5)
Gate F state sources: ['FULL_PINN_TRAINED']
Gate F FE smoke proxy used: False


,alpha,log10_support_ratio,mode_order,full_pinn_omega,fe_omega,frequency_relative_error,displacement_mac,weak_residual_max,weak_residual_rms,training_runtime_seconds,training_loss,normalization_loss,orthogonality_loss,resumed,checkpoint,state_source
0,0.00,0.0,1,0.017451,0.017441,0.000544,0.999998,0.013282,0.005987,305.958790,0.894661,1.107042e-09,9.625768e-10,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
1,0.00,0.0,2,0.046615,0.046598,0.000352,0.999995,0.011046,0.004465,305.958790,0.894661,1.107042e-09,9.625768e-10,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
2,0.00,0.0,3,0.062427,0.062384,0.000682,0.999982,0.007328,0.002352,305.958790,0.894661,1.107042e-09,9.625768e-10,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
3,0.00,0.0,4,0.074055,0.073962,0.001264,0.999965,0.007590,0.003302,305.958790,0.894661,1.107042e-09,9.625768e-10,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
4,0.00,-2.0,1,0.017299,0.017295,0.000269,1.000000,0.008061,0.003895,327.115894,0.824266,2.720511e-11,5.213799e-12,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
5,0.00,-2.0,2,0.046546,0.046542,0.000094,0.999999,0.005257,0.001956,327.115894,0.824266,2.720511e-11,5.213799e-12,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
6,0.00,-2.0,3,0.060063,0.060026,0.000616,0.999986,0.007344,0.002285,327.115894,0.824266,2.720511e-11,5.213799e-12,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
7,0.00,-2.0,4,0.069658,0.069617,0.000591,0.999985,0.004201,0.001707,327.115894,0.824266,2.720511e-11,5.213799e-12,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
8,0.65,-2.0,1,0.016241,0.016219,0.001357,0.999989,0.023123,0.012082,288.395359,0.758816,1.261779e-09,2.367557e-09,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED
9,0.65,-2.0,2,0.045491,0.045477,0.000318,0.999995,0.009073,0.003420,288.395359,0.758816,1.261779e-09,2.367557e-09,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,FULL_PINN_TRAINED


,basis_size,reduced_rank,selected_alpha,selected_log10_support_ratio,maximum_unselected_weak_residual,worst_candidate_alpha,worst_candidate_log10_support_ratio,basis_mass_orthogonality_error
0,1,4,0.00,0.0,0.208743,0.00,-2.0,8.881784e-16
1,2,8,0.00,-2.0,0.074984,0.65,-2.0,3.673762e-09
2,3,12,0.65,-2.0,0.015831,0.65,-1.5,2.922045e-09


,role,candidate_source,alpha,log10_support_ratio,mode_order,passed_state_gate,failed_checks,frequency_relative_error,displacement_mac,weak_residual_max,weak_residual_rms,normalization_loss,orthogonality_loss,warm_epochs,projected_epochs,lbfgs_iterations,resumed,checkpoint,checkpoint_provenance
0,healthy_initial_state,trained_in_current_run,0.00,0.0,1,True,,0.000544,0.999998,0.013282,0.005987,1.107042e-09,9.625768e-10,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
1,healthy_initial_state,trained_in_current_run,0.00,0.0,2,True,,0.000352,0.999995,0.011046,0.004465,1.107042e-09,9.625768e-10,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
2,healthy_initial_state,trained_in_current_run,0.00,0.0,3,True,,0.000682,0.999982,0.007328,0.002352,1.107042e-09,9.625768e-10,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
3,healthy_initial_state,trained_in_current_run,0.00,0.0,4,True,,0.001264,0.999965,0.007590,0.003302,1.107042e-09,9.625768e-10,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
4,greedy_state_2,trained_in_current_run,0.00,-2.0,1,True,,0.000269,1.000000,0.008061,0.003895,2.720511e-11,5.213799e-12,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
5,greedy_state_2,trained_in_current_run,0.00,-2.0,2,True,,0.000094,0.999999,0.005257,0.001956,2.720511e-11,5.213799e-12,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
6,greedy_state_2,trained_in_current_run,0.00,-2.0,3,True,,0.000616,0.999986,0.007344,0.002285,2.720511e-11,5.213799e-12,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
7,greedy_state_2,trained_in_current_run,0.00,-2.0,4,True,,0.000591,0.999985,0.004201,0.001707,2.720511e-11,5.213799e-12,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
8,greedy_state_3,trained_in_current_run,0.65,-2.0,1,True,,0.001357,0.999989,0.023123,0.012082,1.261779e-09,2.367557e-09,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run
9,greedy_state_3,trained_in_current_run,0.65,-2.0,2,True,,0.000318,0.999995,0.009073,0.003420,1.261779e-09,2.367557e-09,1800,4000,800,False,/home/a.riyahi/GPT_PINN/outputs/v0_6_final/che...,trained_in_current_run


## 21. Independent GPT-PINN accuracy and speed validation

The independent validation set contains the 16 cell midpoints of the `5×5` candidate grid; none is a candidate or Full-PINN training state. Four modes give 64 comparisons against a 116-element FE reference. Acceptance requires frequency error ≤1%, displacement MAC ≥0.99, weak residual ≤3%, and a well-orthonormalized reduced mass matrix.

Timing separates three costs:

- total offline Full-PINN basis construction;
- a warm GPT-PINN projected query;
- a direct 116-element FE eigensolve.

The GPT/Full-PINN speed-up is the primary meta-model comparison. The FE baseline is also reported explicitly: for this small one-dimensional model, FE remains a strong competitor and must not be hidden.

In [28]:
def gate_f_validate_gpt_query(result: dict, basis: dict,
                               alpha: float, log10_support_ratio: float) -> list[dict]:
    nodes, fe_omegas, fe_modes = gate_f_fe_reference(
        alpha, log10_support_ratio, n_elements=116)
    x = nodes/GATE_F_LENGTH
    weak_rows = gate_f_weak_residuals(result, basis, alpha, log10_support_ratio)
    rows = []
    for mode_index in range(4):
        predicted = np.interp(x, basis["s"], result["shapes"][:, mode_index])
        reference = fe_modes[0::2, mode_index]
        predicted /= np.sqrt(numpy_trapezoid(predicted**2, x))
        reference /= np.sqrt(numpy_trapezoid(reference**2, x))
        if numpy_trapezoid(predicted*reference, x) < 0.0:
            predicted = -predicted
        mac = float(numpy_trapezoid(predicted*reference, x)**2)
        rows.append({
            "alpha": alpha, "log10_support_ratio": log10_support_ratio,
            "mode_order": mode_index+1,
            "gpt_omega": result["omegas"][mode_index],
            "fe_omega": fe_omegas[mode_index],
            "frequency_relative_error": float(abs(
                result["omegas"][mode_index]-fe_omegas[mode_index])/fe_omegas[mode_index]),
            "displacement_mac": mac,
            **weak_rows[mode_index],
        })
    return rows

alpha_midpoints = 0.5*(candidate_alpha[:-1]+candidate_alpha[1:])
support_midpoints = 0.5*(candidate_log_support[:-1]+candidate_log_support[1:])
gate_f_validation_rows = []
for alpha in alpha_midpoints:
    for log_ratio in support_midpoints:
        query_start = time.perf_counter()
        query = gate_f_gpt_query(gate_f_basis, float(alpha), float(log_ratio))
        query_seconds = time.perf_counter()-query_start
        rows = gate_f_validate_gpt_query(query, gate_f_basis, float(alpha), float(log_ratio))
        for row in rows:
            row["query_seconds_single"] = query_seconds
            gate_f_validation_rows.append(row)
gate_f_validation = pd.DataFrame(gate_f_validation_rows)
gate_f_validation.to_csv(
    TABLE_ROOT/f"gate_f_gpt_independent_validation_{gate_f_mode}.csv", index=False)
display(gate_f_validation)

# Warm-up and isolated timing: query time excludes validation and file I/O.
for _ in range(10):
    gate_f_gpt_query(gate_f_basis, 0.33, -0.88)
    gate_f_fe_reference(0.33, -0.88, n_elements=116)
query_repeats = 1000 if gate_f_mode == "full" else 100
fe_repeats = 40 if gate_f_mode == "full" else 5
start = time.perf_counter()
for _ in range(query_repeats):
    gate_f_gpt_query(gate_f_basis, 0.33, -0.88)
gate_f_query_seconds = (time.perf_counter()-start)/query_repeats
start = time.perf_counter()
for _ in range(fe_repeats):
    gate_f_fe_reference(0.33, -0.88, n_elements=116)
gate_f_fe_seconds = (time.perf_counter()-start)/fe_repeats
gate_f_state_runtime_seconds = np.asarray(
    [float(state["runtime_seconds"]) for state in gate_f_states],
    dtype=float,
)
gate_f_proxy_generation_seconds = float(np.sum(gate_f_state_runtime_seconds))
if GATE_F_SMOKE_PROXY_USED:
    # FE-proxy construction time is not Full-PINN training time.
    gate_f_full_mean_seconds = np.nan
    gate_f_offline_seconds = np.nan
    gate_f_speedup_full = np.nan
else:
    gate_f_full_mean_seconds = float(np.mean(gate_f_state_runtime_seconds))
    gate_f_offline_seconds = float(np.sum(gate_f_state_runtime_seconds))
    gate_f_speedup_full = gate_f_full_mean_seconds/gate_f_query_seconds
gate_f_speedup_fe = gate_f_fe_seconds/gate_f_query_seconds
gate_f_timing = pd.DataFrame([{
    "basis_state_count": len(gate_f_states), "reduced_rank": gate_f_basis["rank"],
    "offline_full_pinn_seconds": gate_f_offline_seconds,
    "mean_full_pinn_training_seconds": gate_f_full_mean_seconds,
    "gpt_query_seconds": gate_f_query_seconds, "fe_116_query_seconds": gate_f_fe_seconds,
    "speedup_vs_full_pinn_training": gate_f_speedup_full,
    "speedup_vs_fe_116": gate_f_speedup_fe,
    "smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "fe_proxy_generation_seconds": (
        gate_f_proxy_generation_seconds
        if GATE_F_SMOKE_PROXY_USED
        else np.nan
    ),
    "query_repeats": query_repeats, "fe_repeats": fe_repeats,
}])
gate_f_timing.to_csv(TABLE_ROOT/f"gate_f_speedup_{gate_f_mode}.csv", index=False)
display(gate_f_timing)

gate_f_gpt_checks = {
    "greedy_residual_le_3pct": bool(gate_f_greedy_history[
        "maximum_unselected_weak_residual"].iloc[-1] <=
        CONFIG["gate_f_greedy_weak_residual_tolerance"]),
    "basis_mass_orthogonality_error_le_1e_5": bool(
        gate_f_basis["mass_orthogonality_error"] <=
        CONFIG["gate_f_max_mass_orthogonality_error"]),
    "independent_frequency_error_le_1pct": bool((gate_f_validation[
        "frequency_relative_error"] <= CONFIG["gate_f_max_gpt_frequency_error"]).all()),
    "independent_mac_ge_0p99": bool((gate_f_validation[
        "displacement_mac"] >= CONFIG["gate_f_min_gpt_mac"]).all()),
    "independent_weak_residual_le_3pct": bool((gate_f_validation[
        "weak_residual_max"] <= CONFIG["gate_f_max_gpt_weak_residual"]).all()),
    "speedup_vs_full_pinn_ge_100": bool(
        (not GATE_F_SMOKE_PROXY_USED)
        and np.isfinite(gate_f_speedup_full)
        and gate_f_speedup_full >= CONFIG["gate_f_min_speedup_vs_full_pinn"]),
    "speedup_vs_fe_ge_2": bool(
        gate_f_speedup_fe >= CONFIG["gate_f_min_speedup_vs_fe"]),
}
gate_f_checks = {
    "full_execution": gate_f_mode == "full",
    **{f"full_pinn_{key}": value for key, value in gate_f_full_checks.items()},
    **{f"gpt_{key}": value for key, value in gate_f_gpt_checks.items()},
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
}
if gate_f_mode == "smoke":
    gate_f_decision = (
        "SMOKE_FE_PROXY_COMPLETED_REQUIRES_FULL_SLURM_RUN"
        if GATE_F_SMOKE_PROXY_USED
        else "SMOKE_COMPLETED_REQUIRES_FULL_RUN"
    )
else:
    gate_f_decision = ("GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL"
                       if all(gate_f_checks.values())
                       else "NO_GO_REVISE_MULTIPARAMETRIC_GPT_PINN")

gate_f_diagnostic = {
    "version": "V0.6 FINAL", "decision": gate_f_decision,
    "execution_mode": gate_f_mode, "execution_boundary": CONFIG["current_execution_boundary"],
    "prior_gate_e_decision": gate_e_decision,
    "parameter_domain": {"alpha": CONFIG["gate_f_alpha_domain"],
                         "log10_support_ratio": CONFIG["gate_f_log10_support_domain"]},
    "selected_full_pinn_states": [list(item) for item in gate_f_selected_parameters],
    "basis_state_count": len(gate_f_states), "reduced_rank": gate_f_basis["rank"],
    "effective_max_basis_states": int(GATE_F_MAX_BASIS_STATES),
    "state_source_classes": GATE_F_STATE_SOURCE_CLASSES,
    "smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "current_run_gate_f_publication_evidence": bool(
        gate_f_mode == "full" and not GATE_F_SMOKE_PROXY_USED
    ),
    "checkpoint_quality_audit_file": str(
        TABLE_ROOT / f"gate_f_checkpoint_quality_audit_{gate_f_mode}.csv"
    ),
    "checkpoint_provenance": [
        {
            "alpha": float(state["alpha"]),
            "log10_support_ratio": float(state["log10_support_ratio"]),
            "source": str(state.get("checkpoint_provenance", state["checkpoint"])),
            "training_signature": state.get("training_signature", {}),
            "quality_verified": bool(state.get("checkpoint_quality_verified", False)),
            "state_source": str(state.get("state_source", "FULL_PINN")),
            "publication_evidence": bool(state.get("publication_evidence", False)),
        }
        for state in gate_f_states
    ],
    "overqualified_checkpoint_reuse_in_smoke": bool(
        gate_f_mode == "smoke"
        and any(
            int(state.get("training_signature", {}).get("warm_epochs", 0))
            > int(CONFIG["gate_f_smoke_warm_adam_epochs"])
            for state in gate_f_states
        )
    ),
    "maximum_final_candidate_weak_residual": float(gate_f_greedy_history[
        "maximum_unselected_weak_residual"].iloc[-1]),
    "full_pinn": {
        "maximum_frequency_relative_error": float(gate_f_full_verification[
            "frequency_relative_error"].max()),
        "minimum_mac": float(gate_f_full_verification["displacement_mac"].min()),
        "maximum_weak_residual": float(gate_f_full_verification["weak_residual_max"].max()),
        "offline_runtime_seconds": (
            None if GATE_F_SMOKE_PROXY_USED else gate_f_offline_seconds
        ),
        "fe_proxy_generation_seconds": (
            gate_f_proxy_generation_seconds
            if GATE_F_SMOKE_PROXY_USED
            else None
        ),
        "state_source": (
            "FE_SMOKE_PROXY"
            if GATE_F_SMOKE_PROXY_USED
            else "FULL_PINN"
        ),
    },
    "independent_gpt_validation": {
        "parameter_point_count": int(len(alpha_midpoints)*len(support_midpoints)),
        "mode_comparison_count": int(len(gate_f_validation)),
        "maximum_frequency_relative_error": float(gate_f_validation[
            "frequency_relative_error"].max()),
        "minimum_mac": float(gate_f_validation["displacement_mac"].min()),
        "maximum_weak_residual": float(gate_f_validation["weak_residual_max"].max()),
    },
    "timing": gate_f_timing.iloc[0].to_dict(),
    "checks": gate_f_checks,
    "fe_baseline_disclosed": True,
    "online_z24_update_performed": False,
    "capacity_or_rul_claim_performed": False,
    "raw_z24_signature_unchanged": gate_f_checks["raw_z24_signature_unchanged"],
    "limitations": [
        "Full-PINN and GPT-PINN solve the calibrated one-dimensional multispan model, not a 3-D bridge",
        "validation is numerical against the FE reference; independent experimental mode shapes remain unavailable",
        "the 2-D rectangle extrapolates model-updating coordinates beyond the calibrated Z24 trajectory",
        "offline Full-PINN cost is substantial and reported explicitly",
        "online Z24 state updating and uncertainty propagation remain the next gate",
        (
            "this smoke run used an FE-snapshot workflow proxy and therefore "
            "does not validate Full-PINN/GPT-PINN training"
            if GATE_F_SMOKE_PROXY_USED
            else "no FE smoke proxy was used"
        ),
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT/"gate_f_diagnostic.json").write_text(
    json_dumps_safe(gate_f_diagnostic, indent=2, sort_keys=True)+"\n", encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.5))
mode_colors = ["#264653", "#2a9d8f", "#e9c46a", "#e76f51"]
for mode_order, color in zip(range(1, 5), mode_colors):
    group = gate_f_validation[gate_f_validation["mode_order"] == mode_order]
    axes[0].scatter(100*group["frequency_relative_error"], group["displacement_mac"],
                    color=color, label=f"mode {mode_order}", alpha=0.85)
axes[0].axvline(100*CONFIG["gate_f_max_gpt_frequency_error"], linestyle="--", color="#555555")
axes[0].axhline(CONFIG["gate_f_min_gpt_mac"], linestyle="--", color="#555555")
axes[0].set_xlabel("Frequency relative error [%]"); axes[0].set_ylabel("Displacement MAC")
axes[0].set_title("64 independent off-grid modal comparisons"); axes[0].legend()
for mode_order, color in zip(range(1, 5), mode_colors):
    group = gate_f_validation[gate_f_validation["mode_order"] == mode_order]
    axes[1].scatter(group["alpha"], 100*group["weak_residual_max"], color=color,
                    label=f"mode {mode_order}", alpha=0.85)
axes[1].axhline(100*CONFIG["gate_f_max_gpt_weak_residual"], linestyle="--", color="#e76f51")
axes[1].set_xlabel(r"Local $EI$ coordinate $\alpha$"); axes[1].set_ylabel("Maximum weak residual [%]")
axes[1].set_title("Independent physics adequacy"); axes[1].legend()
fig.suptitle("Figure 16 — Gate F GPT-PINN accuracy on unseen 2-D states", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_16_gate_f_accuracy_{gate_f_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

worst = gate_f_validation.loc[gate_f_validation["frequency_relative_error"].idxmax()]
worst_query = gate_f_gpt_query(gate_f_basis, float(worst.alpha), float(worst.log10_support_ratio))
nodes, _, fe_modes = gate_f_fe_reference(float(worst.alpha), float(worst.log10_support_ratio), 116)
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.5))
for mode_index, color in enumerate(mode_colors):
    prediction = worst_query["shapes"][:, mode_index]
    reference = fe_modes[0::2, mode_index]
    reference /= np.sqrt(numpy_trapezoid(reference**2, nodes/GATE_F_LENGTH))
    if numpy_trapezoid(np.interp(nodes/GATE_F_LENGTH, gate_f_basis["s"], prediction)*reference,
                nodes/GATE_F_LENGTH) < 0.0:
        prediction = -prediction
    axes[0].plot(gate_f_basis["s"]*GATE_F_LENGTH, prediction, color=color,
                 label=f"GPT mode {mode_index+1}")
    axes[0].plot(nodes, reference, linestyle="--", color=color, linewidth=1)
axes[0].set_xlabel("Bridge coordinate [m]"); axes[0].set_ylabel("Mass-normalized displacement")
axes[0].set_title("Worst-frequency validation state: GPT vs FE"); axes[0].legend(ncol=2, fontsize=8)
if GATE_F_SMOKE_PROXY_USED:
    labels = ["FE proxy\nbuild", "FE 116\nquery", "Reduced proxy\nquery"]
    values = [
        max(gate_f_proxy_generation_seconds / max(len(gate_f_states), 1), 1e-12),
        gate_f_fe_seconds,
        gate_f_query_seconds,
    ]
    timing_title = (
        f"Smoke workflow only: {gate_f_speedup_fe:.1f}× reduced-query/FE ratio; "
        "no Full-PINN speed-up claim"
    )
else:
    labels = ["Full-PINN\ntraining", "FE 116\nquery", "GPT-PINN\nquery"]
    values = [gate_f_full_mean_seconds, gate_f_fe_seconds, gate_f_query_seconds]
    timing_title = (
        f"Speed-up: {gate_f_speedup_full:,.0f}× vs training; "
        f"{gate_f_speedup_fe:.1f}× vs FE"
    )
axes[1].bar(labels, values, color=["#6d597a", "#457b9d", "#e76f51"])
axes[1].set_yscale("log"); axes[1].set_ylabel("Wall time [s], logarithmic")
axes[1].set_title(timing_title)
fig.suptitle(f"Figure 17 — Gate F accuracy and timing: {gate_f_decision.replace('_', ' ')}", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_17_gate_f_speedup_{gate_f_mode}.{suffix}", bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_f_diagnostic, indent=2))

,alpha,log10_support_ratio,mode_order,gpt_omega,fe_omega,frequency_relative_error,displacement_mac,weak_residual_max,weak_residual_rms,query_seconds_single
0,0.08125,-1.75,1,0.017261,0.017259,0.000128,1.000000,0.005333,0.002393,0.000933
1,0.08125,-1.75,2,0.046476,0.046473,0.000080,1.000000,0.003454,0.001899,0.000933
2,0.08125,-1.75,3,0.061001,0.060980,0.000348,0.999990,0.006116,0.002923,0.000933
3,0.08125,-1.75,4,0.071091,0.071059,0.000443,0.999987,0.004800,0.002212,0.000933
4,0.08125,-1.25,1,0.017316,0.017314,0.000128,1.000000,0.004553,0.002294,0.000382
...,...,...,...,...,...,...,...,...,...,...
59,0.56875,-0.75,4,0.070293,0.070243,0.000712,0.999955,0.009186,0.003818,0.000552
60,0.56875,-0.25,1,0.016520,0.016514,0.000385,0.999998,0.007467,0.002589,0.000423
61,0.56875,-0.25,2,0.045713,0.045706,0.000156,0.999998,0.003349,0.001428,0.000423
62,0.56875,-0.25,3,0.059277,0.059220,0.000958,0.999945,0.010416,0.004413,0.000423


,basis_state_count,reduced_rank,offline_full_pinn_seconds,mean_full_pinn_training_seconds,gpt_query_seconds,fe_116_query_seconds,speedup_vs_full_pinn_training,speedup_vs_fe_116,smoke_fe_proxy_used,fe_proxy_generation_seconds,query_repeats,fe_repeats
0,3,12,921.470042,307.156681,0.000115,0.019005,2.660167e+06,164.597222,False,NaN,1000,40


{
  "version": "V0.6 FINAL",
  "decision": "GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "prior_gate_e_decision": "GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL",
  "parameter_domain": {
    "alpha": [
      0.0,
      0.65
    ],
    "log10_support_ratio": [
      -2.0,
      0.0
    ]
  },
  "selected_full_pinn_states": [
    [
      0.0,
      0.0
    ],
    [
      0.0,
      -2.0
    ],
    [
      0.65,
      -2.0
    ]
  ],
  "basis_state_count": 3,
  "reduced_rank": 12,
  "effective_max_basis_states": 5,
  "state_source_classes": [
    "FULL_PINN_TRAINED"
  ],
  "smoke_fe_proxy_used": false,
  "current_run_gate_f_publication_evidence": true,
  "checkpoint_quality_audit_file": "/home/a.riyahi/GPT_PINN/outputs/v0_6_final/runs/V0_6_FINAL_R3_FULL_508150_20260906T011342Z/tables/gate_f_checkpoint_quality_audit_full.csv",
  "checkpoint_provenance": [
    {
      "alpha": 0.0,
     

## 22. Gate G observation model and preregistered online checks

The causal state is $\theta_t=(\alpha_t,\eta_t)$, where $\eta_t=\log_{10}(r_{K,t})$. The GPT-PINN observation operator returns the healthy-normalized modal-frequency vector for equivalent modes 1, 3 and 4. The experimental covariance uses the audited marginal Z24 bootstrap standard deviations; a conservative surrogate floor is added in quadrature from the independent Gate F GPT/FE errors. Cross-mode covariance is unavailable and is therefore not fabricated.

Scenario 01 is the normalization reference and is anchored at $(0,0)$. For later states, the independent estimate uses only the current modal vector. The online estimate is a filtered MAP update using only the previous filtered state and the current measurement. Its random-walk standard deviation is one half of each parameter-domain width. Because scenarios 01/03/04/05/06 are a known progressive-settlement experiment, the admissible transition is $\alpha_t\geq\alpha_{t-1}$ and $\eta_t\leq\eta_{t-1}$. This directional constraint must not be transferred automatically to an operational bridge that may undergo repairs.

Before execution, Gate G requires: state RMSE at most 1%, residual at most 3 standard deviations, a 99% chi-square innovation check, local Jacobian rank two with condition number at most 300, coherent monotone state, stable bootstrap quantiles, and online update time below 2 s. No threshold is changed after seeing the full execution.

In [29]:
gate_g_mode = CONFIG["gate_g_mode"]
if gate_g_mode not in {"full", "smoke"}:
    raise ValueError("GPT_PINN_GATE_G_MODE must be 'full' or 'smoke'")
if gate_g_mode == "full" and gate_f_decision != "GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL":
    raise RuntimeError("Full Gate G requires the executed Gate F GO decision")

gate_g_parameter_lower = np.array([
    CONFIG["gate_f_alpha_domain"][0], CONFIG["gate_f_log10_support_domain"][0]], dtype=float)
gate_g_parameter_upper = np.array([
    CONFIG["gate_f_alpha_domain"][1], CONFIG["gate_f_log10_support_domain"][1]], dtype=float)
gate_g_parameter_span = gate_g_parameter_upper-gate_g_parameter_lower
gate_g_process_sd = CONFIG["gate_g_process_domain_fraction"]*gate_g_parameter_span
gate_g_process_precision = np.diag(1.0/gate_g_process_sd**2)
gate_g_retained_zero_based = np.array([0, 2, 3], dtype=int)
gate_g_experimental_modes = np.array([1, 3, 4], dtype=int)

gate_g_observation_frame = gate_d_observed_ratios[
    gate_d_observed_ratios["experimental_mode"].isin(gate_g_experimental_modes)
].sort_values(["delta_pier_mm", "experimental_mode"]).reset_index(drop=True)
gate_g_deltas = np.sort(gate_g_observation_frame["delta_pier_mm"].unique())
gate_g_scenarios = [str(gate_g_observation_frame[
    gate_g_observation_frame["delta_pier_mm"] == delta]["scenario"].iloc[0]).zfill(2)
                    for delta in gate_g_deltas]
gate_g_observed = np.vstack([
    gate_g_observation_frame[gate_g_observation_frame["delta_pier_mm"] == delta]
    .sort_values("experimental_mode")["frequency_ratio"].to_numpy(dtype=float)
    for delta in gate_g_deltas
])
gate_g_sigma_observation = np.vstack([
    gate_g_observation_frame[gate_g_observation_frame["delta_pier_mm"] == delta]
    .sort_values("experimental_mode")["sigma_frequency_ratio"].to_numpy(dtype=float)
    for delta in gate_g_deltas
])

# Paired-setup cross-mode covariance estimated at Gate A. Diagonal values
# are conservatively raised to the marginal uncertainty used by Gate D.
gate_g_cov_observation = np.stack([
    gate_a_ratio_covariance_by_scenario[scenario].copy()
    for scenario in gate_g_scenarios
])
for state_index in range(len(gate_g_cov_observation)):
    diagonal_addition = np.maximum(
        gate_g_sigma_observation[state_index]**2
        - np.diag(gate_g_cov_observation[state_index]),
        0.0,
    )
    gate_g_cov_observation[state_index] += np.diag(diagonal_addition)

def stable_covariance_cholesky(covariance: np.ndarray) -> np.ndarray:
    covariance = 0.5 * (np.asarray(covariance, dtype=float) + np.asarray(covariance, dtype=float).T)
    eigenvalues = np.linalg.eigvalsh(covariance)
    floor = max(1e-12, 1e-8 * max(1.0, float(eigenvalues.max())))
    if eigenvalues.min() < floor:
        covariance = covariance + np.eye(len(covariance)) * (floor - eigenvalues.min())
    return np.linalg.cholesky(covariance)

def whiten_modal_residual(residual: np.ndarray, covariance: np.ndarray) -> np.ndarray:
    return np.linalg.solve(stable_covariance_cholesky(covariance), np.asarray(residual, dtype=float))

gate_g_healthy_omegas = gate_f_gpt_query(gate_f_basis, 0.0, 0.0)[
    "omegas"][gate_g_retained_zero_based]

def gate_g_forward(parameters: np.ndarray) -> np.ndarray:
    alpha, log_support = np.asarray(parameters, dtype=float)
    if np.any(np.asarray(parameters) < gate_g_parameter_lower-1e-12) or np.any(
            np.asarray(parameters) > gate_g_parameter_upper+1e-12):
        raise ValueError("Gate G query outside the validated Gate F rectangle")
    omegas = gate_f_gpt_query(gate_f_basis, float(alpha), float(log_support))["omegas"]
    return omegas[gate_g_retained_zero_based]/gate_g_healthy_omegas

# Ratio errors contain a current-state and a healthy-denominator surrogate contribution.
gate_g_sigma_surrogate = np.array([
    np.sqrt(np.mean(np.square(
        gate_f_validation[gate_f_validation["mode_order"] == mode]["gpt_omega"].to_numpy()/
        gate_f_validation[gate_f_validation["mode_order"] == mode]["fe_omega"].to_numpy()-1.0
    )))*np.sqrt(2.0)
    for mode in gate_g_experimental_modes
])
gate_g_cov_total = (
    gate_g_cov_observation
    + np.diag(
        gate_g_sigma_surrogate**2 + gate_e_model_discrepancy_sd**2
    )[None, :, :]
)
gate_g_sigma_total = np.sqrt(np.diagonal(gate_g_cov_total, axis1=1, axis2=2))

gate_g_uncertainty_rows = []
for state_index, (scenario, delta) in enumerate(zip(gate_g_scenarios, gate_g_deltas)):
    for mode_index, experimental_mode in enumerate(gate_g_experimental_modes):
        gate_g_uncertainty_rows.append({
            "scenario": scenario, "delta_pier_mm": delta,
            "experimental_mode": int(experimental_mode),
            "observed_frequency_ratio": gate_g_observed[state_index, mode_index],
            "sigma_observation": gate_g_sigma_observation[state_index, mode_index],
            "sigma_gpt_ratio_floor": gate_g_sigma_surrogate[mode_index],
            "sigma_gate_e_model_discrepancy": gate_e_model_discrepancy_sd,
            "sigma_total_diagonal": gate_g_sigma_total[state_index, mode_index],
            "maximum_absolute_cross_mode_correlation": float(np.max(np.abs(
                gate_g_cov_observation[state_index]
                / np.sqrt(np.outer(
                    np.diag(gate_g_cov_observation[state_index]),
                    np.diag(gate_g_cov_observation[state_index]),
                ))
                - np.eye(len(retained_modes))
            ))),
            "cross_mode_covariance_status": "paired-setup bootstrap covariance with shrinkage; used in likelihood",
        })
gate_g_uncertainty = pd.DataFrame(gate_g_uncertainty_rows)
gate_g_uncertainty.to_csv(
    TABLE_ROOT/f"gate_g_observation_uncertainty_{gate_g_mode}.csv", index=False)
display(gate_g_uncertainty)
print("Gate G process SD [alpha, log10 support]:", gate_g_process_sd)
print("Gate G conservative GPT ratio-error floors:", gate_g_sigma_surrogate)

,scenario,delta_pier_mm,experimental_mode,observed_frequency_ratio,sigma_observation,sigma_gpt_ratio_floor,sigma_gate_e_model_discrepancy,sigma_total_diagonal,maximum_absolute_cross_mode_correlation,cross_mode_covariance_status
0,01,0.0,1,1.000000,0.003125,0.000322,0.004948,0.005861,0.000000,paired-setup bootstrap covariance with shrinka...
1,01,0.0,3,1.000000,0.001225,0.000866,0.004948,0.005170,0.000000,paired-setup bootstrap covariance with shrinka...
2,01,0.0,4,1.000000,0.004185,0.000833,0.004948,0.006534,0.000000,paired-setup bootstrap covariance with shrinka...
3,03,20.0,1,0.987500,0.003125,0.000322,0.004948,0.005861,0.228835,paired-setup bootstrap covariance with shrinka...
4,03,20.0,3,0.982843,0.001393,0.000866,0.004948,0.005213,0.228835,paired-setup bootstrap covariance with shrinka...
5,03,20.0,4,0.981439,0.003839,0.000833,0.004948,0.006318,0.228835,paired-setup bootstrap covariance with shrinka...
6,04,40.0,1,0.987500,0.003125,0.000322,0.004948,0.005861,0.116696,paired-setup bootstrap covariance with shrinka...
7,04,40.0,3,0.977941,0.001392,0.000866,0.004948,0.005212,0.116696,paired-setup bootstrap covariance with shrinka...
8,04,40.0,4,0.974478,0.004530,0.000833,0.004948,0.006760,0.116696,paired-setup bootstrap covariance with shrinka...
9,05,80.0,1,0.962500,0.003125,0.000322,0.004948,0.005861,0.077346,paired-setup bootstrap covariance with shrinka...


Gate G process SD [alpha, log10 support]: [0.325 1.   ]
Gate G conservative GPT ratio-error floors: [0.00032227 0.00086601 0.00083304]


## 23. Independent and temporally regularized two-parameter updates

The independent bounded fit is retained as an ablation. The selected online algorithm minimizes the current three-mode Mahalanobis mismatch plus a scaled random-walk penalty from the previous state. It never accesses a future scenario. Multiple deterministic starts guard against a local minimum, while monotone bounds encode only the known direction of this controlled progression. Local finite-difference Jacobians quantify whether the two coordinates remain distinguishable.

In [30]:
from scipy.optimize import least_squares, minimize

def gate_g_standardized_residual(parameters: np.ndarray, state_index: int,
                                 observations: np.ndarray = gate_g_observed) -> np.ndarray:
    return whiten_modal_residual(
        gate_g_forward(parameters)-observations[state_index],
        gate_g_cov_total[state_index],
    )

gate_g_multistarts = np.array([
    [alpha, log_support]
    for alpha in np.linspace(gate_g_parameter_lower[0], gate_g_parameter_upper[0], 4)
    for log_support in np.linspace(gate_g_parameter_lower[1], gate_g_parameter_upper[1], 4)
])

gate_g_independent_path = [np.array([0.0, 0.0])]
gate_g_independent_runtime = [0.0]
for state_index in range(1, len(gate_g_deltas)):
    update_start = time.perf_counter()
    fits = [least_squares(
        gate_g_standardized_residual, start,
        args=(state_index, gate_g_observed),
        bounds=(gate_g_parameter_lower, gate_g_parameter_upper),
        method="trf", max_nfev=250, xtol=1e-11, ftol=1e-11, gtol=1e-11)
        for start in gate_g_multistarts]
    finite_fits = [fit for fit in fits if np.all(np.isfinite(fit.x)) and np.isfinite(
        np.dot(fit.fun, fit.fun))]
    if not finite_fits:
        raise RuntimeError(f"No finite independent Gate G fit at state {state_index}")
    best = min(finite_fits, key=lambda fit: np.dot(fit.fun, fit.fun))
    gate_g_independent_path.append(best.x.copy())
    gate_g_independent_runtime.append(time.perf_counter()-update_start)
gate_g_independent_path = np.vstack(gate_g_independent_path)

def gate_g_filter(observations: np.ndarray, process_fraction: float,
                  initialization: np.ndarray, measure_runtime: bool = False,
                  include_midpoint_start: bool = True):
    process_sd = process_fraction*gate_g_parameter_span
    precision = np.diag(1.0/process_sd**2)
    path = [np.array([0.0, 0.0])]
    records = [{"state_index": 0, "runtime_seconds": 0.0,
                "objective": 0.0, "optimizer_success": True, "nfev": 0}]
    for state_index in range(1, len(gate_g_deltas)):
        previous = path[-1]
        monotone_bounds = [(previous[0], gate_g_parameter_upper[0]),
                           (gate_g_parameter_lower[1], previous[1])]
        lower = np.array([bound[0] for bound in monotone_bounds])
        upper = np.array([bound[1] for bound in monotone_bounds])
        def objective(parameters):
            measurement = gate_g_standardized_residual(
                parameters, state_index, observations)
            transition = parameters-previous
            return float(measurement@measurement+transition@precision@transition)
        candidate_starts = [
            np.clip(initialization[state_index], lower, upper), previous.copy()]
        if include_midpoint_start:
            candidate_starts.append(0.5*(lower+upper))
        start_time = time.perf_counter()
        candidates = [minimize(
            objective, start, method="L-BFGS-B", bounds=monotone_bounds,
            options={"ftol": 1e-11, "gtol": 1e-8, "maxiter": 160})
            for start in candidate_starts]
        finite_candidates = [candidate for candidate in candidates
                             if np.all(np.isfinite(candidate.x)) and np.isfinite(candidate.fun)]
        if not finite_candidates:
            raise RuntimeError(f"No finite sequential Gate G fit at state {state_index}")
        best = min(finite_candidates, key=lambda candidate: candidate.fun)
        elapsed = time.perf_counter()-start_time if measure_runtime else np.nan
        path.append(best.x.copy())
        records.append({"state_index": state_index, "runtime_seconds": elapsed,
                        "objective": float(best.fun),
                        "optimizer_success": bool(best.success), "nfev": int(best.nfev)})
    return np.vstack(path), pd.DataFrame(records)

gate_g_sequential_path, gate_g_update_audit = gate_g_filter(
    gate_g_observed, CONFIG["gate_g_process_domain_fraction"],
    gate_g_independent_path, measure_runtime=True)

def gate_g_performance(path: np.ndarray) -> np.ndarray:
    alpha_health = 1.0-path[:, 0]/gate_g_parameter_span[0]
    support_health = (path[:, 1]-gate_g_parameter_lower[1])/gate_g_parameter_span[1]
    return np.clip(0.5*(alpha_health+support_health), 0.0, 1.0)

gate_g_independent_predictions = np.vstack([
    gate_g_forward(parameters) for parameters in gate_g_independent_path])
gate_g_sequential_predictions = np.vstack([
    gate_g_forward(parameters) for parameters in gate_g_sequential_path])
gate_g_standardized_matrix = np.vstack([
    whiten_modal_residual(
        gate_g_sequential_predictions[state_index]-gate_g_observed[state_index],
        gate_g_cov_total[state_index],
    )
    for state_index in range(len(gate_g_deltas))
])
gate_g_state_rmse = np.sqrt(np.mean(
    (gate_g_sequential_predictions-gate_g_observed)**2, axis=1))
gate_g_nis = np.sum(gate_g_standardized_matrix**2, axis=1)
gate_g_nis_limit = float(stats.chi2.ppf(CONFIG["gate_g_nis_probability"], df=3))
gate_g_performance_path = gate_g_performance(gate_g_sequential_path)
gate_g_batch_path = np.vstack([
    np.array([0.0, 0.0]) if delta == 0.0 else
    np.asarray(gate_e_state(gate_e_parameters, float(delta)))
    for delta in gate_g_deltas
])
gate_g_batch_performance = gate_g_performance(gate_g_batch_path)

def gate_g_weighted_jacobian(parameters: np.ndarray, state_index: int) -> np.ndarray:
    steps = np.array([1e-4, 2e-4])
    columns = []
    for parameter_index, step in enumerate(steps):
        lower = parameters.copy(); upper = parameters.copy()
        lower[parameter_index] = max(
            gate_g_parameter_lower[parameter_index], lower[parameter_index]-step)
        upper[parameter_index] = min(
            gate_g_parameter_upper[parameter_index], upper[parameter_index]+step)
        denominator = upper[parameter_index]-lower[parameter_index]
        if denominator <= 0.0:
            raise RuntimeError("Degenerate finite-difference Gate G Jacobian step")
        columns.append((gate_g_forward(upper)-gate_g_forward(lower))/denominator)
    raw_jacobian = np.column_stack(columns)
    cholesky = stable_covariance_cholesky(gate_g_cov_total[state_index])
    return np.linalg.solve(cholesky, raw_jacobian)

gate_g_identifiability_rows = []
for state_index, parameters in enumerate(gate_g_sequential_path):
    jacobian = gate_g_weighted_jacobian(parameters, state_index)
    singular_values = np.linalg.svd(jacobian, compute_uv=False)
    gate_g_identifiability_rows.append({
        "scenario": gate_g_scenarios[state_index],
        "delta_pier_mm": gate_g_deltas[state_index],
        "weighted_jacobian_rank": int(np.linalg.matrix_rank(jacobian)),
        "weighted_jacobian_condition": float(singular_values[0]/singular_values[-1]),
        "singular_value_1": singular_values[0], "singular_value_2": singular_values[1],
    })
gate_g_identifiability = pd.DataFrame(gate_g_identifiability_rows)

def gate_g_scaled_total_variation(path: np.ndarray) -> float:
    return float(np.sum(np.linalg.norm(
        np.diff(path, axis=0)/gate_g_parameter_span, axis=1)))

gate_g_temporal_ablation_rows = [{
    "method": "independent", "process_domain_fraction": np.nan,
    "scaled_total_variation": gate_g_scaled_total_variation(gate_g_independent_path),
    "alpha_monotonic_violations": int(np.sum(np.diff(gate_g_independent_path[:, 0]) < -1e-9)),
    "support_monotonic_violations": int(np.sum(np.diff(gate_g_independent_path[:, 1]) > 1e-9)),
    "maximum_state_rmse": float(np.max(np.sqrt(np.mean(
        (gate_g_independent_predictions-gate_g_observed)**2, axis=1)))),
    "final_performance_index": float(gate_g_performance(gate_g_independent_path)[-1]),
}]
for process_fraction in (0.25, 0.5, 1.0):
    path, _ = gate_g_filter(
        gate_g_observed, process_fraction, gate_g_independent_path, measure_runtime=False)
    predictions = np.vstack([gate_g_forward(parameters) for parameters in path])
    gate_g_temporal_ablation_rows.append({
        "method": "causal_monotone_MAP", "process_domain_fraction": process_fraction,
        "scaled_total_variation": gate_g_scaled_total_variation(path),
        "alpha_monotonic_violations": int(np.sum(np.diff(path[:, 0]) < -1e-9)),
        "support_monotonic_violations": int(np.sum(np.diff(path[:, 1]) > 1e-9)),
        "maximum_state_rmse": float(np.max(np.sqrt(np.mean(
            (predictions-gate_g_observed)**2, axis=1)))),
        "final_performance_index": float(gate_g_performance(path)[-1]),
    })
gate_g_temporal_ablation = pd.DataFrame(gate_g_temporal_ablation_rows)

gate_g_state_rows = []
gate_g_modal_rows = []
for state_index, (scenario, delta) in enumerate(zip(gate_g_scenarios, gate_g_deltas)):
    gate_g_state_rows.append({
        "scenario": scenario, "delta_pier_mm": delta,
        "independent_alpha": gate_g_independent_path[state_index, 0],
        "independent_log10_support_ratio": gate_g_independent_path[state_index, 1],
        "sequential_alpha": gate_g_sequential_path[state_index, 0],
        "sequential_log10_support_ratio": gate_g_sequential_path[state_index, 1],
        "sequential_support_ratio": 10.0**gate_g_sequential_path[state_index, 1],
        "batch_gate_e_alpha_reference": gate_g_batch_path[state_index, 0],
        "batch_gate_e_log10_support_reference": gate_g_batch_path[state_index, 1],
        "performance_index": gate_g_performance_path[state_index],
        "batch_gate_e_performance_reference": gate_g_batch_performance[state_index],
        "state_modal_ratio_rmse": gate_g_state_rmse[state_index],
        "normalized_innovation_squared": gate_g_nis[state_index],
        "nis_99pct_limit_df3": gate_g_nis_limit,
        "online_update_seconds": gate_g_update_audit.iloc[state_index]["runtime_seconds"],
    })
    for mode_index, experimental_mode in enumerate(gate_g_experimental_modes):
        gate_g_modal_rows.append({
            "scenario": scenario, "delta_pier_mm": delta,
            "experimental_mode": int(experimental_mode),
            "observed_frequency_ratio": gate_g_observed[state_index, mode_index],
            "sequential_gpt_frequency_ratio": gate_g_sequential_predictions[state_index, mode_index],
            "raw_residual": gate_g_sequential_predictions[state_index, mode_index]-
                            gate_g_observed[state_index, mode_index],
            "sigma_total_diagonal": gate_g_sigma_total[state_index, mode_index],
            "standardized_residual": gate_g_standardized_matrix[state_index, mode_index],
        })
gate_g_state_estimates = pd.DataFrame(gate_g_state_rows)
gate_g_modal_fit = pd.DataFrame(gate_g_modal_rows)
gate_g_state_estimates.to_csv(
    TABLE_ROOT/f"gate_g_online_state_estimates_{gate_g_mode}.csv", index=False)
gate_g_modal_fit.to_csv(TABLE_ROOT/f"gate_g_modal_fit_{gate_g_mode}.csv", index=False)
gate_g_identifiability.to_csv(
    TABLE_ROOT/f"gate_g_local_identifiability_{gate_g_mode}.csv", index=False)
gate_g_temporal_ablation.to_csv(
    TABLE_ROOT/f"gate_g_temporal_regularization_ablation_{gate_g_mode}.csv", index=False)
display(gate_g_state_estimates)
display(gate_g_identifiability)
display(gate_g_temporal_ablation)

,scenario,delta_pier_mm,independent_alpha,independent_log10_support_ratio,sequential_alpha,sequential_log10_support_ratio,sequential_support_ratio,batch_gate_e_alpha_reference,batch_gate_e_log10_support_reference,performance_index,batch_gate_e_performance_reference,state_modal_ratio_rmse,normalized_innovation_squared,nis_99pct_limit_df3,online_update_seconds
0,01,0.0,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,11.344867,0.000000
1,03,20.0,0.176701,-1.098706,0.214411,-0.144590,0.716820,0.117967,-0.856364,0.798921,0.695165,0.003242,1.052297,11.344867,0.112305
2,04,40.0,0.166209,-1.450402,0.214411,-1.217276,0.060635,0.235935,-1.211082,0.530749,0.515741,0.003884,1.428485,11.344867,0.124142
3,05,80.0,0.382077,-1.917966,0.378305,-1.920373,0.012012,0.471869,-1.712729,0.228903,0.208841,0.000700,0.032458,11.344867,0.057378
4,06,95.0,0.610594,-1.686365,0.532863,-1.920373,0.012012,0.560345,-1.866403,0.110012,0.102365,0.006960,3.803170,11.344867,0.079595


,scenario,delta_pier_mm,weighted_jacobian_rank,weighted_jacobian_condition,singular_value_1,singular_value_2
0,01,0.0,2,142.829809,20.011704,0.140109
1,03,20.0,2,125.513311,23.229117,0.185073
2,04,40.0,2,11.274275,23.590117,2.092384
3,05,80.0,2,3.470867,32.106335,9.250235
4,06,95.0,2,4.350702,37.410528,8.598734


,method,process_domain_fraction,scaled_total_variation,alpha_monotonic_violations,support_monotonic_violations,maximum_state_rmse,final_performance_index
0,independent,NaN,1.565806,1,1,0.006706,0.108721
1,causal_monotone_MAP,0.25,1.547413,0,0,0.006500,0.108736
2,causal_monotone_MAP,0.50,1.544442,0,0,0.006960,0.110012
3,causal_monotone_MAP,1.00,1.407554,0,0,0.006919,0.109139


## 24. Sequential bootstrap uncertainty and effective-state trajectory

Measurement and GPT-surrogate uncertainty are propagated by a deterministic-seed parametric bootstrap. Each replicate perturbs only the measurements available up to that update and reruns the same causal filter; scenario 01 remains the normalization anchor. The reported 95% intervals are percentile intervals. Their split-half stability is checked on the executed replicas.

For communication only, an equal-weight bounded indicator combines the two normalized effective coordinates,

$$H_t=\frac{1}{2}\left(1-\frac{\alpha_t}{0.65}\right)+\frac{1}{2}\left(\frac{\eta_t+2}{2}\right).$$

$H$ is an effective model-updating state, not measured stiffness, load-carrying capacity, damage probability, or remaining useful life.

In [31]:
require_compute_node("Gate G two-parameter bootstrap")
gate_g_bootstrap_replicates = (
    CONFIG["gate_g_full_bootstrap_replicates"] if gate_g_mode == "full"
    else CONFIG["gate_g_smoke_bootstrap_replicates"])
gate_g_rng = np.random.default_rng(CONFIG["seed"]+7000)
gate_g_bootstrap_paths = np.empty((gate_g_bootstrap_replicates, len(gate_g_deltas), 2))
gate_g_bootstrap_start = time.perf_counter()
for replicate in range(gate_g_bootstrap_replicates):
    sampled_observations = gate_g_observed.copy()
    for state_index in range(1, len(gate_g_deltas)):
        sampled_observations[state_index] += gate_g_rng.multivariate_normal(
            mean=np.zeros(len(gate_g_experimental_modes)),
            cov=gate_g_cov_total[state_index],
        )
    sampled_path, _ = gate_g_filter(
        sampled_observations, CONFIG["gate_g_process_domain_fraction"],
        gate_g_sequential_path, measure_runtime=False, include_midpoint_start=False)
    if not np.all(np.isfinite(sampled_path)):
        raise FloatingPointError(f"Non-finite Gate G bootstrap path at replicate {replicate}")
    gate_g_bootstrap_paths[replicate] = sampled_path
gate_g_bootstrap_seconds = time.perf_counter()-gate_g_bootstrap_start
gate_g_bootstrap_performance = np.stack([
    gate_g_performance(path) for path in gate_g_bootstrap_paths])
gate_g_parameter_quantiles = np.percentile(
    gate_g_bootstrap_paths, [2.5, 50.0, 97.5], axis=0)
gate_g_performance_quantiles = np.percentile(
    gate_g_bootstrap_performance, [2.5, 50.0, 97.5], axis=0)

half = gate_g_bootstrap_replicates//2
first_quantiles = np.percentile(
    gate_g_bootstrap_paths[:half], [2.5, 50.0, 97.5], axis=0)
second_quantiles = np.percentile(
    gate_g_bootstrap_paths[half:], [2.5, 50.0, 97.5], axis=0)
gate_g_quantile_split_scaled_difference = float(np.max(
    np.abs(first_quantiles-second_quantiles)/gate_g_parameter_span[None, None, :]))
first_performance_quantiles = np.percentile(
    gate_g_bootstrap_performance[:half], [2.5, 50.0, 97.5], axis=0)
second_performance_quantiles = np.percentile(
    gate_g_bootstrap_performance[half:], [2.5, 50.0, 97.5], axis=0)
gate_g_performance_split_difference = np.abs(
    first_performance_quantiles-second_performance_quantiles)

gate_g_stability_rows = []
for quantile_index, quantile in enumerate((2.5, 50.0, 97.5)):
    for state_index, (scenario, delta) in enumerate(zip(gate_g_scenarios, gate_g_deltas)):
        for parameter_index, parameter in enumerate(("alpha", "log10_support_ratio")):
            gate_g_stability_rows.append({
                "quantile_percent": quantile, "scenario": scenario,
                "delta_pier_mm": delta, "parameter": parameter,
                "first_half_quantile": first_quantiles[
                    quantile_index, state_index, parameter_index],
                "second_half_quantile": second_quantiles[
                    quantile_index, state_index, parameter_index],
                "absolute_difference": abs(first_quantiles[
                    quantile_index, state_index, parameter_index]-second_quantiles[
                    quantile_index, state_index, parameter_index]),
                "difference_scaled_by_domain": abs(first_quantiles[
                    quantile_index, state_index, parameter_index]-second_quantiles[
                    quantile_index, state_index, parameter_index])/
                    gate_g_parameter_span[parameter_index],
            })
gate_g_bootstrap_stability = pd.DataFrame(gate_g_stability_rows)
gate_g_bootstrap_stability.to_csv(
    TABLE_ROOT/f"gate_g_bootstrap_stability_{gate_g_mode}.csv", index=False)

gate_g_bootstrap_samples = pd.DataFrame({
    "replicate": np.repeat(np.arange(gate_g_bootstrap_replicates), len(gate_g_deltas)),
    "scenario": np.tile(np.asarray(gate_g_scenarios), gate_g_bootstrap_replicates),
    "delta_pier_mm": np.tile(gate_g_deltas, gate_g_bootstrap_replicates),
    "alpha": gate_g_bootstrap_paths[:, :, 0].ravel(),
    "log10_support_ratio": gate_g_bootstrap_paths[:, :, 1].ravel(),
    "performance_index": gate_g_bootstrap_performance.ravel(),
})
gate_g_bootstrap_samples.to_csv(
    TABLE_ROOT/f"gate_g_bootstrap_samples_{gate_g_mode}.csv", index=False)
gate_g_early_index = int(np.flatnonzero(gate_g_deltas == 20.0)[0])
gate_g_early_branch_threshold = -1.0
gate_g_early_low_support_probability = float(np.mean(
    gate_g_bootstrap_paths[:, gate_g_early_index, 1] <= gate_g_early_branch_threshold))
gate_g_early_low_support_probability_first = float(np.mean(
    gate_g_bootstrap_paths[:half, gate_g_early_index, 1] <= gate_g_early_branch_threshold))
gate_g_early_low_support_probability_second = float(np.mean(
    gate_g_bootstrap_paths[half:, gate_g_early_index, 1] <= gate_g_early_branch_threshold))

gate_g_interval_rows = []
for state_index, (scenario, delta) in enumerate(zip(gate_g_scenarios, gate_g_deltas)):
    gate_g_interval_rows.append({
        "scenario": scenario, "delta_pier_mm": delta,
        "alpha_point": gate_g_sequential_path[state_index, 0],
        "alpha_bootstrap_median": gate_g_parameter_quantiles[1, state_index, 0],
        "alpha_ci95_lower": gate_g_parameter_quantiles[0, state_index, 0],
        "alpha_ci95_upper": gate_g_parameter_quantiles[2, state_index, 0],
        "log10_support_point": gate_g_sequential_path[state_index, 1],
        "log10_support_bootstrap_median": gate_g_parameter_quantiles[1, state_index, 1],
        "log10_support_ci95_lower": gate_g_parameter_quantiles[0, state_index, 1],
        "log10_support_ci95_upper": gate_g_parameter_quantiles[2, state_index, 1],
        "performance_point": gate_g_performance_path[state_index],
        "performance_bootstrap_median": gate_g_performance_quantiles[1, state_index],
        "performance_ci95_lower": gate_g_performance_quantiles[0, state_index],
        "performance_ci95_upper": gate_g_performance_quantiles[2, state_index],
    })
gate_g_intervals = pd.DataFrame(gate_g_interval_rows)
gate_g_intervals.to_csv(
    TABLE_ROOT/f"gate_g_bootstrap_intervals_{gate_g_mode}.csv", index=False)
display(gate_g_intervals)

gate_g_independent_variation = gate_g_scaled_total_variation(gate_g_independent_path)
gate_g_sequential_variation = gate_g_scaled_total_variation(gate_g_sequential_path)
gate_g_max_update_seconds = float(gate_g_update_audit["runtime_seconds"].fillna(0.0).max())
gate_g_max_batch_performance_difference = float(np.max(np.abs(
    gate_g_performance_path-gate_g_batch_performance)))
gate_g_checks = {
    "full_execution": gate_g_mode == "full",
    "prior_gate_f_go": gate_f_decision == "GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL",
    "finite_state_and_uncertainty": bool(
        np.all(np.isfinite(gate_g_sequential_path)) and
        np.all(np.isfinite(gate_g_parameter_quantiles))),
    "maximum_state_modal_ratio_rmse_le_1pct": bool(
        gate_g_state_rmse.max() <= CONFIG["gate_g_max_state_modal_ratio_rmse"]),
    "maximum_standardized_residual_le_3sigma": bool(
        np.abs(gate_g_standardized_matrix).max() <= CONFIG["gate_g_max_standardized_residual"]),
    "all_nis_below_99pct_chi_square_limit": bool((gate_g_nis <= gate_g_nis_limit).all()),
    "all_local_weighted_jacobians_rank_two": bool(
        (gate_g_identifiability["weighted_jacobian_rank"] == 2).all()),
    "maximum_local_weighted_jacobian_condition_le_300": bool(
        gate_g_identifiability["weighted_jacobian_condition"].max() <=
        CONFIG["gate_g_max_weighted_jacobian_condition"]),
    "alpha_non_decreasing": bool(np.all(np.diff(gate_g_sequential_path[:, 0]) >= -1e-9)),
    "support_log_ratio_non_increasing": bool(
        np.all(np.diff(gate_g_sequential_path[:, 1]) <= 1e-9)),
    "performance_index_non_increasing": bool(
        np.all(np.diff(gate_g_performance_path) <= 1e-9)),
    "final_performance_index_le_0p25": bool(
        gate_g_performance_path[-1] <= CONFIG["gate_g_max_final_performance_index"]),
    "temporal_total_variation_not_above_independent": bool(
        gate_g_sequential_variation <= gate_g_independent_variation+1e-9),
    "bootstrap_valid_fraction_ge_0p99": bool(
        np.mean(np.all(np.isfinite(gate_g_bootstrap_paths), axis=(1, 2))) >=
        CONFIG["gate_g_min_bootstrap_valid_fraction"]),
    "bootstrap_split_quantile_difference_le_0p15_domain": bool(
        gate_g_quantile_split_scaled_difference <=
        CONFIG["gate_g_max_quantile_split_scaled_difference"]),
    "performance_agreement_with_batch_gate_e_le_0p05": bool(
        gate_g_max_batch_performance_difference <=
        CONFIG["gate_g_max_performance_deviation_from_batch"]),
    "maximum_online_update_time_le_2s": bool(
        gate_g_max_update_seconds <= CONFIG["gate_g_max_online_update_seconds"]),
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
}
if gate_g_mode == "smoke":
    gate_g_decision = "SMOKE_COMPLETED_REQUIRES_FULL_RUN"
else:
    gate_g_decision = ("GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL"
                       if all(gate_g_checks.values())
                       else "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE")

gate_g_diagnostic = {
    "version": "V0.6 FINAL", "decision": gate_g_decision,
    "execution_mode": gate_g_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "prior_gate_f_decision": gate_f_decision,
    "state_definition": {
        "alpha": "effective local deck-rigidity reduction coordinate",
        "log10_support_ratio": "effective Koppigen restraint log10 ratio",
        "performance_index": "equal-weight normalized effective model state; not capacity",
    },
    "chronology": {"scenarios": gate_g_scenarios,
                   "settlement_mm_for_audit_only": gate_g_deltas.tolist(),
                   "future_measurements_used_by_filter": False,
                   "healthy_state_anchored_by_ratio_normalization": True},
    "uncertainty": {
        "method": "1000-replicate causal parametric bootstrap" if gate_g_mode == "full"
                  else "smoke causal parametric bootstrap",
        "replicates": gate_g_bootstrap_replicates,
        "bootstrap_runtime_seconds": gate_g_bootstrap_seconds,
        "surrogate_ratio_sigma_by_mode": gate_g_sigma_surrogate.tolist(),
        "cross_mode_covariance": "paired-setup bootstrap covariance with shrinkage; used for sampling and Mahalanobis likelihood",
        "maximum_split_half_quantile_difference_scaled_by_domain":
            gate_g_quantile_split_scaled_difference,
        "maximum_split_half_performance_quantile_difference": float(
            gate_g_performance_split_difference.max()),
        "early_20mm_low_support_branch_threshold": gate_g_early_branch_threshold,
        "early_20mm_low_support_branch_probability": gate_g_early_low_support_probability,
        "early_20mm_branch_probability_first_half":
            gate_g_early_low_support_probability_first,
        "early_20mm_branch_probability_second_half":
            gate_g_early_low_support_probability_second,
        "interpretation": "two competing early-state parameter branches; not Monte Carlo under-sampling",
    },
    "online_fit": {
        "maximum_state_modal_ratio_rmse": float(gate_g_state_rmse.max()),
        "maximum_absolute_standardized_residual": float(
            np.abs(gate_g_standardized_matrix).max()),
        "maximum_normalized_innovation_squared": float(gate_g_nis.max()),
        "nis_99pct_limit_df3": gate_g_nis_limit,
        "maximum_update_seconds": gate_g_max_update_seconds,
        "independent_scaled_total_variation": gate_g_independent_variation,
        "sequential_scaled_total_variation": gate_g_sequential_variation,
    },
    "identifiability": {
        "minimum_local_rank": int(gate_g_identifiability[
            "weighted_jacobian_rank"].min()),
        "maximum_local_condition_number": float(gate_g_identifiability[
            "weighted_jacobian_condition"].max()),
        "warning": "individual coordinates are weakly separated near the healthy state",
    },
    "performance_state": {
        "point_trajectory": gate_g_performance_path.tolist(),
        "ci95_lower": gate_g_performance_quantiles[0].tolist(),
        "ci95_upper": gate_g_performance_quantiles[2].tolist(),
        "final_point": float(gate_g_performance_path[-1]),
        "maximum_difference_from_batch_gate_e_reference":
            gate_g_max_batch_performance_difference,
    },
    "checks": gate_g_checks,
    "online_z24_update_performed": True,
    "future_data_leakage_performed": False,
    "conditional_prognosis_performed": False,
    "capacity_or_rul_claim_performed": False,
    "raw_z24_signature_unchanged": gate_g_checks["raw_z24_signature_unchanged"],
    "limitations": [
        "only five controlled Z24 benchmark states are available and the analysis is retrospective",
        "scenario 01 is anchored because every observation is normalized by that same reference",
        "the temporal direction is valid for this progressive experiment, not automatically for repaired operation",
        "cross-mode dependence is estimated from paired physical-setup bootstraps with shrinkage and propagated through the likelihood",
        "the two effective coordinates compensate near healthy states; the 20 mm bootstrap is bimodal and its median is unstable",
        "the performance index is not stiffness, load capacity, damage probability, or remaining useful life",
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT/"gate_g_diagnostic.json").write_text(
    json_dumps_safe(gate_g_diagnostic, indent=2, sort_keys=True)+"\n", encoding="utf-8")

mode_colors = ["#264653", "#2a9d8f", "#e76f51"]
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
for mode_index, (mode, color) in enumerate(zip(gate_g_experimental_modes, mode_colors)):
    axes[0].errorbar(gate_g_deltas, gate_g_observed[:, mode_index],
                     yerr=1.96*gate_g_sigma_total[:, mode_index], fmt="o", color=color,
                     capsize=3, label=f"observed mode {mode}")
    axes[0].plot(gate_g_deltas, gate_g_sequential_predictions[:, mode_index],
                 "-", color=color, label=f"online GPT mode {mode}")
    axes[1].plot(gate_g_deltas, gate_g_standardized_matrix[:, mode_index],
                 "o-", color=color, label=f"mode {mode}")
axes[0].set_xlabel("Controlled settlement [mm]")
axes[0].set_ylabel("Healthy-normalized frequency")
axes[0].set_title("Causal GPT-PINN modal reconstruction")
axes[0].legend(ncol=2, fontsize=7)
axes[1].axhspan(-3.0, 3.0, color="#2a9d8f", alpha=0.10, label="±3σ gate")
axes[1].axhline(0.0, color="#555555", linewidth=1)
axes[1].set_xlabel("Controlled settlement [mm]")
axes[1].set_ylabel("Standardized residual")
axes[1].set_title("Innovation audit")
axes[1].legend(fontsize=8)
fig.suptitle("Figure 18 — Gate G online Z24 modal updates", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_18_gate_g_online_modal_fit_{gate_g_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
axes[0].fill_between(gate_g_deltas, gate_g_parameter_quantiles[0, :, 0],
                     gate_g_parameter_quantiles[2, :, 0], color="#e76f51", alpha=0.20,
                     label="95% bootstrap CI")
axes[0].plot(gate_g_deltas, gate_g_sequential_path[:, 0], "o-", color="#e76f51",
             label="causal MAP")
axes[0].scatter(gate_g_deltas, gate_g_independent_path[:, 0], marker="x", color="#264653",
                label="independent")
axes[0].plot(gate_g_deltas, gate_g_batch_path[:, 0], ":", color="#555555",
             label="Gate E batch reference")
axes[0].set_xlabel("Controlled settlement [mm]"); axes[0].set_ylabel(r"Effective $\alpha$")
axes[0].set_title("Local-rigidity coordinate")
axes[1].fill_between(gate_g_deltas, gate_g_parameter_quantiles[0, :, 1],
                     gate_g_parameter_quantiles[2, :, 1], color="#2a9d8f", alpha=0.20,
                     label="95% bootstrap CI")
axes[1].plot(gate_g_deltas, gate_g_sequential_path[:, 1], "o-", color="#2a9d8f",
             label="causal MAP")
axes[1].scatter(gate_g_deltas, gate_g_independent_path[:, 1], marker="x", color="#264653",
                label="independent")
axes[1].plot(gate_g_deltas, gate_g_batch_path[:, 1], ":", color="#555555",
             label="Gate E batch reference")
axes[1].set_xlabel("Controlled settlement [mm]")
axes[1].set_ylabel(r"$\log_{10}(r_K)$")
axes[1].set_title("Koppigen-restraint coordinate")
for axis in axes:
    axis.legend(fontsize=8)
fig.suptitle("Figure 19 — Gate G two-parameter trajectory and uncertainty", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_19_gate_g_parameter_uncertainty_{gate_g_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
axes[0].fill_between(gate_g_deltas, gate_g_performance_quantiles[0],
                     gate_g_performance_quantiles[2], color="#457b9d", alpha=0.22,
                     label="95% bootstrap CI")
axes[0].plot(gate_g_deltas, gate_g_performance_path, "o-", color="#264653",
             label="causal effective state")
axes[0].plot(gate_g_deltas, gate_g_batch_performance, ":", color="#e76f51",
             label="Gate E batch reference")
axes[0].set_ylim(-0.03, 1.05); axes[0].set_xlabel("Controlled settlement [mm]")
axes[0].set_ylabel("Effective performance index H")
axes[0].set_title("Bounded model-state trajectory — not capacity")
axes[0].legend(fontsize=8)
selected_ablation = gate_g_temporal_ablation.copy()
labels = ["independent" if row.method == "independent" else
          f"MAP q={row.process_domain_fraction:g}"
          for row in selected_ablation.itertuples()]
axes[1].bar(labels, selected_ablation["scaled_total_variation"],
            color=["#6d597a", "#8ecae6", "#457b9d", "#264653"])
axes[1].axhline(gate_g_sequential_variation, linestyle="--", color="#e76f51",
                label="selected q=0.5")
axes[1].set_ylabel("Scaled path total variation")
axes[1].set_title("Temporal-regularization ablation")
axes[1].tick_params(axis="x", rotation=18); axes[1].legend(fontsize=8)
fig.suptitle(f"Figure 20 — Gate G state coherence: {gate_g_decision.replace('_', ' ')}",
             weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_20_gate_g_state_coherence_{gate_g_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

early_alpha = gate_g_bootstrap_paths[:, gate_g_early_index, 0]
early_support = gate_g_bootstrap_paths[:, gate_g_early_index, 1]
early_performance = gate_g_bootstrap_performance[:, gate_g_early_index]
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
scatter = axes[0].scatter(early_alpha, early_support, c=early_performance,
                          s=18, alpha=0.45, cmap="viridis")
axes[0].axhline(gate_g_early_branch_threshold, color="#e76f51", linestyle="--",
                label=r"branch audit $\eta=-1$")
axes[0].scatter(gate_g_sequential_path[gate_g_early_index, 0],
                gate_g_sequential_path[gate_g_early_index, 1], marker="*", s=180,
                color="#e76f51", edgecolor="white", label="nominal causal MAP")
fig.colorbar(scatter, ax=axes[0], label="Effective state H")
axes[0].set_xlabel(r"Effective $\alpha$ at 20 mm")
axes[0].set_ylabel(r"$\log_{10}(r_K)$ at 20 mm")
axes[0].set_title("Two compensating bootstrap branches")
axes[0].legend(fontsize=8)
axes[1].hist(early_performance, bins=36, color="#457b9d", alpha=0.8,
             edgecolor="white")
axes[1].axvline(gate_g_performance_path[gate_g_early_index], color="#e76f51",
                linestyle="--", label="nominal point estimate")
axes[1].set_xlabel("Effective state H at 20 mm")
axes[1].set_ylabel("Bootstrap count")
axes[1].set_title(
    f"Low-support branch probability = {gate_g_early_low_support_probability:.1%}")
axes[1].legend(fontsize=8)
fig.suptitle("Figure 21 — Gate G uncertainty topology causing the NO-GO", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_21_gate_g_uncertainty_topology_{gate_g_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_g_diagnostic, indent=2))

,scenario,delta_pier_mm,alpha_point,alpha_bootstrap_median,alpha_ci95_lower,alpha_ci95_upper,log10_support_point,log10_support_bootstrap_median,log10_support_ci95_lower,log10_support_ci95_upper,performance_point,performance_bootstrap_median,performance_ci95_lower,performance_ci95_upper
0,01,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
1,03,20.0,0.214411,0.195626,0.043672,0.285981,-0.144590,-0.142878,-1.563744,0.000000,0.798921,0.790029,0.500965,0.878409
2,04,40.0,0.214411,0.221878,0.093000,0.312017,-1.217276,-1.245158,-1.648295,0.000000,0.530749,0.536121,0.445078,0.801893
3,05,80.0,0.378305,0.382321,0.270104,0.508695,-1.920373,-1.917209,-2.000000,-1.642045,0.228903,0.230609,0.172489,0.297640
4,06,95.0,0.532863,0.532758,0.446430,0.620258,-1.920373,-1.924908,-2.000000,-1.702164,0.110012,0.112840,0.069854,0.161742


{
  "version": "V0.6 FINAL",
  "decision": "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "prior_gate_f_decision": "GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL",
  "state_definition": {
    "alpha": "effective local deck-rigidity reduction coordinate",
    "log10_support_ratio": "effective Koppigen restraint log10 ratio",
    "performance_index": "equal-weight normalized effective model state; not capacity"
  },
  "chronology": {
    "scenarios": [
      "01",
      "03",
      "04",
      "05",
      "06"
    ],
    "settlement_mm_for_audit_only": [
      0.0,
      20.0,
      40.0,
      80.0,
      95.0
    ],
    "future_measurements_used_by_filter": false,
    "healthy_state_anchored_by_ratio_normalization": true
  },
  "uncertainty": {
    "method": "1000-replicate causal parametric bootstrap",
    "replicates": 1000,
    "bootstrap_runtime_seconds": 269.0388274679426,
    "s

## 25. Gate G.1 — identifiable one-coordinate physical trajectory

Gate G established a scientific NO-GO for free two-coordinate inference: the early Z24 likelihood has two compensating branches. Gate G.1 does not select a branch post hoc. It restricts the update to the calibrated physical trajectory

$$\alpha(\xi)=a\xi,\qquad \eta(\xi)=\log_{10}(r_K)=-b\sqrt{\xi},\qquad 0\leq\xi\leq1,$$

using the fixed Gate E exponents. The amplitudes $(a,b)$ are fitted with the GPT-PINN observation operator. For the full retrospective synchronization they use all four nonhealthy states; for every LOSO result they are refitted without the held-out severity. The target $\delta/95$ is used only to calibrate or audit the trajectory, never by the held-out inverse optimizer.

The likelihood includes Z24 modal uncertainty, the independent Gate F surrogate error, and a training-only manifold-discrepancy term. Because each fold has only three calibration states, its residual RMS is inflated by $\sqrt{(n+1)/(n-1)}$ to obtain a conservative small-sample predictive scale.

In [32]:
from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize_scalar

gate_g1_mode = CONFIG["gate_g1_mode"]
if gate_g1_mode not in {"full", "smoke"}:
    raise ValueError("GPT_PINN_GATE_G1_MODE must be 'full' or 'smoke'")
if gate_g1_mode == "full" and gate_g_decision != "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE":
    raise RuntimeError("Gate G.1 is the authorized correction of the executed two-parameter NO-GO")

gate_g1_truth = gate_g_deltas/95.0
gate_g1_amplitude_lower = np.array([0.05, 0.05])
gate_g1_amplitude_upper = np.array([0.65, 2.0])
gate_g1_base_cov = gate_g_cov_total.copy()
gate_g1_base_sigma = np.sqrt(np.diagonal(gate_g1_base_cov, axis1=1, axis2=2))

def gate_g1_coordinates(xi: float, amplitudes: np.ndarray) -> np.ndarray:
    xi = float(xi)
    amplitude_alpha, amplitude_support = np.asarray(amplitudes, dtype=float)
    return np.array([amplitude_alpha*xi, -amplitude_support*np.sqrt(max(0.0, xi))])

def gate_g1_forward(xi: float, amplitudes: np.ndarray) -> np.ndarray:
    return gate_g_forward(gate_g1_coordinates(xi, amplitudes))

def gate_g1_fit_amplitudes(indices, observations=gate_g_observed,
                           starts=None, enforce_three_sigma=True):
    indices = list(indices)
    if starts is None:
        starts = (np.array([0.40, 1.00]), np.array([0.60, 1.70]),
                  np.array([0.64, 1.95]))
    def residual(amplitudes):
        return np.concatenate([
            whiten_modal_residual(
                gate_g1_forward(gate_g1_truth[index], amplitudes)-observations[index],
                gate_g1_base_cov[index],
            )
            for index in indices
        ])
    fits = [least_squares(
        residual, np.asarray(start),
        bounds=(gate_g1_amplitude_lower, gate_g1_amplitude_upper),
        method="trf", max_nfev=300, xtol=1e-11, ftol=1e-11, gtol=1e-11)
        for start in starts]
    best = min(fits, key=lambda fit: np.dot(fit.fun, fit.fun))
    if not enforce_three_sigma:
        return best, residual(best.x)
    sigma_limit = CONFIG["gate_g1_max_calibration_standardized_residual"]-1e-6
    constraints = (
        {"type": "ineq", "fun": lambda amplitudes: sigma_limit-residual(amplitudes)},
        {"type": "ineq", "fun": lambda amplitudes: sigma_limit+residual(amplitudes)},
    )
    constrained = minimize(
        lambda amplitudes: float(residual(amplitudes)@residual(amplitudes)),
        best.x, method="SLSQP",
        bounds=list(zip(gate_g1_amplitude_lower, gate_g1_amplitude_upper)),
        constraints=constraints, options={"ftol": 1e-12, "maxiter": 500})
    constrained_residual = residual(constrained.x)
    if np.max(np.abs(constrained_residual)) > CONFIG[
            "gate_g1_max_calibration_standardized_residual"]:
        raise RuntimeError("No feasible three-sigma Gate G.1 manifold calibration")
    return constrained, constrained_residual

def gate_g1_predictive_discrepancy(indices, observations, amplitudes):
    indices = list(indices)
    residuals = np.vstack([
        gate_g1_forward(gate_g1_truth[index], amplitudes)-observations[index]
        for index in indices])
    count = len(indices)
    if count <= 1:
        raise ValueError("At least two training states are required for discrepancy")
    inflation = np.sqrt((count+1)/(count-1))
    return np.sqrt(np.mean(residuals**2, axis=0))*inflation

def gate_g1_invert(observation: np.ndarray, state_uncertainty: np.ndarray,
                   amplitudes: np.ndarray, lower: float = 0.0):
    state_uncertainty = np.asarray(state_uncertainty, dtype=float)
    covariance = (
        np.diag(state_uncertainty**2)
        if state_uncertainty.ndim == 1
        else state_uncertainty
    )
    def objective(xi):
        residual = whiten_modal_residual(
            gate_g1_forward(float(xi), amplitudes)-observation,
            covariance,
        )
        return float(residual@residual)
    result = minimize_scalar(
        objective, bounds=(float(lower), 1.0), method="bounded",
        options={"xatol": 1e-10, "maxiter": 160})
    candidates = [(float(result.fun), float(result.x)),
                  (objective(float(lower)), float(lower)), (objective(1.0), 1.0)]
    return min(candidates)[1]

gate_g1_training_indices = list(range(1, len(gate_g_deltas)))
gate_g1_full_fit, gate_g1_full_fit_residual = gate_g1_fit_amplitudes(
    gate_g1_training_indices)
gate_g1_amplitudes = gate_g1_full_fit.x.copy()
gate_g1_discrepancy = gate_g1_predictive_discrepancy(
    gate_g1_training_indices, gate_g_observed, gate_g1_amplitudes)
gate_g1_total_cov = (
    gate_g1_base_cov
    + np.diag(gate_g1_discrepancy**2)[None, :, :]
)
gate_g1_total_sigma = np.sqrt(
    np.diagonal(gate_g1_total_cov, axis1=1, axis2=2)
)

gate_g1_calibration_rows = []
for state_index, (scenario, delta) in enumerate(zip(gate_g_scenarios, gate_g_deltas)):
    coordinates = gate_g1_coordinates(gate_g1_truth[state_index], gate_g1_amplitudes)
    prediction = gate_g1_forward(gate_g1_truth[state_index], gate_g1_amplitudes)
    for mode_index, mode in enumerate(gate_g_experimental_modes):
        gate_g1_calibration_rows.append({
            "scenario": scenario, "delta_pier_mm": delta,
            "trajectory_xi": gate_g1_truth[state_index],
            "alpha": coordinates[0], "log10_support_ratio": coordinates[1],
            "experimental_mode": int(mode),
            "observed_frequency_ratio": gate_g_observed[state_index, mode_index],
            "manifold_gpt_frequency_ratio": prediction[mode_index],
            "raw_residual": prediction[mode_index]-gate_g_observed[state_index, mode_index],
            "base_sigma": gate_g1_base_sigma[state_index, mode_index],
            "predictive_model_discrepancy": gate_g1_discrepancy[mode_index],
            "total_predictive_sigma": gate_g1_total_sigma[state_index, mode_index],
        })
gate_g1_calibration = pd.DataFrame(gate_g1_calibration_rows)
gate_g1_calibration.to_csv(
    TABLE_ROOT/f"gate_g1_manifold_calibration_{gate_g1_mode}.csv", index=False)

gate_g1_sensitivity_rows = []
for state_index, xi in enumerate(gate_g1_truth):
    lower = max(0.0, xi-1e-4); upper = min(1.0, xi+1e-4)
    derivative = (gate_g1_forward(upper, gate_g1_amplitudes)-
                  gate_g1_forward(lower, gate_g1_amplitudes))/(upper-lower)
    weighted_norm = float(np.linalg.norm(
        whiten_modal_residual(derivative, gate_g1_total_cov[state_index])
    ))
    gate_g1_sensitivity_rows.append({
        "scenario": gate_g_scenarios[state_index], "delta_pier_mm": gate_g_deltas[state_index],
        "xi": xi, "weighted_sensitivity_norm": weighted_norm,
        **{f"mode_{mode}_derivative": derivative[index]
           for index, mode in enumerate(gate_g_experimental_modes)},
    })
gate_g1_sensitivity = pd.DataFrame(gate_g1_sensitivity_rows)
gate_g1_sensitivity.to_csv(
    TABLE_ROOT/f"gate_g1_weighted_sensitivity_{gate_g1_mode}.csv", index=False)
display(gate_g1_calibration)
display(gate_g1_sensitivity)
print("Gate G.1 amplitudes [a,b]:", gate_g1_amplitudes)
print("Gate G.1 predictive discrepancy by mode:", gate_g1_discrepancy)

,scenario,delta_pier_mm,trajectory_xi,alpha,log10_support_ratio,experimental_mode,observed_frequency_ratio,manifold_gpt_frequency_ratio,raw_residual,base_sigma,predictive_model_discrepancy,total_predictive_sigma
0,01,0.0,0.000000,0.000000,-0.000000,1,1.000000,1.000000,0.000000,0.005861,0.007923,0.009855
1,01,0.0,0.000000,0.000000,-0.000000,3,1.000000,1.000000,0.000000,0.005170,0.007381,0.009012
2,01,0.0,0.000000,0.000000,-0.000000,4,1.000000,1.000000,0.000000,0.006534,0.006882,0.009490
3,03,20.0,0.210526,0.117966,-0.857220,1,0.987500,0.990719,0.003219,0.005861,0.007923,0.009855
4,03,20.0,0.210526,0.117966,-0.857220,3,0.982843,0.991534,0.008690,0.005213,0.007381,0.009037
5,03,20.0,0.210526,0.117966,-0.857220,4,0.981439,0.986813,0.005374,0.006318,0.006882,0.009342
6,04,40.0,0.421053,0.235931,-1.212292,1,0.987500,0.980400,-0.007100,0.005861,0.007923,0.009855
7,04,40.0,0.421053,0.235931,-1.212292,3,0.977941,0.980471,0.002530,0.005212,0.007381,0.009036
8,04,40.0,0.421053,0.235931,-1.212292,4,0.974478,0.972540,-0.001938,0.006760,0.006882,0.009647
9,05,80.0,0.842105,0.471862,-1.714439,1,0.962500,0.955185,-0.007315,0.005861,0.007923,0.009855


,scenario,delta_pier_mm,xi,weighted_sensitivity_norm,mode_1_derivative,mode_3_derivative,mode_4_derivative
0,01,0.0,0.000000,38.840721,-0.077064,-0.134517,-0.332100
1,03,20.0,0.210526,9.563969,-0.046220,-0.045068,-0.064165
2,04,40.0,0.421053,11.338020,-0.052052,-0.061230,-0.071380
3,05,80.0,0.842105,17.078248,-0.069245,-0.122901,-0.078140
4,06,95.0,1.000000,19.984833,-0.078758,-0.158226,-0.075205


Gate G.1 amplitudes [a,b]: [0.56033657 1.86826705]
Gate G.1 predictive discrepancy by mode: [0.00792306 0.00738148 0.00688226]


## 26. Strict leave-one-severity-out inference and fair baselines

For each nonhealthy state, the manifold amplitudes and discrepancy are estimated only from the other nonhealthy states. The held-out modal vector is then inverted for $\xi$ without its settlement label. The same training cases and held-out vector feed two data-only baselines: monotone PCHIP of the mean frequency-drop indicator and an RBF Gaussian-process regressor whose length scale is selected by nested leave-one-out within the training fold.

Each physics fold uses an additional parametric bootstrap that perturbs its training observations, recalibrates $(a,b)$, perturbs the held-out measurement using the training-only predictive discrepancy, and reinverts $\xi$. This propagates calibration, observation, model-form and GPT-surrogate uncertainty without target leakage.

In [33]:
require_compute_node("Gate G.1 LOSO uncertainty propagation")
def gate_g1_gp_prediction(train_observations, train_targets, test_observation):
    mean = train_observations.mean(axis=0)
    scale = np.maximum(train_observations.std(axis=0), 1e-6)
    normalized = (train_observations-mean)/scale
    test = (test_observation-mean)/scale
    pairwise = np.sum((normalized[:, None]-normalized[None, :])**2, axis=2)
    test_distance = np.sum((normalized-test[None, :])**2, axis=1)
    candidates = []
    for length_scale in (0.25, 0.5, 1.0, 2.0, 4.0):
        kernel = np.exp(-0.5*pairwise/length_scale**2)+1e-6*np.eye(len(normalized))
        loo_errors = []
        for heldout in range(len(normalized)):
            keep = np.arange(len(normalized)) != heldout
            train_kernel = kernel[np.ix_(keep, keep)]
            test_kernel = np.exp(-0.5*np.sum(
                (normalized[keep]-normalized[heldout])**2, axis=1)/length_scale**2)
            prediction = test_kernel@np.linalg.solve(train_kernel, train_targets[keep])
            loo_errors.append((prediction-train_targets[heldout])**2)
        weights = np.exp(-0.5*test_distance/length_scale**2)
        prediction = float(weights@np.linalg.solve(kernel, train_targets))
        candidates.append((float(np.mean(loo_errors)), prediction, length_scale))
    return min(candidates)[1:]

gate_g1_fold_records = []
gate_g1_fold_contexts = []
for heldout_index in range(1, len(gate_g_deltas)):
    training_indices = [index for index in range(1, len(gate_g_deltas))
                        if index != heldout_index]
    fold_fit, fold_fit_residual = gate_g1_fit_amplitudes(training_indices)
    fold_discrepancy = gate_g1_predictive_discrepancy(
        training_indices, gate_g_observed, fold_fit.x)
    fold_cov = (
        gate_g1_base_cov[heldout_index]
        + np.diag(fold_discrepancy**2)
    )
    fold_sigma = np.sqrt(np.diag(fold_cov))
    physics_xi = gate_g1_invert(
        gate_g_observed[heldout_index], fold_cov, fold_fit.x)

    baseline_indices = np.array([0]+training_indices)
    indicator_training = np.mean(1.0-gate_g_observed[baseline_indices], axis=1)
    order = np.argsort(indicator_training)
    indicator_sorted = indicator_training[order]
    truth_sorted = gate_g1_truth[baseline_indices][order]
    unique_indicator, inverse_group = np.unique(indicator_sorted, return_inverse=True)
    unique_truth = np.array([
        np.mean(truth_sorted[inverse_group == group])
        for group in range(len(unique_indicator))
    ])
    if len(unique_indicator) < 2:
        raise RuntimeError("PCHIP baseline requires at least two distinct training indicators")
    pchip = PchipInterpolator(unique_indicator, unique_truth, extrapolate=True)
    pchip_xi = float(np.clip(pchip(np.mean(
        1.0-gate_g_observed[heldout_index])), 0.0, 1.0))
    gp_xi, gp_length_scale = gate_g1_gp_prediction(
        gate_g_observed[baseline_indices], gate_g1_truth[baseline_indices],
        gate_g_observed[heldout_index])
    gp_xi = float(np.clip(gp_xi, 0.0, 1.0))
    fold_type = "extrapolation" if gate_g_deltas[heldout_index] == 95.0 else "interpolation"
    gate_g1_fold_records.append({
        "scenario": gate_g_scenarios[heldout_index],
        "heldout_delta_pier_mm": gate_g_deltas[heldout_index],
        "fold_type": fold_type,
        "training_deltas_mm": json_dumps_safe(gate_g_deltas[training_indices].tolist()),
        "true_xi_for_audit_only": gate_g1_truth[heldout_index],
        "physics_gpt_xi": physics_xi,
        "pchip_xi": pchip_xi, "gp_xi": gp_xi,
        "physics_absolute_xi_error": abs(physics_xi-gate_g1_truth[heldout_index]),
        "pchip_absolute_xi_error": abs(pchip_xi-gate_g1_truth[heldout_index]),
        "gp_absolute_xi_error": abs(gp_xi-gate_g1_truth[heldout_index]),
        "fold_amplitude_a": fold_fit.x[0], "fold_amplitude_b": fold_fit.x[1],
        "fold_max_training_standardized_residual": float(np.max(np.abs(fold_fit_residual))),
        "gp_nested_loo_length_scale": gp_length_scale,
        **{f"mode_{mode}_predictive_discrepancy": fold_discrepancy[index]
           for index, mode in enumerate(gate_g_experimental_modes)},
    })
    gate_g1_fold_contexts.append({
        "heldout_index": heldout_index, "training_indices": training_indices,
        "amplitudes": fold_fit.x.copy(), "fold_sigma": fold_sigma.copy(),
        "fold_cov": fold_cov.copy(),
    })

gate_g1_loso_bootstrap_replicates = (
    CONFIG["gate_g1_full_loso_bootstrap_replicates"] if gate_g1_mode == "full"
    else CONFIG["gate_g1_smoke_loso_bootstrap_replicates"])
gate_g1_loso_rng = np.random.default_rng(CONFIG["seed"]+7110)
gate_g1_loso_sample_rows = []
for context, record in zip(gate_g1_fold_contexts, gate_g1_fold_records):
    heldout_index = context["heldout_index"]
    training_indices = context["training_indices"]
    fold_samples = []
    for replicate in range(gate_g1_loso_bootstrap_replicates):
        sampled = gate_g_observed.copy()
        for training_index in training_indices:
            sampled[training_index] += gate_g1_loso_rng.multivariate_normal(
                mean=np.zeros(len(gate_g_experimental_modes)),
                cov=gate_g1_base_cov[training_index],
            )
        sampled[heldout_index] += gate_g1_loso_rng.multivariate_normal(
            mean=np.zeros(len(gate_g_experimental_modes)),
            cov=context["fold_cov"],
        )
        sampled_fit, _ = gate_g1_fit_amplitudes(
            training_indices, sampled, starts=(context["amplitudes"],),
            enforce_three_sigma=False)
        sampled_discrepancy = gate_g1_predictive_discrepancy(
            training_indices, sampled, sampled_fit.x)
        sampled_cov = (
            gate_g1_base_cov[heldout_index]
            + np.diag(sampled_discrepancy**2)
        )
        sampled_xi = gate_g1_invert(
            sampled[heldout_index], sampled_cov, sampled_fit.x)
        fold_samples.append(sampled_xi)
        gate_g1_loso_sample_rows.append({
            "replicate": replicate,
            "heldout_scenario": gate_g_scenarios[heldout_index],
            "heldout_delta_pier_mm": gate_g_deltas[heldout_index],
            "sampled_xi": sampled_xi,
            "sampled_amplitude_a": sampled_fit.x[0],
            "sampled_amplitude_b": sampled_fit.x[1],
        })
    fold_samples = np.asarray(fold_samples)
    quantiles = np.percentile(fold_samples, [2.5, 50.0, 97.5])
    half = len(fold_samples)//2
    split_difference = float(np.max(np.abs(
        np.percentile(fold_samples[:half], [2.5, 50.0, 97.5])-
        np.percentile(fold_samples[half:], [2.5, 50.0, 97.5]))))
    record.update({
        "physics_bootstrap_median_xi": quantiles[1],
        "physics_ci95_lower_xi": quantiles[0], "physics_ci95_upper_xi": quantiles[2],
        "physics_ci95_covers_true_xi": bool(
            quantiles[0] <= record["true_xi_for_audit_only"] <= quantiles[2]),
        "bootstrap_split_max_quantile_difference": split_difference,
    })

gate_g1_loso = pd.DataFrame(gate_g1_fold_records)
gate_g1_loso_samples = pd.DataFrame(gate_g1_loso_sample_rows)
gate_g1_loso.to_csv(TABLE_ROOT/f"gate_g1_loso_{gate_g1_mode}.csv", index=False)
gate_g1_loso_samples.to_csv(
    TABLE_ROOT/f"gate_g1_loso_bootstrap_samples_{gate_g1_mode}.csv", index=False)
display(gate_g1_loso)

fig, ax = plt.subplots(figsize=(7.2, 5.3))
true_values = gate_g1_loso["true_xi_for_audit_only"].to_numpy()
physics_values = gate_g1_loso["physics_gpt_xi"].to_numpy()
physics_median = gate_g1_loso["physics_bootstrap_median_xi"].to_numpy()
physics_lower = np.maximum(
    0.0, physics_median-gate_g1_loso["physics_ci95_lower_xi"].to_numpy())
physics_upper = np.maximum(
    0.0, gate_g1_loso["physics_ci95_upper_xi"].to_numpy()-physics_median)
ax.errorbar(true_values, physics_median, yerr=np.vstack([physics_lower, physics_upper]),
            fmt="o", capsize=4, color="#e76f51", label="1-D physics GPT-PINN posterior")
ax.scatter(true_values, physics_values, marker="x", color="#9d0208",
           label="1-D physics point estimate")
ax.scatter(true_values, gate_g1_loso["pchip_xi"], marker="s", color="#2a9d8f",
           label="PCHIP baseline")
ax.scatter(true_values, gate_g1_loso["gp_xi"], marker="^", color="#6d597a",
           label="RBF-GP baseline")
ax.plot([0, 1], [0, 1], "--", color="#555555", label="ideal")
ax.set_xlim(0.15, 1.04); ax.set_ylim(0.05, 1.05)
ax.set_xlabel(r"Held-out true $\xi=\delta/95$ (audit only)")
ax.set_ylabel(r"Estimated held-out $\hat\xi$")
ax.set_title("Figure 23 — Gate G.1 strict LOSO and identical-data baselines")
ax.legend(fontsize=8); fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_23_gate_g1_loso_{gate_g1_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

,scenario,heldout_delta_pier_mm,fold_type,training_deltas_mm,true_xi_for_audit_only,physics_gpt_xi,pchip_xi,gp_xi,physics_absolute_xi_error,pchip_absolute_xi_error,...,fold_max_training_standardized_residual,gp_nested_loo_length_scale,mode_1_predictive_discrepancy,mode_3_predictive_discrepancy,mode_4_predictive_discrepancy,physics_bootstrap_median_xi,physics_ci95_lower_xi,physics_ci95_upper_xi,physics_ci95_covers_true_xi,bootstrap_split_max_quantile_difference
0,03,20.0,interpolation,"[40.0, 80.0, 95.0]",0.210526,0.328419,0.353252,0.357516,0.117892,0.142725,...,1.190897,4.0,0.009607,0.006165,0.007319,0.329949,0.104615,0.530741,True,0.027594
1,04,40.0,interpolation,"[20.0, 80.0, 95.0]",0.421053,0.383648,0.270647,0.278158,0.037405,0.150406,...,1.640390,4.0,0.008161,0.008989,0.008811,0.382576,0.115302,0.598032,True,0.037382
2,05,80.0,interpolation,"[20.0, 40.0, 95.0]",0.842105,0.855354,0.938041,1.000000,0.013248,0.095935,...,1.603798,4.0,0.007353,0.007253,0.009966,0.858929,0.683832,1.000000,True,0.030719
3,06,95.0,extrapolation,"[20.0, 40.0, 80.0]",1.000000,0.970563,0.732111,0.442478,0.029437,0.267889,...,1.716668,4.0,0.006682,0.007851,0.005063,0.971938,0.859444,1.000000,True,0.005068


## 27. Causal one-coordinate update, uncertainty topology and Gate G.1 decision

The causal MAP filter uses only the preceding $\xi$ and the current modal vector, with a random-walk scale of 0.5 and the known nondecreasing direction of the controlled-settlement sequence. A 1,000-replicate bootstrap propagates the full predictive scale. Unlike Gate G, the one-dimensional bootstrap is required to be unimodal, stable between halves, cover the known benchmark progression, and remain no wider than half the parameter domain.

The effective communication variable is $H^{(1D)}=1-\xi$. It is a benchmark progression coordinate, not stiffness retention, capacity, probability of failure or remaining useful life.

In [34]:
require_compute_node("Gate G.1 online bootstrap")
def gate_g1_filter(observations, measure_runtime=False):
    path = [0.0]
    records = [{"state_index": 0, "runtime_seconds": 0.0, "objective": 0.0}]
    for state_index in range(1, len(gate_g_deltas)):
        previous = path[-1]
        def objective(xi):
            residual = whiten_modal_residual(
                gate_g1_forward(float(xi), gate_g1_amplitudes)-observations[state_index],
                gate_g1_total_cov[state_index],
            )
            return float(residual@residual+
                         ((float(xi)-previous)/CONFIG["gate_g1_process_sd"])**2)
        start_time = time.perf_counter()
        result = minimize_scalar(
            objective, bounds=(previous, 1.0), method="bounded",
            options={"xatol": 1e-10, "maxiter": 160})
        candidates = [(float(result.fun), float(result.x)),
                      (objective(previous), previous), (objective(1.0), 1.0)]
        best_objective, best_xi = min(candidates)
        path.append(best_xi)
        records.append({
            "state_index": state_index,
            "runtime_seconds": time.perf_counter()-start_time if measure_runtime else np.nan,
            "objective": best_objective,
        })
    return np.asarray(path), pd.DataFrame(records)

gate_g1_independent = np.array([0.0]+[
    gate_g1_invert(gate_g_observed[index], gate_g1_total_cov[index], gate_g1_amplitudes)
    for index in range(1, len(gate_g_deltas))])
gate_g1_filtered, gate_g1_runtime = gate_g1_filter(gate_g_observed, measure_runtime=True)
gate_g1_online_predictions = np.vstack([
    gate_g1_forward(xi, gate_g1_amplitudes) for xi in gate_g1_filtered])
gate_g1_online_residual = gate_g1_online_predictions-gate_g_observed
gate_g1_online_standardized = np.vstack([
    whiten_modal_residual(
        gate_g1_online_residual[state_index],
        gate_g1_total_cov[state_index],
    )
    for state_index in range(len(gate_g_deltas))
])
gate_g1_online_rmse = np.sqrt(np.mean(gate_g1_online_residual**2, axis=1))
gate_g1_online_nis = np.sum(gate_g1_online_standardized**2, axis=1)
gate_g1_effective_health = 1.0-gate_g1_filtered

gate_g1_bootstrap_replicates = (
    CONFIG["gate_g1_full_bootstrap_replicates"] if gate_g1_mode == "full"
    else CONFIG["gate_g1_smoke_bootstrap_replicates"])
gate_g1_rng = np.random.default_rng(CONFIG["seed"]+7100)
gate_g1_bootstrap_paths = np.empty((gate_g1_bootstrap_replicates, len(gate_g_deltas)))
gate_g1_bootstrap_start = time.perf_counter()
for replicate in range(gate_g1_bootstrap_replicates):
    sampled = gate_g_observed.copy()
    for state_index in range(1, len(gate_g_deltas)):
        sampled[state_index] += gate_g1_rng.multivariate_normal(
            mean=np.zeros(len(gate_g_experimental_modes)),
            cov=gate_g1_total_cov[state_index],
        )
    sampled_path, _ = gate_g1_filter(sampled, measure_runtime=False)
    if not np.all(np.isfinite(sampled_path)):
        raise FloatingPointError(f"Non-finite Gate G.1 bootstrap at replicate {replicate}")
    gate_g1_bootstrap_paths[replicate] = sampled_path
gate_g1_bootstrap_seconds = time.perf_counter()-gate_g1_bootstrap_start
gate_g1_quantiles = np.percentile(
    gate_g1_bootstrap_paths, [2.5, 50.0, 97.5], axis=0)
half = gate_g1_bootstrap_replicates//2
gate_g1_first_quantiles = np.percentile(
    gate_g1_bootstrap_paths[:half], [2.5, 50.0, 97.5], axis=0)
gate_g1_second_quantiles = np.percentile(
    gate_g1_bootstrap_paths[half:], [2.5, 50.0, 97.5], axis=0)
gate_g1_split_difference = float(np.max(np.abs(
    gate_g1_first_quantiles-gate_g1_second_quantiles)))

def gate_g1_kde_mode_count(samples):
    samples = np.asarray(samples, dtype=float)
    if len(samples) < 3 or np.std(samples) < 1e-8:
        return 1
    grid = np.linspace(0.0, 1.0, 401)
    try:
        density = stats.gaussian_kde(samples, bw_method="scott")(grid)
    except (np.linalg.LinAlgError, ValueError):
        counts, edges = np.histogram(samples, bins=min(40, max(8, int(np.sqrt(len(samples))))),
                                     range=(0.0, 1.0), density=True)
        centers = 0.5*(edges[:-1]+edges[1:])
        density = np.interp(grid, centers, counts, left=counts[0], right=counts[-1])
        density = signal.savgol_filter(density, 9 if len(density) >= 9 else 5, 2)
    prominence = 0.10*max(float(np.max(density)), np.finfo(float).eps)
    peaks = list(signal.find_peaks(density, prominence=prominence)[0])
    if density[0] > density[1] and density[0] >= 0.15*density.max():
        peaks.append(0)
    if density[-1] > density[-2] and density[-1] >= 0.15*density.max():
        peaks.append(len(grid)-1)
    return max(1, len(set(peaks)))

gate_g1_mode_counts = np.array([
    gate_g1_kde_mode_count(gate_g1_bootstrap_paths[:, index])
    for index in range(len(gate_g_deltas))])
gate_g1_interval_rows = []
for state_index, (scenario, delta) in enumerate(zip(gate_g_scenarios, gate_g_deltas)):
    gate_g1_interval_rows.append({
        "scenario": scenario, "delta_pier_mm": delta,
        "true_xi_for_audit_only": gate_g1_truth[state_index],
        "independent_xi": gate_g1_independent[state_index],
        "causal_xi": gate_g1_filtered[state_index],
        "causal_absolute_xi_error": abs(gate_g1_filtered[state_index]-gate_g1_truth[state_index]),
        "bootstrap_median_xi": gate_g1_quantiles[1, state_index],
        "ci95_lower_xi": gate_g1_quantiles[0, state_index],
        "ci95_upper_xi": gate_g1_quantiles[2, state_index],
        "ci95_width": gate_g1_quantiles[2, state_index]-gate_g1_quantiles[0, state_index],
        "ci95_covers_true_xi": bool(gate_g1_quantiles[0, state_index] <=
                                    gate_g1_truth[state_index] <=
                                    gate_g1_quantiles[2, state_index]),
        "kde_mode_count": gate_g1_mode_counts[state_index],
        "effective_health_1d": gate_g1_effective_health[state_index],
        "state_modal_ratio_rmse": gate_g1_online_rmse[state_index],
        "normalized_innovation_squared": gate_g1_online_nis[state_index],
        "online_update_seconds": gate_g1_runtime.iloc[state_index]["runtime_seconds"],
    })
gate_g1_intervals = pd.DataFrame(gate_g1_interval_rows)
gate_g1_bootstrap_samples = pd.DataFrame({
    "replicate": np.repeat(np.arange(gate_g1_bootstrap_replicates), len(gate_g_deltas)),
    "scenario": np.tile(np.asarray(gate_g_scenarios), gate_g1_bootstrap_replicates),
    "delta_pier_mm": np.tile(gate_g_deltas, gate_g1_bootstrap_replicates),
    "sampled_xi": gate_g1_bootstrap_paths.ravel(),
    "sampled_effective_health_1d": (1.0-gate_g1_bootstrap_paths).ravel(),
})
gate_g1_intervals.to_csv(
    TABLE_ROOT/f"gate_g1_online_state_estimates_{gate_g1_mode}.csv", index=False)
gate_g1_bootstrap_samples.to_csv(
    TABLE_ROOT/f"gate_g1_bootstrap_samples_{gate_g1_mode}.csv", index=False)
display(gate_g1_intervals)

gate_g1_interpolation_folds = gate_g1_loso[gate_g1_loso["fold_type"] == "interpolation"]
gate_g1_extrapolation_folds = gate_g1_loso[gate_g1_loso["fold_type"] == "extrapolation"]
gate_g1_loso_spearman = float(stats.spearmanr(
    gate_g1_loso["true_xi_for_audit_only"], gate_g1_loso["physics_gpt_xi"]).statistic)
gate_g1_physics_wins = np.mean(
    (gate_g1_loso["physics_absolute_xi_error"] < gate_g1_loso["pchip_absolute_xi_error"]) &
    (gate_g1_loso["physics_absolute_xi_error"] < gate_g1_loso["gp_absolute_xi_error"]))
gate_g1_checks = {
    "full_execution": gate_g1_mode == "full",
    "prior_gate_g_no_go_preserved": gate_g_decision ==
        "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE",
    "manifold_calibration_max_residual_le_3sigma": bool(
        np.max(np.abs(gate_g1_full_fit_residual)) <=
        CONFIG["gate_g1_max_calibration_standardized_residual"]),
    "minimum_weighted_sensitivity_ge_2": bool(
        gate_g1_sensitivity["weighted_sensitivity_norm"].min() >=
        CONFIG["gate_g1_min_weighted_sensitivity"]),
    "online_modal_ratio_rmse_le_1pct": bool(
        gate_g1_online_rmse.max() <= CONFIG["gate_g1_max_online_modal_ratio_rmse"]),
    "online_standardized_residual_le_3sigma": bool(
        np.abs(gate_g1_online_standardized).max() <=
        CONFIG["gate_g1_max_online_standardized_residual"]),
    "online_nis_below_99pct_limit": bool((gate_g1_online_nis <= gate_g_nis_limit).all()),
    "online_absolute_xi_error_le_0p15": bool(
        np.abs(gate_g1_filtered-gate_g1_truth).max() <=
        CONFIG["gate_g1_max_online_absolute_xi_error"]),
    "causal_xi_non_decreasing": bool(np.all(np.diff(gate_g1_filtered) >= -1e-9)),
    "online_bootstrap_all_ci_cover_truth": bool(gate_g1_intervals[
        "ci95_covers_true_xi"].all()),
    "online_bootstrap_split_difference_le_0p05": bool(
        gate_g1_split_difference <= CONFIG["gate_g1_max_bootstrap_split_difference"]),
    "online_bootstrap_unimodal": bool(
        gate_g1_mode_counts.max() <= CONFIG["gate_g1_max_kde_mode_count"]),
    "online_bootstrap_ci_width_le_0p5": bool(
        gate_g1_intervals["ci95_width"].max() <= CONFIG["gate_g1_max_ci_width"]),
    "loso_interpolation_absolute_xi_error_le_0p15": bool(
        gate_g1_interpolation_folds["physics_absolute_xi_error"].max() <=
        CONFIG["gate_g1_max_loso_interpolation_absolute_xi_error"]),
    "loso_extrapolation_absolute_xi_error_le_0p15": bool(
        gate_g1_extrapolation_folds["physics_absolute_xi_error"].max() <=
        CONFIG["gate_g1_max_loso_extrapolation_absolute_xi_error"]),
    "loso_ci_coverage_100pct": bool(
        gate_g1_loso["physics_ci95_covers_true_xi"].mean() >=
        CONFIG["gate_g1_min_loso_ci_coverage"]),
    "loso_bootstrap_split_difference_le_0p10": bool(
        gate_g1_loso["bootstrap_split_max_quantile_difference"].max() <=
        CONFIG["gate_g1_max_loso_bootstrap_split_difference"]),
    "loso_spearman_ge_0p9": bool(
        gate_g1_loso_spearman >= CONFIG["gate_g1_min_loso_spearman"]),
    "physics_beats_both_baselines_in_at_least_75pct_folds": bool(
        gate_g1_physics_wins >= CONFIG["gate_g1_min_physics_baseline_win_fraction"]),
    "maximum_online_update_time_le_1s": bool(
        gate_g1_runtime["runtime_seconds"].fillna(0.0).max() <=
        CONFIG["gate_g1_max_online_update_seconds"]),
    "no_heldout_target_used_by_inverse_optimizer": True,
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
}
if gate_g1_mode == "smoke":
    gate_g1_decision = "SMOKE_COMPLETED_REQUIRES_FULL_RUN"
else:
    gate_g1_decision = ("GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL"
                        if all(gate_g1_checks.values())
                        else "NO_GO_REVISE_ONE_DIMENSIONAL_STATE")

gate_g1_diagnostic = {
    "version": "V0.6 FINAL", "decision": gate_g1_decision,
    "execution_mode": gate_g1_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "prior_gate_g_decision": gate_g_decision,
    "trajectory": {
        "definition": "alpha=a*xi; log10(rK)=-b*sqrt(xi)",
        "fixed_exponents": [1.0, 0.5],
        "fitted_amplitudes_a_b": gate_g1_amplitudes.tolist(),
        "interpretation": "settlement-progression coordinate; not stiffness or capacity",
    },
    "online": {
        "point_xi": gate_g1_filtered.tolist(),
        "effective_health_1d": gate_g1_effective_health.tolist(),
        "maximum_absolute_xi_error_against_benchmark": float(
            np.abs(gate_g1_filtered-gate_g1_truth).max()),
        "maximum_modal_ratio_rmse": float(gate_g1_online_rmse.max()),
        "maximum_standardized_residual": float(
            np.abs(gate_g1_online_standardized).max()),
        "maximum_update_seconds": float(
            gate_g1_runtime["runtime_seconds"].fillna(0.0).max()),
    },
    "uncertainty": {
        "online_bootstrap_replicates": gate_g1_bootstrap_replicates,
        "loso_bootstrap_replicates_per_fold": gate_g1_loso_bootstrap_replicates,
        "online_bootstrap_seconds": gate_g1_bootstrap_seconds,
        "maximum_online_split_quantile_difference": gate_g1_split_difference,
        "maximum_kde_mode_count": int(gate_g1_mode_counts.max()),
        "maximum_ci_width": float(gate_g1_intervals["ci95_width"].max()),
        "loso_ci_coverage_fraction": float(
            gate_g1_loso["physics_ci95_covers_true_xi"].mean()),
        "model_discrepancy_by_mode": gate_g1_discrepancy.tolist(),
    },
    "strict_loso": {
        "maximum_interpolation_absolute_xi_error": float(
            gate_g1_interpolation_folds["physics_absolute_xi_error"].max()),
        "extrapolation_95mm_absolute_xi_error": float(
            gate_g1_extrapolation_folds["physics_absolute_xi_error"].iloc[0]),
        "spearman": gate_g1_loso_spearman,
        "physics_mean_absolute_xi_error": float(
            gate_g1_loso["physics_absolute_xi_error"].mean()),
        "pchip_mean_absolute_xi_error": float(
            gate_g1_loso["pchip_absolute_xi_error"].mean()),
        "gp_mean_absolute_xi_error": float(
            gate_g1_loso["gp_absolute_xi_error"].mean()),
        "physics_win_fraction": float(gate_g1_physics_wins),
        "target_used_by_inverse_optimizer": False,
    },
    "checks": gate_g1_checks,
    "two_parameter_identifiability_claim": False,
    "conditional_prognosis_performed": False,
    "capacity_or_rul_claim_performed": False,
    "raw_z24_signature_unchanged": gate_g1_checks["raw_z24_signature_unchanged"],
    "limitations": [
        "the one-dimensional manifold uses fixed exponents selected during earlier model development",
        "full-sequence online synchronization is retrospective; strict LOSO is the generalization evidence",
        "only four nonhealthy settlement severities are available",
        "predictive discrepancy is added to a paired-setup covariance model; only the model-discrepancy increment remains diagonal",
        "xi is a benchmark progression coordinate and H=1-xi is not structural capacity or remaining life",
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT/"gate_g1_diagnostic.json").write_text(
    json_dumps_safe(gate_g1_diagnostic, indent=2, sort_keys=True)+"\n", encoding="utf-8")

xi_grid = np.linspace(0.0, 1.0, 160)
coordinate_grid = np.vstack([
    gate_g1_coordinates(xi, gate_g1_amplitudes) for xi in xi_grid])
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
axes[0].plot(coordinate_grid[:, 0], coordinate_grid[:, 1], color="#264653")
state_coordinates = np.vstack([
    gate_g1_coordinates(xi, gate_g1_amplitudes) for xi in gate_g1_filtered])
scatter = axes[0].scatter(state_coordinates[:, 0], state_coordinates[:, 1],
                          c=gate_g_deltas, cmap="viridis", s=80, zorder=3)
fig.colorbar(scatter, ax=axes[0], label="Controlled settlement [mm]")
axes[0].set_xlabel(r"Effective $\alpha$")
axes[0].set_ylabel(r"$\log_{10}(r_K)$")
axes[0].set_title("Identifiable one-dimensional manifold")
axes[1].plot(gate_g1_sensitivity["xi"], gate_g1_sensitivity[
    "weighted_sensitivity_norm"], "o-", color="#e76f51")
axes[1].axhline(CONFIG["gate_g1_min_weighted_sensitivity"], linestyle="--",
                color="#555555", label="minimum gate")
axes[1].set_xlabel(r"Trajectory coordinate $\xi$")
axes[1].set_ylabel("Weighted observation sensitivity")
axes[1].set_title("Local one-coordinate identifiability")
axes[1].legend(fontsize=8)
fig.suptitle("Figure 22 — Gate G.1 physical trajectory and sensitivity", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_22_gate_g1_manifold_{gate_g1_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))
axes[0].fill_between(gate_g_deltas, gate_g1_quantiles[0], gate_g1_quantiles[2],
                     color="#457b9d", alpha=0.22, label="95% predictive CI")
axes[0].plot(gate_g_deltas, gate_g1_filtered, "o-", color="#264653",
             label="causal 1-D update")
axes[0].plot(gate_g_deltas, gate_g1_truth, "--", color="#e76f51",
             label="known benchmark progression")
axes[0].set_xlabel("Controlled settlement [mm]"); axes[0].set_ylabel(r"$\xi$")
axes[0].set_title("Causal state with unimodal uncertainty")
axes[0].legend(fontsize=8)
for mode_index, (mode, color) in enumerate(zip(gate_g_experimental_modes, mode_colors)):
    axes[1].errorbar(gate_g_deltas, gate_g_observed[:, mode_index],
                     yerr=1.96*gate_g1_total_sigma[:, mode_index], fmt="o", capsize=3,
                     color=color, label=f"observed mode {mode}")
    axes[1].plot(gate_g_deltas, gate_g1_online_predictions[:, mode_index],
                 "-", color=color, label=f"1-D GPT mode {mode}")
axes[1].set_xlabel("Controlled settlement [mm]")
axes[1].set_ylabel("Healthy-normalized frequency")
axes[1].set_title("Predictive modal reconstruction")
axes[1].legend(ncol=2, fontsize=7)
fig.suptitle(f"Figure 24 — Gate G.1: {gate_g1_decision.replace('_', ' ')}", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_24_gate_g1_online_state_{gate_g1_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_g1_diagnostic, indent=2))

,scenario,delta_pier_mm,true_xi_for_audit_only,independent_xi,causal_xi,causal_absolute_xi_error,bootstrap_median_xi,ci95_lower_xi,ci95_upper_xi,ci95_width,ci95_covers_true_xi,kde_mode_count,effective_health_1d,state_modal_ratio_rmse,normalized_innovation_squared,online_update_seconds
0,01,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,1,1.000000,0.000000,0.000000,0.000000
1,03,20.0,0.210526,0.317832,0.306128,0.095602,0.309372,0.101799,0.490348,0.388549,True,1,0.693872,0.002533,0.232283,0.005184
2,04,40.0,0.421053,0.393225,0.390397,0.030655,0.414229,0.261719,0.565430,0.303711,True,1,0.609603,0.004064,0.543994,0.008909
3,05,80.0,0.842105,0.861989,0.855823,0.013718,0.854847,0.731488,0.960797,0.229309,True,1,0.144177,0.005650,0.996976,0.005099
4,06,95.0,1.000000,0.994309,0.992946,0.007054,0.992658,0.898824,1.000000,0.101176,True,1,0.007054,0.006326,1.002096,0.004441


{
  "version": "V0.6 FINAL",
  "decision": "GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "prior_gate_g_decision": "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE",
  "trajectory": {
    "definition": "alpha=a*xi; log10(rK)=-b*sqrt(xi)",
    "fixed_exponents": [
      1.0,
      0.5
    ],
    "fitted_amplitudes_a_b": [
      0.5603365657142761,
      1.8682670486346296
    ],
    "interpretation": "settlement-progression coordinate; not stiffness or capacity"
  },
  "online": {
    "point_xi": [
      0.0,
      0.30612835906555275,
      0.39039749702910714,
      0.8558229466807709,
      0.9929463037090195
    ],
    "effective_health_1d": [
      1.0,
      0.6938716409344472,
      0.6096025029708929,
      0.14417705331922914,
      0.007053696290980516
    ],
    "maximum_absolute_xi_error_against_benchmark": 0.09560204327607907,
    "maximum_modal_ratio_rmse": 0.0063258

## 28. Gate G.1 decision and next authorized stage

The preceding code cell generates the current Gate G.1 decision from the executed full/smoke evidence. A pass authorizes only the identifiable one-dimensional Z24 settlement-progression coordinate and the conditional benchmark-stage forecast in Gate H.

The earlier free two-coordinate result remains scientifically separate: `NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE`. Restricting the inference to a calibrated physical trajectory does not establish independent two-parameter identifiability and does not authorize structural-capacity, probability-of-failure, calendar-time RUL or maintenance-cost claims.


## 29. Gate H — leakage-free conditional response propagation

Gate H is deliberately narrower than a prognostic or RUL claim. At each rolling origin it uses only past Z24 AVT measurements, the locked one-coordinate physics trajectory, and a **declared next controlled-settlement increment**. It predicts the next benchmark coordinate and modal response.

The target is opened only after the forecast object is frozen. Point-estimate latency and the 1,000-replicate uncertainty audit are timed separately. The schedule-nominal comparator remains mandatory because \(\xi=\delta/95\) makes it intrinsically strong. A successful Gate H supports conditional response propagation under a known intervention/load increment; it does not support autonomous natural-aging prediction.


In [35]:
gate_h_mode = CONFIG["gate_h_mode"]
if gate_h_mode not in {"full", "smoke"}:
    raise ValueError("GPT_PINN_GATE_H_MODE must be 'full' or 'smoke'")
if gate_h_mode == "full" and gate_g1_decision != "GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL":
    raise RuntimeError("Full Gate H requires the executed Gate G.1 GO decision")

gate_h_origins = np.arange(1, len(gate_g_deltas)-1, dtype=int)
gate_h_targets = gate_h_origins+1
gate_h_beta_bounds = np.asarray(CONFIG["gate_h_beta_bounds"], dtype=float)

def gate_h_training_discrepancy(indices, observations, amplitudes):
    indices = list(indices)
    residuals = np.vstack([
        gate_g1_forward(gate_g1_truth[index], amplitudes)-observations[index]
        for index in indices])
    if len(indices) == 1:
        inflation = 2.0
    else:
        inflation = np.sqrt((len(indices)+1)/(len(indices)-1))
    rms = np.sqrt(np.mean(residuals**2, axis=0))*inflation
    return np.maximum(rms, CONFIG["gate_h_model_discrepancy_floor"])

def gate_h_filter_prefix(observations, amplitudes, discrepancy, origin):
    path = [0.0]
    for state_index in range(1, origin+1):
        previous = path[-1]
        covariance = (
            gate_g1_base_cov[state_index] + np.diag(discrepancy**2)
        )
        def objective(xi):
            residual = whiten_modal_residual(
                gate_g1_forward(float(xi), amplitudes)-observations[state_index],
                covariance,
            )
            return float(residual@residual+
                         ((float(xi)-previous)/CONFIG["gate_g1_process_sd"])**2)
        result = minimize_scalar(
            objective, bounds=(previous, 1.0), method="bounded",
            options={"xatol": 1e-10, "maxiter": 160})
        candidates = [(float(result.fun), float(result.x)),
                      (objective(previous), previous), (objective(1.0), 1.0)]
        path.append(min(candidates)[1])
    return np.asarray(path)

def gate_h_beta_posterior(state_path, delta_prefix):
    controls = np.diff(np.asarray(delta_prefix, dtype=float))/95.0
    increments = np.diff(np.asarray(state_path, dtype=float))
    prior_mean = CONFIG["gate_h_beta_prior_mean"]
    prior_sd = CONFIG["gate_h_beta_prior_sd"]
    process_sd = CONFIG["gate_h_process_sd"]
    precision = 1.0/prior_sd**2+np.dot(controls, controls)/process_sd**2
    variance = 1.0/precision
    mean = variance*(prior_mean/prior_sd**2+
                     np.dot(controls, increments)/process_sd**2)
    return float(np.clip(mean, *gate_h_beta_bounds)), float(np.sqrt(variance))

def gate_h_rbf_gp_predict(control_history, state_history, next_control):
    x = np.asarray(control_history, dtype=float)
    y = np.asarray(state_history, dtype=float)
    candidates = []
    for length_scale in (0.10, 0.20, 0.40, 0.80):
        distance = (x[:, None]-x[None, :])**2
        kernel = np.exp(-0.5*distance/length_scale**2)+0.03**2*np.eye(len(x))
        loo_errors = []
        for heldout in range(len(x)):
            keep = np.arange(len(x)) != heldout
            train_kernel = kernel[np.ix_(keep, keep)]
            test_kernel = np.exp(-0.5*(x[keep]-x[heldout])**2/length_scale**2)
            prediction = test_kernel@np.linalg.solve(train_kernel, y[keep])
            loo_errors.append((prediction-y[heldout])**2)
        test_kernel = np.exp(-0.5*(x-next_control)**2/length_scale**2)
        prediction = float(test_kernel@np.linalg.solve(kernel, y))
        candidates.append((float(np.mean(loo_errors)), prediction, length_scale))
    _, prediction, length_scale = min(candidates)
    return float(np.clip(prediction, y[-1], 1.0)), float(length_scale)

gate_h_contexts = []
gate_h_point_rows = []
for origin, target in zip(gate_h_origins, gate_h_targets):
    forecast_start = time.perf_counter()
    training_indices = list(range(1, origin+1))
    fold_fit, fold_residual = gate_g1_fit_amplitudes(training_indices)
    fold_discrepancy = gate_h_training_discrepancy(
        training_indices, gate_g_observed, fold_fit.x)
    state_path = gate_h_filter_prefix(
        gate_g_observed[:origin+1], fold_fit.x, fold_discrepancy, origin)
    beta_mean, beta_sd = gate_h_beta_posterior(
        state_path, gate_g_deltas[:origin+1])
    next_control_increment = (gate_g_deltas[target]-gate_g_deltas[origin])/95.0
    physics_xi = float(np.clip(
        state_path[-1]+beta_mean*next_control_increment, state_path[-1], 1.0))
    schedule_xi = float(np.clip(
        state_path[-1]+next_control_increment, state_path[-1], 1.0))
    persistence_xi = float(state_path[-1])
    control_history = gate_g_deltas[:origin+1]/95.0
    linear_slope = float(max(0.0, np.dot(control_history, state_path)/
                             max(np.dot(control_history, control_history), 1e-12)))
    linear_xi = float(np.clip(
        linear_slope*(gate_g_deltas[target]/95.0), state_path[-1], 1.0))
    gp_xi, gp_length_scale = gate_h_rbf_gp_predict(
        control_history, state_path, gate_g_deltas[target]/95.0)
    point_modal = gate_g1_forward(physics_xi, fold_fit.x)
    schedule_modal = gate_g1_forward(schedule_xi, fold_fit.x)
    forecast_seconds = time.perf_counter()-forecast_start
    gate_h_contexts.append({
        "origin": origin, "target": target, "training_indices": training_indices,
        "amplitudes": fold_fit.x.copy(), "discrepancy": fold_discrepancy.copy(),
        "state_path": state_path.copy(), "beta_mean": beta_mean, "beta_sd": beta_sd,
        "next_control_increment": next_control_increment,
        "point_modal": point_modal.copy(), "schedule_modal": schedule_modal.copy(),
    })
    gate_h_point_rows.append({
        "origin_scenario": gate_g_scenarios[origin],
        "origin_delta_pier_mm": gate_g_deltas[origin],
        "target_scenario": gate_g_scenarios[target],
        "target_delta_pier_mm": gate_g_deltas[target],
        "past_measurement_indices": json_dumps_safe(list(range(origin+1))),
        "declared_next_control_increment": next_control_increment,
        "origin_causal_xi": state_path[-1],
        "beta_posterior_mean": beta_mean, "beta_posterior_sd": beta_sd,
        "physics_forecast_xi": physics_xi,
        "schedule_nominal_xi": schedule_xi,
        "persistence_xi": persistence_xi,
        "linear_trend_xi": linear_xi,
        "rbf_gp_xi": gp_xi, "rbf_gp_length_scale": gp_length_scale,
        "fold_amplitude_a": fold_fit.x[0], "fold_amplitude_b": fold_fit.x[1],
        "fold_max_training_standardized_residual": float(np.max(np.abs(fold_residual))),
        **{f"mode_{mode}_training_discrepancy": fold_discrepancy[index]
           for index, mode in enumerate(gate_g_experimental_modes)},
        "point_forecast_seconds": forecast_seconds,
    })

# Targets are opened only after every forecast object above is frozen.
for row, context in zip(gate_h_point_rows, gate_h_contexts):
    target = context["target"]
    target_xi = gate_g1_truth[target]
    observed_modal = gate_g_observed[target]
    row.update({
        "true_xi_for_audit_only": target_xi,
        "physics_absolute_xi_error": abs(row["physics_forecast_xi"]-target_xi),
        "schedule_absolute_xi_error": abs(row["schedule_nominal_xi"]-target_xi),
        "persistence_absolute_xi_error": abs(row["persistence_xi"]-target_xi),
        "linear_absolute_xi_error": abs(row["linear_trend_xi"]-target_xi),
        "rbf_gp_absolute_xi_error": abs(row["rbf_gp_xi"]-target_xi),
        "physics_modal_rmse": float(np.sqrt(np.mean(
            (context["point_modal"]-observed_modal)**2))),
        "schedule_modal_rmse": float(np.sqrt(np.mean(
            (context["schedule_modal"]-observed_modal)**2))),
    })

gate_h_rolling = pd.DataFrame(gate_h_point_rows)
display(gate_h_rolling)

,origin_scenario,origin_delta_pier_mm,target_scenario,target_delta_pier_mm,past_measurement_indices,declared_next_control_increment,origin_causal_xi,beta_posterior_mean,beta_posterior_sd,physics_forecast_xi,...,mode_4_training_discrepancy,point_forecast_seconds,true_xi_for_audit_only,physics_absolute_xi_error,schedule_absolute_xi_error,persistence_absolute_xi_error,linear_absolute_xi_error,rbf_gp_absolute_xi_error,physics_modal_rmse,schedule_modal_rmse
0,03,20.0,04,40.0,"[0, 1]",0.210526,0.251816,1.085287,0.375880,0.480298,...,0.01,0.037114,0.421053,0.059245,0.041290,0.169236,0.082580,0.169236,0.011394,0.010086
1,04,40.0,05,80.0,"[0, 1, 2]",0.421053,0.375463,0.934370,0.313794,0.768882,...,0.01,0.130167,0.842105,0.073223,0.045590,0.466642,0.007929,0.466642,0.006391,0.004419
2,05,80.0,06,95.0,"[0, 1, 2, 3]",0.157895,0.837524,1.011770,0.210972,0.997277,...,0.01,0.268483,1.000000,0.002723,0.004581,0.162476,0.000000,0.017237,0.008842,0.008759


## 30. Rolling-origin predictive uncertainty and benchmark warning risk

For every rolling origin, training observations and trajectory amplitudes are recalibrated from the available prefix only. A parametric bootstrap propagates modal-observation uncertainty, training-only model discrepancy, trajectory calibration, transition uncertainty and future process variation. The bootstrap is an uncertainty audit and is not conflated with the point-query latency.

The event \(H^{(1D)}<0.20\) is a benchmark-coordinate warning event. It is not a bridge failure probability.


In [36]:
require_compute_node("Gate H probabilistic response propagation")
def gate_h_empirical_crps(samples, target):
    values = np.sort(np.asarray(samples, dtype=float))
    count = len(values)
    indices = np.arange(1, count+1)
    pairwise_mean = 2.0*np.sum((2*indices-count-1)*values)/count**2
    return float(np.mean(np.abs(values-target))-0.5*pairwise_mean)

def gate_h_gaussian_nll(samples, target):
    values = np.asarray(samples, dtype=float)
    variance = max(float(np.var(values, ddof=1)), 1e-8)
    return float(0.5*np.log(2*np.pi*variance)+0.5*(target-np.mean(values))**2/variance)

gate_h_bootstrap_replicates = (
    CONFIG["gate_h_full_bootstrap_replicates"] if gate_h_mode == "full"
    else CONFIG["gate_h_smoke_bootstrap_replicates"])
gate_h_rng = np.random.default_rng(CONFIG["seed"]+7200)
gate_h_sample_rows = []
gate_h_modal_rows = []
gate_h_risk_rows = []

for row, context in zip(gate_h_point_rows, gate_h_contexts):
    probabilistic_start = time.perf_counter()
    origin, target = context["origin"], context["target"]
    training_indices = context["training_indices"]
    future_indices = list(range(target, len(gate_g_deltas)))
    xi_samples = np.empty(gate_h_bootstrap_replicates)
    modal_samples = np.empty((gate_h_bootstrap_replicates, len(gate_g_experimental_modes)))
    future_paths = np.empty((gate_h_bootstrap_replicates, len(future_indices)))
    for replicate in range(gate_h_bootstrap_replicates):
        sampled_prefix = gate_g_observed[:origin+1].copy()
        for training_index in training_indices:
            sampled_prefix[training_index] += gate_h_rng.multivariate_normal(
                mean=np.zeros(len(gate_g_experimental_modes)),
                cov=gate_g1_base_cov[training_index],
            )
        sampled_fit, _ = gate_g1_fit_amplitudes(
            training_indices, sampled_prefix, starts=(context["amplitudes"],),
            enforce_three_sigma=False)
        sampled_discrepancy = gate_h_training_discrepancy(
            training_indices, sampled_prefix, sampled_fit.x)
        sampled_state_path = gate_h_filter_prefix(
            sampled_prefix, sampled_fit.x, sampled_discrepancy, origin)
        beta_mean, beta_sd = gate_h_beta_posterior(
            sampled_state_path, gate_g_deltas[:origin+1])
        beta_sample = float(np.clip(
            gate_h_rng.normal(beta_mean, beta_sd), *gate_h_beta_bounds))
        current_xi = sampled_state_path[-1]
        for future_position, future_index in enumerate(future_indices):
            previous_index = future_index-1
            control_increment = (gate_g_deltas[future_index]-
                                 gate_g_deltas[previous_index])/95.0
            innovation = gate_h_rng.normal(0.0, CONFIG["gate_h_process_sd"])
            current_xi = float(np.clip(
                current_xi+beta_sample*control_increment+innovation,
                current_xi, 1.0))
            future_paths[replicate, future_position] = current_xi
        xi_samples[replicate] = future_paths[replicate, 0]
        modal_mean = gate_g1_forward(xi_samples[replicate], sampled_fit.x)
        # Use the most conservative training-state covariance in the PSD sense:
        # start from the state with the largest trace and add model discrepancy.
        conservative_index = max(
            training_indices,
            key=lambda index: float(np.trace(gate_g1_base_cov[index])),
        )
        predictive_covariance = (
            gate_g1_base_cov[conservative_index]
            + np.diag(sampled_discrepancy**2)
        )
        modal_samples[replicate] = modal_mean + gate_h_rng.multivariate_normal(
            mean=np.zeros(len(gate_g_experimental_modes)),
            cov=predictive_covariance,
        )
        gate_h_sample_rows.append({
            "replicate": replicate,
            "origin_delta_pier_mm": gate_g_deltas[origin],
            "target_delta_pier_mm": gate_g_deltas[target],
            "sampled_xi": xi_samples[replicate],
            "sampled_effective_health_1d": 1.0-xi_samples[replicate],
            **{f"sampled_mode_{mode}_ratio": modal_samples[replicate, mode_index]
               for mode_index, mode in enumerate(gate_g_experimental_modes)},
        })

    xi_quantiles = np.percentile(xi_samples, [2.5, 50.0, 97.5])
    half = gate_h_bootstrap_replicates//2
    split_difference = float(np.max(np.abs(
        np.percentile(xi_samples[:half], [2.5, 50.0, 97.5])-
        np.percentile(xi_samples[half:], [2.5, 50.0, 97.5]))))
    target_xi = gate_g1_truth[target]
    row.update({
        "bootstrap_median_xi": xi_quantiles[1],
        "xi_ci95_lower": xi_quantiles[0], "xi_ci95_upper": xi_quantiles[2],
        "xi_ci95_width": xi_quantiles[2]-xi_quantiles[0],
        "xi_ci95_covers_true": bool(xi_quantiles[0] <= target_xi <= xi_quantiles[2]),
        "xi_crps": gate_h_empirical_crps(xi_samples, target_xi),
        "xi_gaussian_nll": gate_h_gaussian_nll(xi_samples, target_xi),
        "bootstrap_split_quantile_difference": split_difference,
        "probability_next_health_below_warning": float(np.mean(
            1.0-xi_samples < CONFIG["gate_h_warning_health"])),
    })
    for mode_index, mode in enumerate(gate_g_experimental_modes):
        quantiles = np.percentile(modal_samples[:, mode_index], [2.5, 50.0, 97.5])
        observed = gate_g_observed[target, mode_index]
        gate_h_modal_rows.append({
            "origin_delta_pier_mm": gate_g_deltas[origin],
            "target_delta_pier_mm": gate_g_deltas[target],
            "experimental_mode": int(mode),
            "point_forecast_ratio": context["point_modal"][mode_index],
            "observed_ratio_for_scoring_only": observed,
            "predictive_median_ratio": quantiles[1],
            "ci95_lower_ratio": quantiles[0], "ci95_upper_ratio": quantiles[2],
            "ci95_covers_observed": bool(quantiles[0] <= observed <= quantiles[2]),
            "crps": gate_h_empirical_crps(modal_samples[:, mode_index], observed),
            "gaussian_nll": gate_h_gaussian_nll(modal_samples[:, mode_index], observed),
        })

    warning_xi = 1.0-CONFIG["gate_h_warning_health"]
    crossed = future_paths >= warning_xi
    any_crossing = np.any(crossed, axis=1)
    first_crossing = np.where(
        any_crossing,
        np.argmax(crossed, axis=1)+1,
        len(future_indices)+1,
    ).astype(float)
    probability_within = float(np.mean(np.any(crossed, axis=1)))
    restricted_expected = float(np.mean(first_crossing))
    for future_position, future_index in enumerate(future_indices):
        future_quantiles = np.percentile(
            future_paths[:, future_position], [2.5, 50.0, 97.5])
        gate_h_risk_rows.append({
            "origin_delta_pier_mm": gate_g_deltas[origin],
            "future_delta_pier_mm": gate_g_deltas[future_index],
            "horizon_stages": future_position+1,
            "probability_health_below_warning_by_stage": float(np.mean(
                np.any(crossed[:, :future_position+1], axis=1))),
            "future_xi_median": future_quantiles[1],
            "future_xi_ci95_lower": future_quantiles[0],
            "future_xi_ci95_upper": future_quantiles[2],
            "probability_crossing_within_remaining_schedule": probability_within,
            "restricted_expected_stages_to_warning": restricted_expected,
            "warning_health_threshold": CONFIG["gate_h_warning_health"],
            "benchmark_truth_health_for_audit_only": 1.0-gate_g1_truth[future_index],
        })
    row["probabilistic_forecast_seconds"] = time.perf_counter()-probabilistic_start

gate_h_rolling = pd.DataFrame(gate_h_point_rows)
gate_h_modal = pd.DataFrame(gate_h_modal_rows)
gate_h_samples = pd.DataFrame(gate_h_sample_rows)
gate_h_risk = pd.DataFrame(gate_h_risk_rows)
gate_h_rolling.to_csv(TABLE_ROOT/f"gate_h_rolling_forecasts_{gate_h_mode}.csv", index=False)
gate_h_modal.to_csv(TABLE_ROOT/f"gate_h_modal_forecasts_{gate_h_mode}.csv", index=False)
gate_h_samples.to_csv(TABLE_ROOT/f"gate_h_predictive_samples_{gate_h_mode}.csv", index=False)
gate_h_risk.to_csv(TABLE_ROOT/f"gate_h_threshold_risk_{gate_h_mode}.csv", index=False)
display(gate_h_rolling)
display(gate_h_modal)
display(gate_h_risk)

,origin_scenario,origin_delta_pier_mm,target_scenario,target_delta_pier_mm,past_measurement_indices,declared_next_control_increment,origin_causal_xi,beta_posterior_mean,beta_posterior_sd,physics_forecast_xi,...,bootstrap_median_xi,xi_ci95_lower,xi_ci95_upper,xi_ci95_width,xi_ci95_covers_true,xi_crps,xi_gaussian_nll,bootstrap_split_quantile_difference,probability_next_health_below_warning,probabilistic_forecast_seconds
0,03,20.0,04,40.0,"[0, 1]",0.210526,0.251816,1.085287,0.375880,0.480298,...,0.455821,0.201772,0.757521,0.555749,True,0.041098,-0.938941,0.011610,0.011,12.018132
1,04,40.0,05,80.0,"[0, 1, 2]",0.421053,0.375463,0.934370,0.313794,0.768882,...,0.784781,0.413991,1.000000,0.586009,True,0.049473,-0.794858,0.018770,0.466,34.786012
2,05,80.0,06,95.0,"[0, 1, 2, 3]",0.157895,0.837524,1.011770,0.210972,0.997277,...,0.987004,0.820871,1.000000,0.179129,True,0.016727,-1.554879,0.004008,0.996,41.592159


,origin_delta_pier_mm,target_delta_pier_mm,experimental_mode,point_forecast_ratio,observed_ratio_for_scoring_only,predictive_median_ratio,ci95_lower_ratio,ci95_upper_ratio,ci95_covers_observed,crps,gaussian_nll
0,20.0,40.0,1,0.973046,0.987500,0.974246,0.938774,1.007296,True,0.007811,-2.834542
1,20.0,40.0,3,0.971338,0.977941,0.973116,0.920118,1.015494,True,0.005434,-2.786323
2,20.0,40.0,4,0.962775,0.974478,0.965255,0.920958,1.002868,True,0.006266,-2.838370
3,40.0,80.0,1,0.960264,0.962500,0.958967,0.921795,0.997258,True,0.004978,-2.994899
4,40.0,80.0,3,0.947290,0.938725,0.947848,0.892323,0.992077,True,0.007408,-2.695953
5,40.0,80.0,4,0.941683,0.935035,0.942919,0.905683,0.987222,True,0.006311,-2.828383
6,80.0,95.0,1,0.947447,0.937500,0.950168,0.921744,0.976193,True,0.007602,-2.948262
7,80.0,95.0,3,0.915040,0.926471,0.922664,0.892247,0.959060,True,0.004392,-3.130428
8,80.0,95.0,4,0.923334,0.921114,0.927220,0.898719,0.956324,True,0.004394,-3.212184


,origin_delta_pier_mm,future_delta_pier_mm,horizon_stages,probability_health_below_warning_by_stage,future_xi_median,future_xi_ci95_lower,future_xi_ci95_upper,probability_crossing_within_remaining_schedule,restricted_expected_stages_to_warning,warning_health_threshold,benchmark_truth_health_for_audit_only
0,20.0,40.0,1,0.011,0.455821,0.201772,0.757521,0.768,2.598,0.2,0.578947
1,20.0,80.0,2,0.623,0.889381,0.361807,1.000000,0.768,2.598,0.2,0.157895
2,20.0,95.0,3,0.768,1.000000,0.384518,1.000000,0.768,2.598,0.2,0.000000
3,40.0,80.0,1,0.466,0.784781,0.413991,1.000000,0.734,1.800,0.2,0.157895
4,40.0,95.0,2,0.734,0.949904,0.445155,1.000000,0.734,1.800,0.2,0.000000
5,80.0,95.0,1,0.996,0.987004,0.820871,1.000000,0.996,1.004,0.2,0.000000


## 31. Gate H decision — conditional response evidence versus autonomous prognosis

Gate H issues two separate decisions:

1. whether the conditional next-response forecasts meet the predeclared accuracy, calibration and timing checks;
2. whether an increment-free autonomous progression forecast has been demonstrated.

The second decision is expected to remain a NO-GO for Z24 because the experiment contains only a few deliberately imposed settlement states and no natural calendar-time degradation trajectory.


In [37]:
gate_h_physics_mae = float(gate_h_rolling["physics_absolute_xi_error"].mean())
gate_h_schedule_mae = float(gate_h_rolling["schedule_absolute_xi_error"].mean())
gate_h_persistence_mae = float(gate_h_rolling["persistence_absolute_xi_error"].mean())
gate_h_linear_mae = float(gate_h_rolling["linear_absolute_xi_error"].mean())
gate_h_gp_mae = float(gate_h_rolling["rbf_gp_absolute_xi_error"].mean())
gate_h_physics_modal_mean = float(gate_h_rolling["physics_modal_rmse"].mean())
gate_h_schedule_modal_mean = float(gate_h_rolling["schedule_modal_rmse"].mean())
gate_h_schedule_wins = float(np.mean(
    gate_h_rolling["physics_absolute_xi_error"] <
    gate_h_rolling["schedule_absolute_xi_error"]))

gate_h_checks = {
    "full_execution": gate_h_mode == "full",
    "prior_gate_g1_go_preserved": gate_g1_decision ==
        "GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL",
    "three_rolling_origins_executed": len(gate_h_rolling) == 3,
    "future_measurements_not_used_by_forecaster": True,
    "future_targets_used_only_for_scoring": True,
    "future_control_increment_declared_as_condition": True,
    "maximum_absolute_xi_error_le_0p15": bool(
        gate_h_rolling["physics_absolute_xi_error"].max() <=
        CONFIG["gate_h_max_absolute_xi_error"]),
    "mean_modal_rmse_le_1p5pct": bool(
        gate_h_physics_modal_mean <= CONFIG["gate_h_max_mean_modal_rmse"]),
    "maximum_modal_rmse_le_2pct": bool(
        gate_h_rolling["physics_modal_rmse"].max() <=
        CONFIG["gate_h_max_single_modal_rmse"]),
    "xi_predictive_coverage_ge_2of3": bool(
        gate_h_rolling["xi_ci95_covers_true"].mean() >=
        CONFIG["gate_h_min_xi_ci_coverage"]),
    "modal_predictive_coverage_ge_80pct": bool(
        gate_h_modal["ci95_covers_observed"].mean() >=
        CONFIG["gate_h_min_modal_ci_coverage"]),
    "mean_xi_crps_le_0p15": bool(
        gate_h_rolling["xi_crps"].mean() <= CONFIG["gate_h_max_mean_xi_crps"]),
    "bootstrap_split_quantile_difference_le_0p08": bool(
        gate_h_rolling["bootstrap_split_quantile_difference"].max() <=
        CONFIG["gate_h_max_split_quantile_difference"]),
    "maximum_xi_ci_width_le_0p65": bool(
        gate_h_rolling["xi_ci95_width"].max() <= CONFIG["gate_h_max_xi_ci_width"]),
    "maximum_point_forecast_time_le_0p1s": bool(
        gate_h_rolling["point_forecast_seconds"].max() <=
        CONFIG["gate_h_max_point_forecast_seconds"]),
    "maximum_probabilistic_forecast_time_le_predeclared_limit": bool(
        gate_h_rolling["probabilistic_forecast_seconds"].max() <=
        CONFIG["gate_h_max_probabilistic_forecast_seconds"]),
    "state_mae_not_materially_worse_than_schedule_nominal": bool(
        gate_h_physics_mae-gate_h_schedule_mae <=
        CONFIG["gate_h_max_excess_mae_over_schedule_baseline"]),
    "modal_rmse_not_materially_worse_than_schedule_nominal": bool(
        gate_h_physics_modal_mean-gate_h_schedule_modal_mean <=
        CONFIG["gate_h_max_excess_modal_rmse_over_schedule_baseline"]),
    "finite_probabilistic_scores": bool(np.isfinite(
        gate_h_rolling[["xi_crps", "xi_gaussian_nll"]].to_numpy()).all() and
        np.isfinite(gate_h_modal[["crps", "gaussian_nll"]].to_numpy()).all()),
    "benchmark_warning_probabilities_bounded": bool(
        gate_h_risk["probability_health_below_warning_by_stage"].between(0, 1).all()),
    "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
}
gate_h_autonomy_checks = {
    "mean_error_below_schedule_nominal": gate_h_physics_mae < gate_h_schedule_mae,
    "wins_against_schedule_nominal_in_at_least_2of3_origins":
        gate_h_schedule_wins >= CONFIG[
            "gate_h_min_schedule_baseline_win_fraction_for_autonomy"],
    "increment_free_forecast_demonstrated": False,
}
if gate_h_mode == "smoke":
    gate_h_decision = "SMOKE_COMPLETED_REQUIRES_FULL_RUN"
elif all(gate_h_checks.values()):
    gate_h_decision = "GO_Z24_CONDITIONAL_RESPONSE_EVIDENCE_COMPLETE_V0_6_FINAL"
else:
    gate_h_decision = "NO_GO_CONDITIONAL_RESPONSE_PROPAGATION_REVISE"
gate_h_autonomy_decision = (
    "GO_AUTONOMOUS_PROGRESSION" if all(gate_h_autonomy_checks.values())
    else "NO_GO_AUTONOMOUS_OR_CALENDAR_TIME_PROGNOSIS")

gate_h_diagnostic = {
    "version": "V0.6 FINAL", "decision": gate_h_decision,
    "autonomy_decision": gate_h_autonomy_decision,
    "execution_mode": gate_h_mode,
    "execution_boundary": CONFIG["current_execution_boundary"],
    "forecast_definition": {
        "target": "next Z24 modal response and xi under a declared controlled-settlement increment",
        "transition": "delta_xi=beta*delta_settlement/95+epsilon; beta>=0",
        "beta_prior_mean_sd": [CONFIG["gate_h_beta_prior_mean"],
                               CONFIG["gate_h_beta_prior_sd"]],
        "process_sd": CONFIG["gate_h_process_sd"],
        "warning_health_threshold": CONFIG["gate_h_warning_health"],
        "future_measurements_used": False,
        "future_control_schedule_used_as_condition": True,
    },
    "rolling_origin": {
        "origins_mm": gate_h_rolling["origin_delta_pier_mm"].tolist(),
        "targets_mm": gate_h_rolling["target_delta_pier_mm"].tolist(),
        "point_xi": gate_h_rolling["physics_forecast_xi"].tolist(),
        "truth_xi_for_audit_only": gate_h_rolling["true_xi_for_audit_only"].tolist(),
        "maximum_absolute_xi_error": float(
            gate_h_rolling["physics_absolute_xi_error"].max()),
        "mean_modal_rmse": gate_h_physics_modal_mean,
        "maximum_modal_rmse": float(gate_h_rolling["physics_modal_rmse"].max()),
        "maximum_point_forecast_seconds": float(
            gate_h_rolling["point_forecast_seconds"].max()),
        "maximum_probabilistic_forecast_seconds": float(
            gate_h_rolling["probabilistic_forecast_seconds"].max()),
    },
    "uncertainty": {
        "bootstrap_replicates_per_origin": gate_h_bootstrap_replicates,
        "xi_ci_coverage_fraction": float(gate_h_rolling["xi_ci95_covers_true"].mean()),
        "modal_ci_coverage_fraction": float(gate_h_modal["ci95_covers_observed"].mean()),
        "mean_xi_crps": float(gate_h_rolling["xi_crps"].mean()),
        "maximum_split_quantile_difference": float(
            gate_h_rolling["bootstrap_split_quantile_difference"].max()),
        "maximum_xi_ci_width": float(gate_h_rolling["xi_ci95_width"].max()),
    },
    "baselines": {
        "physics_mean_absolute_xi_error": gate_h_physics_mae,
        "schedule_nominal_mean_absolute_xi_error": gate_h_schedule_mae,
        "persistence_mean_absolute_xi_error": gate_h_persistence_mae,
        "linear_trend_mean_absolute_xi_error": gate_h_linear_mae,
        "rbf_gp_mean_absolute_xi_error": gate_h_gp_mae,
        "physics_schedule_win_fraction": gate_h_schedule_wins,
        "physics_mean_modal_rmse": gate_h_physics_modal_mean,
        "schedule_nominal_mean_modal_rmse": gate_h_schedule_modal_mean,
    },
    "checks": gate_h_checks,
    "autonomy_checks": gate_h_autonomy_checks,
    "conditional_response_propagation_performed": True,
    "autonomous_or_calendar_time_prognosis_performed": False,
    "capacity_or_rul_claim_performed": False,
    "maintenance_decision_performed": False,
    "raw_z24_signature_unchanged": gate_h_checks["raw_z24_signature_unchanged"],
    "limitations": [
        "only three rolling forecast origins and four nonhealthy states are available",
        "future controlled-settlement increments are declared conditioning inputs, not predictions",
        "xi=delta/95 makes the schedule-nominal comparator intrinsically strong",
        "the transition prior is intentionally informative because the progression sample is tiny",
        "warning risk is a benchmark-coordinate event, not structural failure probability",
        "no natural aging time, capacity or remaining-life target exists in Z24",
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT/"gate_h_diagnostic.json").write_text(
    json_dumps_safe(gate_h_diagnostic, indent=2, sort_keys=True)+"\n", encoding="utf-8")

fig, ax = plt.subplots(figsize=(7.4, 5.4))
truth = gate_h_rolling["true_xi_for_audit_only"].to_numpy()
median = gate_h_rolling["bootstrap_median_xi"].to_numpy()
lower = gate_h_rolling["xi_ci95_lower"].to_numpy()
upper = gate_h_rolling["xi_ci95_upper"].to_numpy()
ax.errorbar(truth, median, yerr=np.vstack([median-lower, upper-median]),
            fmt="o", color="#e76f51", capsize=4, label="conditional physics posterior")
ax.scatter(truth, gate_h_rolling["schedule_nominal_xi"], marker="s", s=55,
           color="#2a9d8f", label="schedule nominal")
ax.scatter(truth, gate_h_rolling["persistence_xi"], marker="x", s=65,
           color="#6d597a", label="persistence")
ax.scatter(truth, gate_h_rolling["linear_trend_xi"], marker="^", s=55,
           color="#457b9d", label="linear trend")
ax.scatter(truth, gate_h_rolling["rbf_gp_xi"], marker="D", s=45,
           color="#f4a261", label="RBF-GP")
ax.plot([0, 1], [0, 1], "--", color="#555555", label="ideal")
ax.set(xlabel=r"Next benchmark $\xi$ (audit only)", ylabel=r"Rolling forecast $\hat\xi$",
       title="Figure 25 — Gate H rolling-origin conditional forecasts")
ax.legend(fontsize=8)
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_25_gate_h_rolling_forecast_{gate_h_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

fig, ax = plt.subplots(figsize=(9.6, 5.2))
colors = {1: "#264653", 3: "#2a9d8f", 4: "#e76f51"}
offsets = {1: -1.3, 3: 0.0, 4: 1.3}
for mode in gate_g_experimental_modes:
    frame = gate_h_modal[gate_h_modal["experimental_mode"] == mode]
    x = frame["target_delta_pier_mm"].to_numpy()+offsets[int(mode)]
    med = frame["predictive_median_ratio"].to_numpy()
    lo = frame["ci95_lower_ratio"].to_numpy()
    hi = frame["ci95_upper_ratio"].to_numpy()
    ax.errorbar(x, med, yerr=np.vstack([med-lo, hi-med]), fmt="o-", capsize=3,
                color=colors[int(mode)], label=f"forecast mode {mode}")
    ax.scatter(x, frame["observed_ratio_for_scoring_only"], marker="x", s=60,
               color=colors[int(mode)], label=f"observed mode {mode}")
ax.set(xlabel="Target controlled settlement [mm]",
       ylabel="Healthy-normalized frequency ratio",
       title="Figure 26 — Past-only modal forecast distributions")
ax.legend(ncol=3, fontsize=7)
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_26_gate_h_modal_forecast_{gate_h_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

fig, ax = plt.subplots(figsize=(8.4, 5.2))
for origin, frame in gate_h_risk.groupby("origin_delta_pier_mm"):
    ax.plot(frame["future_delta_pier_mm"],
            frame["probability_health_below_warning_by_stage"], "o-",
            label=f"origin {origin:g} mm")
ax.set(xlabel="Declared future settlement stage [mm]",
       ylabel=r"$P(H^{(1D)}<0.20)$ by stage", ylim=(-0.03, 1.03),
       title="Figure 27 — Conditional benchmark warning probability")
ax.legend()
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(FIGURE_ROOT/f"figure_27_gate_h_warning_risk_{gate_h_mode}.{suffix}",
                bbox_inches="tight")
plt.show()
plt.close("all")

print(json_dumps_safe(gate_h_diagnostic, indent=2))

{
  "version": "V0.6 FINAL",
  "decision": "NO_GO_CONDITIONAL_RESPONSE_PROPAGATION_REVISE",
  "autonomy_decision": "NO_GO_AUTONOMOUS_OR_CALENDAR_TIME_PROGNOSIS",
  "execution_mode": "full",
  "execution_boundary": "STOP_AFTER_V0_6_FINAL_CONSOLIDATION_AND_CODE_FREEZE",
  "forecast_definition": {
    "target": "next Z24 modal response and xi under a declared controlled-settlement increment",
    "transition": "delta_xi=beta*delta_settlement/95+epsilon; beta>=0",
    "beta_prior_mean_sd": [
      1.0,
      0.5
    ],
    "process_sd": 0.12,
    "warning_health_threshold": 0.2,
    "future_measurements_used": false,
    "future_control_schedule_used_as_condition": true
  },
  "rolling_origin": {
    "origins_mm": [
      20.0,
      40.0,
      80.0
    ],
    "targets_mm": [
      40.0,
      80.0,
      95.0
    ],
    "point_xi": [
      0.4802976795699456,
      0.7688818954098047,
      0.9972774851918298
    ],
    "truth_xi_for_audit_only": [
      0.42105263157894735,
      0.8421

## 32. Gate I — locked Z24 FVT external-excitation validation

The AVT pipeline above is the complete development path. Gate I now opens the Z24 forced-vibration tests (FVT) for the same controlled settlement states \(01,03,04,05,06\).

This is a **locked validation**:

- the AVT-retained modal tracks remain fixed;
- the three-span physics model, GPT-PINN basis, trajectory exponents, amplitudes and discrepancy are not refitted to FVT;
- FVT scenario 01 is used only as the excitation-specific normalization reference;
- FVT target settlement labels are used only after inversion for scoring;
- no FVT result is allowed to modify an upstream AVT choice.

A pass supports robustness across two excitation regimes of the same bridge. It is not cross-bridge validation.


In [38]:
gate_i_mode = CONFIG["gate_i_mode"]
if gate_i_mode not in {"full", "smoke"}:
    raise ValueError("GPT_PINN_GATE_I_MODE must be 'full' or 'smoke'")
if gate_i_mode == "full" and gate_g1_decision != "GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL":
    raise RuntimeError("Full Gate I requires the executed Gate G.1 GO decision")
require_compute_node("Gate I locked Z24 FVT validation")

gate_i_frequency_cube = np.full_like(frequency_cube, np.nan)
gate_i_mac_cube = np.full_like(frequency_cube, np.nan)
gate_i_extraction_log = []
gate_i_selected_label_rows = []

def gate_i_healthy_mode_candidates(
    frequency: np.ndarray,
    leading: np.ndarray,
    vectors: np.ndarray,
    setup_index: int,
) -> list[tuple[float, np.ndarray] | None]:
    smoothed = smooth_log_spectrum(frequency, leading)
    selected: list[tuple[float, np.ndarray] | None] = []
    used_indices: set[int] = set()
    for mode_index, (low, high) in enumerate(CONFIG["candidate_mode_bands_hz"]):
        band_indices = np.flatnonzero((frequency >= low) & (frequency <= high))
        if not len(band_indices):
            selected.append(None)
            continue
        local_peaks, properties = signal.find_peaks(
            smoothed[band_indices], prominence=0.01)
        candidate_indices = band_indices[local_peaks]
        if not len(candidate_indices):
            candidate_indices = np.array(
                [band_indices[int(np.argmax(smoothed[band_indices]))]])
        avt_anchor = frequency_cube[0, setup_index, mode_index]
        candidate_indices = np.asarray([
            index for index in candidate_indices if index not in used_indices
        ], dtype=int)
        if not len(candidate_indices):
            selected.append(None)
            continue
        distance = np.abs(frequency[candidate_indices]-avt_anchor)
        within = distance <= CONFIG["gate_i_healthy_anchor_radius_hz"]
        if not np.any(within):
            selected.append(None)
            continue
        candidate_indices = candidate_indices[within]
        distance = distance[within]
        chosen = int(candidate_indices[int(np.argmin(distance))])
        used_indices.add(chosen)
        selected.append((float(frequency[chosen]), vectors[chosen]))
    return selected

for setup_index in range(setup_count):
    # setup_index is the physical AVT zone index used by the development cube.
    # FVT file numbering is reversed spatially and must be mapped explicitly.
    physical_avt_setup = setup_index + 1
    fvt_file_setup = ACTIVE_AVT_TO_FVT_SETUP[physical_avt_setup]
    setup = physical_avt_setup
    paths = {
        scenario: z24_physical_setup_path(
            scenario, CONFIG["gate_i_excitation"], physical_avt_setup
        )
        for scenario in scenarios
    }
    try:
        vertical_by_scenario = {}
        shaker_by_scenario = {}
        for scenario, path in paths.items():
            if not path.exists():
                raise FileNotFoundError(path)
            _, labels = load_z24_mat(path)
            shaker_by_scenario[scenario] = sorted(
                set(labels).intersection(FVT_SHAKER_CHANNELS)
            )
            vertical_by_scenario[scenario] = response_vertical_labels(labels)
        common_labels = sorted(set.intersection(
            *(set(values) for values in vertical_by_scenario.values())))
        if any(label in FVT_SHAKER_CHANNELS for label in common_labels):
            raise RuntimeError("FVT driving-point channels leaked into response FDD")
        if len(common_labels) < 3:
            raise RuntimeError(
                f"Too few common vertical FVT response channels in physical setup "
                f"{physical_avt_setup} / FVT file setup {fvt_file_setup}: {len(common_labels)}")
        gate_i_selected_label_rows.append({
            "physical_avt_setup": physical_avt_setup,
            "fvt_file_setup": fvt_file_setup,
            "mapping_status": SETUP_MAPPING_STATUS,
            "common_vertical_count": len(common_labels),
            "labels": ";".join(common_labels),
            "excluded_shaker_labels": ";".join(sorted(set(
                label for values in shaker_by_scenario.values() for label in values
            ))),
        })
        setup_data = {
            scenario: compute_fdd(path, common_labels)
            for scenario, path in paths.items()
        }
        healthy_modes = gate_i_healthy_mode_candidates(
            *setup_data["01"], setup_index)
        for mode_index, mode in enumerate(healthy_modes):
            if mode is not None:
                gate_i_frequency_cube[0, setup_index, mode_index] = mode[0]
                gate_i_mac_cube[0, setup_index, mode_index] = 1.0

        previous_modes = list(healthy_modes)
        for scenario_index, scenario in enumerate(scenarios[1:], start=1):
            frequency, leading, vectors = setup_data[scenario]
            smoothed = smooth_log_spectrum(frequency, leading)
            analysis_indices = np.flatnonzero((frequency >= 3.0) & (frequency <= 14.0))
            peaks, _ = signal.find_peaks(
                smoothed[analysis_indices], prominence=0.005)
            candidates = analysis_indices[peaks]
            if not len(candidates):
                gate_i_extraction_log.append({
                    "physical_avt_setup": physical_avt_setup,
                    "fvt_file_setup": fvt_file_setup,
                    "scenario": scenario,
                    "status": "NO_DETECTED_FVT_PEAKS",
                })
                continue
            cost = np.full((mode_count, len(candidates)), 99.0)
            mac_values = np.full_like(cost, np.nan)
            for mode_index, reference in enumerate(previous_modes):
                if reference is None:
                    continue
                reference_frequency, reference_vector = reference
                for candidate_index, spectral_index in enumerate(candidates):
                    relative_shift = abs(
                        frequency[spectral_index]-reference_frequency
                    ) / max(reference_frequency, 1e-12)
                    mac = modal_assurance_criterion(
                        reference_vector, vectors[spectral_index])
                    mac_values[mode_index, candidate_index] = mac
                    if relative_shift <= CONFIG["gate_i_tracking_max_relative_shift"]:
                        cost[mode_index, candidate_index] = (
                            CONFIG["mode_tracking_frequency_weight"]*relative_shift
                            + CONFIG["mode_tracking_mac_weight"]*(1.0-mac)
                        )
            rows, columns = linear_sum_assignment(cost)
            next_modes = list(previous_modes)
            for row_index, column_index in zip(rows, columns):
                if cost[row_index, column_index] >= 99.0:
                    continue
                spectral_index = int(candidates[column_index])
                gate_i_frequency_cube[
                    scenario_index, setup_index, row_index
                ] = float(frequency[spectral_index])
                gate_i_mac_cube[
                    scenario_index, setup_index, row_index
                ] = float(mac_values[row_index, column_index])
                next_modes[row_index] = (
                    float(frequency[spectral_index]), vectors[spectral_index])
            previous_modes = next_modes
            gate_i_extraction_log.append({
                "physical_avt_setup": physical_avt_setup,
                "fvt_file_setup": fvt_file_setup,
                "scenario": scenario,
                "status": "TRACKED_WITH_EXPLICIT_ABSTENTION",
                "detected_peak_count": int(len(candidates)),
            })
    except Exception as exc:
        gate_i_extraction_log.append({
            "physical_avt_setup": physical_avt_setup,
            "fvt_file_setup": fvt_file_setup,
            "scenario": "ALL",
            "status": "SETUP_EXTRACTION_FAILED",
            "error": repr(exc),
        })

gate_i_selected_labels = pd.DataFrame(gate_i_selected_label_rows)
gate_i_log = pd.DataFrame(gate_i_extraction_log)
gate_i_selected_labels.to_csv(
    TABLE_ROOT/"gate_i_fvt_common_vertical_channels.csv", index=False)
gate_i_log.to_csv(TABLE_ROOT/"gate_i_fvt_extraction_log.csv", index=False)
display(gate_i_selected_labels)
display(gate_i_log)

gate_i_summary_rows = []
for mode_index in range(mode_count):
    medians = np.array([
        np.nanmedian(row) if np.isfinite(row).any() else np.nan
        for row in gate_i_frequency_cube[:, :, mode_index]
    ])
    mads = np.array([
        np.nanmedian(np.abs(row-median)) if np.isfinite(row).any() else np.nan
        for row, median in zip(gate_i_frequency_cube[:, :, mode_index], medians)
    ])
    tolerances = np.maximum(0.15, 3.0*1.4826*mads)
    support = np.sum(
        np.abs(gate_i_frequency_cube[:, :, mode_index]-medians[:, None])
        <= tolerances[:, None],
        axis=1,
    )
    for state_index, scenario in enumerate(scenarios):
        gate_i_summary_rows.append({
            "scenario": scenario,
            "delta_pier_mm": settlements[state_index],
            "candidate_mode": mode_index+1,
            "median_frequency_hz": medians[state_index],
            "mad_frequency_hz": mads[state_index],
            "supported_setups": int(support[state_index]),
            "median_sequential_mac": (
                float(np.nanmedian(gate_i_mac_cube[state_index, :, mode_index]))
                if np.isfinite(gate_i_mac_cube[state_index, :, mode_index]).any()
                else np.nan),
        })
gate_i_mode_summary = pd.DataFrame(gate_i_summary_rows)
gate_i_mode_summary.to_csv(
    TABLE_ROOT/"gate_i_fvt_mode_summary.csv", index=False)
display(gate_i_mode_summary.pivot(
    index=["scenario", "delta_pier_mm"],
    columns="candidate_mode",
    values="median_frequency_hz",
))

gate_i_bootstrap_replicates = (
    CONFIG["gate_i_full_bootstrap_replicates"] if gate_i_mode == "full"
    else CONFIG["gate_i_smoke_bootstrap_replicates"])
gate_i_rng = np.random.default_rng(CONFIG["seed"]+7300)
gate_i_observation_rows = []
for state_index, scenario in enumerate(scenarios):
    for mode in retained_modes:
        mode_index = mode-1
        healthy_values = gate_i_frequency_cube[0, :, mode_index]
        current_values = gate_i_frequency_cube[state_index, :, mode_index]
        paired = np.flatnonzero(
            np.isfinite(healthy_values) & np.isfinite(current_values))
        if len(paired) < CONFIG["gate_i_minimum_setup_support"]:
            gate_i_observation_rows.append({
                "scenario": scenario,
                "delta_pier_mm": settlements[state_index],
                "mode": mode,
                "frequency_ratio": np.nan,
                "sigma_frequency_ratio": np.nan,
                "paired_setup_support": int(len(paired)),
            })
            continue
        current_median = float(np.median(current_values[paired]))
        healthy_median_fvt = float(np.median(healthy_values[paired]))
        ratio = current_median/healthy_median_fvt
        if state_index == 0:
            ratio_draws = np.ones(gate_i_bootstrap_replicates)
        else:
            sampled_positions = gate_i_rng.choice(
                paired,
                size=(gate_i_bootstrap_replicates, len(paired)),
                replace=True,
            )
            numerator = np.median(current_values[sampled_positions], axis=1)
            denominator = np.median(healthy_values[sampled_positions], axis=1)
            ratio_draws = numerator/denominator
        resolution_floor = 0.5*frequency_resolution_hz/max(healthy_median_fvt, 1e-12)
        sigma_ratio = max(
            float(np.std(ratio_draws, ddof=1)) if state_index else 0.0,
            resolution_floor,
        )
        gate_i_observation_rows.append({
            "scenario": scenario,
            "delta_pier_mm": settlements[state_index],
            "mode": mode,
            "median_frequency_hz": current_median,
            "healthy_fvt_frequency_hz": healthy_median_fvt,
            "frequency_ratio": ratio,
            "sigma_frequency_ratio": sigma_ratio,
            "ci95_low_ratio": float(np.quantile(ratio_draws, 0.025)),
            "ci95_high_ratio": float(np.quantile(ratio_draws, 0.975)),
            "paired_setup_support": int(len(paired)),
        })

gate_i_observations_long = pd.DataFrame(gate_i_observation_rows)
gate_i_observations_long.to_csv(
    TABLE_ROOT/"gate_i_fvt_locked_observations.csv", index=False)
display(gate_i_observations_long)


# Joint FVT ratio covariance using the same physical setup resampling for
# all retained modes. This preserves cross-mode dependence.
gate_i_ratio_covariance_by_scenario: dict[str, np.ndarray] = {}
gate_i_covariance_rows = []
gate_i_joint_rng = np.random.default_rng(CONFIG["seed"] + 7311)
for state_index, scenario in enumerate(scenarios):
    state_draws = []
    attempts = 0
    while len(state_draws) < gate_i_bootstrap_replicates:
        attempts += 1
        if attempts > gate_i_bootstrap_replicates * 20:
            raise RuntimeError(
                f"Unable to form joint FVT covariance draws for scenario {scenario}"
            )
        sampled_setups = gate_i_joint_rng.integers(0, setup_count, size=setup_count)
        ratios = []
        valid_draw = True
        for mode in retained_modes:
            mode_index = mode - 1
            current = gate_i_frequency_cube[state_index, sampled_setups, mode_index]
            baseline = gate_i_frequency_cube[0, sampled_setups, mode_index]
            valid = np.isfinite(current) & np.isfinite(baseline)
            if np.count_nonzero(valid) < max(3, CONFIG["gate_i_minimum_setup_support"] // 2):
                valid_draw = False
                break
            ratios.append(float(np.median(current[valid]) / np.median(baseline[valid])))
        if valid_draw:
            state_draws.append(ratios)

    state_draws = np.asarray(state_draws, dtype=float)
    covariance = np.atleast_2d(np.cov(state_draws, rowvar=False, ddof=1))
    marginal = gate_i_observations_long[
        gate_i_observations_long["scenario"] == scenario
    ].set_index("mode").loc[retained_modes, "sigma_frequency_ratio"].to_numpy()
    diagonal_addition = np.maximum(marginal**2 - np.diag(covariance), 0.0)
    covariance += np.diag(diagonal_addition)
    shrinkage = float(CONFIG["modal_ratio_covariance_shrinkage"])
    covariance = (
        (1.0 - shrinkage) * covariance
        + shrinkage * np.diag(np.diag(covariance))
    )
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    covariance = (
        eigenvectors
        @ np.diag(np.maximum(eigenvalues, max(1e-12, 1e-6 * eigenvalues.max())))
        @ eigenvectors.T
    )
    gate_i_ratio_covariance_by_scenario[scenario] = covariance
    for row_index, mode_i in enumerate(retained_modes):
        for column_index, mode_j in enumerate(retained_modes):
            gate_i_covariance_rows.append({
                "scenario": scenario,
                "mode_i": int(mode_i),
                "mode_j": int(mode_j),
                "covariance": float(covariance[row_index, column_index]),
                "correlation": float(
                    covariance[row_index, column_index]
                    / np.sqrt(
                        covariance[row_index, row_index]
                        * covariance[column_index, column_index]
                    )
                ),
            })
gate_i_ratio_covariance = pd.DataFrame(gate_i_covariance_rows)
gate_i_ratio_covariance.to_csv(
    TABLE_ROOT / "gate_i_fvt_joint_modal_ratio_covariance.csv", index=False
)

gate_i_extraction_pass = bool(
    gate_i_observations_long["frequency_ratio"].notna().all()
    and (gate_i_observations_long["paired_setup_support"]
         >= CONFIG["gate_i_minimum_setup_support"]).all()
)
print("Gate I extraction pass:", gate_i_extraction_pass)

,physical_avt_setup,fvt_file_setup,mapping_status,common_vertical_count,labels,excluded_shaker_labels
0,1,9,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,19,100V;101V;102V;103V;199V;200V;201V;202V;203V;2...,DP1V;DP2V
1,2,8,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,104V;105V;106V;107V;108V;204V;205V;206V;207V;2...,DP1V;DP2V
2,3,7,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,109V;110V;111V;112V;113V;209V;210V;211V;212V;2...,DP1V;DP2V
3,4,6,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,114V;115V;116V;117V;118V;214V;215V;216V;217V;2...,DP1V;DP2V
4,5,5,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,18,119V;120V;121V;122V;123V;219V;220V;221V;222V;2...,DP1V;DP2V
5,6,4,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,124V;125V;126V;127V;128V;224V;225V;226V;227V;2...,DP1V;DP2V
6,7,3,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,129V;130V;131V;132V;133V;229V;230V;231V;232V;2...,DP1V;DP2V
7,8,2,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,134V;135V;136V;137V;138V;234V;235V;236V;237V;2...,DP1V;DP2V
8,9,1,DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL...,20,139V;140V;141V;142V;143V;239V;240V;241V;242V;2...,DP1V;DP2V


,physical_avt_setup,fvt_file_setup,scenario,status,detected_peak_count
0,1,9,03,TRACKED_WITH_EXPLICIT_ABSTENTION,17
1,1,9,04,TRACKED_WITH_EXPLICIT_ABSTENTION,14
2,1,9,05,TRACKED_WITH_EXPLICIT_ABSTENTION,19
3,1,9,06,TRACKED_WITH_EXPLICIT_ABSTENTION,14
4,2,8,03,TRACKED_WITH_EXPLICIT_ABSTENTION,11
5,2,8,04,TRACKED_WITH_EXPLICIT_ABSTENTION,14
6,2,8,05,TRACKED_WITH_EXPLICIT_ABSTENTION,15
7,2,8,06,TRACKED_WITH_EXPLICIT_ABSTENTION,10
8,3,7,03,TRACKED_WITH_EXPLICIT_ABSTENTION,15
9,3,7,04,TRACKED_WITH_EXPLICIT_ABSTENTION,17


,candidate_mode,1,2,3,4,5
scenario,delta_pier_mm,,,,,
01,0.0,3.906250,5.029297,9.912109,10.595703,12.597656
03,20.0,3.857422,4.931641,9.814453,10.424805,12.597656
04,40.0,3.857422,4.833984,9.741211,10.375977,12.597656
05,80.0,3.759766,4.907227,9.326172,10.449219,12.573242
06,95.0,3.686523,4.809570,9.155273,10.424805,12.573242


,scenario,delta_pier_mm,mode,median_frequency_hz,healthy_fvt_frequency_hz,frequency_ratio,sigma_frequency_ratio,ci95_low_ratio,ci95_high_ratio,paired_setup_support
0,01,0.0,1,3.906250,3.906250,1.000000,0.003125,1.000000,1.000000,9
1,01,0.0,3,9.912109,9.912109,1.000000,0.001232,1.000000,1.000000,9
2,01,0.0,4,10.595703,10.595703,1.000000,0.001152,1.000000,1.000000,9
3,03,20.0,1,3.857422,3.906250,0.987500,0.003125,0.987500,0.987500,9
4,03,20.0,3,9.814453,9.912109,0.990148,0.001491,0.987624,0.992593,9
5,03,20.0,4,10.424805,10.595703,0.983871,0.005437,0.977064,0.988453,9
6,04,40.0,1,3.857422,3.906250,0.987500,0.003125,0.987500,0.993750,9
7,04,40.0,3,9.741211,9.912109,0.982759,0.003183,0.972906,0.985185,9
8,04,40.0,4,10.375977,10.595703,0.979263,0.010546,0.970252,0.986079,9
9,05,80.0,1,3.759766,3.906250,0.962500,0.003125,0.962500,0.962500,9


Gate I extraction pass: True


## 33. Gate I locked inverse update, uncertainty and baselines

When FVT modal extraction passes, the AVT-developed trajectory amplitudes \((a,b)\), GPT-PINN observation operator, surrogate uncertainty and model discrepancy are frozen. No FVT target state changes them.

The same FVT observations are also evaluated by two AVT-trained data-only comparators:

- a monotone PCHIP mapping from the AVT mean frequency-drop indicator to \(\xi\);
- the AVT RBF-GP baseline.

The primary evidence is the independently recovered FVT progression coordinate, its uncertainty, modal reconstruction error, ranking, and comparison with the locked baselines.


In [39]:
gate_i_diagnostic = {
    "version": "V0.6 FINAL",
    "gate": "I_Z24_FVT_LOCKED_VALIDATION",
    "execution_mode": gate_i_mode,
    "development_excitation": "AVT",
    "validation_excitation": "FVT",
    "avt_fvt_setup_mapping_status": SETUP_MAPPING_STATUS,
    "avt_to_fvt_setup_map": ACTIVE_AVT_TO_FVT_SETUP,
    "fvt_shaker_channels_excluded": sorted(FVT_SHAKER_CHANNELS),
    "cross_mode_covariance": "paired physical-setup bootstrap covariance with shrinkage",
    "target_refit_performed": False,
    "upstream_model_changed_from_fvt": False,
}

if not gate_i_extraction_pass:
    gate_i_decision = "NO_GO_Z24_FVT_MODAL_EXTRACTION"
    gate_i_diagnostic.update({
        "decision": gate_i_decision,
        "reason": "Insufficient locked FVT modal support for one or more state/mode pairs",
        "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT)
            == RAW_SIGNATURE_BEFORE,
    })
else:
    gate_i_ratio_pivot = gate_i_observations_long.pivot(
        index="delta_pier_mm", columns="mode", values="frequency_ratio"
    ).loc[settlements, retained_modes]
    gate_i_sigma_pivot = gate_i_observations_long.pivot(
        index="delta_pier_mm", columns="mode", values="sigma_frequency_ratio"
    ).loc[settlements, retained_modes]
    gate_i_observed = gate_i_ratio_pivot.to_numpy(dtype=float)
    gate_i_sigma_observation = gate_i_sigma_pivot.to_numpy(dtype=float)
    gate_i_cov_observation = np.stack([
        gate_i_ratio_covariance_by_scenario[str(scenario).zfill(2)].copy()
        for scenario in scenarios
    ])
    for state_index in range(len(gate_i_cov_observation)):
        diagonal_addition = np.maximum(
            gate_i_sigma_observation[state_index]**2
            - np.diag(gate_i_cov_observation[state_index]),
            0.0,
        )
        gate_i_cov_observation[state_index] += np.diag(diagonal_addition)
    gate_i_total_cov = (
        gate_i_cov_observation
        + np.diag(
            gate_g1_discrepancy**2 + gate_g_sigma_surrogate**2 + gate_e_model_discrepancy_sd**2
        )[None, :, :]
    )
    gate_i_total_sigma = np.sqrt(
        np.diagonal(gate_i_total_cov, axis1=1, axis2=2)
    )

    gate_i_independent_xi = np.array([0.0]+[
        gate_g1_invert(
            gate_i_observed[index],
            gate_i_total_cov[index],
            gate_g1_amplitudes,
            lower=0.0,
        )
        for index in range(1, len(settlements))
    ])
    gate_i_causal_xi = [0.0]
    gate_i_point_runtime = [0.0]
    for index in range(1, len(settlements)):
        start = time.perf_counter()
        estimate = gate_g1_invert(
            gate_i_observed[index],
            gate_i_total_cov[index],
            gate_g1_amplitudes,
            lower=gate_i_causal_xi[-1],
        )
        gate_i_point_runtime.append(time.perf_counter()-start)
        gate_i_causal_xi.append(estimate)
    gate_i_causal_xi = np.asarray(gate_i_causal_xi)

    gate_i_modal_prediction = np.vstack([
        gate_g1_forward(xi, gate_g1_amplitudes)
        for xi in gate_i_causal_xi
    ])
    gate_i_modal_rmse = np.sqrt(np.mean(
        (gate_i_modal_prediction-gate_i_observed)**2, axis=1))

    # Locked AVT-trained PCHIP comparator.
    avt_indicator = np.mean(1.0-gate_g_observed, axis=1)
    order = np.argsort(avt_indicator)
    unique_indicator, inverse = np.unique(avt_indicator[order], return_inverse=True)
    unique_truth = np.array([
        np.mean(gate_g1_truth[order][inverse == group_index])
        for group_index in range(len(unique_indicator))
    ])
    gate_i_pchip_model = PchipInterpolator(
        unique_indicator, unique_truth, extrapolate=True)
    gate_i_indicator = np.mean(1.0-gate_i_observed, axis=1)
    gate_i_pchip_xi = np.clip(gate_i_pchip_model(gate_i_indicator), 0.0, 1.0)

    # Locked AVT-trained RBF-GP comparator.
    gate_i_gp_xi = np.array([
        np.clip(gate_g1_gp_prediction(
            gate_g_observed, gate_g1_truth, observation)[0], 0.0, 1.0)
        for observation in gate_i_observed
    ])

    gate_i_bootstrap_samples = np.empty(
        (gate_i_bootstrap_replicates, len(settlements)))
    bootstrap_start = time.perf_counter()
    for replicate in range(gate_i_bootstrap_replicates):
        sampled = gate_i_observed.copy()
        for state_index in range(len(settlements)):
            sampled[state_index] += gate_i_rng.multivariate_normal(
                mean=np.zeros(len(retained_modes)),
                cov=gate_i_total_cov[state_index],
            )
        path = [0.0]
        for index in range(1, len(settlements)):
            path.append(gate_g1_invert(
                sampled[index],
                gate_i_total_cov[index],
                gate_g1_amplitudes,
                lower=path[-1],
            ))
        gate_i_bootstrap_samples[replicate] = path
    gate_i_bootstrap_seconds = time.perf_counter()-bootstrap_start
    gate_i_quantiles = np.percentile(
        gate_i_bootstrap_samples, [2.5, 50.0, 97.5], axis=0)
    half = gate_i_bootstrap_replicates//2
    gate_i_split_difference = float(np.max(np.abs(
        np.percentile(gate_i_bootstrap_samples[:half], [2.5, 50.0, 97.5], axis=0)
        - np.percentile(gate_i_bootstrap_samples[half:], [2.5, 50.0, 97.5], axis=0)
    )))
    gate_i_nonhealthy_mask = gate_g1_truth > 0.0
    gate_i_coverage = np.mean(
        (gate_i_quantiles[0, gate_i_nonhealthy_mask]
         <= gate_g1_truth[gate_i_nonhealthy_mask])
        & (gate_g1_truth[gate_i_nonhealthy_mask]
           <= gate_i_quantiles[2, gate_i_nonhealthy_mask])
    )

    gate_i_result_rows = []
    for index, (scenario, delta) in enumerate(zip(scenarios, settlements)):
        gate_i_result_rows.append({
            "scenario": scenario,
            "delta_pier_mm": delta,
            "true_xi_for_audit_only": gate_g1_truth[index],
            "locked_independent_xi": gate_i_independent_xi[index],
            "locked_causal_xi": gate_i_causal_xi[index],
            "physics_absolute_xi_error": abs(
                gate_i_causal_xi[index]-gate_g1_truth[index]),
            "pchip_xi": gate_i_pchip_xi[index],
            "pchip_absolute_xi_error": abs(
                gate_i_pchip_xi[index]-gate_g1_truth[index]),
            "rbf_gp_xi": gate_i_gp_xi[index],
            "rbf_gp_absolute_xi_error": abs(
                gate_i_gp_xi[index]-gate_g1_truth[index]),
            "modal_ratio_rmse": gate_i_modal_rmse[index],
            "ci95_lower_xi": gate_i_quantiles[0, index],
            "ci95_median_xi": gate_i_quantiles[1, index],
            "ci95_upper_xi": gate_i_quantiles[2, index],
            "ci95_covers_true_xi": bool(
                gate_i_quantiles[0, index] <= gate_g1_truth[index]
                <= gate_i_quantiles[2, index]),
            "point_inverse_seconds": gate_i_point_runtime[index],
        })
    gate_i_results = pd.DataFrame(gate_i_result_rows)
    gate_i_results.to_csv(
        TABLE_ROOT/"gate_i_fvt_locked_inverse_results.csv", index=False)
    pd.DataFrame({
        "replicate": np.repeat(
            np.arange(gate_i_bootstrap_replicates), len(settlements)),
        "scenario": np.tile(np.asarray(scenarios), gate_i_bootstrap_replicates),
        "delta_pier_mm": np.tile(settlements, gate_i_bootstrap_replicates),
        "sampled_xi": gate_i_bootstrap_samples.ravel(),
    }).to_csv(TABLE_ROOT/"gate_i_fvt_bootstrap_samples.csv", index=False)
    display(gate_i_results)

    nonhealthy = gate_i_results["delta_pier_mm"] > 0.0
    gate_i_physics_mae = float(gate_i_results.loc[
        nonhealthy, "physics_absolute_xi_error"].mean())
    gate_i_pchip_mae = float(gate_i_results.loc[
        nonhealthy, "pchip_absolute_xi_error"].mean())
    gate_i_gp_mae = float(gate_i_results.loc[
        nonhealthy, "rbf_gp_absolute_xi_error"].mean())
    gate_i_best_baseline_mae = min(gate_i_pchip_mae, gate_i_gp_mae)
    gate_i_spearman = float(stats.spearmanr(
        gate_g1_truth[nonhealthy.to_numpy()],
        gate_i_causal_xi[nonhealthy.to_numpy()],
    ).statistic)

    gate_i_checks = {
        "full_execution": gate_i_mode == "full",
        "locked_no_target_refit": True,
        "minimum_setup_support_met": bool(
            (gate_i_observations_long["paired_setup_support"]
             >= CONFIG["gate_i_minimum_setup_support"]).all()),
        "causal_xi_non_decreasing": bool(
            np.all(np.diff(gate_i_causal_xi) >= -1e-9)),
        "spearman_ge_0p90": bool(
            gate_i_spearman >= CONFIG["gate_i_min_spearman"]),
        "mean_absolute_xi_error_le_0p15": bool(
            gate_i_physics_mae <= CONFIG["gate_i_max_mean_absolute_xi_error"]),
        "maximum_absolute_xi_error_le_0p25": bool(
            gate_i_results.loc[nonhealthy, "physics_absolute_xi_error"].max()
            <= CONFIG["gate_i_max_absolute_xi_error"]),
        "maximum_modal_ratio_rmse_le_2p5pct": bool(
            gate_i_modal_rmse.max() <= CONFIG["gate_i_max_modal_ratio_rmse"]),
        "ci95_coverage_ge_0p80": bool(
            gate_i_coverage >= CONFIG["gate_i_min_ci_coverage"]),
        "bootstrap_split_difference_le_0p10": bool(
            gate_i_split_difference
            <= CONFIG["gate_i_max_bootstrap_split_difference"]),
        "physics_not_materially_worse_than_best_locked_baseline": bool(
            gate_i_physics_mae-gate_i_best_baseline_mae
            <= CONFIG["gate_i_max_physics_excess_mae_over_best_baseline"]),
        "raw_z24_signature_unchanged": raw_metadata_signature(DATASET_ROOT)
            == RAW_SIGNATURE_BEFORE,
    }
    if gate_i_mode == "smoke":
        gate_i_decision = "SMOKE_COMPLETED_REQUIRES_FULL_RUN"
    else:
        gate_i_decision = (
            "GO_Z24_FVT_LOCKED_VALIDATION_V0_6_FINAL"
            if all(gate_i_checks.values())
            else "NO_GO_Z24_FVT_EXTERNAL_VALIDATION"
        )

    gate_i_diagnostic.update({
        "decision": gate_i_decision,
        "retained_modes": retained_modes,
        "trajectory_amplitudes_source": "AVT Gate G.1; frozen",
        "trajectory_amplitudes_a_b": gate_g1_amplitudes.tolist(),
        "bootstrap_replicates": gate_i_bootstrap_replicates,
        "bootstrap_runtime_seconds": gate_i_bootstrap_seconds,
        "maximum_point_inverse_seconds": float(max(gate_i_point_runtime)),
        "spearman_nonhealthy": gate_i_spearman,
        "physics_mean_absolute_xi_error_nonhealthy": gate_i_physics_mae,
        "pchip_mean_absolute_xi_error_nonhealthy": gate_i_pchip_mae,
        "rbf_gp_mean_absolute_xi_error_nonhealthy": gate_i_gp_mae,
        "maximum_physics_absolute_xi_error_nonhealthy": float(
            gate_i_results.loc[nonhealthy, "physics_absolute_xi_error"].max()),
        "maximum_modal_ratio_rmse": float(gate_i_modal_rmse.max()),
        "ci95_coverage_fraction_nonhealthy": float(gate_i_coverage),
        "maximum_bootstrap_split_quantile_difference": gate_i_split_difference,
        "checks": gate_i_checks,
        "limitations": [
            "FVT is a different excitation regime of the same bridge, not a second bridge",
            "FVT healthy normalization uses scenario 01 and therefore does not test absolute frequency scale transfer",
            "experimental mode identities remain rank-based because assembled global mode shapes are unavailable",
            "the AVT-developed one-dimensional trajectory remains a model-form assumption",
        ],
    })

(OUTPUT_ROOT/"gate_i_diagnostic.json").write_text(
    json_dumps_safe(gate_i_diagnostic, indent=2, sort_keys=True)+"\n",
    encoding="utf-8",
)

if gate_i_extraction_pass:
    fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.8))
    truth = gate_g1_truth
    median = gate_i_quantiles[1]
    lower = median-gate_i_quantiles[0]
    upper = gate_i_quantiles[2]-median
    axes[0].errorbar(
        truth, median, yerr=np.vstack([lower, upper]),
        fmt="o-", capsize=4, color="#e76f51",
        label="locked AVT→FVT physics update",
    )
    axes[0].scatter(
        truth, gate_i_pchip_xi, marker="s", color="#2a9d8f",
        label="locked AVT PCHIP",
    )
    axes[0].scatter(
        truth, gate_i_gp_xi, marker="^", color="#6d597a",
        label="locked AVT RBF-GP",
    )
    axes[0].plot([0, 1], [0, 1], "--", color="#555555", label="ideal")
    axes[0].set(
        xlabel=r"FVT true $\xi=\delta/95$ (audit only)",
        ylabel=r"Locked estimate $\hat{\xi}$",
        title="AVT-developed model applied to FVT without refitting",
    )
    axes[0].legend(fontsize=8)

    for mode_index, (mode, color) in enumerate(zip(
        gate_g_experimental_modes, ["#264653", "#2a9d8f", "#e76f51"]
    )):
        axes[1].plot(
            settlements,
            gate_i_observed[:, mode_index],
            "o", color=color, label=f"FVT observed mode {mode}",
        )
        axes[1].plot(
            settlements,
            gate_i_modal_prediction[:, mode_index],
            "-", color=color, label=f"locked physics mode {mode}",
        )
    axes[1].set(
        xlabel="Controlled settlement [mm]",
        ylabel="FVT healthy-normalized frequency ratio",
        title="Locked modal reconstruction",
    )
    axes[1].legend(ncol=2, fontsize=7)
    fig.suptitle(
        f"Figure 28 — Gate I: {gate_i_decision.replace('_', ' ')}",
        weight="bold",
    )
    fig.tight_layout()
    for suffix in ("png", "pdf"):
        fig.savefig(
            FIGURE_ROOT/f"figure_28_gate_i_fvt_locked_validation.{suffix}",
            bbox_inches="tight",
        )
    plt.show()
    plt.close("all")

print(json_dumps_safe(gate_i_diagnostic, indent=2))

,scenario,delta_pier_mm,true_xi_for_audit_only,locked_independent_xi,locked_causal_xi,physics_absolute_xi_error,pchip_xi,pchip_absolute_xi_error,rbf_gp_xi,rbf_gp_absolute_xi_error,modal_ratio_rmse,ci95_lower_xi,ci95_median_xi,ci95_upper_xi,ci95_covers_true_xi,point_inverse_seconds
0,01,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,True,0.000000
1,03,20.0,0.210526,0.257330,0.257330,0.046804,0.138735,0.071791,0.110614,0.099912,0.000752,0.002327,0.253505,0.459964,True,0.004516
2,04,40.0,0.421053,0.333389,0.333389,0.087664,0.239603,0.181450,0.272506,0.148547,0.002227,0.166211,0.353488,0.540822,True,0.005647
3,05,80.0,0.842105,0.817386,0.817386,0.024719,0.684735,0.157370,0.000000,0.842105,0.025682,0.667149,0.817078,0.941845,True,0.004158
4,06,95.0,1.000000,0.946172,0.946172,0.053828,0.795013,0.204987,0.000000,1.000000,0.029994,0.832449,0.949618,1.000000,True,0.004114


{
  "version": "V0.6 FINAL",
  "gate": "I_Z24_FVT_LOCKED_VALIDATION",
  "execution_mode": "full",
  "development_excitation": "AVT",
  "validation_excitation": "FVT",
  "avt_fvt_setup_mapping_status": "DOCUMENTED_REVERSED_MAPPING_CONFIRMED_BY_LABEL_OVERLAP",
  "avt_to_fvt_setup_map": {
    "1": 9,
    "2": 8,
    "3": 7,
    "4": 6,
    "5": 5,
    "6": 4,
    "7": 3,
    "8": 2,
    "9": 1
  },
  "fvt_shaker_channels_excluded": [
    "DP1V",
    "DP2V"
  ],
  "cross_mode_covariance": "paired physical-setup bootstrap covariance with shrinkage",
  "target_refit_performed": false,
  "upstream_model_changed_from_fvt": false,
  "decision": "NO_GO_Z24_FVT_EXTERNAL_VALIDATION",
  "retained_modes": [
    1,
    3,
    4
  ],
  "trajectory_amplitudes_source": "AVT Gate G.1; frozen",
  "trajectory_amplitudes_a_b": [
    0.5603365657142761,
    1.8682670486346296
  ],
  "bootstrap_replicates": 1000,
  "bootstrap_runtime_seconds": 24.522582646925002,
  "maximum_point_inverse_seconds": 0.005646625

## 34. Final protocol freeze and reproducibility contract

This section hashes the final protocol and verifies that the scientific quantities agreed before the final run have not drifted. It does not tune any model and does not alter the FVT threshold.


In [40]:
FINAL_PROTOCOL = {
    "article_title": CONFIG["final_article_title"],
    "primary_benchmark": "Z24",
    "development_excitation": "AVT",
    "locked_cross_excitation_audit": "FVT",
    "primary_scenarios": list(CONFIG["primary_scenarios"]),
    "controlled_settlement_mm": [float(value) for value in CONFIG["pier_settlement_mm"]],
    "retained_experimental_modes": [int(value) for value in retained_modes],
    "span_lengths_m": [14.0, 30.0, 14.0],
    "total_length_m": 58.0,
    "avt_to_fvt_physical_setup_map": {
        str(key): int(value) for key, value in ACTIVE_AVT_TO_FVT_SETUP.items()
    },
    "fvt_driving_point_channels_excluded": sorted(FVT_SHAKER_CHANNELS),
    "joint_modal_covariance": "paired physical-setup bootstrap with shrinkage and resolution floor",
    "gate_e_model_discrepancy_cap": float(CONFIG["gate_e_max_model_discrepancy_sd"]),
    "gate_i_modal_ratio_rmse_threshold": float(CONFIG["gate_i_max_modal_ratio_rmse"]),
    "gate_h_role": "supplementary conditional response propagation only",
    "gate_i_role": "locked no-target-refit cross-excitation audit; result is preserved even if NO-GO",
    "prohibited_claims": [
        "actual material stiffness loss",
        "load-carrying capacity",
        "autonomous natural deterioration",
        "calendar-time remaining useful life",
        "structural failure probability",
        "maintenance cost savings",
        "optimal maintenance policy",
        "cross-bridge generalization",
    ],
}
FINAL_PROTOCOL_JSON = json_dumps_safe(
    FINAL_PROTOCOL, sort_keys=True, separators=(",", ":"), ensure_ascii=False
)
FINAL_PROTOCOL_SHA256 = hashlib.sha256(
    FINAL_PROTOCOL_JSON.encode("utf-8")
).hexdigest()

expected_setup_map = {1: 9, 2: 8, 3: 7, 4: 6, 5: 5, 6: 4, 7: 3, 8: 2, 9: 1}
final_protocol_checks = {
    "primary_scenarios_frozen":
        list(CONFIG["primary_scenarios"]) == ["01", "03", "04", "05", "06"],
    "settlement_sequence_frozen":
        np.allclose(CONFIG["pier_settlement_mm"], [0.0, 20.0, 40.0, 80.0, 95.0]),
    "retained_modes_frozen":
        [int(value) for value in retained_modes] == [1, 3, 4],
    "three_span_geometry_frozen":
        np.isclose(sum(FINAL_PROTOCOL["span_lengths_m"]), 58.0),
    "avt_fvt_mapping_frozen":
        {int(key): int(value) for key, value in ACTIVE_AVT_TO_FVT_SETUP.items()}
        == expected_setup_map,
    "fvt_shakers_excluded":
        set(FVT_SHAKER_CHANNELS) == {"DP1V", "DP2V"},
    "gate_e_discrepancy_cap_frozen":
        np.isclose(CONFIG["gate_e_max_model_discrepancy_sd"], 0.01),
    "gate_i_threshold_frozen":
        np.isclose(CONFIG["gate_i_max_modal_ratio_rmse"], 0.025),
    "raw_z24_metadata_unchanged":
        raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE,
}
final_protocol_checks = {
    str(name): bool(value)
    for name, value in final_protocol_checks.items()
}
if not all(final_protocol_checks.values()):
    raise RuntimeError(
        "V0.6 FINAL protocol-freeze violation: "
        + json_dumps_safe(final_protocol_checks, sort_keys=True)
    )

protocol_record = {
    "version": VERSION,
    "patch_level": PATCH_LEVEL,
    "protocol": FINAL_PROTOCOL,
    "protocol_sha256": FINAL_PROTOCOL_SHA256,
    "checks": final_protocol_checks,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "V0_6_FINAL_FROZEN_PROTOCOL.json").write_text(
    json_dumps_safe(protocol_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Frozen protocol SHA-256:", FINAL_PROTOCOL_SHA256)
print(json_dumps_safe(final_protocol_checks, indent=2, sort_keys=True))


Frozen protocol SHA-256: 23eb6a54d0683384c7a7b4c63bfa23224019391623d1c6afa8c46f50fec765dc
{
  "avt_fvt_mapping_frozen": true,
  "fvt_shakers_excluded": true,
  "gate_e_discrepancy_cap_frozen": true,
  "gate_i_threshold_frozen": true,
  "primary_scenarios_frozen": true,
  "raw_z24_metadata_unchanged": true,
  "retained_modes_frozen": true,
  "settlement_sequence_frozen": true,
  "three_span_geometry_frozen": true
}


## 35. Publication-grade online timing and cost amortization

The benchmark separates:

- **offline cost**: construction of the Full-PINN basis;
- **online GPT-PINN query cost**;
- **online FE reference-query cost**;
- **Full-PINN retraining cost per new state**.

Timing is repeated after warm-up. The publication timing certificate is valid only for a full run inside a Slurm allocation on a non-login compute node.


In [41]:
def final_sync_device() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def final_benchmark_callable(callable_object, repeats: int, warmup: int) -> tuple[dict, np.ndarray]:
    if repeats < 1:
        raise ValueError("repeats must be positive")
    for _ in range(max(0, warmup)):
        callable_object()
    final_sync_device()
    gc.collect()
    timings_ns = np.empty(repeats, dtype=np.int64)
    gc_was_enabled = gc.isenabled()
    gc.disable()
    try:
        for index in range(repeats):
            final_sync_device()
            start_ns = time.perf_counter_ns()
            callable_object()
            final_sync_device()
            timings_ns[index] = time.perf_counter_ns() - start_ns
    finally:
        if gc_was_enabled:
            gc.enable()
    timings_s = timings_ns.astype(np.float64) * 1e-9
    statistics = {
        "repeats": int(repeats),
        "warmup_calls": int(warmup),
        "mean_seconds": float(np.mean(timings_s)),
        "std_seconds": float(np.std(timings_s, ddof=1)) if repeats > 1 else 0.0,
        "median_seconds": float(np.median(timings_s)),
        "p05_seconds": float(np.quantile(timings_s, 0.05)),
        "p95_seconds": float(np.quantile(timings_s, 0.95)),
        "minimum_seconds": float(np.min(timings_s)),
        "maximum_seconds": float(np.max(timings_s)),
    }
    return statistics, timings_s

benchmark_points = [
    (0.00, 0.00),
    (0.1625, -1.50),
    (0.3250, -1.00),
    (0.4875, -0.50),
    (0.6500, -2.00),
]
gpt_repeats = (
    CONFIG["final_timing_gpt_repeats_full"]
    if RUN_PROFILE == "full"
    else CONFIG["final_timing_gpt_repeats_smoke"]
)
fe_repeats = (
    CONFIG["final_timing_fe_repeats_full"]
    if RUN_PROFILE == "full"
    else CONFIG["final_timing_fe_repeats_smoke"]
)
timing_rows = []
timing_sample_rows = []
gpt_timing_method_label = (
    "REDUCED_FE_PROXY_SMOKE"
    if GATE_F_SMOKE_PROXY_USED
    else "GPT_PINN"
)

for point_index, (alpha, log_ratio) in enumerate(benchmark_points):
    gpt_stats, gpt_samples = final_benchmark_callable(
        lambda alpha=alpha, log_ratio=log_ratio: gate_f_gpt_query(
            gate_f_basis, alpha, log_ratio
        ),
        repeats=gpt_repeats,
        warmup=CONFIG["final_timing_warmup_calls"],
    )
    fe_stats, fe_samples = final_benchmark_callable(
        lambda alpha=alpha, log_ratio=log_ratio: gate_f_fe_reference(
            alpha, log_ratio, n_elements=116
        ),
        repeats=fe_repeats,
        warmup=max(5, CONFIG["final_timing_warmup_calls"] // 3),
    )
    for method, statistics in (
        (gpt_timing_method_label, gpt_stats),
        ("FE_116", fe_stats),
    ):
        timing_rows.append({
            "point_index": point_index,
            "alpha": alpha,
            "log10_support_ratio": log_ratio,
            "method": method,
            **statistics,
        })
    timing_sample_rows.extend({
        "point_index": point_index,
        "alpha": alpha,
        "log10_support_ratio": log_ratio,
        "method": gpt_timing_method_label,
        "replicate": replicate,
        "seconds": float(value),
    } for replicate, value in enumerate(gpt_samples))
    timing_sample_rows.extend({
        "point_index": point_index,
        "alpha": alpha,
        "log10_support_ratio": log_ratio,
        "method": "FE_116",
        "replicate": replicate,
        "seconds": float(value),
    } for replicate, value in enumerate(fe_samples))

final_timing_by_point = pd.DataFrame(timing_rows)
final_timing_samples = pd.DataFrame(timing_sample_rows)
final_timing_by_point.to_csv(
    TABLE_ROOT / "V0_6_FINAL_timing_by_parameter_point.csv", index=False
)
final_timing_samples.to_csv(
    TABLE_ROOT / "V0_6_FINAL_timing_raw_samples.csv", index=False
)

gpt_median_seconds = float(
    final_timing_by_point.loc[
        final_timing_by_point["method"] == gpt_timing_method_label, "median_seconds"
    ].median()
)
fe_median_seconds = float(
    final_timing_by_point.loc[
        final_timing_by_point["method"] == "FE_116", "median_seconds"
    ].median()
)
full_pinn_training_times = np.asarray(
    [float(state["runtime_seconds"]) for state in gate_f_states], dtype=float
)
if GATE_F_SMOKE_PROXY_USED:
    # The FE proxy permits code-path validation only. Its construction time
    # must not be reported as Full-PINN training or used for amortization.
    offline_basis_seconds = np.nan
    mean_full_pinn_training_seconds = np.nan
    median_full_pinn_training_seconds = np.nan
    smoke_proxy_basis_generation_seconds = float(
        np.sum(full_pinn_training_times)
    )
else:
    offline_basis_seconds = float(np.sum(full_pinn_training_times))
    mean_full_pinn_training_seconds = float(np.mean(full_pinn_training_times))
    median_full_pinn_training_seconds = float(np.median(full_pinn_training_times))
    smoke_proxy_basis_generation_seconds = np.nan

def final_break_even_queries(
    offline_seconds: float,
    baseline_online_seconds: float,
    proposed_online_seconds: float,
) -> int | None:
    savings = baseline_online_seconds - proposed_online_seconds
    if (
        not np.isfinite(offline_seconds)
        or offline_seconds < 0.0
        or not np.isfinite(savings)
        or savings <= 0.0
    ):
        return None
    return int(np.ceil(offline_seconds / savings))

break_even_vs_fe = final_break_even_queries(
    offline_basis_seconds, fe_median_seconds, gpt_median_seconds
)
break_even_vs_full_pinn = final_break_even_queries(
    offline_basis_seconds, median_full_pinn_training_seconds, gpt_median_seconds
)
speedup_vs_fe = fe_median_seconds / gpt_median_seconds
speedup_vs_full_pinn = median_full_pinn_training_seconds / gpt_median_seconds

query_grid = sorted(set(
    [1, 10, 100, 1_000, 10_000, 100_000, 1_000_000]
    + ([break_even_vs_fe] if break_even_vs_fe is not None else [])
))
amortization_rows = []
for query_count in query_grid:
    if GATE_F_SMOKE_PROXY_USED:
        amortization_rows.extend([
            {
                "query_count": int(query_count),
                "method": "REDUCED_FE_PROXY_SMOKE",
                "total_seconds": query_count * gpt_median_seconds,
                "offline_seconds": np.nan,
                "online_seconds": query_count * gpt_median_seconds,
            },
            {
                "query_count": int(query_count),
                "method": "FE_116",
                "total_seconds": query_count * fe_median_seconds,
                "offline_seconds": 0.0,
                "online_seconds": query_count * fe_median_seconds,
            },
        ])
    else:
        amortization_rows.extend([
            {
                "query_count": int(query_count),
                "method": "GPT_PINN",
                "total_seconds": offline_basis_seconds + query_count * gpt_median_seconds,
                "offline_seconds": offline_basis_seconds,
                "online_seconds": query_count * gpt_median_seconds,
            },
            {
                "query_count": int(query_count),
                "method": "FE_116",
                "total_seconds": query_count * fe_median_seconds,
                "offline_seconds": 0.0,
                "online_seconds": query_count * fe_median_seconds,
            },
            {
                "query_count": int(query_count),
                "method": "Full_PINN_retraining",
                "total_seconds": query_count * median_full_pinn_training_seconds,
                "offline_seconds": 0.0,
                "online_seconds": query_count * median_full_pinn_training_seconds,
            },
        ])
final_amortization = pd.DataFrame(amortization_rows)
final_amortization.to_csv(
    TABLE_ROOT / "V0_6_FINAL_cost_amortization.csv", index=False
)

timing_compute_certified = bool(
    RUN_PROFILE == "full"
    and IN_SLURM_ALLOCATION
    and not IS_LOGIN_NODE
    and not ALLOW_LOGIN_NODE_HEAVY
)
final_timing_checks = {
    "timings_finite_and_positive": bool(
        np.isfinite(final_timing_samples["seconds"]).all()
        and (final_timing_samples["seconds"] > 0.0).all()
    ),
    "speedup_vs_fe_ge_10": bool(
        speedup_vs_fe >= CONFIG["final_timing_min_speedup_vs_fe"]
    ),
    "offline_cost_reported": bool(
        (not GATE_F_SMOKE_PROXY_USED)
        and np.isfinite(offline_basis_seconds)
        and offline_basis_seconds > 0.0
    ),
    "real_full_pinn_basis_used": bool(not GATE_F_SMOKE_PROXY_USED),
    "full_profile": RUN_PROFILE == "full",
    "slurm_compute_node_certified": timing_compute_certified,
}
final_timing_summary = {
    "version": VERSION,
    "timing_compute_certified": timing_compute_certified,
    "benchmark_parameter_points": benchmark_points,
    "gpt_query_median_seconds_across_points": gpt_median_seconds,
    "fe_116_query_median_seconds_across_points": fe_median_seconds,
    "full_pinn_training_median_seconds": (
        None if GATE_F_SMOKE_PROXY_USED
        else median_full_pinn_training_seconds
    ),
    "offline_basis_construction_seconds": (
        None if GATE_F_SMOKE_PROXY_USED
        else offline_basis_seconds
    ),
    "smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "smoke_fe_proxy_generation_seconds": (
        smoke_proxy_basis_generation_seconds
        if GATE_F_SMOKE_PROXY_USED
        else None
    ),
    "speedup_vs_fe_116": speedup_vs_fe,
    "speedup_vs_full_pinn_retraining": (
        None if GATE_F_SMOKE_PROXY_USED else speedup_vs_full_pinn
    ),
    "break_even_queries_vs_fe_116": (
        None if GATE_F_SMOKE_PROXY_USED else break_even_vs_fe
    ),
    "break_even_queries_vs_full_pinn_retraining": break_even_vs_full_pinn,
    "checks": final_timing_checks,
    "hostname": HOSTNAME,
    "slurm_job_id": SLURM_JOB_ID,
    "cpu_threads": EFFECTIVE_CPU_THREADS,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "V0_6_FINAL_TIMING_SUMMARY.json").write_text(
    json_dumps_safe(final_timing_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(final_timing_by_point)
print(json_dumps_safe(final_timing_summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.7))
if GATE_F_SMOKE_PROXY_USED:
    method_order = ["REDUCED_FE_PROXY_SMOKE", "FE_116"]
    plot_values = [
        gpt_median_seconds,
        fe_median_seconds,
        max(smoke_proxy_basis_generation_seconds, 1e-12),
    ]
    bar_labels = [
        "Reduced FE proxy\nonline",
        "FE-116\nonline",
        "Proxy basis\nbuild",
    ]
else:
    method_order = ["GPT_PINN", "FE_116"]
    plot_values = [
        gpt_median_seconds,
        fe_median_seconds,
        median_full_pinn_training_seconds,
    ]
    bar_labels = [
        "GPT-PINN\nonline",
        "FE-116\nonline",
        "Full-PINN\ntraining",
    ]
axes[0].bar(bar_labels, plot_values)
axes[0].set_yscale("log")
axes[0].set_ylabel("Median wall time per operation [s]")
axes[0].set_title(
    f"Online speed-up vs FE: {speedup_vs_fe:.1f}×"
)

amortization_methods = (
    ("REDUCED_FE_PROXY_SMOKE", "FE_116")
    if GATE_F_SMOKE_PROXY_USED
    else ("GPT_PINN", "FE_116", "Full_PINN_retraining")
)
for method in amortization_methods:
    group = final_amortization[final_amortization["method"] == method]
    axes[1].loglog(
        group["query_count"], group["total_seconds"], "o-", label=method
    )
if break_even_vs_fe is not None and not GATE_F_SMOKE_PROXY_USED:
    axes[1].axvline(
        break_even_vs_fe, linestyle="--",
        label=f"GPT/FE break-even ≈ {break_even_vs_fe:,} queries"
    )
axes[1].set_xlabel("Number of repeated Digital-Twin queries")
axes[1].set_ylabel("Cumulative wall time [s]")
axes[1].set_title("Offline/online cost amortization")
axes[1].legend(fontsize=8)
fig.suptitle("V0.6 FINAL — reproducible timing and amortization", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT / f"figure_29_final_timing_amortization.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


,point_index,alpha,log10_support_ratio,method,repeats,warmup_calls,mean_seconds,std_seconds,median_seconds,p05_seconds,p95_seconds,minimum_seconds,maximum_seconds
0,0,0.0000,0.0,GPT_PINN,2000,30,0.000137,0.000008,0.000135,0.000134,0.000145,0.000132,0.000426
1,0,0.0000,0.0,FE_116,200,10,0.180303,0.651797,0.009317,0.008199,2.201248,0.007910,2.957200
2,1,0.1625,-1.5,GPT_PINN,2000,30,0.000141,0.000009,0.000140,0.000135,0.000150,0.000133,0.000476
3,1,0.1625,-1.5,FE_116,200,10,0.009303,0.000782,0.009068,0.008600,0.011127,0.008474,0.013780
4,2,0.3250,-1.0,GPT_PINN,2000,30,0.000135,0.000008,0.000133,0.000132,0.000144,0.000131,0.000452
5,2,0.3250,-1.0,FE_116,200,10,0.183129,0.639003,0.009579,0.008718,1.493500,0.008308,2.955414
6,3,0.4875,-0.5,GPT_PINN,2000,30,0.000136,0.000078,0.000133,0.000131,0.000142,0.000130,0.003598
7,3,0.4875,-0.5,FE_116,200,10,0.125165,0.552028,0.009325,0.007956,0.066646,0.007683,2.953016
8,4,0.6500,-2.0,GPT_PINN,2000,30,0.000124,0.000015,0.000121,0.000120,0.000135,0.000119,0.000697
9,4,0.6500,-2.0,FE_116,200,10,0.011462,0.011064,0.008446,0.008083,0.034984,0.007856,0.099073


{
  "version": "V0_6_FINAL",
  "timing_compute_certified": true,
  "benchmark_parameter_points": [
    [
      0.0,
      0.0
    ],
    [
      0.1625,
      -1.5
    ],
    [
      0.325,
      -1.0
    ],
    [
      0.4875,
      -0.5
    ],
    [
      0.65,
      -2.0
    ]
  ],
  "gpt_query_median_seconds_across_points": 0.000133049,
  "fe_116_query_median_seconds_across_points": 0.009317223999999999,
  "full_pinn_training_median_seconds": 305.95878987619653,
  "offline_basis_construction_seconds": 921.4700424701441,
  "smoke_fe_proxy_used": false,
  "smoke_fe_proxy_generation_seconds": null,
  "speedup_vs_fe_116": 70.02851580996474,
  "speedup_vs_full_pinn_retraining": 2299594.8100038073,
  "break_even_queries_vs_fe_116": 100333,
  "break_even_queries_vs_full_pinn_retraining": 4,
  "checks": {
    "timings_finite_and_positive": true,
    "speedup_vs_fe_ge_10": true,
    "offline_cost_reported": true,
    "real_full_pinn_basis_used": true,
    "full_profile": true,
    "slurm_co

## 36. Final ablation study

The ablation is diagnostic, not a new tuning round.

- **Basis-size/order:** greedy prefixes are compared with a geometry-spread ordering of the **same already-trained state pool**. This does not pretend to be an independently trained random-design baseline.
- **Observation model:** full versus diagonal modal covariance, with and without model discrepancy.
- **Mode content:** mode subsets are evaluated in strict leave-one-severity-out point inference.

FVT is not used to choose any ablation configuration.


In [42]:
def final_basis_validation_metrics(
    states: list[dict],
    order_name: str,
    basis_size: int,
) -> dict:
    basis = gate_f_build_reduced_basis(states, gate_f_s_values)
    rows = []
    for alpha in alpha_midpoints:
        for log_ratio in support_midpoints:
            result = gate_f_gpt_query(basis, float(alpha), float(log_ratio))
            rows.extend(
                gate_f_validate_gpt_query(
                    result, basis, float(alpha), float(log_ratio)
                )
            )
    frame = pd.DataFrame(rows)
    return {
        "ordering": order_name,
        "basis_state_count": int(basis_size),
        "reduced_rank": int(basis["rank"]),
        "maximum_frequency_relative_error": float(
            frame["frequency_relative_error"].max()
        ),
        "mean_frequency_relative_error": float(
            frame["frequency_relative_error"].mean()
        ),
        "minimum_mac": float(frame["displacement_mac"].min()),
        "maximum_weak_residual": float(frame["weak_residual_max"].max()),
        "basis_mass_orthogonality_error": float(
            basis["mass_orthogonality_error"]
        ),
    }

greedy_indices = list(range(len(gate_f_states)))
parameter_coordinates = np.asarray(gate_f_selected_parameters, dtype=float)
parameter_scale = np.asarray([
    CONFIG["gate_f_alpha_domain"][1] - CONFIG["gate_f_alpha_domain"][0],
    CONFIG["gate_f_log10_support_domain"][1]
    - CONFIG["gate_f_log10_support_domain"][0],
])
normalized_coordinates = parameter_coordinates / parameter_scale
spread_indices = [0]
remaining = set(range(1, len(gate_f_states)))
while remaining:
    next_index = max(
        remaining,
        key=lambda candidate: min(
            np.linalg.norm(
                normalized_coordinates[candidate]
                - normalized_coordinates[selected]
            )
            for selected in spread_indices
        ),
    )
    spread_indices.append(next_index)
    remaining.remove(next_index)

basis_ablation_rows = []
for ordering_name, ordering in (
    ("greedy_prefix", greedy_indices),
    ("same_pool_geometry_spread_order", spread_indices),
):
    for basis_size in range(1, len(ordering) + 1):
        selected_states = [gate_f_states[index] for index in ordering[:basis_size]]
        basis_ablation_rows.append(
            final_basis_validation_metrics(
                selected_states, ordering_name, basis_size
            )
        )
final_basis_ablation = pd.DataFrame(basis_ablation_rows)
final_basis_ablation.to_csv(
    TABLE_ROOT / "V0_6_FINAL_basis_ablation.csv", index=False
)
display(final_basis_ablation)

def final_subset_covariances(
    covariance_cube: np.ndarray,
    mode_indices: list[int],
    policy: str,
) -> np.ndarray:
    selected = np.take(
        np.take(covariance_cube, mode_indices, axis=1),
        mode_indices,
        axis=2,
    ).copy()
    if policy == "diagonal":
        selected = np.stack([
            np.diag(np.diag(covariance)) for covariance in selected
        ])
    elif policy != "full":
        raise ValueError(f"Unknown covariance policy: {policy}")
    return selected

def final_fit_amplitudes_for_ablation(
    training_indices: list[int],
    observations: np.ndarray,
    covariances: np.ndarray,
    mode_indices: list[int],
):
    starts = (
        np.array([0.40, 1.00]),
        np.array([0.60, 1.70]),
        np.array([0.64, 1.95]),
    )
    def residual(amplitudes):
        return np.concatenate([
            whiten_modal_residual(
                gate_g1_forward(
                    gate_g1_truth[index], amplitudes
                )[mode_indices] - observations[index],
                covariances[index],
            )
            for index in training_indices
        ])
    fits = [
        least_squares(
            residual,
            start,
            bounds=(gate_g1_amplitude_lower, gate_g1_amplitude_upper),
            method="trf",
            max_nfev=400,
            xtol=1e-11,
            ftol=1e-11,
            gtol=1e-11,
        )
        for start in starts
    ]
    best = min(fits, key=lambda fit: float(fit.fun @ fit.fun))
    return best, residual(best.x)

def final_loso_ablation(
    label: str,
    mode_indices: list[int],
    covariance_policy: str,
    include_model_discrepancy: bool,
) -> tuple[dict, pd.DataFrame]:
    observations = gate_g_observed[:, mode_indices]
    base_covariances = final_subset_covariances(
        gate_g1_base_cov, mode_indices, covariance_policy
    )
    fold_rows = []
    for heldout_index in range(1, len(gate_g1_truth)):
        training_indices = [
            index for index in range(1, len(gate_g1_truth))
            if index != heldout_index
        ]
        fit, training_residual = final_fit_amplitudes_for_ablation(
            training_indices,
            observations,
            base_covariances,
            mode_indices,
        )
        raw_training_residuals = np.vstack([
            gate_g1_forward(
                gate_g1_truth[index], fit.x
            )[mode_indices] - observations[index]
            for index in training_indices
        ])
        inflation = np.sqrt(
            (len(training_indices) + 1)
            / max(1, len(training_indices) - 1)
        )
        discrepancy = (
            np.sqrt(np.mean(raw_training_residuals ** 2, axis=0)) * inflation
            if include_model_discrepancy
            else np.zeros(len(mode_indices), dtype=float)
        )
        heldout_covariance = (
            base_covariances[heldout_index]
            + np.diag(discrepancy ** 2)
        )
        def objective(xi):
            residual = whiten_modal_residual(
                gate_g1_forward(float(xi), fit.x)[mode_indices]
                - observations[heldout_index],
                heldout_covariance,
            )
            return float(residual @ residual)
        result = minimize_scalar(
            objective,
            bounds=(0.0, 1.0),
            method="bounded",
            options={"xatol": 1e-10, "maxiter": 200},
        )
        candidates = [
            (float(result.fun), float(result.x)),
            (objective(0.0), 0.0),
            (objective(1.0), 1.0),
        ]
        estimate = min(candidates)[1]
        fold_rows.append({
            "ablation": label,
            "heldout_delta_pier_mm": float(gate_g_deltas[heldout_index]),
            "true_xi": float(gate_g1_truth[heldout_index]),
            "estimated_xi": float(estimate),
            "absolute_xi_error": float(
                abs(estimate - gate_g1_truth[heldout_index])
            ),
            "maximum_training_standardized_residual": float(
                np.max(np.abs(training_residual))
            ),
            "amplitude_a": float(fit.x[0]),
            "amplitude_b": float(fit.x[1]),
        })
    folds = pd.DataFrame(fold_rows)
    spearman = stats.spearmanr(
        folds["true_xi"], folds["estimated_xi"]
    ).statistic
    summary = {
        "ablation": label,
        "modes": [
            int(gate_g_experimental_modes[index]) for index in mode_indices
        ],
        "covariance_policy": covariance_policy,
        "model_discrepancy_included": include_model_discrepancy,
        "mean_absolute_xi_error": float(
            folds["absolute_xi_error"].mean()
        ),
        "maximum_absolute_xi_error": float(
            folds["absolute_xi_error"].max()
        ),
        "spearman": float(spearman) if np.isfinite(spearman) else np.nan,
        "maximum_training_standardized_residual": float(
            folds["maximum_training_standardized_residual"].max()
        ),
    }
    return summary, folds

ablation_specs = [
    ("reference_full_cov_discrepancy_modes_1_3_4", [0, 1, 2], "full", True),
    ("diagonal_cov_discrepancy_modes_1_3_4", [0, 1, 2], "diagonal", True),
    ("full_cov_no_discrepancy_modes_1_3_4", [0, 1, 2], "full", False),
    ("full_cov_discrepancy_modes_1_3", [0, 1], "full", True),
    ("full_cov_discrepancy_mode_1", [0], "full", True),
]
observation_ablation_summaries = []
observation_ablation_folds = []
for specification in ablation_specs:
    summary, folds = final_loso_ablation(*specification)
    observation_ablation_summaries.append(summary)
    observation_ablation_folds.append(folds)

final_observation_ablation = pd.DataFrame(observation_ablation_summaries)
final_observation_ablation_folds = pd.concat(
    observation_ablation_folds, ignore_index=True
)
final_observation_ablation.to_csv(
    TABLE_ROOT / "V0_6_FINAL_observation_ablation.csv", index=False
)
final_observation_ablation_folds.to_csv(
    TABLE_ROOT / "V0_6_FINAL_observation_ablation_folds.csv", index=False
)
display(final_observation_ablation)

final_ablation_checks = {
    "all_basis_evaluations_finite": bool(
        np.isfinite(final_basis_ablation.select_dtypes(include=[np.number])).all().all()
    ),
    "final_greedy_basis_frequency_error_le_0p2pct": bool(
        final_basis_ablation[
            final_basis_ablation["ordering"] == "greedy_prefix"
        ].sort_values("basis_state_count").iloc[-1][
            "maximum_frequency_relative_error"
        ] <= CONFIG["final_ablation_max_frequency_error"]
    ),
    "reference_loso_ablation_finite": bool(
        np.isfinite(
            final_observation_ablation.loc[
                final_observation_ablation["ablation"]
                == "reference_full_cov_discrepancy_modes_1_3_4",
                "mean_absolute_xi_error",
            ]
        ).all()
    ),
}
final_ablation_summary = {
    "version": VERSION,
    "smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "current_run_ablation_publication_evidence": bool(
        RUN_PROFILE == "full" and not GATE_F_SMOKE_PROXY_USED
    ),
    "basis_order_note": (
        "The spread-order comparison reorders the same greedy-trained state pool; "
        "it is not an independently trained random/uniform design."
    ),
    "reference_configuration": (
        "full paired-setup covariance + model discrepancy + experimental modes 1,3,4"
    ),
    "checks": final_ablation_checks,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "V0_6_FINAL_ABLATION_SUMMARY.json").write_text(
    json_dumps_safe(final_ablation_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.7))
for ordering_name, group in final_basis_ablation.groupby("ordering"):
    group = group.sort_values("basis_state_count")
    axes[0].semilogy(
        group["basis_state_count"],
        group["maximum_frequency_relative_error"],
        "o-",
        label=ordering_name,
    )
axes[0].axhline(
    CONFIG["final_ablation_max_frequency_error"],
    linestyle="--",
    label="0.2% final threshold",
)
axes[0].set_xlabel("Full-PINN basis-state count")
axes[0].set_ylabel("Maximum independent frequency relative error")
axes[0].set_title("Basis-size and ordering ablation")
axes[0].legend(fontsize=8)

plot_ablation = final_observation_ablation.sort_values(
    "mean_absolute_xi_error"
)
axes[1].barh(
    plot_ablation["ablation"],
    plot_ablation["mean_absolute_xi_error"],
)
axes[1].set_xlabel(r"Strict LOSO mean absolute error in $\xi$")
axes[1].set_title("Observation-model and mode-content ablation")
fig.suptitle("V0.6 FINAL — minimal scientific ablations", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT / f"figure_30_final_ablation.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")
print(json_dumps_safe(final_ablation_summary, indent=2))


,ordering,basis_state_count,reduced_rank,maximum_frequency_relative_error,mean_frequency_relative_error,minimum_mac,maximum_weak_residual,basis_mass_orthogonality_error
0,greedy_prefix,1,4,0.037038,0.008657,0.945822,0.168335,8.881784e-16
1,greedy_prefix,2,8,0.012230,0.002820,0.996201,0.061425,3.673762e-09
2,greedy_prefix,3,12,0.000958,0.000367,0.999945,0.012926,2.922045e-09
3,same_pool_geometry_spread_order,1,4,0.037038,0.008657,0.945822,0.168335,8.881784e-16
4,same_pool_geometry_spread_order,2,8,0.003978,0.000925,0.998876,0.026599,5.616239e-10
5,same_pool_geometry_spread_order,3,12,0.000958,0.000367,0.999945,0.012926,2.604745e-09


,ablation,modes,covariance_policy,model_discrepancy_included,mean_absolute_xi_error,maximum_absolute_xi_error,spearman,maximum_training_standardized_residual
0,reference_full_cov_discrepancy_modes_1_3_4,"[1, 3, 4]",full,True,0.049496,0.117892,1.0,1.716668
1,diagonal_cov_discrepancy_modes_1_3_4,"[1, 3, 4]",diagonal,True,0.049181,0.117436,1.0,1.726765
2,full_cov_no_discrepancy_modes_1_3_4,"[1, 3, 4]",full,False,0.050791,0.115339,1.0,1.716668
3,full_cov_discrepancy_modes_1_3,"[1, 3]",full,True,0.054959,0.143730,1.0,1.722944
4,full_cov_discrepancy_mode_1,[1],full,True,0.091624,0.147352,0.8,1.377988


{
  "version": "V0_6_FINAL",
  "smoke_fe_proxy_used": false,
  "current_run_ablation_publication_evidence": true,
  "basis_order_note": "The spread-order comparison reorders the same greedy-trained state pool; it is not an independently trained random/uniform design.",
  "reference_configuration": "full paired-setup covariance + model discrepancy + experimental modes 1,3,4",
  "checks": {
    "all_basis_evaluations_finite": true,
    "final_greedy_basis_frequency_error_le_0p2pct": true,
    "reference_loso_ablation_finite": true
  },
  "timestamp_utc": "2026-09-06T03:39:59.715726+00:00"
}


## 37. Final numerical sensitivity analysis

This section does not retune the bridge model. It checks whether the numerical conclusions remain stable when:

- the direct FE mesh is refined;
- the reduced-basis quadrature grid is refined.

The physical zone width, retained modes, FVT threshold, and calibrated trajectory are not changed.


In [43]:
sensitivity_points = [
    (0.0, 0.0),
    (0.325, -1.0),
    (0.65, -2.0),
]
mesh_levels = (
    CONFIG["final_sensitivity_fe_meshes_full"]
    if RUN_PROFILE == "full"
    else CONFIG["final_sensitivity_fe_meshes_smoke"]
)
mesh_reference = max(mesh_levels)
mesh_rows = []
for alpha, log_ratio in sensitivity_points:
    _, reference_omegas, _ = gate_f_fe_reference(
        alpha, log_ratio, n_elements=mesh_reference
    )
    for mesh in mesh_levels:
        _, omegas, _ = gate_f_fe_reference(
            alpha, log_ratio, n_elements=int(mesh)
        )
        for mode_index, (omega, reference) in enumerate(
            zip(omegas, reference_omegas), start=1
        ):
            mesh_rows.append({
                "alpha": alpha,
                "log10_support_ratio": log_ratio,
                "mesh_elements": int(mesh),
                "reference_mesh_elements": int(mesh_reference),
                "mode_order": mode_index,
                "omega": float(omega),
                "reference_omega": float(reference),
                "relative_frequency_change": float(
                    abs(omega - reference) / reference
                ),
            })
final_mesh_sensitivity = pd.DataFrame(mesh_rows)
final_mesh_sensitivity.to_csv(
    TABLE_ROOT / "V0_6_FINAL_fe_mesh_sensitivity.csv", index=False
)

basis_grid_levels = (
    CONFIG["final_sensitivity_basis_points_full"]
    if RUN_PROFILE == "full"
    else CONFIG["final_sensitivity_basis_points_smoke"]
)
basis_grid_rows = []
for grid_points in basis_grid_levels:
    grid = np.linspace(0.0, 1.0, int(grid_points))
    basis = gate_f_build_reduced_basis(gate_f_states, grid)
    validation_rows = []
    for alpha in alpha_midpoints:
        for log_ratio in support_midpoints:
            query = gate_f_gpt_query(
                basis, float(alpha), float(log_ratio)
            )
            validation_rows.extend(
                gate_f_validate_gpt_query(
                    query, basis, float(alpha), float(log_ratio)
                )
            )
    frame = pd.DataFrame(validation_rows)
    basis_grid_rows.append({
        "basis_grid_points": int(grid_points),
        "reduced_rank": int(basis["rank"]),
        "maximum_frequency_relative_error": float(
            frame["frequency_relative_error"].max()
        ),
        "mean_frequency_relative_error": float(
            frame["frequency_relative_error"].mean()
        ),
        "minimum_mac": float(frame["displacement_mac"].min()),
        "maximum_weak_residual": float(frame["weak_residual_max"].max()),
        "mass_orthogonality_error": float(
            basis["mass_orthogonality_error"]
        ),
    })
final_basis_grid_sensitivity = pd.DataFrame(basis_grid_rows)
final_basis_grid_sensitivity.to_csv(
    TABLE_ROOT / "V0_6_FINAL_basis_grid_sensitivity.csv", index=False
)

mesh_116_rows = final_mesh_sensitivity[
    final_mesh_sensitivity["mesh_elements"] == 116
]
mesh_change_at_116 = (
    float(mesh_116_rows["relative_frequency_change"].max())
    if len(mesh_116_rows)
    else np.nan
)
basis_frequency_range = float(
    final_basis_grid_sensitivity[
        "maximum_frequency_relative_error"
    ].max()
    - final_basis_grid_sensitivity[
        "maximum_frequency_relative_error"
    ].min()
)
final_sensitivity_checks = {
    "mesh_results_finite": bool(
        np.isfinite(final_mesh_sensitivity["relative_frequency_change"]).all()
    ),
    "fe_116_change_vs_finest_le_0p2pct": bool(
        np.isfinite(mesh_change_at_116)
        and mesh_change_at_116
        <= CONFIG["final_sensitivity_max_mesh_frequency_change"]
    ),
    "basis_grid_frequency_range_le_0p2pct": bool(
        basis_frequency_range
        <= CONFIG["final_sensitivity_max_basis_frequency_change"]
    ),
    "basis_grid_mac_ge_0p999": bool(
        final_basis_grid_sensitivity["minimum_mac"].min() >= 0.999
    ),
}
final_sensitivity_summary = {
    "version": VERSION,
    "smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "current_run_sensitivity_publication_evidence": bool(
        RUN_PROFILE == "full" and not GATE_F_SMOKE_PROXY_USED
    ),
    "mesh_levels": [int(value) for value in mesh_levels],
    "basis_grid_levels": [int(value) for value in basis_grid_levels],
    "mesh_116_max_relative_change_vs_finest": mesh_change_at_116,
    "basis_grid_max_error_range": basis_frequency_range,
    "checks": final_sensitivity_checks,
    "parameters_intentionally_not_retuned": [
        "three-span geometry",
        "local-zone width",
        "retained experimental modes",
        "trajectory exponents/amplitudes",
        "Gate-I threshold",
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_ROOT / "V0_6_FINAL_SENSITIVITY_SUMMARY.json").write_text(
    json_dumps_safe(final_sensitivity_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(final_mesh_sensitivity)
display(final_basis_grid_sensitivity)
print(json_dumps_safe(final_sensitivity_summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6))
mesh_plot = (
    final_mesh_sensitivity
    .groupby("mesh_elements", as_index=False)[
        "relative_frequency_change"
    ].max()
)
axes[0].loglog(
    mesh_plot["mesh_elements"],
    mesh_plot["relative_frequency_change"],
    "o-",
)
axes[0].axhline(
    CONFIG["final_sensitivity_max_mesh_frequency_change"],
    linestyle="--",
)
axes[0].set_xlabel("FE elements")
axes[0].set_ylabel("Maximum frequency change vs finest mesh")
axes[0].set_title("Direct FE mesh convergence")

axes[1].plot(
    final_basis_grid_sensitivity["basis_grid_points"],
    final_basis_grid_sensitivity[
        "maximum_frequency_relative_error"
    ],
    "o-",
)
axes[1].axhline(
    CONFIG["final_ablation_max_frequency_error"],
    linestyle="--",
)
axes[1].set_xlabel("Reduced-basis quadrature points")
axes[1].set_ylabel("Maximum independent frequency error")
axes[1].set_title("Reduced-basis grid sensitivity")
fig.suptitle("V0.6 FINAL — numerical sensitivity", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT / f"figure_31_final_sensitivity.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


,alpha,log10_support_ratio,mesh_elements,reference_mesh_elements,mode_order,omega,reference_omega,relative_frequency_change
0,0.00,0.0,29,232,1,0.017441,0.017441,0.000003
1,0.00,0.0,29,232,2,0.046599,0.046598,0.000024
2,0.00,0.0,29,232,3,0.062387,0.062384,0.000043
3,0.00,0.0,29,232,4,0.073966,0.073962,0.000060
4,0.00,0.0,29,232,5,0.118124,0.118106,0.000153
...,...,...,...,...,...,...,...,...
67,0.65,-2.0,232,232,2,0.045480,0.045480,0.000000
68,0.65,-2.0,232,232,3,0.055739,0.055739,0.000000
69,0.65,-2.0,232,232,4,0.067871,0.067871,0.000000
70,0.65,-2.0,232,232,5,0.106420,0.106420,0.000000


,basis_grid_points,reduced_rank,maximum_frequency_relative_error,mean_frequency_relative_error,minimum_mac,maximum_weak_residual,mass_orthogonality_error
0,251,12,0.000955,0.000366,0.999945,0.012876,2.129389e-09
1,501,12,0.000958,0.000367,0.999945,0.012926,2.922045e-09
2,1001,12,0.000959,0.000368,0.999944,0.012938,1.029334e-09


{
  "version": "V0_6_FINAL",
  "smoke_fe_proxy_used": false,
  "current_run_sensitivity_publication_evidence": true,
  "mesh_levels": [
    29,
    58,
    116,
    232
  ],
  "basis_grid_levels": [
    251,
    501,
    1001
  ],
  "mesh_116_max_relative_change_vs_finest": 6.421547705363967e-05,
  "basis_grid_max_error_range": 3.269900496648574e-06,
  "checks": {
    "mesh_results_finite": true,
    "fe_116_change_vs_finest_le_0p2pct": true,
    "basis_grid_frequency_range_le_0p2pct": true,
    "basis_grid_mac_ge_0p999": true
  },
  "parameters_intentionally_not_retuned": [
    "three-span geometry",
    "local-zone width",
    "retained experimental modes",
    "trajectory exponents/amplitudes",
    "Gate-I threshold"
  ],
  "timestamp_utc": "2026-09-06T03:40:10.253697+00:00"
}


## 38. Final manuscript tables, claim matrix, and article-ready figures

All values below are pulled from current-run diagnostic objects. No numerical result is hard-coded in Markdown.


In [44]:
def final_nested_get(mapping, path, default=np.nan):
    value = mapping
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return default
        value = value[key]
    return value

gate_objects_for_tables = {
    "A": globals().get("gate_a_diagnostic"),
    "B": globals().get("gate_b_diagnostic"),
    "C": globals().get("gate_c_diagnostic"),
    "D": globals().get("gate_d_diagnostic"),
    "E": globals().get("gate_e_diagnostic"),
    "F": globals().get("gate_f_diagnostic"),
    "G": globals().get("gate_g_diagnostic"),
    "G1": globals().get("gate_g1_diagnostic"),
    "H": globals().get("gate_h_diagnostic"),
    "I": globals().get("gate_i_diagnostic"),
}
main_result_rows = [
    {
        "result": "Full-PINN maximum frequency relative error",
        "value": final_nested_get(
            gate_objects_for_tables["B"],
            ["maximum_frequency_relative_error"],
        ),
        "unit": "fraction",
        "role": "synthetic solver verification",
    },
    {
        "result": "GPT-PINN maximum independent frequency relative error",
        "value": final_nested_get(
            gate_objects_for_tables["F"],
            ["independent_gpt_validation", "maximum_frequency_relative_error"],
        ),
        "unit": "fraction",
        "role": "primary numerical surrogate accuracy",
    },
    {
        "result": "GPT-PINN minimum independent MAC",
        "value": final_nested_get(
            gate_objects_for_tables["F"],
            ["independent_gpt_validation", "minimum_mac"],
        ),
        "unit": "dimensionless",
        "role": "primary numerical surrogate accuracy",
    },
    {
        "result": "Certified median online speed-up vs FE-116",
        "value": speedup_vs_fe,
        "unit": "times",
        "role": "primary computational result",
    },
    {
        "result": "Offline basis construction cost",
        "value": offline_basis_seconds,
        "unit": "seconds",
        "role": "cost disclosure",
    },
    {
        "result": "FE break-even query count",
        "value": break_even_vs_fe,
        "unit": "queries",
        "role": "multi-query Digital-Twin economics",
    },
    {
        "result": "Strict LOSO GPT-PINN mean absolute xi error",
        "value": final_nested_get(
            gate_objects_for_tables["G1"],
            ["strict_loso", "physics_mean_absolute_xi_error"],
        ),
        "unit": "xi",
        "role": "primary experimental state-updating result",
    },
    {
        "result": "Strict LOSO Spearman correlation",
        "value": final_nested_get(
            gate_objects_for_tables["G1"],
            ["strict_loso", "spearman"],
        ),
        "unit": "dimensionless",
        "role": "primary experimental state-updating result",
    },
    {
        "result": "Locked FVT xi mean absolute error",
        "value": final_nested_get(
            gate_objects_for_tables["I"],
            ["physics_mean_absolute_xi_error_nonhealthy"],
        ),
        "unit": "xi",
        "role": "locked cross-excitation audit",
    },
    {
        "result": "Locked FVT maximum modal-ratio RMSE",
        "value": final_nested_get(
            gate_objects_for_tables["I"],
            ["maximum_modal_ratio_rmse"],
        ),
        "unit": "fraction",
        "role": "locked cross-excitation audit",
    },
]
final_main_results = pd.DataFrame(main_result_rows)
final_main_results.to_csv(
    TABLE_ROOT / "V0_6_FINAL_main_results.csv", index=False
)
try:
    (TABLE_ROOT / "V0_6_FINAL_main_results.tex").write_text(
        final_main_results.to_latex(index=False, escape=False),
        encoding="utf-8",
    )
except Exception as exc:
    warnings.warn(
        f"LaTeX table export failed non-critically: {type(exc).__name__}: {exc}",
        RuntimeWarning,
    )

final_claim_matrix = pd.DataFrame([
    {
        "claim": "Bridge-adapted greedy GPT-PINN reduced modal solver",
        "status": "SUPPORTED_IF_FINAL_CORE_CHECKS_PASS",
        "evidence": "Gates B, C, E and F",
    },
    {
        "claim": "Rapid online modal Digital-Twin query relative to the frozen FE-116 baseline",
        "status": "SUPPORTED_ONLY_BY_COMPUTE_NODE_CERTIFIED_TIMING",
        "evidence": "Final timing benchmark",
    },
    {
        "claim": "One-dimensional identifiable progression coordinate on Z24 settlement states",
        "status": "SUPPORTED_IF_GATE_G1_PASSES",
        "evidence": "Gate G.1 strict LOSO",
    },
    {
        "claim": "Free two-parameter physical identifiability",
        "status": "NOT_SUPPORTED",
        "evidence": "Gate G NO-GO is preserved",
    },
    {
        "claim": "Locked AVT-to-FVT cross-excitation assessment",
        "status": "SUPPORTED_AS_AN_AUDIT_REGARDLESS_OF_GO_OR_NO_GO",
        "evidence": "Gate I, no target refit",
    },
    {
        "claim": "Autonomous prognosis or calendar-time RUL",
        "status": "NOT_SUPPORTED",
        "evidence": "Gate H is conditional response propagation only",
    },
    {
        "claim": "Actual capacity, failure probability or optimal maintenance",
        "status": "NOT_SUPPORTED",
        "evidence": "Z24 controlled benchmark does not supply those targets",
    },
])
final_claim_matrix.to_csv(
    TABLE_ROOT / "V0_6_FINAL_claim_matrix.csv", index=False
)
display(final_main_results)
display(final_claim_matrix)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.8))
accuracy_value = final_nested_get(
    gate_objects_for_tables["F"],
    ["independent_gpt_validation", "maximum_frequency_relative_error"],
)
axes[0].scatter(
    [gpt_median_seconds, fe_median_seconds],
    [100 * accuracy_value, 0.0],
    s=[180, 120],
)
axes[0].annotate(
    "GPT-PINN", (gpt_median_seconds, 100 * accuracy_value),
    xytext=(8, 8), textcoords="offset points",
)
axes[0].annotate(
    "FE-116 reference", (fe_median_seconds, 0.0),
    xytext=(8, 8), textcoords="offset points",
)
axes[0].set_xscale("log")
axes[0].set_xlabel("Median online wall time [s]")
axes[0].set_ylabel("Maximum frequency error vs FE [%]")
axes[0].set_title("Accuracy–latency position")

if "gate_g1_loso" in globals():
    axes[1].plot(
        gate_g1_loso["true_xi_for_audit_only"],
        gate_g1_loso["physics_gpt_xi"],
        "o-",
        label="GPT-PINN strict LOSO",
    )
    axes[1].plot([0, 1], [0, 1], "--", label="ideal")
    axes[1].set_xlabel(r"True audit coordinate $\xi=\delta/95$")
    axes[1].set_ylabel(r"Estimated $\hat{\xi}$")
    axes[1].set_title("Identifiable Z24 progression coordinate")
    axes[1].legend()
fig.suptitle("V0.6 FINAL — principal article evidence", weight="bold")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FIGURE_ROOT / f"figure_32_final_principal_evidence.{suffix}",
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


,result,value,unit,role
0,Full-PINN maximum frequency relative error,0.000007,fraction,synthetic solver verification
1,GPT-PINN maximum independent frequency relativ...,0.000958,fraction,primary numerical surrogate accuracy
2,GPT-PINN minimum independent MAC,0.999945,dimensionless,primary numerical surrogate accuracy
3,Certified median online speed-up vs FE-116,70.028516,times,primary computational result
4,Offline basis construction cost,921.470042,seconds,cost disclosure
5,FE break-even query count,100333.000000,queries,multi-query Digital-Twin economics
6,Strict LOSO GPT-PINN mean absolute xi error,0.049496,xi,primary experimental state-updating result
7,Strict LOSO Spearman correlation,1.000000,dimensionless,primary experimental state-updating result
8,Locked FVT xi mean absolute error,0.053254,xi,locked cross-excitation audit
9,Locked FVT maximum modal-ratio RMSE,0.029994,fraction,locked cross-excitation audit


,claim,status,evidence
0,Bridge-adapted greedy GPT-PINN reduced modal s...,SUPPORTED_IF_FINAL_CORE_CHECKS_PASS,"Gates B, C, E and F"
1,Rapid online modal Digital-Twin query relative...,SUPPORTED_ONLY_BY_COMPUTE_NODE_CERTIFIED_TIMING,Final timing benchmark
2,One-dimensional identifiable progression coord...,SUPPORTED_IF_GATE_G1_PASSES,Gate G.1 strict LOSO
3,Free two-parameter physical identifiability,NOT_SUPPORTED,Gate G NO-GO is preserved
4,Locked AVT-to-FVT cross-excitation assessment,SUPPORTED_AS_AN_AUDIT_REGARDLESS_OF_GO_OR_NO_GO,"Gate I, no target refit"
5,Autonomous prognosis or calendar-time RUL,NOT_SUPPORTED,Gate H is conditional response propagation only
6,"Actual capacity, failure probability or optima...",NOT_SUPPORTED,Z24 controlled benchmark does not supply those...


## 39. Final evidence package and irreversible code-freeze decision

The article core may be manuscript-ready even when Gate I is a NO-GO, because the locked FVT result is an audit rather than a tuning target. Publication timing, however, is certified only on a Slurm compute node.

After `READY_FOR_MANUSCRIPT_AND_CODE_FREEZE`, the physical model, retained modes, thresholds, and FVT protocol must not be altered for this article.


In [45]:
def final_extract_decision(value):
    if value is None:
        return "NOT_EXECUTED"
    for key in ("decision", "status"):
        if isinstance(value, dict) and key in value:
            return value[key]
    return "EXECUTED_DECISION_KEY_NOT_FOUND"

gate_objects = {
    "A_data_and_modal_observations": globals().get("gate_a_diagnostic"),
    "B_full_pinn": globals().get("gate_b_diagnostic"),
    "C_gpt_pinn_pilot": globals().get("gate_c_diagnostic"),
    "D_simple_parameterization": globals().get("gate_d_diagnostic"),
    "E_three_span_parameterization": globals().get("gate_e_diagnostic"),
    "F_multimode_gpt_pinn": globals().get("gate_f_diagnostic"),
    "G_free_two_parameter_update": globals().get("gate_g_diagnostic"),
    "G1_identifiable_trajectory": globals().get("gate_g1_diagnostic"),
    "H_conditional_response_propagation": globals().get("gate_h_diagnostic"),
    "I_z24_fvt_locked_audit": globals().get("gate_i_diagnostic"),
}
gate_decisions = {
    name: final_extract_decision(value)
    for name, value in gate_objects.items()
}
raw_unchanged_final = bool(
    raw_metadata_signature(DATASET_ROOT) == RAW_SIGNATURE_BEFORE
)
all_gates_executed = all(
    decision != "NOT_EXECUTED" for decision in gate_decisions.values()
)
gate_i_value = gate_objects["I_z24_fvt_locked_audit"] or {}
gate_i_locked_audit_executed = bool(
    gate_decisions["I_z24_fvt_locked_audit"] in {
        "GO_Z24_FVT_LOCKED_VALIDATION_V0_6_FINAL",
        "NO_GO_Z24_FVT_EXTERNAL_VALIDATION",
    }
    and gate_i_value.get("target_refit_performed") is False
    and gate_i_value.get("upstream_model_changed_from_fvt") is False
)

core_scientific_requirements = {
    "gate_b_full_pinn_go":
        gate_decisions["B_full_pinn"] == "GO_TO_GPT_PINN_PILOT",
    "gate_c_gpt_pinn_go":
        gate_decisions["C_gpt_pinn_pilot"]
        == "GO_TO_Z24_CALIBRATION_PILOT_WITH_FE_BASELINE_REQUIRED",
    "gate_e_three_span_go":
        gate_decisions["E_three_span_parameterization"]
        == "GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL",
    "gate_f_multimode_gpt_go":
        gate_decisions["F_multimode_gpt_pinn"]
        == "GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL",
    "gate_f_real_full_pinn_basis_no_smoke_proxy":
        bool(not GATE_F_SMOKE_PROXY_USED),
    "gate_g_two_parameter_no_go_preserved":
        gate_decisions["G_free_two_parameter_update"]
        == "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNSTABLE",
    "gate_g1_identifiable_trajectory_go":
        gate_decisions["G1_identifiable_trajectory"]
        == "GO_TO_CONDITIONAL_RESPONSE_PROPAGATION_V0_6_FINAL",
    "gate_i_locked_audit_executed_without_refit":
        gate_i_locked_audit_executed,
    "final_protocol_frozen":
        all(final_protocol_checks.values()),
    "final_timing_scientifically_valid":
        all(
            value for key, value in final_timing_checks.items()
            if key not in {"full_profile", "slurm_compute_node_certified"}
        ),
    "final_ablation_completed":
        all(final_ablation_checks.values()),
    "final_sensitivity_completed":
        all(final_sensitivity_checks.values()),
    "raw_z24_metadata_unchanged":
        raw_unchanged_final,
    "checkpoint_io_probe_passed":
        bool(CHECKPOINT_IO_PROBE.get("passed")),
    "checkpoint_io_no_save_failures":
        not any(
            event.get("status") == "FAIL"
            for event in CHECKPOINT_IO_EVENTS
        ),
    "full_profile":
        RUN_PROFILE == "full",
}
scientific_manuscript_ready = bool(
    all(core_scientific_requirements.values())
)
publication_compute_certified = bool(
    scientific_manuscript_ready
    and timing_compute_certified
    and RUN_PROFILE == "full"
)
code_freeze_authorized = publication_compute_certified

gate_h_value = gate_objects[
    "H_conditional_response_propagation"
] or {}
conditional_response_status = gate_decisions[
    "H_conditional_response_propagation"
]
autonomous_prognosis_supported = bool(
    gate_h_value.get("autonomy_decision") == "GO_AUTONOMOUS_PROGRESSION"
)

if code_freeze_authorized:
    article_evidence_status = "READY_FOR_MANUSCRIPT_AND_CODE_FREEZE"
elif scientific_manuscript_ready:
    article_evidence_status = (
        "SCIENTIFIC_CORE_READY_REEXECUTE_ON_SLURM_FOR_PUBLICATION_TIMING"
    )
else:
    article_evidence_status = "NOT_READY_REVIEW_FAILED_OR_UNEXECUTED_REQUIREMENTS"

source_notebook_sha256 = (
    hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest()
    if NOTEBOOK_PATH.exists()
    else None
)
execution_certificate = {
    "version": VERSION,
    "patch_level": PATCH_LEVEL,
    "run_id": RUN_ID,
    "run_profile": RUN_PROFILE,
    "hostname": HOSTNAME,
    "is_login_node": IS_LOGIN_NODE,
    "slurm_job_id": SLURM_JOB_ID,
    "slurm_node_list": os.environ.get("SLURM_NODELIST"),
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "notebook_path": str(NOTEBOOK_PATH),
    "source_notebook_sha256_at_runtime": source_notebook_sha256,
    "frozen_protocol_sha256": FINAL_PROTOCOL_SHA256,
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "effective_cpu_threads": EFFECTIVE_CPU_THREADS,
    "gate_decisions": gate_decisions,
    "all_gate_objects_executed": all_gates_executed,
    "raw_z24_metadata_signature_before":
        RAW_SIGNATURE_BEFORE[0],
    "raw_z24_metadata_unchanged": raw_unchanged_final,
    "full_numerical_run_on_marwan_demonstrated": bool(
        RUN_PROFILE == "full"
        and all_gates_executed
        and raw_unchanged_final
    ),
    "full_run_on_marwan_compute_node_demonstrated": bool(
        RUN_PROFILE == "full"
        and IN_SLURM_ALLOCATION
        and not IS_LOGIN_NODE
        and all_gates_executed
        and raw_unchanged_final
        and not ALLOW_LOGIN_NODE_HEAVY
    ),
    "publication_timing_compute_certified":
        timing_compute_certified,
    "gate_f_smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "gate_f_state_source_classes": GATE_F_STATE_SOURCE_CLASSES,
    "resource_policy_events": RESOURCE_POLICY_EVENTS,
    "checkpoint_io_probe": CHECKPOINT_IO_PROBE,
    "checkpoint_io_event_count": len(CHECKPOINT_IO_EVENTS),
    "checkpoint_io_failure_count": sum(
        event.get("status") == "FAIL"
        for event in CHECKPOINT_IO_EVENTS
    ),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
certificate_path = (
    OUTPUT_ROOT / "V0_6_FINAL_EXECUTION_CERTIFICATE.json"
)
certificate_path.write_text(
    json_dumps_safe(execution_certificate, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

final_evidence = {
    "version": VERSION,
    "article_title": CONFIG["final_article_title"],
    "article_evidence_status": article_evidence_status,
    "scientific_manuscript_ready": scientific_manuscript_ready,
    "publication_compute_certified": publication_compute_certified,
    "code_freeze_authorized": code_freeze_authorized,
    "primary_benchmark": "Z24",
    "development_excitation": "AVT",
    "locked_cross_excitation_audit": "FVT",
    "gate_decisions": gate_decisions,
    "core_scientific_requirements": core_scientific_requirements,
    "frozen_protocol_sha256": FINAL_PROTOCOL_SHA256,
    "timing": final_timing_summary,
    "ablation": final_ablation_summary,
    "sensitivity": final_sensitivity_summary,
    "locked_fvt_audit_decision":
        gate_decisions["I_z24_fvt_locked_audit"],
    "gate_h_supplementary_status":
        conditional_response_status,
    "autonomous_prognosis_supported":
        autonomous_prognosis_supported,
    "gate_f_smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "current_run_evidence_class": (
        "SMOKE_WORKFLOW_WITH_FE_PROXY_NO_GPT_PINN_PUBLICATION_EVIDENCE"
        if GATE_F_SMOKE_PROXY_USED
        else (
            "FULL_GPT_PINN_PUBLICATION_CANDIDATE"
            if RUN_PROFILE == "full"
            else "SMOKE_WITH_VALIDATED_FULL_PINN_CHECKPOINTS"
        )
    ),
    "supported_claims": [
        "bridge-adapted Full-PINN and greedy GPT-PINN reduced modal solver",
        "rapid repeated modal Digital-Twin queries relative to the frozen FE-116 baseline when certified on a Slurm compute node",
        "strict Z24 AVT leave-one-severity-out inference on an identifiable one-dimensional progression coordinate",
        "explicit demonstration that good modal fit does not imply free two-parameter identifiability",
        "locked AVT-to-FVT cross-excitation audit without target refitting, reported whether GO or NO-GO",
    ],
    "unsupported_claims": FINAL_PROTOCOL["prohibited_claims"],
    "stop_rule": (
        "After READY_FOR_MANUSCRIPT_AND_CODE_FREEZE, no further model, "
        "mode, threshold, trajectory, or FVT-protocol tuning is permitted "
        "for this article."
    ),
    "execution_certificate": str(certificate_path),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
final_evidence_path = (
    OUTPUT_ROOT / "V0_6_FINAL_EVIDENCE_SUMMARY.json"
)
final_evidence_path.write_text(
    json_dumps_safe(final_evidence, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

pd.DataFrame([
    {"gate": gate, "decision": decision}
    for gate, decision in gate_decisions.items()
]).to_csv(
    TABLE_ROOT / "V0_6_FINAL_gate_decisions.csv", index=False
)
pd.DataFrame([
    {"requirement": key, "passed": value}
    for key, value in core_scientific_requirements.items()
]).to_csv(
    TABLE_ROOT / "V0_6_FINAL_readiness_requirements.csv", index=False
)

checkpoint_io_audit_path = (
    TABLE_ROOT / "V0_6_FINAL_checkpoint_io_audit.csv"
)
pd.DataFrame(CHECKPOINT_IO_EVENTS).to_csv(
    checkpoint_io_audit_path, index=False
)
checkpoint_io_json_path = (
    OUTPUT_ROOT / "V0_6_FINAL_CHECKPOINT_IO_AUDIT.json"
)
checkpoint_io_json_path.write_text(
    json_dumps_safe(CHECKPOINT_IO_EVENTS, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

code_freeze_record = {
    "version": VERSION,
    "authorized": code_freeze_authorized,
    "article_evidence_status": article_evidence_status,
    "frozen_protocol_sha256": FINAL_PROTOCOL_SHA256,
    "source_notebook_sha256_at_runtime": source_notebook_sha256,
    "gate_f_smoke_fe_proxy_used": bool(GATE_F_SMOKE_PROXY_USED),
    "locked_fvt_result_must_not_be_reclassified_post_hoc": True,
    "future_work_outside_this_article": [
        "Route345 or OpenLab methodology transfer",
        "calendar-time degradation/RUL data",
        "maintenance-decision optimization",
    ],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
code_freeze_path = OUTPUT_ROOT / "V0_6_FINAL_CODE_FREEZE.json"
code_freeze_path.write_text(
    json_dumps_safe(code_freeze_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

manifest_path = OUTPUT_ROOT / "V0_6_FINAL_ARTIFACT_MANIFEST.csv"
manifest_rows = []
for path in sorted(
    candidate for candidate in OUTPUT_ROOT.rglob("*")
    if candidate.is_file() and candidate != manifest_path
):
    manifest_rows.append({
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "bytes": int(path.stat().st_size),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
pd.DataFrame(manifest_rows).to_csv(
    manifest_path, index=False
)

latest_pointer = VERSION_ROOT / "LATEST_RUN.json"
latest_pointer.write_text(
    json_dumps_safe({
        "version": VERSION,
        "run_id": RUN_ID,
        "output_root": str(OUTPUT_ROOT),
        "article_evidence_status": article_evidence_status,
        "final_evidence": str(final_evidence_path),
        "execution_certificate": str(certificate_path),
        "code_freeze": str(code_freeze_path),
        "artifact_manifest": str(manifest_path),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(json_dumps_safe(final_evidence, indent=2))
print("Execution certificate:", certificate_path)
print("Final evidence:", final_evidence_path)
print("Code-freeze record:", code_freeze_path)
print("Artifact manifest:", manifest_path)
print("Latest-run pointer:", latest_pointer)


{
  "version": "V0_6_FINAL",
  "article_title": "A Greedy GPT-PINN Reduced Digital Twin for Rapid Bridge Modal Updating: Identifiability and Cross-Excitation Assessment on the Z24 Benchmark",
  "article_evidence_status": "READY_FOR_MANUSCRIPT_AND_CODE_FREEZE",
  "scientific_manuscript_ready": true,
  "publication_compute_certified": true,
  "code_freeze_authorized": true,
  "primary_benchmark": "Z24",
  "development_excitation": "AVT",
  "locked_cross_excitation_audit": "FVT",
  "gate_decisions": {
    "A_data_and_modal_observations": "GO_TO_SYNTHETIC_PHYSICS_PILOT",
    "B_full_pinn": "GO_TO_GPT_PINN_PILOT",
    "C_gpt_pinn_pilot": "GO_TO_Z24_CALIBRATION_PILOT_WITH_FE_BASELINE_REQUIRED",
    "D_simple_parameterization": "NO_GO_REVISE_PHYSICAL_PARAMETERIZATION",
    "E_three_span_parameterization": "GO_TO_MULTIPARAMETRIC_GPT_PINN_V0_6_FINAL",
    "F_multimode_gpt_pinn": "GO_TO_Z24_ONLINE_STATE_UPDATING_V0_6_FINAL",
    "G_free_two_parameter_update": "NO_GO_TWO_PARAMETER_UNCERTAINTY_UNS

## 40. Final execution instructions and stopping rule

### Publication run

Run this notebook in a Jupyter server whose kernel is inside a Slurm allocation. The default settings are:

```bash
export GPT_PINN_RUN_PROFILE=auto
export GPT_PINN_COMPUTE_GUARD=strict
```

- on a non-login Slurm compute node, `auto` selects `full`;
- on `login02`, `auto` selects `smoke`;
- forcing `full` on `login02` is blocked unless an emergency override is deliberately set, and such a run is never publication-certified.

Open:

```text
/home/a.riyahi/GPT_PINN/GPT_PINN_Z24_Physics_Digital_Twin_V0_6_FINAL_R3.ipynb
```

then use:

```text
Kernel → Restart & Clear Output
Run → Run All Cells
```

The definitive status is read from:

```text
outputs/v0_6_final/runs/<RUN_ID>/V0_6_FINAL_EVIDENCE_SUMMARY.json
```

The development phase ends only when:

```text
article_evidence_status = READY_FOR_MANUSCRIPT_AND_CODE_FREEZE
```

At that point, do not add new algorithms or retune FVT-facing parameters. Proceed to manuscript preparation.
